# LA Studio Unified Dubbing Coordinator (optional)

This is a real one-URL, one-token coordinator for the selected Dubbing models. It starts each selected **exact CUDA worker** privately, verifies its ordinary `/health`, then exposes it through one Cloudflare tunnel. It never substitutes local CPU or API Gateway routes. The coordinator and exact worker notebooks are embedded in this notebook; no LA Studio repository clone or repository token is required at runtime.

Keep the normal per-model notebooks if you prefer them. This notebook is optional and does not replace those routes.


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'fastapi==0.115.12', 'uvicorn==0.34.3', 'httpx==0.28.1'], check=True)
print('Installed the unified coordinator runtime; application sources are embedded below.')


In [ ]:
import json
                from pathlib import Path

                # Keep only the exact models you intend to select in LA Studio.
                # Every generated exact notebook is embedded below, so this list
                # can be edited without cloning the desktop repository.
                UNIFIED_WORKERS = [
    {
        "capability": "voice-isolation",
        "model": "sherpa-onnx-spleeter-2stems-fp16"
    },
    {
        "capability": "stt",
        "model": "whisper.cpp"
    },
    {
        "capability": "subtitle-ocr",
        "model": "pp-ocrv5-multilingual-3.1"
    },
    {
        "capability": "translation",
        "model": "m2m100-418m"
    },
    {
        "capability": "tts",
        "model": "kokoro"
    },
    {
        "capability": "forced-alignment",
        "model": "mms-forced-aligner-onnx"
    }
]
                CONFIG_PATH = Path('/content/la_studio_unified_workers.json')
                CONFIG_PATH.write_text(json.dumps(UNIFIED_WORKERS, indent=2), encoding='utf-8')
                print('Will prewarm:', ', '.join(f"{row['capability']}/{row['model']}" for row in UNIFIED_WORKERS))


In [ ]:
import shutil
from pathlib import Path

EMBEDDED_UNIFIED_FILES = {
    'notebooks/alignment/LA_STUDIO_ALIGNMENT_CANARY_CTC_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio forced-alignment \\u2014 Canary CTC Aligner\\n",\n        "\\n",\n        "This notebook loads exactly `canary-ctc-aligner` (`cstr/canary-ctc-aligner-GGUF`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "!git clone --quiet --recursive https://github.com/CrispStrobe/CrispASR.git /content/CrispASR\\n",\n        "!git -C /content/CrispASR checkout --quiet 754b67289cf1137e3ed722885705f94132fc614f\\n",\n        "!git -C /content/CrispASR submodule update --init --recursive\\n",\n        "!cmake -S /content/CrispASR -B /content/CrispASR/build -G Ninja -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON\\n",\n        "!cmake --build /content/CrispASR/build --target crispasr -j2\\n",\n        "%pip install -q \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"python-multipart==0.0.20\\"\\n",\n        "\\n",\n        "!wget -q --show-progress -O /content/canary-ctc-aligner-q4_k.gguf https://huggingface.co/cstr/canary-ctc-aligner-GGUF/resolve/main/canary-ctc-aligner-q4_k.gguf\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_alignment_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport re\\\\nimport subprocess\\\\nimport tempfile\\\\nimport threading\\\\nfrom pathlib import Path\\\\n\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nimport json\\\\n\\\\nMODEL_ID = \\"canary-ctc-aligner\\"\\\\nMODEL_NAME = \\"Canary CTC Aligner\\"\\\\nUPSTREAM_MODEL = \\"cstr/canary-ctc-aligner-GGUF\\"\\\\nSUPPORTED_LANGUAGES = [\\"bg\\", \\"cs\\", \\"da\\", \\"de\\", \\"el\\", \\"en\\", \\"es\\", \\"et\\", \\"fi\\", \\"fr\\", \\"hr\\", \\"hu\\", \\"it\\", \\"lt\\", \\"lv\\", \\"mt\\", \\"nl\\", \\"pl\\", \\"pt\\", \\"ro\\", \\"ru\\", \\"sk\\", \\"sl\\", \\"sv\\", \\"uk\\"]\\\\nCRISPASR = \\"/content/CrispASR/build/bin/crispasr\\"\\\\nALIGNER_MODEL = \\"/content/canary-ctc-aligner-q4_k.gguf\\"\\\\n\\\\nif not Path(CRISPASR).is_file() or not Path(ALIGNER_MODEL).is_file():\\\\n    raise RuntimeError(\\"CrispASR CUDA runtime or the exact aligner model is missing\\")\\\\n\\\\ndef _collect_crisp_segments(payload):\\\\n    if isinstance(payload, list):\\\\n        entries = payload\\\\n    elif isinstance(payload, dict):\\\\n        entries = payload.get(\\"segments\\", payload.get(\\"alignment\\", payload.get(\\"results\\", [])))\\\\n    else:\\\\n        entries = []\\\\n    output = []\\\\n    for entry in entries:\\\\n        if not isinstance(entry, dict):\\\\n            continue\\\\n        words = entry.get(\\"words\\")\\\\n        if isinstance(words, list) and words:\\\\n            output.extend(words)\\\\n        else:\\\\n            output.append(entry)\\\\n    return output\\\\n\\\\ndef align_exact(source_path: str, transcript: str, language: str):\\\\n    if MODEL_ID == \\"wav2vec2-aligner-zh\\" and language not in {\\"zh\\", \\"zho\\", \\"chi\\", \\"cmn\\", \\"zh-cn\\"}:\\\\n        raise HTTPException(status_code=422, detail=\\"the selected Wav2Vec2 aligner supports Mandarin Chinese only\\")\\\\n    with tempfile.TemporaryDirectory(prefix=\\"la-studio-crisp-align-\\") as directory:\\\\n        wav_path = str(Path(directory) / \\"source.wav\\")\\\\n        output_path = str(Path(directory) / \\"alignment.json\\")\\\\n        subprocess.run(\\\\n            [\\"ffmpeg\\", \\"-y\\", \\"-v\\", \\"error\\", \\"-i\\", source_path, \\"-vn\\", \\"-ac\\", \\"1\\", \\"-ar\\", \\"16000\\", wav_path],\\\\n            check=True,\\\\n        )\\\\n        command = [\\\\n            CRISPASR, \\"--align-only\\", \\"-am\\", ALIGNER_MODEL, \\"-f\\", wav_path,\\\\n            \\"--ref-text\\", transcript, \\"--align-format\\", \\"json\\",\\\\n            \\"--align-granularity\\", \\"word\\", \\"--align-output\\", output_path,\\\\n            \\"--gpu-backend\\", \\"cuda\\", \\"--strict-pipeline\\", \\"-l\\", language,\\\\n        ]\\\\n        result = subprocess.run(command, text=True, capture_output=True)\\\\n        if result.returncode != 0 or not Path(output_path).is_file():\\\\n            raise RuntimeError(\\"CrispASR CUDA aligner failed: \\" + (result.stderr or result.stdout)[-1600:])\\\\n        payload = json.loads(Path(output_path).read_text(encoding=\\"utf-8\\"))\\\\n        return _collect_crisp_segments(payload)\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_ALIGNMENT_TOKEN\\"]\\\\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\\\\nMAX_AUDIO_SECONDS = 300\\\\nALLOWED_CONTENT_TYPES = {\\\\n    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp4\\", \\"audio/webm\\",\\\\n    \\"audio/ogg\\", \\"audio/flac\\", \\"application/octet-stream\\",\\\\n}\\\\nALLOWED_EXTENSIONS = {\\".wav\\", \\".mp3\\", \\".m4a\\", \\".mp4\\", \\".webm\\", \\".ogg\\", \\".flac\\"}\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\nMODEL_LOCK = threading.Lock()\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef media_duration_seconds(path: str) -> float:\\\\n    probe = subprocess.run(\\\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-show_entries\\", \\"format=duration\\",\\\\n         \\"-of\\", \\"default=nokey=1:noprint_wrappers=1\\", path],\\\\n        text=True, capture_output=True,\\\\n    )\\\\n    try:\\\\n        duration = float(probe.stdout.strip())\\\\n    except ValueError:\\\\n        duration = 0.0\\\\n    if probe.returncode != 0 or duration <= 0.0:\\\\n        raise HTTPException(status_code=415, detail=\\"audio is unsupported or could not be decoded\\")\\\\n    return duration\\\\n\\\\ndef validate_segments(raw_segments) -> list[dict]:\\\\n    segments = []\\\\n    previous_end = 0.0\\\\n    for raw in raw_segments:\\\\n        text = str(raw.get(\\"text\\", \\"\\")).strip()\\\\n        if not text:\\\\n            continue\\\\n        start = float(raw.get(\\"start\\", raw.get(\\"start_time\\", 0.0)))\\\\n        end = float(raw.get(\\"end\\", raw.get(\\"end_time\\", start)))\\\\n        score = max(0.0, min(1.0, float(raw.get(\\"score\\", raw.get(\\"confidence\\", 1.0)))))\\\\n        if start < 0.0 or end < start or start + 0.002 < previous_end:\\\\n            raise RuntimeError(\\"aligner returned non-monotonic timestamps\\")\\\\n        segments.append({\\"text\\": text, \\"start\\": start, \\"end\\": end, \\"score\\": score, \\"kind\\": raw.get(\\"kind\\", \\"word\\")})\\\\n        previous_end = end\\\\n    if not segments:\\\\n        raise RuntimeError(\\"aligner returned no timestamped tokens\\")\\\\n    return segments\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Alignment - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"forced-alignment\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"max_audio_seconds\\": MAX_AUDIO_SECONDS,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/alignments\\")\\\\nasync def align(\\\\n    audio: UploadFile = File(...),\\\\n    transcript: str = Form(...),\\\\n    language: str = Form(\\"en\\"),\\\\n    model: str = Form(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    text = transcript.strip()\\\\n    if not text:\\\\n        raise HTTPException(status_code=422, detail=\\"transcript is required\\")\\\\n    suffix = Path(audio.filename or \\"audio.wav\\").suffix.lower() or \\".wav\\"\\\\n    if suffix not in ALLOWED_EXTENSIONS:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio filename extension\\")\\\\n    if audio.content_type and audio.content_type not in ALLOWED_CONTENT_TYPES:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio MIME type\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab alignment worker is busy; retry shortly\\")\\\\n    source_path = None\\\\n    try:\\\\n        descriptor, source_path = tempfile.mkstemp(suffix=suffix)\\\\n        with os.fdopen(descriptor, \\"wb\\") as output:\\\\n            while chunk := await audio.read(1024 * 1024):\\\\n                output.write(chunk)\\\\n                if output.tell() > MAX_UPLOAD_BYTES:\\\\n                    raise HTTPException(status_code=413, detail=\\"audio exceeds 512 MB upload limit\\")\\\\n        duration = media_duration_seconds(source_path)\\\\n        if duration > MAX_AUDIO_SECONDS:\\\\n            raise HTTPException(status_code=413, detail=\\"audio exceeds the five minute duration limit\\")\\\\n        with MODEL_LOCK:\\\\n            raw_segments = align_exact(source_path, text, language.strip().lower() or \\"en\\")\\\\n        segments = validate_segments(raw_segments)\\\\n        return {\\"duration\\": duration, \\"segments\\": segments, \\"unaligned_tokens\\": []}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} alignment failed: {type(error).__name__}: {str(error)[:300]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        if source_path:\\\\n            Path(source_path).unlink(missing_ok=True)\\\\n        await audio.close()\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'canary-ctc-aligner\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Forced Alignment\'\\n",\n        "MODEL_ID = \'canary-ctc-aligner\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_alignment_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_alignment_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_alignment_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "forced-alignment",\n      "family_id": "canary-ctc-aligner",\n      "upstream_model": "cstr/canary-ctc-aligner-GGUF",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/alignment/LA_STUDIO_ALIGNMENT_MMS_ONNX_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio forced-alignment \\u2014 MMS Forced Aligner ONNX\\n",\n        "\\n",\n        "This notebook loads exactly `mms-forced-aligner-onnx` (`onnx-community/mms-300m-1130-forced-aligner-ONNX`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"onnxruntime-gpu==1.22.0\\" \\"transformers==4.57.6\\" \\"huggingface-hub==0.36.0\\" \\"git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git@11855d1de76af2b490dd2e8e2db2661805ae90a0\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"python-multipart==0.0.20\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_alignment_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport re\\\\nimport subprocess\\\\nimport tempfile\\\\nimport threading\\\\nfrom pathlib import Path\\\\n\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nimport math\\\\nimport numpy as np\\\\nimport onnxruntime as ort\\\\nfrom huggingface_hub import snapshot_download\\\\nfrom transformers import AutoConfig, AutoTokenizer\\\\nfrom ctc_forced_aligner import get_alignments, get_spans, postprocess_results, preprocess_text\\\\n\\\\nMODEL_ID = \\"mms-forced-aligner-onnx\\"\\\\nMODEL_NAME = \\"MMS Forced Aligner ONNX\\"\\\\nUPSTREAM_MODEL = \\"onnx-community/mms-300m-1130-forced-aligner-ONNX\\"\\\\nSUPPORTED_LANGUAGES = [\\"158 languages (ISO 639-1 or ISO 639-3)\\"]\\\\nMODEL_DIR = snapshot_download(\\\\n    UPSTREAM_MODEL,\\\\n    allow_patterns=[\\\\n        \\"config.json\\", \\"preprocessor_config.json\\", \\"special_tokens_map.json\\",\\\\n        \\"tokenizer.json\\", \\"tokenizer_config.json\\", \\"vocab.json\\", \\"onnx/model_fp16.onnx\\",\\\\n    ],\\\\n)\\\\nSESSION = ort.InferenceSession(\\\\n    str(Path(MODEL_DIR) / \\"onnx/model_fp16.onnx\\"),\\\\n    providers=[\\"CUDAExecutionProvider\\"],\\\\n)\\\\nif \\"CUDAExecutionProvider\\" not in SESSION.get_providers():\\\\n    raise RuntimeError(\\"MMS ONNX did not activate CUDAExecutionProvider\\")\\\\nTOKENIZER = AutoTokenizer.from_pretrained(MODEL_DIR, word_delimiter_token=None)\\\\nCONFIG = AutoConfig.from_pretrained(MODEL_DIR)\\\\nRATIO = int(CONFIG.inputs_to_logits_ratio)\\\\nSAMPLE_RATE = 16000\\\\n\\\\nISO3 = {\\\\n    \\"ar\\": \\"ara\\", \\"be\\": \\"bel\\", \\"bg\\": \\"bul\\", \\"de\\": \\"deu\\", \\"el\\": \\"ell\\", \\"en\\": \\"eng\\",\\\\n    \\"fa\\": \\"fas\\", \\"he\\": \\"heb\\", \\"kk\\": \\"kaz\\", \\"ky\\": \\"kir\\", \\"lv\\": \\"lav\\", \\"lt\\": \\"lit\\",\\\\n    \\"mk\\": \\"mkd\\", \\"mn\\": \\"mon\\", \\"ru\\": \\"rus\\", \\"sr\\": \\"srp\\", \\"th\\": \\"tha\\", \\"tr\\": \\"tur\\",\\\\n    \\"ug\\": \\"uig\\", \\"uk\\": \\"ukr\\", \\"yi\\": \\"yid\\", \\"vi\\": \\"vie\\", \\"zh\\": \\"chi\\", \\"ja\\": \\"jpn\\",\\\\n    \\"fr\\": \\"fra\\", \\"es\\": \\"spa\\", \\"it\\": \\"ita\\", \\"pt\\": \\"por\\", \\"ko\\": \\"kor\\",\\\\n}\\\\n\\\\ndef _load_mono(path: str):\\\\n    raw = subprocess.run(\\\\n        [\\"ffmpeg\\", \\"-nostdin\\", \\"-threads\\", \\"0\\", \\"-i\\", path, \\"-f\\", \\"f32le\\",\\\\n         \\"-ac\\", \\"1\\", \\"-ar\\", str(SAMPLE_RATE), \\"-\\"],\\\\n        check=True, capture_output=True,\\\\n    ).stdout\\\\n    return np.frombuffer(raw, dtype=np.float32).copy()\\\\n\\\\ndef _onnx_emissions(audio: np.ndarray):\\\\n    window = 30 * SAMPLE_RATE\\\\n    context = 2 * SAMPLE_RATE\\\\n    context_frames = context // RATIO\\\\n    window_frames = window // RATIO\\\\n    if audio.size < window:\\\\n        chunks = [audio]\\\\n        extension = 0\\\\n        use_context = False\\\\n    else:\\\\n        count = math.ceil(audio.size / window)\\\\n        extension = count * window - audio.size\\\\n        padded = np.pad(audio, (context, context + extension))\\\\n        chunks = [padded[index * window:index * window + window + 2 * context] for index in range(count)]\\\\n        use_context = True\\\\n    outputs = []\\\\n    input_name = SESSION.get_inputs()[0].name\\\\n    for chunk in chunks:\\\\n        logits = SESSION.run(None, {input_name: chunk[None, :].astype(np.float32)})[0][0]\\\\n        if use_context:\\\\n            logits = logits[context_frames:context_frames + window_frames]\\\\n        outputs.append(logits)\\\\n    emissions = np.concatenate(outputs, axis=0)\\\\n    if extension:\\\\n        emissions = emissions[:-(extension // RATIO)]\\\\n    values = torch.from_numpy(emissions).float().log_softmax(-1)\\\\n    values = torch.cat([values, torch.zeros(values.size(0), 1)], dim=1)\\\\n    return values, RATIO * 1000.0 / SAMPLE_RATE\\\\n\\\\ndef align_exact(source_path: str, transcript: str, language: str):\\\\n    iso = ISO3.get(language, language)\\\\n    emissions, stride = _onnx_emissions(_load_mono(source_path))\\\\n    tokens, text_tokens = preprocess_text(transcript, romanize=True, language=iso)\\\\n    segments, scores, blank = get_alignments(emissions, tokens, TOKENIZER)\\\\n    spans = get_spans(tokens, segments, blank)\\\\n    return postprocess_results(text_tokens, spans, stride, scores)\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_ALIGNMENT_TOKEN\\"]\\\\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\\\\nMAX_AUDIO_SECONDS = 300\\\\nALLOWED_CONTENT_TYPES = {\\\\n    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp4\\", \\"audio/webm\\",\\\\n    \\"audio/ogg\\", \\"audio/flac\\", \\"application/octet-stream\\",\\\\n}\\\\nALLOWED_EXTENSIONS = {\\".wav\\", \\".mp3\\", \\".m4a\\", \\".mp4\\", \\".webm\\", \\".ogg\\", \\".flac\\"}\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\nMODEL_LOCK = threading.Lock()\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef media_duration_seconds(path: str) -> float:\\\\n    probe = subprocess.run(\\\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-show_entries\\", \\"format=duration\\",\\\\n         \\"-of\\", \\"default=nokey=1:noprint_wrappers=1\\", path],\\\\n        text=True, capture_output=True,\\\\n    )\\\\n    try:\\\\n        duration = float(probe.stdout.strip())\\\\n    except ValueError:\\\\n        duration = 0.0\\\\n    if probe.returncode != 0 or duration <= 0.0:\\\\n        raise HTTPException(status_code=415, detail=\\"audio is unsupported or could not be decoded\\")\\\\n    return duration\\\\n\\\\ndef validate_segments(raw_segments) -> list[dict]:\\\\n    segments = []\\\\n    previous_end = 0.0\\\\n    for raw in raw_segments:\\\\n        text = str(raw.get(\\"text\\", \\"\\")).strip()\\\\n        if not text:\\\\n            continue\\\\n        start = float(raw.get(\\"start\\", raw.get(\\"start_time\\", 0.0)))\\\\n        end = float(raw.get(\\"end\\", raw.get(\\"end_time\\", start)))\\\\n        score = max(0.0, min(1.0, float(raw.get(\\"score\\", raw.get(\\"confidence\\", 1.0)))))\\\\n        if start < 0.0 or end < start or start + 0.002 < previous_end:\\\\n            raise RuntimeError(\\"aligner returned non-monotonic timestamps\\")\\\\n        segments.append({\\"text\\": text, \\"start\\": start, \\"end\\": end, \\"score\\": score, \\"kind\\": raw.get(\\"kind\\", \\"word\\")})\\\\n        previous_end = end\\\\n    if not segments:\\\\n        raise RuntimeError(\\"aligner returned no timestamped tokens\\")\\\\n    return segments\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Alignment - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"forced-alignment\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"max_audio_seconds\\": MAX_AUDIO_SECONDS,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/alignments\\")\\\\nasync def align(\\\\n    audio: UploadFile = File(...),\\\\n    transcript: str = Form(...),\\\\n    language: str = Form(\\"en\\"),\\\\n    model: str = Form(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    text = transcript.strip()\\\\n    if not text:\\\\n        raise HTTPException(status_code=422, detail=\\"transcript is required\\")\\\\n    suffix = Path(audio.filename or \\"audio.wav\\").suffix.lower() or \\".wav\\"\\\\n    if suffix not in ALLOWED_EXTENSIONS:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio filename extension\\")\\\\n    if audio.content_type and audio.content_type not in ALLOWED_CONTENT_TYPES:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio MIME type\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab alignment worker is busy; retry shortly\\")\\\\n    source_path = None\\\\n    try:\\\\n        descriptor, source_path = tempfile.mkstemp(suffix=suffix)\\\\n        with os.fdopen(descriptor, \\"wb\\") as output:\\\\n            while chunk := await audio.read(1024 * 1024):\\\\n                output.write(chunk)\\\\n                if output.tell() > MAX_UPLOAD_BYTES:\\\\n                    raise HTTPException(status_code=413, detail=\\"audio exceeds 512 MB upload limit\\")\\\\n        duration = media_duration_seconds(source_path)\\\\n        if duration > MAX_AUDIO_SECONDS:\\\\n            raise HTTPException(status_code=413, detail=\\"audio exceeds the five minute duration limit\\")\\\\n        with MODEL_LOCK:\\\\n            raw_segments = align_exact(source_path, text, language.strip().lower() or \\"en\\")\\\\n        segments = validate_segments(raw_segments)\\\\n        return {\\"duration\\": duration, \\"segments\\": segments, \\"unaligned_tokens\\": []}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} alignment failed: {type(error).__name__}: {str(error)[:300]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        if source_path:\\\\n            Path(source_path).unlink(missing_ok=True)\\\\n        await audio.close()\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'mms-forced-aligner-onnx\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Forced Alignment\'\\n",\n        "MODEL_ID = \'mms-forced-aligner-onnx\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_alignment_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_alignment_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_alignment_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "forced-alignment",\n      "family_id": "mms-forced-aligner-onnx",\n      "upstream_model": "onnx-community/mms-300m-1130-forced-aligner-ONNX",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/alignment/LA_STUDIO_ALIGNMENT_QWEN3_0_6B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio forced-alignment \\u2014 Qwen3 ForcedAligner 0.6B\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3-forced-aligner-0.6b` (`Qwen/Qwen3-ForcedAligner-0.6B`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"git+https://github.com/QwenLM/Qwen3-ASR.git@7c6daf77a2421100f5fb066495372c00129d39ff\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"python-multipart==0.0.20\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_alignment_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport re\\\\nimport subprocess\\\\nimport tempfile\\\\nimport threading\\\\nfrom pathlib import Path\\\\n\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nfrom qwen_asr import Qwen3ForcedAligner\\\\n\\\\nMODEL_ID = \\"qwen3-forced-aligner-0.6b\\"\\\\nMODEL_NAME = \\"Qwen3 ForcedAligner 0.6B\\"\\\\nUPSTREAM_MODEL = \\"Qwen/Qwen3-ForcedAligner-0.6B\\"\\\\nLANGUAGE_NAMES = {\\\\n    \\"zh\\": \\"Chinese\\", \\"zho\\": \\"Chinese\\", \\"chi\\": \\"Chinese\\", \\"en\\": \\"English\\", \\"eng\\": \\"English\\",\\\\n    \\"yue\\": \\"Cantonese\\", \\"ja\\": \\"Japanese\\", \\"jpn\\": \\"Japanese\\",\\\\n    \\"ko\\": \\"Korean\\", \\"kor\\": \\"Korean\\", \\"de\\": \\"German\\", \\"deu\\": \\"German\\",\\\\n    \\"fr\\": \\"French\\", \\"fra\\": \\"French\\", \\"ru\\": \\"Russian\\", \\"rus\\": \\"Russian\\",\\\\n    \\"pt\\": \\"Portuguese\\", \\"por\\": \\"Portuguese\\", \\"es\\": \\"Spanish\\", \\"spa\\": \\"Spanish\\",\\\\n    \\"it\\": \\"Italian\\", \\"ita\\": \\"Italian\\",\\\\n}\\\\nSUPPORTED_LANGUAGES = sorted(set(LANGUAGE_NAMES.values()))\\\\nALIGNER = Qwen3ForcedAligner.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    dtype=torch.float16,\\\\n    device_map=\\"cuda:0\\",\\\\n)\\\\n\\\\ndef align_exact(source_path: str, transcript: str, language: str):\\\\n    language_name = LANGUAGE_NAMES.get(language)\\\\n    if not language_name:\\\\n        raise HTTPException(status_code=422, detail=\\"unsupported Qwen3 ForcedAligner language\\")\\\\n    results = ALIGNER.align(audio=source_path, text=transcript, language=language_name)\\\\n    if not results:\\\\n        return []\\\\n    return [\\\\n        {\\"text\\": item.text, \\"start\\": item.start_time, \\"end\\": item.end_time, \\"score\\": 1.0}\\\\n        for item in results[0]\\\\n    ]\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_ALIGNMENT_TOKEN\\"]\\\\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\\\\nMAX_AUDIO_SECONDS = 300\\\\nALLOWED_CONTENT_TYPES = {\\\\n    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp4\\", \\"audio/webm\\",\\\\n    \\"audio/ogg\\", \\"audio/flac\\", \\"application/octet-stream\\",\\\\n}\\\\nALLOWED_EXTENSIONS = {\\".wav\\", \\".mp3\\", \\".m4a\\", \\".mp4\\", \\".webm\\", \\".ogg\\", \\".flac\\"}\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\nMODEL_LOCK = threading.Lock()\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef media_duration_seconds(path: str) -> float:\\\\n    probe = subprocess.run(\\\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-show_entries\\", \\"format=duration\\",\\\\n         \\"-of\\", \\"default=nokey=1:noprint_wrappers=1\\", path],\\\\n        text=True, capture_output=True,\\\\n    )\\\\n    try:\\\\n        duration = float(probe.stdout.strip())\\\\n    except ValueError:\\\\n        duration = 0.0\\\\n    if probe.returncode != 0 or duration <= 0.0:\\\\n        raise HTTPException(status_code=415, detail=\\"audio is unsupported or could not be decoded\\")\\\\n    return duration\\\\n\\\\ndef validate_segments(raw_segments) -> list[dict]:\\\\n    segments = []\\\\n    previous_end = 0.0\\\\n    for raw in raw_segments:\\\\n        text = str(raw.get(\\"text\\", \\"\\")).strip()\\\\n        if not text:\\\\n            continue\\\\n        start = float(raw.get(\\"start\\", raw.get(\\"start_time\\", 0.0)))\\\\n        end = float(raw.get(\\"end\\", raw.get(\\"end_time\\", start)))\\\\n        score = max(0.0, min(1.0, float(raw.get(\\"score\\", raw.get(\\"confidence\\", 1.0)))))\\\\n        if start < 0.0 or end < start or start + 0.002 < previous_end:\\\\n            raise RuntimeError(\\"aligner returned non-monotonic timestamps\\")\\\\n        segments.append({\\"text\\": text, \\"start\\": start, \\"end\\": end, \\"score\\": score, \\"kind\\": raw.get(\\"kind\\", \\"word\\")})\\\\n        previous_end = end\\\\n    if not segments:\\\\n        raise RuntimeError(\\"aligner returned no timestamped tokens\\")\\\\n    return segments\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Alignment - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"forced-alignment\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"max_audio_seconds\\": MAX_AUDIO_SECONDS,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/alignments\\")\\\\nasync def align(\\\\n    audio: UploadFile = File(...),\\\\n    transcript: str = Form(...),\\\\n    language: str = Form(\\"en\\"),\\\\n    model: str = Form(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    text = transcript.strip()\\\\n    if not text:\\\\n        raise HTTPException(status_code=422, detail=\\"transcript is required\\")\\\\n    suffix = Path(audio.filename or \\"audio.wav\\").suffix.lower() or \\".wav\\"\\\\n    if suffix not in ALLOWED_EXTENSIONS:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio filename extension\\")\\\\n    if audio.content_type and audio.content_type not in ALLOWED_CONTENT_TYPES:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio MIME type\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab alignment worker is busy; retry shortly\\")\\\\n    source_path = None\\\\n    try:\\\\n        descriptor, source_path = tempfile.mkstemp(suffix=suffix)\\\\n        with os.fdopen(descriptor, \\"wb\\") as output:\\\\n            while chunk := await audio.read(1024 * 1024):\\\\n                output.write(chunk)\\\\n                if output.tell() > MAX_UPLOAD_BYTES:\\\\n                    raise HTTPException(status_code=413, detail=\\"audio exceeds 512 MB upload limit\\")\\\\n        duration = media_duration_seconds(source_path)\\\\n        if duration > MAX_AUDIO_SECONDS:\\\\n            raise HTTPException(status_code=413, detail=\\"audio exceeds the five minute duration limit\\")\\\\n        with MODEL_LOCK:\\\\n            raw_segments = align_exact(source_path, text, language.strip().lower() or \\"en\\")\\\\n        segments = validate_segments(raw_segments)\\\\n        return {\\"duration\\": duration, \\"segments\\": segments, \\"unaligned_tokens\\": []}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} alignment failed: {type(error).__name__}: {str(error)[:300]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        if source_path:\\\\n            Path(source_path).unlink(missing_ok=True)\\\\n        await audio.close()\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'qwen3-forced-aligner-0.6b\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Forced Alignment\'\\n",\n        "MODEL_ID = \'qwen3-forced-aligner-0.6b\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_alignment_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_alignment_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_alignment_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "forced-alignment",\n      "family_id": "qwen3-forced-aligner-0.6b",\n      "upstream_model": "Qwen/Qwen3-ForcedAligner-0.6B",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/alignment/LA_STUDIO_ALIGNMENT_WAV2VEC2_ZH_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio forced-alignment \\u2014 Wav2Vec2 Chinese Aligner\\n",\n        "\\n",\n        "This notebook loads exactly `wav2vec2-aligner-zh` (`cstr/wav2vec2-large-xlsr-53-chinese-zh-cn-GGUF`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "!git clone --quiet --recursive https://github.com/CrispStrobe/CrispASR.git /content/CrispASR\\n",\n        "!git -C /content/CrispASR checkout --quiet 754b67289cf1137e3ed722885705f94132fc614f\\n",\n        "!git -C /content/CrispASR submodule update --init --recursive\\n",\n        "!cmake -S /content/CrispASR -B /content/CrispASR/build -G Ninja -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON\\n",\n        "!cmake --build /content/CrispASR/build --target crispasr -j2\\n",\n        "%pip install -q \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"python-multipart==0.0.20\\"\\n",\n        "\\n",\n        "!wget -q --show-progress -O /content/wav2vec2-aligner-zh-q4_k.gguf https://huggingface.co/cstr/wav2vec2-large-xlsr-53-chinese-zh-cn-GGUF/resolve/main/wav2vec2-large-xlsr-53-chinese-zh-cn-q4_k.gguf\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_alignment_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport re\\\\nimport subprocess\\\\nimport tempfile\\\\nimport threading\\\\nfrom pathlib import Path\\\\n\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nimport json\\\\n\\\\nMODEL_ID = \\"wav2vec2-aligner-zh\\"\\\\nMODEL_NAME = \\"Wav2Vec2 Chinese Aligner\\"\\\\nUPSTREAM_MODEL = \\"cstr/wav2vec2-large-xlsr-53-chinese-zh-cn-GGUF\\"\\\\nSUPPORTED_LANGUAGES = [\\"zh\\", \\"zho\\", \\"chi\\", \\"cmn\\", \\"zh-cn\\"]\\\\nCRISPASR = \\"/content/CrispASR/build/bin/crispasr\\"\\\\nALIGNER_MODEL = \\"/content/wav2vec2-aligner-zh-q4_k.gguf\\"\\\\n\\\\nif not Path(CRISPASR).is_file() or not Path(ALIGNER_MODEL).is_file():\\\\n    raise RuntimeError(\\"CrispASR CUDA runtime or the exact aligner model is missing\\")\\\\n\\\\ndef _collect_crisp_segments(payload):\\\\n    if isinstance(payload, list):\\\\n        entries = payload\\\\n    elif isinstance(payload, dict):\\\\n        entries = payload.get(\\"segments\\", payload.get(\\"alignment\\", payload.get(\\"results\\", [])))\\\\n    else:\\\\n        entries = []\\\\n    output = []\\\\n    for entry in entries:\\\\n        if not isinstance(entry, dict):\\\\n            continue\\\\n        words = entry.get(\\"words\\")\\\\n        if isinstance(words, list) and words:\\\\n            output.extend(words)\\\\n        else:\\\\n            output.append(entry)\\\\n    return output\\\\n\\\\ndef align_exact(source_path: str, transcript: str, language: str):\\\\n    if MODEL_ID == \\"wav2vec2-aligner-zh\\" and language not in {\\"zh\\", \\"zho\\", \\"chi\\", \\"cmn\\", \\"zh-cn\\"}:\\\\n        raise HTTPException(status_code=422, detail=\\"the selected Wav2Vec2 aligner supports Mandarin Chinese only\\")\\\\n    with tempfile.TemporaryDirectory(prefix=\\"la-studio-crisp-align-\\") as directory:\\\\n        wav_path = str(Path(directory) / \\"source.wav\\")\\\\n        output_path = str(Path(directory) / \\"alignment.json\\")\\\\n        subprocess.run(\\\\n            [\\"ffmpeg\\", \\"-y\\", \\"-v\\", \\"error\\", \\"-i\\", source_path, \\"-vn\\", \\"-ac\\", \\"1\\", \\"-ar\\", \\"16000\\", wav_path],\\\\n            check=True,\\\\n        )\\\\n        command = [\\\\n            CRISPASR, \\"--align-only\\", \\"-am\\", ALIGNER_MODEL, \\"-f\\", wav_path,\\\\n            \\"--ref-text\\", transcript, \\"--align-format\\", \\"json\\",\\\\n            \\"--align-granularity\\", \\"word\\", \\"--align-output\\", output_path,\\\\n            \\"--gpu-backend\\", \\"cuda\\", \\"--strict-pipeline\\", \\"-l\\", language,\\\\n        ]\\\\n        result = subprocess.run(command, text=True, capture_output=True)\\\\n        if result.returncode != 0 or not Path(output_path).is_file():\\\\n            raise RuntimeError(\\"CrispASR CUDA aligner failed: \\" + (result.stderr or result.stdout)[-1600:])\\\\n        payload = json.loads(Path(output_path).read_text(encoding=\\"utf-8\\"))\\\\n        return _collect_crisp_segments(payload)\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_ALIGNMENT_TOKEN\\"]\\\\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\\\\nMAX_AUDIO_SECONDS = 300\\\\nALLOWED_CONTENT_TYPES = {\\\\n    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp4\\", \\"audio/webm\\",\\\\n    \\"audio/ogg\\", \\"audio/flac\\", \\"application/octet-stream\\",\\\\n}\\\\nALLOWED_EXTENSIONS = {\\".wav\\", \\".mp3\\", \\".m4a\\", \\".mp4\\", \\".webm\\", \\".ogg\\", \\".flac\\"}\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\nMODEL_LOCK = threading.Lock()\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef media_duration_seconds(path: str) -> float:\\\\n    probe = subprocess.run(\\\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-show_entries\\", \\"format=duration\\",\\\\n         \\"-of\\", \\"default=nokey=1:noprint_wrappers=1\\", path],\\\\n        text=True, capture_output=True,\\\\n    )\\\\n    try:\\\\n        duration = float(probe.stdout.strip())\\\\n    except ValueError:\\\\n        duration = 0.0\\\\n    if probe.returncode != 0 or duration <= 0.0:\\\\n        raise HTTPException(status_code=415, detail=\\"audio is unsupported or could not be decoded\\")\\\\n    return duration\\\\n\\\\ndef validate_segments(raw_segments) -> list[dict]:\\\\n    segments = []\\\\n    previous_end = 0.0\\\\n    for raw in raw_segments:\\\\n        text = str(raw.get(\\"text\\", \\"\\")).strip()\\\\n        if not text:\\\\n            continue\\\\n        start = float(raw.get(\\"start\\", raw.get(\\"start_time\\", 0.0)))\\\\n        end = float(raw.get(\\"end\\", raw.get(\\"end_time\\", start)))\\\\n        score = max(0.0, min(1.0, float(raw.get(\\"score\\", raw.get(\\"confidence\\", 1.0)))))\\\\n        if start < 0.0 or end < start or start + 0.002 < previous_end:\\\\n            raise RuntimeError(\\"aligner returned non-monotonic timestamps\\")\\\\n        segments.append({\\"text\\": text, \\"start\\": start, \\"end\\": end, \\"score\\": score, \\"kind\\": raw.get(\\"kind\\", \\"word\\")})\\\\n        previous_end = end\\\\n    if not segments:\\\\n        raise RuntimeError(\\"aligner returned no timestamped tokens\\")\\\\n    return segments\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Alignment - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"forced-alignment\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"max_audio_seconds\\": MAX_AUDIO_SECONDS,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/alignments\\")\\\\nasync def align(\\\\n    audio: UploadFile = File(...),\\\\n    transcript: str = Form(...),\\\\n    language: str = Form(\\"en\\"),\\\\n    model: str = Form(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    text = transcript.strip()\\\\n    if not text:\\\\n        raise HTTPException(status_code=422, detail=\\"transcript is required\\")\\\\n    suffix = Path(audio.filename or \\"audio.wav\\").suffix.lower() or \\".wav\\"\\\\n    if suffix not in ALLOWED_EXTENSIONS:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio filename extension\\")\\\\n    if audio.content_type and audio.content_type not in ALLOWED_CONTENT_TYPES:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported audio MIME type\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab alignment worker is busy; retry shortly\\")\\\\n    source_path = None\\\\n    try:\\\\n        descriptor, source_path = tempfile.mkstemp(suffix=suffix)\\\\n        with os.fdopen(descriptor, \\"wb\\") as output:\\\\n            while chunk := await audio.read(1024 * 1024):\\\\n                output.write(chunk)\\\\n                if output.tell() > MAX_UPLOAD_BYTES:\\\\n                    raise HTTPException(status_code=413, detail=\\"audio exceeds 512 MB upload limit\\")\\\\n        duration = media_duration_seconds(source_path)\\\\n        if duration > MAX_AUDIO_SECONDS:\\\\n            raise HTTPException(status_code=413, detail=\\"audio exceeds the five minute duration limit\\")\\\\n        with MODEL_LOCK:\\\\n            raw_segments = align_exact(source_path, text, language.strip().lower() or \\"en\\")\\\\n        segments = validate_segments(raw_segments)\\\\n        return {\\"duration\\": duration, \\"segments\\": segments, \\"unaligned_tokens\\": []}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} alignment failed: {type(error).__name__}: {str(error)[:300]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        if source_path:\\\\n            Path(source_path).unlink(missing_ok=True)\\\\n        await audio.close()\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'wav2vec2-aligner-zh\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Forced Alignment\'\\n",\n        "MODEL_ID = \'wav2vec2-aligner-zh\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_ALIGNMENT_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_alignment_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_alignment_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_alignment_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "forced-alignment",\n      "family_id": "wav2vec2-aligner-zh",\n      "upstream_model": "cstr/wav2vec2-large-xlsr-53-chinese-zh-cn-GGUF",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/llm/LA_STUDIO_LLM_QWEN3_5_2B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio llm-chat \\u2014 Qwen3.5 2B\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3.5-2b` (`Qwen/Qwen3.5-2B`) on CUDA.\\n",\n        "It is independent from API Gateway and rejects every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into the matching LA Studio feature.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"transformers>=5.6.0,<6\\" \\"accelerate>=1.12,<2\\" \\"safetensors>=0.6,<1\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_chat_worker.py\')\\n",\n        "WORKER.write_text(\'import json\\\\nimport os\\\\nimport secrets\\\\nimport threading\\\\n\\\\nimport torch\\\\nfrom fastapi import Depends, FastAPI, Header, HTTPException, Request\\\\nfrom fastapi.responses import StreamingResponse\\\\nfrom pydantic import BaseModel, Field\\\\nfrom transformers import (\\\\n    AutoModelForMultimodalLM,\\\\n    AutoProcessor,\\\\n    StoppingCriteria,\\\\n    StoppingCriteriaList,\\\\n    TextIteratorStreamer,\\\\n)\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.\\")\\\\n\\\\nMODEL_ID = \\"qwen3.5-2b\\"\\\\nMODEL_NAME = \\"Qwen3.5 2B\\"\\\\nUPSTREAM_MODEL = \\"Qwen/Qwen3.5-2B\\"\\\\nUPSTREAM_REVISION = \\"15852e8c16360a2fea060d615a32b45270f8a8fc\\"\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_CHAT_TOKEN\\"]\\\\nMAX_CHAT_MESSAGES = 64\\\\nMAX_CHAT_CHARS = 50000\\\\nMAX_CHAT_TOKENS = 32768\\\\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nPROCESSOR = AutoProcessor.from_pretrained(UPSTREAM_MODEL, revision=UPSTREAM_REVISION)\\\\nMODEL = AutoModelForMultimodalLM.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    revision=UPSTREAM_REVISION,\\\\n    torch_dtype=torch.float16,\\\\n    device_map={\\"\\": 0},\\\\n    low_cpu_mem_usage=True,\\\\n).eval()\\\\nTOKENIZER = PROCESSOR.tokenizer\\\\n\\\\ndef authorize(authorization: str = Header(default=\\"\\")):\\\\n    if not secrets.compare_digest(authorization, \\"Bearer \\" + TOKEN):\\\\n        raise HTTPException(status_code=401, detail=\\"invalid or missing bearer token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\nclass ChatMessage(BaseModel):\\\\n    role: str = Field(pattern=\\"^(system|user|assistant)$\\")\\\\n    content: str = Field(min_length=1, max_length=8000)\\\\n\\\\nclass ChatRequest(BaseModel):\\\\n    model: str\\\\n    messages: list[ChatMessage]\\\\n    stream: bool = True\\\\n    max_tokens: int = Field(default=1024, ge=1, le=MAX_CHAT_TOKENS)\\\\n    context_tokens: int = Field(default=4096, ge=512, le=131072)\\\\n    temperature: float = Field(default=0.7, ge=0.01, le=2.0)\\\\n    top_p: float = Field(default=0.8, ge=0.01, le=1.0)\\\\n    top_k: int = Field(default=20, ge=1, le=200)\\\\n    repeat_penalty: float = Field(default=1.05, ge=0.8, le=2.0)\\\\n\\\\nclass DisconnectStop(StoppingCriteria):\\\\n    def __init__(self, cancelled: threading.Event):\\\\n        self.cancelled = cancelled\\\\n    def __call__(self, input_ids, scores, **kwargs):\\\\n        return self.cancelled.is_set()\\\\n\\\\napp = FastAPI(title=\\"LA Studio Chat - Qwen3.5 2B\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"llm-chat\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/chat/completions\\")\\\\nasync def chat(request: ChatRequest, http_request: Request, _: None = Depends(authorize)):\\\\n    require_exact_model(request.model)\\\\n    if not request.stream:\\\\n        raise HTTPException(status_code=400, detail=\\"this direct worker requires stream=true\\")\\\\n    if not request.messages:\\\\n        raise HTTPException(status_code=400, detail=\\"messages must not be empty\\")\\\\n    if len(request.messages) > MAX_CHAT_MESSAGES or sum(len(item.content) for item in request.messages) > MAX_CHAT_CHARS:\\\\n        raise HTTPException(status_code=413, detail=\\"chat request is too large\\")\\\\n    if not INFERENCE_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"worker is busy; retry shortly\\")\\\\n    try:\\\\n        messages = [{\\"role\\": item.role, \\"content\\": item.content} for item in request.messages]\\\\n        inputs = PROCESSOR.apply_chat_template(\\\\n            messages,\\\\n            add_generation_prompt=True,\\\\n            tokenize=True,\\\\n            return_dict=True,\\\\n            return_tensors=\\"pt\\",\\\\n            truncation=True,\\\\n            max_length=request.context_tokens,\\\\n        ).to(\\"cuda\\")\\\\n        streamer = TextIteratorStreamer(TOKENIZER, skip_prompt=True, skip_special_tokens=True)\\\\n        cancelled = threading.Event()\\\\n        generation_errors = []\\\\n        def generate():\\\\n            try:\\\\n                MODEL.generate(\\\\n                    **inputs,\\\\n                    streamer=streamer,\\\\n                    max_new_tokens=request.max_tokens,\\\\n                    do_sample=True,\\\\n                    temperature=request.temperature,\\\\n                    top_p=request.top_p,\\\\n                    top_k=request.top_k,\\\\n                    repetition_penalty=request.repeat_penalty,\\\\n                    pad_token_id=TOKENIZER.eos_token_id,\\\\n                    stopping_criteria=StoppingCriteriaList([DisconnectStop(cancelled)]),\\\\n                )\\\\n            except Exception as error:\\\\n                generation_errors.append(error)\\\\n                streamer.end()\\\\n        generation = threading.Thread(target=generate, daemon=True)\\\\n        generation.start()\\\\n    except Exception:\\\\n        INFERENCE_SLOTS.release()\\\\n        raise\\\\n\\\\n    async def events():\\\\n        try:\\\\n            for token in streamer:\\\\n                if await http_request.is_disconnected():\\\\n                    cancelled.set()\\\\n                    break\\\\n                yield \\"data: \\" + json.dumps({\\"choices\\": [{\\"delta\\": {\\"content\\": token}}]}, ensure_ascii=False) + \\"\\\\\\\\n\\\\\\\\n\\"\\\\n            generation.join(timeout=10)\\\\n            if generation_errors:\\\\n                message = f\\"{type(generation_errors[0]).__name__}: {str(generation_errors[0])[:300]}\\"\\\\n                yield \\"data: \\" + json.dumps({\\"error\\": {\\"message\\": message}}) + \\"\\\\\\\\n\\\\\\\\n\\"\\\\n            yield \\"data: [DONE]\\\\\\\\n\\\\\\\\n\\"\\\\n        finally:\\\\n            cancelled.set()\\\\n            INFERENCE_SLOTS.release()\\\\n    return StreamingResponse(events(), media_type=\\"text/event-stream\\", headers={\\"Cache-Control\\": \\"no-store\\"})\' + \'\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'qwen3.5-2b\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'llm-chat\'\\n",\n        "MODEL_ID = \'qwen3.5-2b\'\\n",\n        "PORT = 3944\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_CHAT_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_CHAT_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_CHAT_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_chat_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_chat_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_chat_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "llm-chat",\n      "family_id": "qwen3.5-2b",\n      "upstream_model": "Qwen/Qwen3.5-2B",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/stt/LA_STUDIO_STT_NEMOTRON_3_5_0_6B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio \\u2014 Nemotron-3.5 ASR Streaming 0.6B Colab GPU worker\\n",\n        "\\n",\n        "This notebook loads exactly `nemotron-3.5-asr-streaming-0.6b` on the Colab GPU.\\n",\n        "It rejects transcription requests for every other model ID.\\n",\n        "\\n",\n        "Long recordings use an asynchronous GPU job: the app uploads the\\n",\n        "audio once, then polls short status requests until this exact model\\n",\n        "completes. This avoids Cloudflare\'s 120-second proxy response limit.\\n",\n        "\\n",\n        "Run every cell in order, then paste the printed URL and TOKEN into\\n",\n        "LA Studio. The tunnel is public, but every worker endpoint requires\\n",\n        "the random session token printed by the last cell.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "%pip install -q transformers==5.14.1 accelerate==1.10.1 librosa==0.11.0 fastapi==0.115.12 uvicorn==0.34.3 python-multipart==0.0.20 soundfile==0.13.1\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import asyncio\\n",\n        "import os\\n",\n        "import secrets\\n",\n        "import tempfile\\n",\n        "import threading\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "import soundfile as sf\\n",\n        "import torch\\n",\n        "from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile\\n",\n        "from fastapi.responses import JSONResponse\\n",\n        "\\n",\n        "if not torch.cuda.is_available():\\n",\n        "    raise RuntimeError(\\"A Colab GPU runtime is required. Choose Runtime > Change runtime type > GPU.\\")\\n",\n        "\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "WORKER_REVISION = \\"stt-2026-07-30.2\\"\\n",\n        "MAX_UPLOAD_BYTES = 512 * 1024 * 1024\\n",\n        "MAX_AUDIO_SECONDS = 30 * 60\\n",\n        "ALLOWED_CONTENT_TYPES = {\\n",\n        "    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp3\\", \\"audio/mp4\\",\\n",\n        "    \\"audio/flac\\", \\"audio/ogg\\", \\"audio/webm\\", \\"application/octet-stream\\",\\n",\n        "}\\n",\n        "REQUEST_SLOTS = threading.BoundedSemaphore(1)\\n",\n        "JOB_TTL_SECONDS = 30 * 60\\n",\n        "UPLOAD_TTL_SECONDS = 10 * 60\\n",\n        "CHUNK_UPLOAD_BYTES = 2 * 1024 * 1024\\n",\n        "JOB_LOCK = threading.Lock()\\n",\n        "JOBS: dict[str, dict] = {}\\n",\n        "UPLOADS: dict[str, dict] = {}\\n",\n        "\\n",\n        "import librosa\\n",\n        "from transformers import AutoModelForRNNT, AutoProcessor\\n",\n        "\\n",\n        "MODEL_ID = \\"nemotron-3.5-asr-streaming-0.6b\\"\\n",\n        "MODEL_NAME = \\"Nemotron-3.5 ASR Streaming 0.6B\\"\\n",\n        "UPSTREAM_MODEL = \\"nvidia/nemotron-3.5-asr-streaming-0.6b\\"\\n",\n        "processor = AutoProcessor.from_pretrained(UPSTREAM_MODEL)\\n",\n        "stt_model = AutoModelForRNNT.from_pretrained(\\n",\n        "    UPSTREAM_MODEL,\\n",\n        "    device_map=\\"auto\\",\\n",\n        "    dtype=torch.float16,\\n",\n        ")\\n",\n        "if not next(stt_model.parameters()).is_cuda:\\n",\n        "    raise RuntimeError(\\"Nemotron was not loaded on CUDA\\")\\n",\n        "\\n",\n        "def run_transcription(path: str, language: str):\\n",\n        "    sampling_rate = processor.feature_extractor.sampling_rate\\n",\n        "    audio, _ = librosa.load(path, sr=sampling_rate, mono=True)\\n",\n        "    requested = language.strip() or \\"auto\\"\\n",\n        "    inputs = processor(\\n",\n        "        audio,\\n",\n        "        sampling_rate=sampling_rate,\\n",\n        "        language=requested,\\n",\n        "        return_tensors=\\"pt\\",\\n",\n        "    )\\n",\n        "    inputs = inputs.to(stt_model.device, dtype=stt_model.dtype)\\n",\n        "    with torch.inference_mode():\\n",\n        "        output = stt_model.generate(**inputs, return_dict_in_generate=True)\\n",\n        "    text = processor.decode(output.sequences, skip_special_tokens=True).strip()\\n",\n        "    return {\\"text\\": text, \\"segments\\": [], \\"language\\": requested}\\n",\n        "\\n",\n        "app = FastAPI(title=f\\"LA Studio STT \\u2014 {MODEL_NAME}\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.exception_handler(Exception)\\n",\n        "async def unhandled_exception(request: Request, error: Exception):\\n",\n        "    # A tunnel 500 without a response body is impossible to act on from the\\n",\n        "    # desktop app. Keep the detail bounded, and also print it in the Colab\\n",\n        "    # cell so the notebook owns the operational diagnosis.\\n",\n        "    detail = f\\"STT worker internal error: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "    print(detail, flush=True)\\n",\n        "    return JSONResponse(status_code=500, content={\\"detail\\": detail})\\n",\n        "\\n",\n        "\\n",\n        "def require_token(authorization: str | None) -> None:\\n",\n        "    if authorization != f\\"Bearer {TOKEN}\\":\\n",\n        "        raise HTTPException(status_code=401, detail=\\"Invalid or missing Colab session token\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/health\\")\\n",\n        "def health():\\n",\n        "    return {\\n",\n        "        \\"ok\\": True,\\n",\n        "        \\"ready\\": True,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"gpu\\": torch.cuda.get_device_name(0),\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "        \\"variant\\": \\"fixed\\",\\n",\n        "        \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v1/capabilities\\")\\n",\n        "def capabilities(authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    return {\\n",\n        "        \\"contract_version\\": 1,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"cuda\\": True,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "        \\"endpoints\\": {\\n",\n        "            \\"transcription_jobs\\": \\"/v2/jobs/transcriptions\\",\\n",\n        "            \\"chunked_transcription_uploads\\": \\"/v2/uploads/stt\\",\\n",\n        "        },\\n",\n        "        \\"chunked_uploads\\": True,\\n",\n        "        \\"capabilities\\": [{\\n",\n        "            \\"id\\": \\"stt\\",\\n",\n        "            \\"models\\": [{\\n",\n        "                \\"id\\": MODEL_ID,\\n",\n        "                \\"name\\": MODEL_NAME,\\n",\n        "                \\"variant\\": \\"fixed\\",\\n",\n        "                \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "                \\"loaded\\": True,\\n",\n        "                \\"device\\": \\"cuda\\",\\n",\n        "            }],\\n",\n        "        }],\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "async def save_upload(file: UploadFile) -> tuple[Path, int]:\\n",\n        "    suffix = Path(file.filename or \\"audio.wav\\").suffix or \\".wav\\"\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=suffix, delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    total = 0\\n",\n        "    try:\\n",\n        "        with handle:\\n",\n        "            while True:\\n",\n        "                chunk = await file.read(1024 * 1024)\\n",\n        "                if not chunk:\\n",\n        "                    break\\n",\n        "                total += len(chunk)\\n",\n        "                if total > MAX_UPLOAD_BYTES:\\n",\n        "                    raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "                handle.write(chunk)\\n",\n        "        return path, total\\n",\n        "    except Exception:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "        raise\\n",\n        "\\n",\n        "\\n",\n        "def validate_model(model: str) -> str:\\n",\n        "    requested = model.strip().lower()\\n",\n        "    if requested != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    return requested\\n",\n        "\\n",\n        "\\n",\n        "def validate_audio_duration(path: Path) -> None:\\n",\n        "    try:\\n",\n        "        info = sf.info(str(path))\\n",\n        "        if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "    except HTTPException:\\n",\n        "        raise\\n",\n        "    except Exception:\\n",\n        "        # Compressed formats may not be readable by libsndfile; the\\n",\n        "        # model-specific decoder remains the source of truth.\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def snapshot_job(job: dict) -> dict:\\n",\n        "    response = {\\n",\n        "        \\"job_id\\": job[\\"job_id\\"],\\n",\n        "        \\"status\\": job[\\"status\\"],\\n",\n        "        \\"progress\\": job[\\"progress\\"],\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "    }\\n",\n        "    if job.get(\\"detail\\"):\\n",\n        "        response[\\"detail\\"] = job[\\"detail\\"]\\n",\n        "    if job.get(\\"result\\") is not None:\\n",\n        "        response[\\"result\\"] = job[\\"result\\"]\\n",\n        "    return response\\n",\n        "\\n",\n        "\\n",\n        "def prune_finished_jobs() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            job_id for job_id, job in JOBS.items()\\n",\n        "            if job.get(\\"finished_at\\") and now - job[\\"finished_at\\"] > JOB_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for job_id in expired:\\n",\n        "            job = JOBS.pop(job_id)\\n",\n        "            if job.get(\\"path\\"):\\n",\n        "                expired_paths.append(Path(job[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def prune_expired_uploads() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            upload_id for upload_id, upload in UPLOADS.items()\\n",\n        "            if now - upload[\\"created_at\\"] > UPLOAD_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for upload_id in expired:\\n",\n        "            upload = UPLOADS.pop(upload_id)\\n",\n        "            expired_paths.append(Path(upload[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def run_job(job_id: str) -> None:\\n",\n        "    try:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is None:\\n",\n        "                return\\n",\n        "            if job.get(\\"cancel_requested\\"):\\n",\n        "                job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "                return\\n",\n        "            job[\\"status\\"] = \\"running\\"\\n",\n        "            job[\\"progress\\"] = 15\\n",\n        "            path = job[\\"path\\"]\\n",\n        "            language = job[\\"language\\"]\\n",\n        "            response_format = job[\\"response_format\\"]\\n",\n        "        result = await asyncio.to_thread(run_transcription, path, language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise RuntimeError(\\"The loaded model returned an empty transcript\\")\\n",\n        "        payload = {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"succeeded\\"\\n",\n        "                    job[\\"result\\"] = payload\\n",\n        "                    job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    except Exception as error:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"failed\\"\\n",\n        "                    job[\\"detail\\"] = f\\"{MODEL_NAME} transcription failed: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    finally:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            source_path = Path(job[\\"path\\"]) if job and job.get(\\"path\\") else None\\n",\n        "            if job is not None:\\n",\n        "                job[\\"path\\"] = None\\n",\n        "        if source_path is not None:\\n",\n        "            source_path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt\\", status_code=201)\\n",\n        "async def begin_chunked_stt_upload(payload: dict,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    model = validate_model(str(payload.get(\\"model\\", \\"\\")))\\n",\n        "    try:\\n",\n        "        size_bytes = int(payload.get(\\"size_bytes\\", 0))\\n",\n        "    except (TypeError, ValueError):\\n",\n        "        raise HTTPException(status_code=422, detail=\\"Audio upload size must be an integer\\")\\n",\n        "    if size_bytes <= 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "    if size_bytes > MAX_UPLOAD_BYTES:\\n",\n        "        raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=\\".wav\\", delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    handle.close()\\n",\n        "    upload_id = secrets.token_urlsafe(18)\\n",\n        "    with JOB_LOCK:\\n",\n        "        UPLOADS[upload_id] = {\\n",\n        "            \\"path\\": str(path),\\n",\n        "            \\"size_bytes\\": size_bytes,\\n",\n        "            \\"received_bytes\\": 0,\\n",\n        "            \\"next_chunk\\": 0,\\n",\n        "            \\"model\\": model,\\n",\n        "            \\"language\\": str(payload.get(\\"language\\", \\"auto\\")),\\n",\n        "            \\"response_format\\": str(payload.get(\\"response_format\\", \\"verbose_json\\")),\\n",\n        "            \\"created_at\\": asyncio.get_running_loop().time(),\\n",\n        "        }\\n",\n        "    return {\\"upload_id\\": upload_id, \\"chunk_bytes\\": CHUNK_UPLOAD_BYTES}\\n",\n        "\\n",\n        "\\n",\n        "@app.put(\\"/v2/uploads/stt/{upload_id}/chunks/{chunk_index}\\")\\n",\n        "async def upload_chunked_stt_audio(upload_id: str, chunk_index: int, request: Request,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        remaining = upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]\\n",\n        "    chunks = []\\n",\n        "    total = 0\\n",\n        "    async for piece in request.stream():\\n",\n        "        total += len(piece)\\n",\n        "        if total > CHUNK_UPLOAD_BYTES or total > remaining:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload chunk is too large\\")\\n",\n        "        chunks.append(piece)\\n",\n        "    if total == 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"STT upload chunk is empty\\")\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        if total > upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload exceeds its declared size\\")\\n",\n        "        try:\\n",\n        "            with Path(upload[\\"path\\"]).open(\\"ab\\") as handle:\\n",\n        "                for piece in chunks:\\n",\n        "                    handle.write(piece)\\n",\n        "        except OSError as error:\\n",\n        "            raise HTTPException(status_code=500, detail=f\\"Unable to store STT upload chunk: {error}\\")\\n",\n        "        upload[\\"received_bytes\\"] += total\\n",\n        "        upload[\\"next_chunk\\"] += 1\\n",\n        "        return {\\n",\n        "            \\"received_bytes\\": upload[\\"received_bytes\\"],\\n",\n        "            \\"next_chunk\\": upload[\\"next_chunk\\"],\\n",\n        "        }\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/uploads/stt/{upload_id}\\")\\n",\n        "async def cancel_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.pop(upload_id, None)\\n",\n        "    if upload is not None:\\n",\n        "        Path(upload[\\"path\\"]).unlink(missing_ok=True)\\n",\n        "    return {\\"status\\": \\"cancelled\\"}\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt/{upload_id}/commit\\", status_code=202)\\n",\n        "async def commit_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if upload[\\"received_bytes\\"] != upload[\\"size_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload is incomplete\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = Path(upload[\\"path\\"])\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        validate_audio_duration(path)\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            upload = UPLOADS.pop(upload_id, None)\\n",\n        "            if upload is None:\\n",\n        "                raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": upload[\\"language\\"],\\n",\n        "                \\"response_format\\": upload[\\"response_format\\"],\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        if not queued:\\n",\n        "            with JOB_LOCK:\\n",\n        "                UPLOADS.pop(upload_id, None)\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/jobs/transcriptions\\", status_code=202)\\n",\n        "async def create_transcription_job(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": language,\\n",\n        "                \\"response_format\\": response_format,\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        await file.close()\\n",\n        "        if not queued:\\n",\n        "            if path is not None:\\n",\n        "                path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def transcription_job_status(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    prune_finished_jobs()\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def cancel_transcription_job(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        if job[\\"status\\"] in {\\"queued\\", \\"running\\"}:\\n",\n        "            job[\\"cancel_requested\\"] = True\\n",\n        "            job[\\"status\\"] = \\"cancelled\\"\\n",\n        "            job[\\"detail\\"] = \\"Transcription cancellation requested\\"\\n",\n        "            job[\\"progress\\"] = 100\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v1/audio/transcriptions\\")\\n",\n        "async def transcriptions(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "\\n",\n        "        # Compatibility endpoint for older desktop builds. New LA Studio builds\\n",\n        "        # use /v2/jobs/transcriptions so a long GPU run cannot hit the\\n",\n        "        # Cloudflare 120-second proxy response limit.\\n",\n        "        result = await asyncio.to_thread(run_transcription, str(path), language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise HTTPException(status_code=502, detail=\\"The loaded model returned an empty transcript\\")\\n",\n        "        return {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "    finally:\\n",\n        "        if path is not None:\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import subprocess\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.request\\n",\n        "\\n",\n        "\\n",\n        "def read_health(url: str, timeout: float = 2.0):\\n",\n        "    try:\\n",\n        "        with urllib.request.urlopen(url.rstrip(\\"/\\") + \\"/health\\", timeout=timeout) as response:\\n",\n        "            if response.status != 200:\\n",\n        "                return None\\n",\n        "            payload = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            return payload if isinstance(payload, dict) else None\\n",\n        "    except Exception:\\n",\n        "        return None\\n",\n        "\\n",\n        "\\n",\n        "def is_exact_worker(payload) -> bool:\\n",\n        "    return bool(\\n",\n        "        isinstance(payload, dict)\\n",\n        "        and payload.get(\\"ready\\") is True\\n",\n        "        and str(payload.get(\\"device\\", \\"\\")).strip().lower() == \\"cuda\\"\\n",\n        "        and str(payload.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "        and str(payload.get(\\"worker_revision\\", \\"\\")) == WORKER_REVISION\\n",\n        "    )\\n",\n        "\\n",\n        "\\n",\n        "def wait_for_local_health():\\n",\n        "    deadline = time.time() + 60\\n",\n        "    while time.time() < deadline:\\n",\n        "        health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "        if health is not None:\\n",\n        "            return health\\n",\n        "        time.sleep(0.5)\\n",\n        "    raise RuntimeError(\\"The local STT worker did not become healthy\\")\\n",\n        "\\n",\n        "\\n",\n        "def run_server():\\n",\n        "    import uvicorn\\n",\n        "    uvicorn.run(app, host=\\"127.0.0.1\\", port=8000, log_level=\\"warning\\")\\n",\n        "\\n",\n        "\\n",\n        "# Re-running this cell in the same Colab runtime must not create a second\\n",\n        "# Uvicorn server.  The existing app functions use the refreshed notebook\\n",\n        "# globals, so the same exact model can safely be reused.  A different model\\n",\n        "# must use a fresh Colab runtime to avoid silently serving the wrong worker.\\n",\n        "local_health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "if local_health is None:\\n",\n        "    threading.Thread(target=run_server, daemon=True).start()\\n",\n        "    local_health = wait_for_local_health()\\n",\n        "    print(\\"Started local LA Studio STT worker on port 8000\\")\\n",\n        "elif is_exact_worker(local_health):\\n",\n        "    print(\\"Reusing the existing local LA Studio STT worker on port 8000\\")\\n",\n        "else:\\n",\n        "    current_model = str(local_health.get(\\"model\\", \\"unknown\\"))\\n",\n        "    current_revision = str(local_health.get(\\"worker_revision\\", \\"unknown\\"))\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"Port 8000 serves LA Studio model \'{current_model}\' (worker revision \'{current_revision}\'), \\"\\n",\n        "        f\\"not the required \'{MODEL_ID}\' revision \'{WORKER_REVISION}\'. \\"\\n",\n        "        \\"Use Runtime > Disconnect and delete runtime, then Run all for this exact model.\\"\\n",\n        "    )\\n",\n        "\\n",\n        "if not is_exact_worker(local_health):\\n",\n        "    raise RuntimeError(\\"The local worker did not confirm the selected exact CUDA model\\")\\n",\n        "\\n",\n        "\\n",\n        "def valid_cloudflared(path: str) -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [path, \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> str:\\n",\n        "    path = \\"/content/cloudflared\\"\\n",\n        "    if valid_cloudflared(path):\\n",\n        "        return path\\n",\n        "    result = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if result.returncode == 0:\\n",\n        "        subprocess.run([\\"chmod\\", \\"+x\\", path], check=True)\\n",\n        "    if result.returncode != 0 or not valid_cloudflared(path):\\n",\n        "        detail = result.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not obtain a working cloudflared binary: \\" + detail)\\n",\n        "    return path\\n",\n        "\\n",\n        "\\n",\n        "existing_url = os.environ.get(\\"LA_STUDIO_COLAB_STT_URL\\", \\"\\").strip()\\n",\n        "existing_health = read_health(existing_url, timeout=4.0) if existing_url else None\\n",\n        "if is_exact_worker(existing_health):\\n",\n        "    worker_url = existing_url\\n",\n        "    print(\\"Reusing the existing public Cloudflare tunnel\\")\\n",\n        "else:\\n",\n        "    cloudflared_path = ensure_cloudflared()\\n",\n        "    process = subprocess.Popen(\\n",\n        "        [cloudflared_path, \\"tunnel\\", \\"--url\\", \\"http://127.0.0.1:8000\\", \\"--no-autoupdate\\"],\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        text=True,\\n",\n        "        bufsize=1,\\n",\n        "    )\\n",\n        "    lines = queue.Queue()\\n",\n        "\\n",\n        "    def collect_output():\\n",\n        "        assert process.stdout is not None\\n",\n        "        for line in process.stdout:\\n",\n        "            lines.put(line)\\n",\n        "\\n",\n        "    threading.Thread(target=collect_output, daemon=True).start()\\n",\n        "    worker_url = \\"\\"\\n",\n        "    deadline = time.time() + 90\\n",\n        "    while time.time() < deadline and not worker_url:\\n",\n        "        if process.poll() is not None:\\n",\n        "            raise RuntimeError(\\"cloudflared exited before creating a public tunnel\\")\\n",\n        "        try:\\n",\n        "            line = lines.get(timeout=1)\\n",\n        "        except queue.Empty:\\n",\n        "            continue\\n",\n        "        match = re.search(r\\"https://[a-z0-9-]+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "        if match:\\n",\n        "            # Colab\'s self-request to its own public quick tunnel is not a\\n",\n        "            # reliable readiness test. The desktop Check Colab action is the\\n",\n        "            # authoritative public endpoint + token + exact-model verification.\\n",\n        "            worker_url = match.group(0)\\n",\n        "            print(\\"Cloudflare tunnel URL created. Verify it with Check Colab in LA Studio.\\")\\n",\n        "\\n",\n        "    if not worker_url:\\n",\n        "        process.terminate()\\n",\n        "        raise RuntimeError(\\n",\n        "            \\"cloudflared did not publish a trycloudflare URL within 90 seconds. \\"\\n",\n        "            \\"Run the launch cell once more; if it repeats, reset the Colab runtime and check that internet access is available.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_URL\\"] = worker_url\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_TOKEN\\"] = TOKEN\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_MODEL\\"] = MODEL_ID\\n",\n        "\\n",\n        "print(\\"\\\\nLA Studio Colab STT worker is ready\\")\\n",\n        "print(\\"MODEL:\\", MODEL_ID)\\n",\n        "print(\\"URL:\\", worker_url)\\n",\n        "print(\\"TOKEN:\\", TOKEN)\\n",\n        "print(\\"\\\\nPaste the URL and TOKEN into LA Studio. Keep this cell running.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "language": "python",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "stt",\n      "family_id": "nemotron-3.5-asr-streaming-0.6b",\n      "upstream_model": "nvidia/nemotron-3.5-asr-streaming-0.6b",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/stt/LA_STUDIO_STT_QWEN3_ASR_0_6B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio \\u2014 Qwen3-ASR 0.6B Colab GPU worker\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3-asr-0.6b` on the Colab GPU.\\n",\n        "It rejects transcription requests for every other model ID.\\n",\n        "\\n",\n        "Long recordings use an asynchronous GPU job: the app uploads the\\n",\n        "audio once, then polls short status requests until this exact model\\n",\n        "completes. This avoids Cloudflare\'s 120-second proxy response limit.\\n",\n        "\\n",\n        "Run every cell in order, then paste the printed URL and TOKEN into\\n",\n        "LA Studio. The tunnel is public, but every worker endpoint requires\\n",\n        "the random session token printed by the last cell.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "%pip install -q qwen-asr==0.0.6 fastapi==0.115.12 uvicorn==0.34.3 python-multipart==0.0.20 soundfile==0.13.1\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import asyncio\\n",\n        "import os\\n",\n        "import secrets\\n",\n        "import tempfile\\n",\n        "import threading\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "import soundfile as sf\\n",\n        "import torch\\n",\n        "from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile\\n",\n        "from fastapi.responses import JSONResponse\\n",\n        "\\n",\n        "if not torch.cuda.is_available():\\n",\n        "    raise RuntimeError(\\"A Colab GPU runtime is required. Choose Runtime > Change runtime type > GPU.\\")\\n",\n        "\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "WORKER_REVISION = \\"stt-2026-07-30.2\\"\\n",\n        "MAX_UPLOAD_BYTES = 512 * 1024 * 1024\\n",\n        "MAX_AUDIO_SECONDS = 30 * 60\\n",\n        "ALLOWED_CONTENT_TYPES = {\\n",\n        "    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp3\\", \\"audio/mp4\\",\\n",\n        "    \\"audio/flac\\", \\"audio/ogg\\", \\"audio/webm\\", \\"application/octet-stream\\",\\n",\n        "}\\n",\n        "REQUEST_SLOTS = threading.BoundedSemaphore(1)\\n",\n        "JOB_TTL_SECONDS = 30 * 60\\n",\n        "UPLOAD_TTL_SECONDS = 10 * 60\\n",\n        "CHUNK_UPLOAD_BYTES = 2 * 1024 * 1024\\n",\n        "JOB_LOCK = threading.Lock()\\n",\n        "JOBS: dict[str, dict] = {}\\n",\n        "UPLOADS: dict[str, dict] = {}\\n",\n        "\\n",\n        "from qwen_asr import Qwen3ASRModel\\n",\n        "\\n",\n        "MODEL_ID = \\"qwen3-asr-0.6b\\"\\n",\n        "MODEL_NAME = \\"Qwen3-ASR 0.6B\\"\\n",\n        "UPSTREAM_MODEL = \\"Qwen/Qwen3-ASR-0.6B\\"\\n",\n        "stt_model = Qwen3ASRModel.from_pretrained(\\n",\n        "    UPSTREAM_MODEL,\\n",\n        "    dtype=torch.bfloat16,\\n",\n        "    device_map=\\"cuda:0\\",\\n",\n        "    max_inference_batch_size=1,\\n",\n        "    max_new_tokens=2048,\\n",\n        ")\\n",\n        "\\n",\n        "QWEN_LANGUAGE_NAMES = {\\n",\n        "    \\"ar\\": \\"Arabic\\", \\"cs\\": \\"Czech\\", \\"da\\": \\"Danish\\", \\"de\\": \\"German\\",\\n",\n        "    \\"en\\": \\"English\\", \\"es\\": \\"Spanish\\", \\"fa\\": \\"Persian\\", \\"fi\\": \\"Finnish\\",\\n",\n        "    \\"fil\\": \\"Filipino\\", \\"fr\\": \\"French\\", \\"el\\": \\"Greek\\", \\"hi\\": \\"Hindi\\",\\n",\n        "    \\"hu\\": \\"Hungarian\\", \\"id\\": \\"Indonesian\\", \\"it\\": \\"Italian\\", \\"ja\\": \\"Japanese\\",\\n",\n        "    \\"ko\\": \\"Korean\\", \\"ms\\": \\"Malay\\", \\"nl\\": \\"Dutch\\", \\"pl\\": \\"Polish\\",\\n",\n        "    \\"pt\\": \\"Portuguese\\", \\"ro\\": \\"Romanian\\", \\"ru\\": \\"Russian\\", \\"sv\\": \\"Swedish\\",\\n",\n        "    \\"th\\": \\"Thai\\", \\"tr\\": \\"Turkish\\", \\"vi\\": \\"Vietnamese\\", \\"yue\\": \\"Cantonese\\",\\n",\n        "    \\"zh\\": \\"Chinese\\",\\n",\n        "}\\n",\n        "\\n",\n        "def run_transcription(path: str, language: str):\\n",\n        "    requested = language.strip().lower()\\n",\n        "    language_name = None if not requested or requested == \\"auto\\" else QWEN_LANGUAGE_NAMES.get(requested)\\n",\n        "    results = stt_model.transcribe(audio=path, language=language_name)\\n",\n        "    result = results[0]\\n",\n        "    return {\\"text\\": result.text, \\"segments\\": [], \\"language\\": result.language}\\n",\n        "\\n",\n        "app = FastAPI(title=f\\"LA Studio STT \\u2014 {MODEL_NAME}\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.exception_handler(Exception)\\n",\n        "async def unhandled_exception(request: Request, error: Exception):\\n",\n        "    # A tunnel 500 without a response body is impossible to act on from the\\n",\n        "    # desktop app. Keep the detail bounded, and also print it in the Colab\\n",\n        "    # cell so the notebook owns the operational diagnosis.\\n",\n        "    detail = f\\"STT worker internal error: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "    print(detail, flush=True)\\n",\n        "    return JSONResponse(status_code=500, content={\\"detail\\": detail})\\n",\n        "\\n",\n        "\\n",\n        "def require_token(authorization: str | None) -> None:\\n",\n        "    if authorization != f\\"Bearer {TOKEN}\\":\\n",\n        "        raise HTTPException(status_code=401, detail=\\"Invalid or missing Colab session token\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/health\\")\\n",\n        "def health():\\n",\n        "    return {\\n",\n        "        \\"ok\\": True,\\n",\n        "        \\"ready\\": True,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"gpu\\": torch.cuda.get_device_name(0),\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "        \\"variant\\": \\"fixed\\",\\n",\n        "        \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v1/capabilities\\")\\n",\n        "def capabilities(authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    return {\\n",\n        "        \\"contract_version\\": 1,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"cuda\\": True,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "        \\"endpoints\\": {\\n",\n        "            \\"transcription_jobs\\": \\"/v2/jobs/transcriptions\\",\\n",\n        "            \\"chunked_transcription_uploads\\": \\"/v2/uploads/stt\\",\\n",\n        "        },\\n",\n        "        \\"chunked_uploads\\": True,\\n",\n        "        \\"capabilities\\": [{\\n",\n        "            \\"id\\": \\"stt\\",\\n",\n        "            \\"models\\": [{\\n",\n        "                \\"id\\": MODEL_ID,\\n",\n        "                \\"name\\": MODEL_NAME,\\n",\n        "                \\"variant\\": \\"fixed\\",\\n",\n        "                \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "                \\"loaded\\": True,\\n",\n        "                \\"device\\": \\"cuda\\",\\n",\n        "            }],\\n",\n        "        }],\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "async def save_upload(file: UploadFile) -> tuple[Path, int]:\\n",\n        "    suffix = Path(file.filename or \\"audio.wav\\").suffix or \\".wav\\"\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=suffix, delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    total = 0\\n",\n        "    try:\\n",\n        "        with handle:\\n",\n        "            while True:\\n",\n        "                chunk = await file.read(1024 * 1024)\\n",\n        "                if not chunk:\\n",\n        "                    break\\n",\n        "                total += len(chunk)\\n",\n        "                if total > MAX_UPLOAD_BYTES:\\n",\n        "                    raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "                handle.write(chunk)\\n",\n        "        return path, total\\n",\n        "    except Exception:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "        raise\\n",\n        "\\n",\n        "\\n",\n        "def validate_model(model: str) -> str:\\n",\n        "    requested = model.strip().lower()\\n",\n        "    if requested != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    return requested\\n",\n        "\\n",\n        "\\n",\n        "def validate_audio_duration(path: Path) -> None:\\n",\n        "    try:\\n",\n        "        info = sf.info(str(path))\\n",\n        "        if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "    except HTTPException:\\n",\n        "        raise\\n",\n        "    except Exception:\\n",\n        "        # Compressed formats may not be readable by libsndfile; the\\n",\n        "        # model-specific decoder remains the source of truth.\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def snapshot_job(job: dict) -> dict:\\n",\n        "    response = {\\n",\n        "        \\"job_id\\": job[\\"job_id\\"],\\n",\n        "        \\"status\\": job[\\"status\\"],\\n",\n        "        \\"progress\\": job[\\"progress\\"],\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "    }\\n",\n        "    if job.get(\\"detail\\"):\\n",\n        "        response[\\"detail\\"] = job[\\"detail\\"]\\n",\n        "    if job.get(\\"result\\") is not None:\\n",\n        "        response[\\"result\\"] = job[\\"result\\"]\\n",\n        "    return response\\n",\n        "\\n",\n        "\\n",\n        "def prune_finished_jobs() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            job_id for job_id, job in JOBS.items()\\n",\n        "            if job.get(\\"finished_at\\") and now - job[\\"finished_at\\"] > JOB_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for job_id in expired:\\n",\n        "            job = JOBS.pop(job_id)\\n",\n        "            if job.get(\\"path\\"):\\n",\n        "                expired_paths.append(Path(job[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def prune_expired_uploads() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            upload_id for upload_id, upload in UPLOADS.items()\\n",\n        "            if now - upload[\\"created_at\\"] > UPLOAD_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for upload_id in expired:\\n",\n        "            upload = UPLOADS.pop(upload_id)\\n",\n        "            expired_paths.append(Path(upload[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def run_job(job_id: str) -> None:\\n",\n        "    try:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is None:\\n",\n        "                return\\n",\n        "            if job.get(\\"cancel_requested\\"):\\n",\n        "                job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "                return\\n",\n        "            job[\\"status\\"] = \\"running\\"\\n",\n        "            job[\\"progress\\"] = 15\\n",\n        "            path = job[\\"path\\"]\\n",\n        "            language = job[\\"language\\"]\\n",\n        "            response_format = job[\\"response_format\\"]\\n",\n        "        result = await asyncio.to_thread(run_transcription, path, language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise RuntimeError(\\"The loaded model returned an empty transcript\\")\\n",\n        "        payload = {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"succeeded\\"\\n",\n        "                    job[\\"result\\"] = payload\\n",\n        "                    job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    except Exception as error:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"failed\\"\\n",\n        "                    job[\\"detail\\"] = f\\"{MODEL_NAME} transcription failed: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    finally:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            source_path = Path(job[\\"path\\"]) if job and job.get(\\"path\\") else None\\n",\n        "            if job is not None:\\n",\n        "                job[\\"path\\"] = None\\n",\n        "        if source_path is not None:\\n",\n        "            source_path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt\\", status_code=201)\\n",\n        "async def begin_chunked_stt_upload(payload: dict,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    model = validate_model(str(payload.get(\\"model\\", \\"\\")))\\n",\n        "    try:\\n",\n        "        size_bytes = int(payload.get(\\"size_bytes\\", 0))\\n",\n        "    except (TypeError, ValueError):\\n",\n        "        raise HTTPException(status_code=422, detail=\\"Audio upload size must be an integer\\")\\n",\n        "    if size_bytes <= 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "    if size_bytes > MAX_UPLOAD_BYTES:\\n",\n        "        raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=\\".wav\\", delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    handle.close()\\n",\n        "    upload_id = secrets.token_urlsafe(18)\\n",\n        "    with JOB_LOCK:\\n",\n        "        UPLOADS[upload_id] = {\\n",\n        "            \\"path\\": str(path),\\n",\n        "            \\"size_bytes\\": size_bytes,\\n",\n        "            \\"received_bytes\\": 0,\\n",\n        "            \\"next_chunk\\": 0,\\n",\n        "            \\"model\\": model,\\n",\n        "            \\"language\\": str(payload.get(\\"language\\", \\"auto\\")),\\n",\n        "            \\"response_format\\": str(payload.get(\\"response_format\\", \\"verbose_json\\")),\\n",\n        "            \\"created_at\\": asyncio.get_running_loop().time(),\\n",\n        "        }\\n",\n        "    return {\\"upload_id\\": upload_id, \\"chunk_bytes\\": CHUNK_UPLOAD_BYTES}\\n",\n        "\\n",\n        "\\n",\n        "@app.put(\\"/v2/uploads/stt/{upload_id}/chunks/{chunk_index}\\")\\n",\n        "async def upload_chunked_stt_audio(upload_id: str, chunk_index: int, request: Request,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        remaining = upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]\\n",\n        "    chunks = []\\n",\n        "    total = 0\\n",\n        "    async for piece in request.stream():\\n",\n        "        total += len(piece)\\n",\n        "        if total > CHUNK_UPLOAD_BYTES or total > remaining:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload chunk is too large\\")\\n",\n        "        chunks.append(piece)\\n",\n        "    if total == 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"STT upload chunk is empty\\")\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        if total > upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload exceeds its declared size\\")\\n",\n        "        try:\\n",\n        "            with Path(upload[\\"path\\"]).open(\\"ab\\") as handle:\\n",\n        "                for piece in chunks:\\n",\n        "                    handle.write(piece)\\n",\n        "        except OSError as error:\\n",\n        "            raise HTTPException(status_code=500, detail=f\\"Unable to store STT upload chunk: {error}\\")\\n",\n        "        upload[\\"received_bytes\\"] += total\\n",\n        "        upload[\\"next_chunk\\"] += 1\\n",\n        "        return {\\n",\n        "            \\"received_bytes\\": upload[\\"received_bytes\\"],\\n",\n        "            \\"next_chunk\\": upload[\\"next_chunk\\"],\\n",\n        "        }\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/uploads/stt/{upload_id}\\")\\n",\n        "async def cancel_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.pop(upload_id, None)\\n",\n        "    if upload is not None:\\n",\n        "        Path(upload[\\"path\\"]).unlink(missing_ok=True)\\n",\n        "    return {\\"status\\": \\"cancelled\\"}\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt/{upload_id}/commit\\", status_code=202)\\n",\n        "async def commit_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if upload[\\"received_bytes\\"] != upload[\\"size_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload is incomplete\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = Path(upload[\\"path\\"])\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        validate_audio_duration(path)\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            upload = UPLOADS.pop(upload_id, None)\\n",\n        "            if upload is None:\\n",\n        "                raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": upload[\\"language\\"],\\n",\n        "                \\"response_format\\": upload[\\"response_format\\"],\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        if not queued:\\n",\n        "            with JOB_LOCK:\\n",\n        "                UPLOADS.pop(upload_id, None)\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/jobs/transcriptions\\", status_code=202)\\n",\n        "async def create_transcription_job(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": language,\\n",\n        "                \\"response_format\\": response_format,\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        await file.close()\\n",\n        "        if not queued:\\n",\n        "            if path is not None:\\n",\n        "                path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def transcription_job_status(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    prune_finished_jobs()\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def cancel_transcription_job(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        if job[\\"status\\"] in {\\"queued\\", \\"running\\"}:\\n",\n        "            job[\\"cancel_requested\\"] = True\\n",\n        "            job[\\"status\\"] = \\"cancelled\\"\\n",\n        "            job[\\"detail\\"] = \\"Transcription cancellation requested\\"\\n",\n        "            job[\\"progress\\"] = 100\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v1/audio/transcriptions\\")\\n",\n        "async def transcriptions(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "\\n",\n        "        # Compatibility endpoint for older desktop builds. New LA Studio builds\\n",\n        "        # use /v2/jobs/transcriptions so a long GPU run cannot hit the\\n",\n        "        # Cloudflare 120-second proxy response limit.\\n",\n        "        result = await asyncio.to_thread(run_transcription, str(path), language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise HTTPException(status_code=502, detail=\\"The loaded model returned an empty transcript\\")\\n",\n        "        return {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "    finally:\\n",\n        "        if path is not None:\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import subprocess\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.request\\n",\n        "\\n",\n        "\\n",\n        "def read_health(url: str, timeout: float = 2.0):\\n",\n        "    try:\\n",\n        "        with urllib.request.urlopen(url.rstrip(\\"/\\") + \\"/health\\", timeout=timeout) as response:\\n",\n        "            if response.status != 200:\\n",\n        "                return None\\n",\n        "            payload = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            return payload if isinstance(payload, dict) else None\\n",\n        "    except Exception:\\n",\n        "        return None\\n",\n        "\\n",\n        "\\n",\n        "def is_exact_worker(payload) -> bool:\\n",\n        "    return bool(\\n",\n        "        isinstance(payload, dict)\\n",\n        "        and payload.get(\\"ready\\") is True\\n",\n        "        and str(payload.get(\\"device\\", \\"\\")).strip().lower() == \\"cuda\\"\\n",\n        "        and str(payload.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "        and str(payload.get(\\"worker_revision\\", \\"\\")) == WORKER_REVISION\\n",\n        "    )\\n",\n        "\\n",\n        "\\n",\n        "def wait_for_local_health():\\n",\n        "    deadline = time.time() + 60\\n",\n        "    while time.time() < deadline:\\n",\n        "        health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "        if health is not None:\\n",\n        "            return health\\n",\n        "        time.sleep(0.5)\\n",\n        "    raise RuntimeError(\\"The local STT worker did not become healthy\\")\\n",\n        "\\n",\n        "\\n",\n        "def run_server():\\n",\n        "    import uvicorn\\n",\n        "    uvicorn.run(app, host=\\"127.0.0.1\\", port=8000, log_level=\\"warning\\")\\n",\n        "\\n",\n        "\\n",\n        "# Re-running this cell in the same Colab runtime must not create a second\\n",\n        "# Uvicorn server.  The existing app functions use the refreshed notebook\\n",\n        "# globals, so the same exact model can safely be reused.  A different model\\n",\n        "# must use a fresh Colab runtime to avoid silently serving the wrong worker.\\n",\n        "local_health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "if local_health is None:\\n",\n        "    threading.Thread(target=run_server, daemon=True).start()\\n",\n        "    local_health = wait_for_local_health()\\n",\n        "    print(\\"Started local LA Studio STT worker on port 8000\\")\\n",\n        "elif is_exact_worker(local_health):\\n",\n        "    print(\\"Reusing the existing local LA Studio STT worker on port 8000\\")\\n",\n        "else:\\n",\n        "    current_model = str(local_health.get(\\"model\\", \\"unknown\\"))\\n",\n        "    current_revision = str(local_health.get(\\"worker_revision\\", \\"unknown\\"))\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"Port 8000 serves LA Studio model \'{current_model}\' (worker revision \'{current_revision}\'), \\"\\n",\n        "        f\\"not the required \'{MODEL_ID}\' revision \'{WORKER_REVISION}\'. \\"\\n",\n        "        \\"Use Runtime > Disconnect and delete runtime, then Run all for this exact model.\\"\\n",\n        "    )\\n",\n        "\\n",\n        "if not is_exact_worker(local_health):\\n",\n        "    raise RuntimeError(\\"The local worker did not confirm the selected exact CUDA model\\")\\n",\n        "\\n",\n        "\\n",\n        "def valid_cloudflared(path: str) -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [path, \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> str:\\n",\n        "    path = \\"/content/cloudflared\\"\\n",\n        "    if valid_cloudflared(path):\\n",\n        "        return path\\n",\n        "    result = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if result.returncode == 0:\\n",\n        "        subprocess.run([\\"chmod\\", \\"+x\\", path], check=True)\\n",\n        "    if result.returncode != 0 or not valid_cloudflared(path):\\n",\n        "        detail = result.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not obtain a working cloudflared binary: \\" + detail)\\n",\n        "    return path\\n",\n        "\\n",\n        "\\n",\n        "existing_url = os.environ.get(\\"LA_STUDIO_COLAB_STT_URL\\", \\"\\").strip()\\n",\n        "existing_health = read_health(existing_url, timeout=4.0) if existing_url else None\\n",\n        "if is_exact_worker(existing_health):\\n",\n        "    worker_url = existing_url\\n",\n        "    print(\\"Reusing the existing public Cloudflare tunnel\\")\\n",\n        "else:\\n",\n        "    cloudflared_path = ensure_cloudflared()\\n",\n        "    process = subprocess.Popen(\\n",\n        "        [cloudflared_path, \\"tunnel\\", \\"--url\\", \\"http://127.0.0.1:8000\\", \\"--no-autoupdate\\"],\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        text=True,\\n",\n        "        bufsize=1,\\n",\n        "    )\\n",\n        "    lines = queue.Queue()\\n",\n        "\\n",\n        "    def collect_output():\\n",\n        "        assert process.stdout is not None\\n",\n        "        for line in process.stdout:\\n",\n        "            lines.put(line)\\n",\n        "\\n",\n        "    threading.Thread(target=collect_output, daemon=True).start()\\n",\n        "    worker_url = \\"\\"\\n",\n        "    deadline = time.time() + 90\\n",\n        "    while time.time() < deadline and not worker_url:\\n",\n        "        if process.poll() is not None:\\n",\n        "            raise RuntimeError(\\"cloudflared exited before creating a public tunnel\\")\\n",\n        "        try:\\n",\n        "            line = lines.get(timeout=1)\\n",\n        "        except queue.Empty:\\n",\n        "            continue\\n",\n        "        match = re.search(r\\"https://[a-z0-9-]+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "        if match:\\n",\n        "            # Colab\'s self-request to its own public quick tunnel is not a\\n",\n        "            # reliable readiness test. The desktop Check Colab action is the\\n",\n        "            # authoritative public endpoint + token + exact-model verification.\\n",\n        "            worker_url = match.group(0)\\n",\n        "            print(\\"Cloudflare tunnel URL created. Verify it with Check Colab in LA Studio.\\")\\n",\n        "\\n",\n        "    if not worker_url:\\n",\n        "        process.terminate()\\n",\n        "        raise RuntimeError(\\n",\n        "            \\"cloudflared did not publish a trycloudflare URL within 90 seconds. \\"\\n",\n        "            \\"Run the launch cell once more; if it repeats, reset the Colab runtime and check that internet access is available.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_URL\\"] = worker_url\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_TOKEN\\"] = TOKEN\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_MODEL\\"] = MODEL_ID\\n",\n        "\\n",\n        "print(\\"\\\\nLA Studio Colab STT worker is ready\\")\\n",\n        "print(\\"MODEL:\\", MODEL_ID)\\n",\n        "print(\\"URL:\\", worker_url)\\n",\n        "print(\\"TOKEN:\\", TOKEN)\\n",\n        "print(\\"\\\\nPaste the URL and TOKEN into LA Studio. Keep this cell running.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "language": "python",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "stt",\n      "family_id": "qwen3-asr-0.6b",\n      "upstream_model": "Qwen/Qwen3-ASR-0.6B",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/stt/LA_STUDIO_STT_QWEN3_ASR_1_7B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio \\u2014 Qwen3-ASR 1.7B Colab GPU worker\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3-asr-1.7b` on the Colab GPU.\\n",\n        "It rejects transcription requests for every other model ID.\\n",\n        "\\n",\n        "Long recordings use an asynchronous GPU job: the app uploads the\\n",\n        "audio once, then polls short status requests until this exact model\\n",\n        "completes. This avoids Cloudflare\'s 120-second proxy response limit.\\n",\n        "\\n",\n        "Run every cell in order, then paste the printed URL and TOKEN into\\n",\n        "LA Studio. The tunnel is public, but every worker endpoint requires\\n",\n        "the random session token printed by the last cell.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "%pip install -q qwen-asr==0.0.6 fastapi==0.115.12 uvicorn==0.34.3 python-multipart==0.0.20 soundfile==0.13.1\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import asyncio\\n",\n        "import os\\n",\n        "import secrets\\n",\n        "import tempfile\\n",\n        "import threading\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "import soundfile as sf\\n",\n        "import torch\\n",\n        "from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile\\n",\n        "from fastapi.responses import JSONResponse\\n",\n        "\\n",\n        "if not torch.cuda.is_available():\\n",\n        "    raise RuntimeError(\\"A Colab GPU runtime is required. Choose Runtime > Change runtime type > GPU.\\")\\n",\n        "\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "WORKER_REVISION = \\"stt-2026-07-30.2\\"\\n",\n        "MAX_UPLOAD_BYTES = 512 * 1024 * 1024\\n",\n        "MAX_AUDIO_SECONDS = 30 * 60\\n",\n        "ALLOWED_CONTENT_TYPES = {\\n",\n        "    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp3\\", \\"audio/mp4\\",\\n",\n        "    \\"audio/flac\\", \\"audio/ogg\\", \\"audio/webm\\", \\"application/octet-stream\\",\\n",\n        "}\\n",\n        "REQUEST_SLOTS = threading.BoundedSemaphore(1)\\n",\n        "JOB_TTL_SECONDS = 30 * 60\\n",\n        "UPLOAD_TTL_SECONDS = 10 * 60\\n",\n        "CHUNK_UPLOAD_BYTES = 2 * 1024 * 1024\\n",\n        "JOB_LOCK = threading.Lock()\\n",\n        "JOBS: dict[str, dict] = {}\\n",\n        "UPLOADS: dict[str, dict] = {}\\n",\n        "\\n",\n        "from qwen_asr import Qwen3ASRModel\\n",\n        "\\n",\n        "MODEL_ID = \\"qwen3-asr-1.7b\\"\\n",\n        "MODEL_NAME = \\"Qwen3-ASR 1.7B\\"\\n",\n        "UPSTREAM_MODEL = \\"Qwen/Qwen3-ASR-1.7B\\"\\n",\n        "stt_model = Qwen3ASRModel.from_pretrained(\\n",\n        "    UPSTREAM_MODEL,\\n",\n        "    dtype=torch.bfloat16,\\n",\n        "    device_map=\\"cuda:0\\",\\n",\n        "    max_inference_batch_size=1,\\n",\n        "    max_new_tokens=2048,\\n",\n        ")\\n",\n        "\\n",\n        "QWEN_LANGUAGE_NAMES = {\\n",\n        "    \\"ar\\": \\"Arabic\\", \\"cs\\": \\"Czech\\", \\"da\\": \\"Danish\\", \\"de\\": \\"German\\",\\n",\n        "    \\"en\\": \\"English\\", \\"es\\": \\"Spanish\\", \\"fa\\": \\"Persian\\", \\"fi\\": \\"Finnish\\",\\n",\n        "    \\"fil\\": \\"Filipino\\", \\"fr\\": \\"French\\", \\"el\\": \\"Greek\\", \\"hi\\": \\"Hindi\\",\\n",\n        "    \\"hu\\": \\"Hungarian\\", \\"id\\": \\"Indonesian\\", \\"it\\": \\"Italian\\", \\"ja\\": \\"Japanese\\",\\n",\n        "    \\"ko\\": \\"Korean\\", \\"ms\\": \\"Malay\\", \\"nl\\": \\"Dutch\\", \\"pl\\": \\"Polish\\",\\n",\n        "    \\"pt\\": \\"Portuguese\\", \\"ro\\": \\"Romanian\\", \\"ru\\": \\"Russian\\", \\"sv\\": \\"Swedish\\",\\n",\n        "    \\"th\\": \\"Thai\\", \\"tr\\": \\"Turkish\\", \\"vi\\": \\"Vietnamese\\", \\"yue\\": \\"Cantonese\\",\\n",\n        "    \\"zh\\": \\"Chinese\\",\\n",\n        "}\\n",\n        "\\n",\n        "def run_transcription(path: str, language: str):\\n",\n        "    requested = language.strip().lower()\\n",\n        "    language_name = None if not requested or requested == \\"auto\\" else QWEN_LANGUAGE_NAMES.get(requested)\\n",\n        "    results = stt_model.transcribe(audio=path, language=language_name)\\n",\n        "    result = results[0]\\n",\n        "    return {\\"text\\": result.text, \\"segments\\": [], \\"language\\": result.language}\\n",\n        "\\n",\n        "app = FastAPI(title=f\\"LA Studio STT \\u2014 {MODEL_NAME}\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.exception_handler(Exception)\\n",\n        "async def unhandled_exception(request: Request, error: Exception):\\n",\n        "    # A tunnel 500 without a response body is impossible to act on from the\\n",\n        "    # desktop app. Keep the detail bounded, and also print it in the Colab\\n",\n        "    # cell so the notebook owns the operational diagnosis.\\n",\n        "    detail = f\\"STT worker internal error: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "    print(detail, flush=True)\\n",\n        "    return JSONResponse(status_code=500, content={\\"detail\\": detail})\\n",\n        "\\n",\n        "\\n",\n        "def require_token(authorization: str | None) -> None:\\n",\n        "    if authorization != f\\"Bearer {TOKEN}\\":\\n",\n        "        raise HTTPException(status_code=401, detail=\\"Invalid or missing Colab session token\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/health\\")\\n",\n        "def health():\\n",\n        "    return {\\n",\n        "        \\"ok\\": True,\\n",\n        "        \\"ready\\": True,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"gpu\\": torch.cuda.get_device_name(0),\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "        \\"variant\\": \\"fixed\\",\\n",\n        "        \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v1/capabilities\\")\\n",\n        "def capabilities(authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    return {\\n",\n        "        \\"contract_version\\": 1,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"cuda\\": True,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "        \\"endpoints\\": {\\n",\n        "            \\"transcription_jobs\\": \\"/v2/jobs/transcriptions\\",\\n",\n        "            \\"chunked_transcription_uploads\\": \\"/v2/uploads/stt\\",\\n",\n        "        },\\n",\n        "        \\"chunked_uploads\\": True,\\n",\n        "        \\"capabilities\\": [{\\n",\n        "            \\"id\\": \\"stt\\",\\n",\n        "            \\"models\\": [{\\n",\n        "                \\"id\\": MODEL_ID,\\n",\n        "                \\"name\\": MODEL_NAME,\\n",\n        "                \\"variant\\": \\"fixed\\",\\n",\n        "                \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "                \\"loaded\\": True,\\n",\n        "                \\"device\\": \\"cuda\\",\\n",\n        "            }],\\n",\n        "        }],\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "async def save_upload(file: UploadFile) -> tuple[Path, int]:\\n",\n        "    suffix = Path(file.filename or \\"audio.wav\\").suffix or \\".wav\\"\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=suffix, delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    total = 0\\n",\n        "    try:\\n",\n        "        with handle:\\n",\n        "            while True:\\n",\n        "                chunk = await file.read(1024 * 1024)\\n",\n        "                if not chunk:\\n",\n        "                    break\\n",\n        "                total += len(chunk)\\n",\n        "                if total > MAX_UPLOAD_BYTES:\\n",\n        "                    raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "                handle.write(chunk)\\n",\n        "        return path, total\\n",\n        "    except Exception:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "        raise\\n",\n        "\\n",\n        "\\n",\n        "def validate_model(model: str) -> str:\\n",\n        "    requested = model.strip().lower()\\n",\n        "    if requested != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    return requested\\n",\n        "\\n",\n        "\\n",\n        "def validate_audio_duration(path: Path) -> None:\\n",\n        "    try:\\n",\n        "        info = sf.info(str(path))\\n",\n        "        if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "    except HTTPException:\\n",\n        "        raise\\n",\n        "    except Exception:\\n",\n        "        # Compressed formats may not be readable by libsndfile; the\\n",\n        "        # model-specific decoder remains the source of truth.\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def snapshot_job(job: dict) -> dict:\\n",\n        "    response = {\\n",\n        "        \\"job_id\\": job[\\"job_id\\"],\\n",\n        "        \\"status\\": job[\\"status\\"],\\n",\n        "        \\"progress\\": job[\\"progress\\"],\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "    }\\n",\n        "    if job.get(\\"detail\\"):\\n",\n        "        response[\\"detail\\"] = job[\\"detail\\"]\\n",\n        "    if job.get(\\"result\\") is not None:\\n",\n        "        response[\\"result\\"] = job[\\"result\\"]\\n",\n        "    return response\\n",\n        "\\n",\n        "\\n",\n        "def prune_finished_jobs() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            job_id for job_id, job in JOBS.items()\\n",\n        "            if job.get(\\"finished_at\\") and now - job[\\"finished_at\\"] > JOB_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for job_id in expired:\\n",\n        "            job = JOBS.pop(job_id)\\n",\n        "            if job.get(\\"path\\"):\\n",\n        "                expired_paths.append(Path(job[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def prune_expired_uploads() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            upload_id for upload_id, upload in UPLOADS.items()\\n",\n        "            if now - upload[\\"created_at\\"] > UPLOAD_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for upload_id in expired:\\n",\n        "            upload = UPLOADS.pop(upload_id)\\n",\n        "            expired_paths.append(Path(upload[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def run_job(job_id: str) -> None:\\n",\n        "    try:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is None:\\n",\n        "                return\\n",\n        "            if job.get(\\"cancel_requested\\"):\\n",\n        "                job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "                return\\n",\n        "            job[\\"status\\"] = \\"running\\"\\n",\n        "            job[\\"progress\\"] = 15\\n",\n        "            path = job[\\"path\\"]\\n",\n        "            language = job[\\"language\\"]\\n",\n        "            response_format = job[\\"response_format\\"]\\n",\n        "        result = await asyncio.to_thread(run_transcription, path, language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise RuntimeError(\\"The loaded model returned an empty transcript\\")\\n",\n        "        payload = {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"succeeded\\"\\n",\n        "                    job[\\"result\\"] = payload\\n",\n        "                    job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    except Exception as error:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"failed\\"\\n",\n        "                    job[\\"detail\\"] = f\\"{MODEL_NAME} transcription failed: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    finally:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            source_path = Path(job[\\"path\\"]) if job and job.get(\\"path\\") else None\\n",\n        "            if job is not None:\\n",\n        "                job[\\"path\\"] = None\\n",\n        "        if source_path is not None:\\n",\n        "            source_path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt\\", status_code=201)\\n",\n        "async def begin_chunked_stt_upload(payload: dict,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    model = validate_model(str(payload.get(\\"model\\", \\"\\")))\\n",\n        "    try:\\n",\n        "        size_bytes = int(payload.get(\\"size_bytes\\", 0))\\n",\n        "    except (TypeError, ValueError):\\n",\n        "        raise HTTPException(status_code=422, detail=\\"Audio upload size must be an integer\\")\\n",\n        "    if size_bytes <= 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "    if size_bytes > MAX_UPLOAD_BYTES:\\n",\n        "        raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=\\".wav\\", delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    handle.close()\\n",\n        "    upload_id = secrets.token_urlsafe(18)\\n",\n        "    with JOB_LOCK:\\n",\n        "        UPLOADS[upload_id] = {\\n",\n        "            \\"path\\": str(path),\\n",\n        "            \\"size_bytes\\": size_bytes,\\n",\n        "            \\"received_bytes\\": 0,\\n",\n        "            \\"next_chunk\\": 0,\\n",\n        "            \\"model\\": model,\\n",\n        "            \\"language\\": str(payload.get(\\"language\\", \\"auto\\")),\\n",\n        "            \\"response_format\\": str(payload.get(\\"response_format\\", \\"verbose_json\\")),\\n",\n        "            \\"created_at\\": asyncio.get_running_loop().time(),\\n",\n        "        }\\n",\n        "    return {\\"upload_id\\": upload_id, \\"chunk_bytes\\": CHUNK_UPLOAD_BYTES}\\n",\n        "\\n",\n        "\\n",\n        "@app.put(\\"/v2/uploads/stt/{upload_id}/chunks/{chunk_index}\\")\\n",\n        "async def upload_chunked_stt_audio(upload_id: str, chunk_index: int, request: Request,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        remaining = upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]\\n",\n        "    chunks = []\\n",\n        "    total = 0\\n",\n        "    async for piece in request.stream():\\n",\n        "        total += len(piece)\\n",\n        "        if total > CHUNK_UPLOAD_BYTES or total > remaining:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload chunk is too large\\")\\n",\n        "        chunks.append(piece)\\n",\n        "    if total == 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"STT upload chunk is empty\\")\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        if total > upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload exceeds its declared size\\")\\n",\n        "        try:\\n",\n        "            with Path(upload[\\"path\\"]).open(\\"ab\\") as handle:\\n",\n        "                for piece in chunks:\\n",\n        "                    handle.write(piece)\\n",\n        "        except OSError as error:\\n",\n        "            raise HTTPException(status_code=500, detail=f\\"Unable to store STT upload chunk: {error}\\")\\n",\n        "        upload[\\"received_bytes\\"] += total\\n",\n        "        upload[\\"next_chunk\\"] += 1\\n",\n        "        return {\\n",\n        "            \\"received_bytes\\": upload[\\"received_bytes\\"],\\n",\n        "            \\"next_chunk\\": upload[\\"next_chunk\\"],\\n",\n        "        }\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/uploads/stt/{upload_id}\\")\\n",\n        "async def cancel_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.pop(upload_id, None)\\n",\n        "    if upload is not None:\\n",\n        "        Path(upload[\\"path\\"]).unlink(missing_ok=True)\\n",\n        "    return {\\"status\\": \\"cancelled\\"}\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt/{upload_id}/commit\\", status_code=202)\\n",\n        "async def commit_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if upload[\\"received_bytes\\"] != upload[\\"size_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload is incomplete\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = Path(upload[\\"path\\"])\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        validate_audio_duration(path)\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            upload = UPLOADS.pop(upload_id, None)\\n",\n        "            if upload is None:\\n",\n        "                raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": upload[\\"language\\"],\\n",\n        "                \\"response_format\\": upload[\\"response_format\\"],\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        if not queued:\\n",\n        "            with JOB_LOCK:\\n",\n        "                UPLOADS.pop(upload_id, None)\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/jobs/transcriptions\\", status_code=202)\\n",\n        "async def create_transcription_job(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": language,\\n",\n        "                \\"response_format\\": response_format,\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        await file.close()\\n",\n        "        if not queued:\\n",\n        "            if path is not None:\\n",\n        "                path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def transcription_job_status(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    prune_finished_jobs()\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def cancel_transcription_job(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        if job[\\"status\\"] in {\\"queued\\", \\"running\\"}:\\n",\n        "            job[\\"cancel_requested\\"] = True\\n",\n        "            job[\\"status\\"] = \\"cancelled\\"\\n",\n        "            job[\\"detail\\"] = \\"Transcription cancellation requested\\"\\n",\n        "            job[\\"progress\\"] = 100\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v1/audio/transcriptions\\")\\n",\n        "async def transcriptions(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "\\n",\n        "        # Compatibility endpoint for older desktop builds. New LA Studio builds\\n",\n        "        # use /v2/jobs/transcriptions so a long GPU run cannot hit the\\n",\n        "        # Cloudflare 120-second proxy response limit.\\n",\n        "        result = await asyncio.to_thread(run_transcription, str(path), language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise HTTPException(status_code=502, detail=\\"The loaded model returned an empty transcript\\")\\n",\n        "        return {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "    finally:\\n",\n        "        if path is not None:\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import subprocess\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.request\\n",\n        "\\n",\n        "\\n",\n        "def read_health(url: str, timeout: float = 2.0):\\n",\n        "    try:\\n",\n        "        with urllib.request.urlopen(url.rstrip(\\"/\\") + \\"/health\\", timeout=timeout) as response:\\n",\n        "            if response.status != 200:\\n",\n        "                return None\\n",\n        "            payload = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            return payload if isinstance(payload, dict) else None\\n",\n        "    except Exception:\\n",\n        "        return None\\n",\n        "\\n",\n        "\\n",\n        "def is_exact_worker(payload) -> bool:\\n",\n        "    return bool(\\n",\n        "        isinstance(payload, dict)\\n",\n        "        and payload.get(\\"ready\\") is True\\n",\n        "        and str(payload.get(\\"device\\", \\"\\")).strip().lower() == \\"cuda\\"\\n",\n        "        and str(payload.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "        and str(payload.get(\\"worker_revision\\", \\"\\")) == WORKER_REVISION\\n",\n        "    )\\n",\n        "\\n",\n        "\\n",\n        "def wait_for_local_health():\\n",\n        "    deadline = time.time() + 60\\n",\n        "    while time.time() < deadline:\\n",\n        "        health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "        if health is not None:\\n",\n        "            return health\\n",\n        "        time.sleep(0.5)\\n",\n        "    raise RuntimeError(\\"The local STT worker did not become healthy\\")\\n",\n        "\\n",\n        "\\n",\n        "def run_server():\\n",\n        "    import uvicorn\\n",\n        "    uvicorn.run(app, host=\\"127.0.0.1\\", port=8000, log_level=\\"warning\\")\\n",\n        "\\n",\n        "\\n",\n        "# Re-running this cell in the same Colab runtime must not create a second\\n",\n        "# Uvicorn server.  The existing app functions use the refreshed notebook\\n",\n        "# globals, so the same exact model can safely be reused.  A different model\\n",\n        "# must use a fresh Colab runtime to avoid silently serving the wrong worker.\\n",\n        "local_health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "if local_health is None:\\n",\n        "    threading.Thread(target=run_server, daemon=True).start()\\n",\n        "    local_health = wait_for_local_health()\\n",\n        "    print(\\"Started local LA Studio STT worker on port 8000\\")\\n",\n        "elif is_exact_worker(local_health):\\n",\n        "    print(\\"Reusing the existing local LA Studio STT worker on port 8000\\")\\n",\n        "else:\\n",\n        "    current_model = str(local_health.get(\\"model\\", \\"unknown\\"))\\n",\n        "    current_revision = str(local_health.get(\\"worker_revision\\", \\"unknown\\"))\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"Port 8000 serves LA Studio model \'{current_model}\' (worker revision \'{current_revision}\'), \\"\\n",\n        "        f\\"not the required \'{MODEL_ID}\' revision \'{WORKER_REVISION}\'. \\"\\n",\n        "        \\"Use Runtime > Disconnect and delete runtime, then Run all for this exact model.\\"\\n",\n        "    )\\n",\n        "\\n",\n        "if not is_exact_worker(local_health):\\n",\n        "    raise RuntimeError(\\"The local worker did not confirm the selected exact CUDA model\\")\\n",\n        "\\n",\n        "\\n",\n        "def valid_cloudflared(path: str) -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [path, \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> str:\\n",\n        "    path = \\"/content/cloudflared\\"\\n",\n        "    if valid_cloudflared(path):\\n",\n        "        return path\\n",\n        "    result = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if result.returncode == 0:\\n",\n        "        subprocess.run([\\"chmod\\", \\"+x\\", path], check=True)\\n",\n        "    if result.returncode != 0 or not valid_cloudflared(path):\\n",\n        "        detail = result.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not obtain a working cloudflared binary: \\" + detail)\\n",\n        "    return path\\n",\n        "\\n",\n        "\\n",\n        "existing_url = os.environ.get(\\"LA_STUDIO_COLAB_STT_URL\\", \\"\\").strip()\\n",\n        "existing_health = read_health(existing_url, timeout=4.0) if existing_url else None\\n",\n        "if is_exact_worker(existing_health):\\n",\n        "    worker_url = existing_url\\n",\n        "    print(\\"Reusing the existing public Cloudflare tunnel\\")\\n",\n        "else:\\n",\n        "    cloudflared_path = ensure_cloudflared()\\n",\n        "    process = subprocess.Popen(\\n",\n        "        [cloudflared_path, \\"tunnel\\", \\"--url\\", \\"http://127.0.0.1:8000\\", \\"--no-autoupdate\\"],\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        text=True,\\n",\n        "        bufsize=1,\\n",\n        "    )\\n",\n        "    lines = queue.Queue()\\n",\n        "\\n",\n        "    def collect_output():\\n",\n        "        assert process.stdout is not None\\n",\n        "        for line in process.stdout:\\n",\n        "            lines.put(line)\\n",\n        "\\n",\n        "    threading.Thread(target=collect_output, daemon=True).start()\\n",\n        "    worker_url = \\"\\"\\n",\n        "    deadline = time.time() + 90\\n",\n        "    while time.time() < deadline and not worker_url:\\n",\n        "        if process.poll() is not None:\\n",\n        "            raise RuntimeError(\\"cloudflared exited before creating a public tunnel\\")\\n",\n        "        try:\\n",\n        "            line = lines.get(timeout=1)\\n",\n        "        except queue.Empty:\\n",\n        "            continue\\n",\n        "        match = re.search(r\\"https://[a-z0-9-]+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "        if match:\\n",\n        "            # Colab\'s self-request to its own public quick tunnel is not a\\n",\n        "            # reliable readiness test. The desktop Check Colab action is the\\n",\n        "            # authoritative public endpoint + token + exact-model verification.\\n",\n        "            worker_url = match.group(0)\\n",\n        "            print(\\"Cloudflare tunnel URL created. Verify it with Check Colab in LA Studio.\\")\\n",\n        "\\n",\n        "    if not worker_url:\\n",\n        "        process.terminate()\\n",\n        "        raise RuntimeError(\\n",\n        "            \\"cloudflared did not publish a trycloudflare URL within 90 seconds. \\"\\n",\n        "            \\"Run the launch cell once more; if it repeats, reset the Colab runtime and check that internet access is available.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_URL\\"] = worker_url\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_TOKEN\\"] = TOKEN\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_MODEL\\"] = MODEL_ID\\n",\n        "\\n",\n        "print(\\"\\\\nLA Studio Colab STT worker is ready\\")\\n",\n        "print(\\"MODEL:\\", MODEL_ID)\\n",\n        "print(\\"URL:\\", worker_url)\\n",\n        "print(\\"TOKEN:\\", TOKEN)\\n",\n        "print(\\"\\\\nPaste the URL and TOKEN into LA Studio. Keep this cell running.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "language": "python",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "stt",\n      "family_id": "qwen3-asr-1.7b",\n      "upstream_model": "Qwen/Qwen3-ASR-1.7B",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/stt/LA_STUDIO_STT_WHISPER_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio \\u2014 Whisper large-v3 (faster-whisper CUDA) Colab GPU worker\\n",\n        "\\n",\n        "This notebook loads exactly `whisper.cpp` on the Colab GPU.\\n",\n        "It rejects transcription requests for every other model ID.\\n",\n        "\\n",\n        "Long recordings use an asynchronous GPU job: the app uploads the\\n",\n        "audio once, then polls short status requests until this exact model\\n",\n        "completes. This avoids Cloudflare\'s 120-second proxy response limit.\\n",\n        "\\n",\n        "Run every cell in order, then paste the printed URL and TOKEN into\\n",\n        "LA Studio. The tunnel is public, but every worker endpoint requires\\n",\n        "the random session token printed by the last cell.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "%pip install -q faster-whisper==1.2.1 fastapi==0.115.12 uvicorn==0.34.3 python-multipart==0.0.20 soundfile==0.13.1\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import asyncio\\n",\n        "import os\\n",\n        "import secrets\\n",\n        "import tempfile\\n",\n        "import threading\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "import soundfile as sf\\n",\n        "import torch\\n",\n        "from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile\\n",\n        "from fastapi.responses import JSONResponse\\n",\n        "\\n",\n        "if not torch.cuda.is_available():\\n",\n        "    raise RuntimeError(\\"A Colab GPU runtime is required. Choose Runtime > Change runtime type > GPU.\\")\\n",\n        "\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "WORKER_REVISION = \\"stt-2026-07-30.2\\"\\n",\n        "MAX_UPLOAD_BYTES = 512 * 1024 * 1024\\n",\n        "MAX_AUDIO_SECONDS = 30 * 60\\n",\n        "ALLOWED_CONTENT_TYPES = {\\n",\n        "    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp3\\", \\"audio/mp4\\",\\n",\n        "    \\"audio/flac\\", \\"audio/ogg\\", \\"audio/webm\\", \\"application/octet-stream\\",\\n",\n        "}\\n",\n        "REQUEST_SLOTS = threading.BoundedSemaphore(1)\\n",\n        "JOB_TTL_SECONDS = 30 * 60\\n",\n        "UPLOAD_TTL_SECONDS = 10 * 60\\n",\n        "CHUNK_UPLOAD_BYTES = 2 * 1024 * 1024\\n",\n        "JOB_LOCK = threading.Lock()\\n",\n        "JOBS: dict[str, dict] = {}\\n",\n        "UPLOADS: dict[str, dict] = {}\\n",\n        "\\n",\n        "from faster_whisper import WhisperModel\\n",\n        "\\n",\n        "MODEL_ID = \\"whisper.cpp\\"\\n",\n        "MODEL_NAME = \\"Whisper large-v3 (faster-whisper CUDA)\\"\\n",\n        "UPSTREAM_MODEL = \\"large-v3\\"\\n",\n        "stt_model = WhisperModel(UPSTREAM_MODEL, device=\\"cuda\\", compute_type=\\"float16\\")\\n",\n        "\\n",\n        "def run_transcription(path: str, language: str):\\n",\n        "    requested_language = language.strip().lower()\\n",\n        "    if not requested_language or requested_language == \\"auto\\":\\n",\n        "        requested_language = None\\n",\n        "    segments, info = stt_model.transcribe(\\n",\n        "        path,\\n",\n        "        language=requested_language,\\n",\n        "        beam_size=5,\\n",\n        "        vad_filter=True,\\n",\n        "    )\\n",\n        "    rows = []\\n",\n        "    text_parts = []\\n",\n        "    for index, segment in enumerate(segments):\\n",\n        "        clean = segment.text.strip()\\n",\n        "        if clean:\\n",\n        "            text_parts.append(clean)\\n",\n        "        rows.append({\\"id\\": index, \\"start\\": float(segment.start), \\"end\\": float(segment.end), \\"text\\": clean})\\n",\n        "    return {\\"text\\": \\" \\".join(text_parts), \\"segments\\": rows, \\"language\\": info.language}\\n",\n        "\\n",\n        "app = FastAPI(title=f\\"LA Studio STT \\u2014 {MODEL_NAME}\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.exception_handler(Exception)\\n",\n        "async def unhandled_exception(request: Request, error: Exception):\\n",\n        "    # A tunnel 500 without a response body is impossible to act on from the\\n",\n        "    # desktop app. Keep the detail bounded, and also print it in the Colab\\n",\n        "    # cell so the notebook owns the operational diagnosis.\\n",\n        "    detail = f\\"STT worker internal error: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "    print(detail, flush=True)\\n",\n        "    return JSONResponse(status_code=500, content={\\"detail\\": detail})\\n",\n        "\\n",\n        "\\n",\n        "def require_token(authorization: str | None) -> None:\\n",\n        "    if authorization != f\\"Bearer {TOKEN}\\":\\n",\n        "        raise HTTPException(status_code=401, detail=\\"Invalid or missing Colab session token\\")\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/health\\")\\n",\n        "def health():\\n",\n        "    return {\\n",\n        "        \\"ok\\": True,\\n",\n        "        \\"ready\\": True,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"gpu\\": torch.cuda.get_device_name(0),\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "        \\"variant\\": \\"fixed\\",\\n",\n        "        \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v1/capabilities\\")\\n",\n        "def capabilities(authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    return {\\n",\n        "        \\"contract_version\\": 1,\\n",\n        "        \\"worker_revision\\": WORKER_REVISION,\\n",\n        "        \\"device\\": \\"cuda\\",\\n",\n        "        \\"cuda\\": True,\\n",\n        "        \\"cpu_fallback\\": False,\\n",\n        "        \\"endpoints\\": {\\n",\n        "            \\"transcription_jobs\\": \\"/v2/jobs/transcriptions\\",\\n",\n        "            \\"chunked_transcription_uploads\\": \\"/v2/uploads/stt\\",\\n",\n        "        },\\n",\n        "        \\"chunked_uploads\\": True,\\n",\n        "        \\"capabilities\\": [{\\n",\n        "            \\"id\\": \\"stt\\",\\n",\n        "            \\"models\\": [{\\n",\n        "                \\"id\\": MODEL_ID,\\n",\n        "                \\"name\\": MODEL_NAME,\\n",\n        "                \\"variant\\": \\"fixed\\",\\n",\n        "                \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "                \\"loaded\\": True,\\n",\n        "                \\"device\\": \\"cuda\\",\\n",\n        "            }],\\n",\n        "        }],\\n",\n        "    }\\n",\n        "\\n",\n        "\\n",\n        "async def save_upload(file: UploadFile) -> tuple[Path, int]:\\n",\n        "    suffix = Path(file.filename or \\"audio.wav\\").suffix or \\".wav\\"\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=suffix, delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    total = 0\\n",\n        "    try:\\n",\n        "        with handle:\\n",\n        "            while True:\\n",\n        "                chunk = await file.read(1024 * 1024)\\n",\n        "                if not chunk:\\n",\n        "                    break\\n",\n        "                total += len(chunk)\\n",\n        "                if total > MAX_UPLOAD_BYTES:\\n",\n        "                    raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "                handle.write(chunk)\\n",\n        "        return path, total\\n",\n        "    except Exception:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "        raise\\n",\n        "\\n",\n        "\\n",\n        "def validate_model(model: str) -> str:\\n",\n        "    requested = model.strip().lower()\\n",\n        "    if requested != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    return requested\\n",\n        "\\n",\n        "\\n",\n        "def validate_audio_duration(path: Path) -> None:\\n",\n        "    try:\\n",\n        "        info = sf.info(str(path))\\n",\n        "        if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "    except HTTPException:\\n",\n        "        raise\\n",\n        "    except Exception:\\n",\n        "        # Compressed formats may not be readable by libsndfile; the\\n",\n        "        # model-specific decoder remains the source of truth.\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def snapshot_job(job: dict) -> dict:\\n",\n        "    response = {\\n",\n        "        \\"job_id\\": job[\\"job_id\\"],\\n",\n        "        \\"status\\": job[\\"status\\"],\\n",\n        "        \\"progress\\": job[\\"progress\\"],\\n",\n        "        \\"model\\": MODEL_ID,\\n",\n        "    }\\n",\n        "    if job.get(\\"detail\\"):\\n",\n        "        response[\\"detail\\"] = job[\\"detail\\"]\\n",\n        "    if job.get(\\"result\\") is not None:\\n",\n        "        response[\\"result\\"] = job[\\"result\\"]\\n",\n        "    return response\\n",\n        "\\n",\n        "\\n",\n        "def prune_finished_jobs() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            job_id for job_id, job in JOBS.items()\\n",\n        "            if job.get(\\"finished_at\\") and now - job[\\"finished_at\\"] > JOB_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for job_id in expired:\\n",\n        "            job = JOBS.pop(job_id)\\n",\n        "            if job.get(\\"path\\"):\\n",\n        "                expired_paths.append(Path(job[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def prune_expired_uploads() -> None:\\n",\n        "    now = asyncio.get_running_loop().time()\\n",\n        "    expired_paths = []\\n",\n        "    with JOB_LOCK:\\n",\n        "        expired = [\\n",\n        "            upload_id for upload_id, upload in UPLOADS.items()\\n",\n        "            if now - upload[\\"created_at\\"] > UPLOAD_TTL_SECONDS\\n",\n        "        ]\\n",\n        "        for upload_id in expired:\\n",\n        "            upload = UPLOADS.pop(upload_id)\\n",\n        "            expired_paths.append(Path(upload[\\"path\\"]))\\n",\n        "    for path in expired_paths:\\n",\n        "        path.unlink(missing_ok=True)\\n",\n        "\\n",\n        "\\n",\n        "async def run_job(job_id: str) -> None:\\n",\n        "    try:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is None:\\n",\n        "                return\\n",\n        "            if job.get(\\"cancel_requested\\"):\\n",\n        "                job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "                return\\n",\n        "            job[\\"status\\"] = \\"running\\"\\n",\n        "            job[\\"progress\\"] = 15\\n",\n        "            path = job[\\"path\\"]\\n",\n        "            language = job[\\"language\\"]\\n",\n        "            response_format = job[\\"response_format\\"]\\n",\n        "        result = await asyncio.to_thread(run_transcription, path, language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise RuntimeError(\\"The loaded model returned an empty transcript\\")\\n",\n        "        payload = {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"succeeded\\"\\n",\n        "                    job[\\"result\\"] = payload\\n",\n        "                    job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    except Exception as error:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            if job is not None:\\n",\n        "                if job.get(\\"cancel_requested\\"):\\n",\n        "                    job[\\"status\\"] = \\"cancelled\\"\\n",\n        "                    job[\\"detail\\"] = \\"Transcription cancelled\\"\\n",\n        "                else:\\n",\n        "                    job[\\"status\\"] = \\"failed\\"\\n",\n        "                    job[\\"detail\\"] = f\\"{MODEL_NAME} transcription failed: {type(error).__name__}: {str(error)[:300]}\\"\\n",\n        "                job[\\"progress\\"] = 100\\n",\n        "                job[\\"finished_at\\"] = asyncio.get_running_loop().time()\\n",\n        "    finally:\\n",\n        "        with JOB_LOCK:\\n",\n        "            job = JOBS.get(job_id)\\n",\n        "            source_path = Path(job[\\"path\\"]) if job and job.get(\\"path\\") else None\\n",\n        "            if job is not None:\\n",\n        "                job[\\"path\\"] = None\\n",\n        "        if source_path is not None:\\n",\n        "            source_path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt\\", status_code=201)\\n",\n        "async def begin_chunked_stt_upload(payload: dict,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    model = validate_model(str(payload.get(\\"model\\", \\"\\")))\\n",\n        "    try:\\n",\n        "        size_bytes = int(payload.get(\\"size_bytes\\", 0))\\n",\n        "    except (TypeError, ValueError):\\n",\n        "        raise HTTPException(status_code=422, detail=\\"Audio upload size must be an integer\\")\\n",\n        "    if size_bytes <= 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "    if size_bytes > MAX_UPLOAD_BYTES:\\n",\n        "        raise HTTPException(status_code=413, detail=\\"Audio upload exceeds 512 MiB\\")\\n",\n        "    handle = tempfile.NamedTemporaryFile(prefix=\\"la-studio-stt-\\", suffix=\\".wav\\", delete=False)\\n",\n        "    path = Path(handle.name)\\n",\n        "    handle.close()\\n",\n        "    upload_id = secrets.token_urlsafe(18)\\n",\n        "    with JOB_LOCK:\\n",\n        "        UPLOADS[upload_id] = {\\n",\n        "            \\"path\\": str(path),\\n",\n        "            \\"size_bytes\\": size_bytes,\\n",\n        "            \\"received_bytes\\": 0,\\n",\n        "            \\"next_chunk\\": 0,\\n",\n        "            \\"model\\": model,\\n",\n        "            \\"language\\": str(payload.get(\\"language\\", \\"auto\\")),\\n",\n        "            \\"response_format\\": str(payload.get(\\"response_format\\", \\"verbose_json\\")),\\n",\n        "            \\"created_at\\": asyncio.get_running_loop().time(),\\n",\n        "        }\\n",\n        "    return {\\"upload_id\\": upload_id, \\"chunk_bytes\\": CHUNK_UPLOAD_BYTES}\\n",\n        "\\n",\n        "\\n",\n        "@app.put(\\"/v2/uploads/stt/{upload_id}/chunks/{chunk_index}\\")\\n",\n        "async def upload_chunked_stt_audio(upload_id: str, chunk_index: int, request: Request,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        remaining = upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]\\n",\n        "    chunks = []\\n",\n        "    total = 0\\n",\n        "    async for piece in request.stream():\\n",\n        "        total += len(piece)\\n",\n        "        if total > CHUNK_UPLOAD_BYTES or total > remaining:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload chunk is too large\\")\\n",\n        "        chunks.append(piece)\\n",\n        "    if total == 0:\\n",\n        "        raise HTTPException(status_code=422, detail=\\"STT upload chunk is empty\\")\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if chunk_index != upload[\\"next_chunk\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload chunk is out of order\\")\\n",\n        "        if total > upload[\\"size_bytes\\"] - upload[\\"received_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=413, detail=\\"STT upload exceeds its declared size\\")\\n",\n        "        try:\\n",\n        "            with Path(upload[\\"path\\"]).open(\\"ab\\") as handle:\\n",\n        "                for piece in chunks:\\n",\n        "                    handle.write(piece)\\n",\n        "        except OSError as error:\\n",\n        "            raise HTTPException(status_code=500, detail=f\\"Unable to store STT upload chunk: {error}\\")\\n",\n        "        upload[\\"received_bytes\\"] += total\\n",\n        "        upload[\\"next_chunk\\"] += 1\\n",\n        "        return {\\n",\n        "            \\"received_bytes\\": upload[\\"received_bytes\\"],\\n",\n        "            \\"next_chunk\\": upload[\\"next_chunk\\"],\\n",\n        "        }\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/uploads/stt/{upload_id}\\")\\n",\n        "async def cancel_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.pop(upload_id, None)\\n",\n        "    if upload is not None:\\n",\n        "        Path(upload[\\"path\\"]).unlink(missing_ok=True)\\n",\n        "    return {\\"status\\": \\"cancelled\\"}\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/uploads/stt/{upload_id}/commit\\", status_code=202)\\n",\n        "async def commit_chunked_stt_upload(upload_id: str,\\n",\n        "                                    authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    await prune_expired_uploads()\\n",\n        "    with JOB_LOCK:\\n",\n        "        upload = UPLOADS.get(upload_id)\\n",\n        "        if upload is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "        if upload[\\"received_bytes\\"] != upload[\\"size_bytes\\"]:\\n",\n        "            raise HTTPException(status_code=409, detail=\\"STT upload is incomplete\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = Path(upload[\\"path\\"])\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        validate_audio_duration(path)\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            upload = UPLOADS.pop(upload_id, None)\\n",\n        "            if upload is None:\\n",\n        "                raise HTTPException(status_code=404, detail=\\"STT upload was not found or has expired\\")\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": upload[\\"language\\"],\\n",\n        "                \\"response_format\\": upload[\\"response_format\\"],\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        if not queued:\\n",\n        "            with JOB_LOCK:\\n",\n        "                UPLOADS.pop(upload_id, None)\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v2/jobs/transcriptions\\", status_code=202)\\n",\n        "async def create_transcription_job(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    queued = False\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "        prune_finished_jobs()\\n",\n        "        job_id = secrets.token_urlsafe(18)\\n",\n        "        with JOB_LOCK:\\n",\n        "            JOBS[job_id] = {\\n",\n        "                \\"job_id\\": job_id,\\n",\n        "                \\"status\\": \\"queued\\",\\n",\n        "                \\"progress\\": 5,\\n",\n        "                \\"path\\": str(path),\\n",\n        "                \\"language\\": language,\\n",\n        "                \\"response_format\\": response_format,\\n",\n        "                \\"cancel_requested\\": False,\\n",\n        "                \\"detail\\": \\"\\",\\n",\n        "                \\"result\\": None,\\n",\n        "            }\\n",\n        "            response = snapshot_job(JOBS[job_id])\\n",\n        "        asyncio.create_task(run_job(job_id))\\n",\n        "        queued = True\\n",\n        "        return response\\n",\n        "    finally:\\n",\n        "        await file.close()\\n",\n        "        if not queued:\\n",\n        "            if path is not None:\\n",\n        "                path.unlink(missing_ok=True)\\n",\n        "            REQUEST_SLOTS.release()\\n",\n        "\\n",\n        "\\n",\n        "@app.get(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def transcription_job_status(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    prune_finished_jobs()\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.delete(\\"/v2/jobs/transcriptions/{job_id}\\")\\n",\n        "async def cancel_transcription_job(job_id: str,\\n",\n        "                                   authorization: str | None = Header(default=None)):\\n",\n        "    require_token(authorization)\\n",\n        "    with JOB_LOCK:\\n",\n        "        job = JOBS.get(job_id)\\n",\n        "        if job is None:\\n",\n        "            raise HTTPException(status_code=404, detail=\\"Transcription job was not found or has expired\\")\\n",\n        "        if job[\\"status\\"] in {\\"queued\\", \\"running\\"}:\\n",\n        "            job[\\"cancel_requested\\"] = True\\n",\n        "            job[\\"status\\"] = \\"cancelled\\"\\n",\n        "            job[\\"detail\\"] = \\"Transcription cancellation requested\\"\\n",\n        "            job[\\"progress\\"] = 100\\n",\n        "        return snapshot_job(job)\\n",\n        "\\n",\n        "\\n",\n        "@app.post(\\"/v1/audio/transcriptions\\")\\n",\n        "async def transcriptions(\\n",\n        "    file: UploadFile = File(...),\\n",\n        "    model: str = Form(...),\\n",\n        "    language: str = Form(default=\\"auto\\"),\\n",\n        "    response_format: str = Form(default=\\"verbose_json\\"),\\n",\n        "    authorization: str | None = Header(default=None),\\n",\n        "):\\n",\n        "    require_token(authorization)\\n",\n        "    if model.strip().lower() != MODEL_ID:\\n",\n        "        raise HTTPException(\\n",\n        "            status_code=409,\\n",\n        "            detail=f\\"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.\\",\\n",\n        "        )\\n",\n        "    content_type = (file.content_type or \\"application/octet-stream\\").lower()\\n",\n        "    if content_type not in ALLOWED_CONTENT_TYPES and not content_type.startswith(\\"audio/\\"):\\n",\n        "        raise HTTPException(status_code=415, detail=f\\"Unsupported audio content type: {content_type}\\")\\n",\n        "    if not REQUEST_SLOTS.acquire(blocking=False):\\n",\n        "        raise HTTPException(status_code=429, detail=\\"The Colab GPU is already processing another transcription\\")\\n",\n        "\\n",\n        "    path = None\\n",\n        "    try:\\n",\n        "        path, size = await save_upload(file)\\n",\n        "        if size == 0:\\n",\n        "            raise HTTPException(status_code=422, detail=\\"The uploaded audio file is empty\\")\\n",\n        "        try:\\n",\n        "            info = sf.info(str(path))\\n",\n        "            if info.duration > MAX_AUDIO_SECONDS:\\n",\n        "                raise HTTPException(status_code=413, detail=\\"Audio is longer than the 30 minute session limit\\")\\n",\n        "        except HTTPException:\\n",\n        "            raise\\n",\n        "        except Exception:\\n",\n        "            # Compressed formats may not be readable by libsndfile; the\\n",\n        "            # model-specific decoder below remains the source of truth.\\n",\n        "            pass\\n",\n        "\\n",\n        "        # Compatibility endpoint for older desktop builds. New LA Studio builds\\n",\n        "        # use /v2/jobs/transcriptions so a long GPU run cannot hit the\\n",\n        "        # Cloudflare 120-second proxy response limit.\\n",\n        "        result = await asyncio.to_thread(run_transcription, str(path), language)\\n",\n        "        text = str(result.get(\\"text\\", \\"\\")).strip()\\n",\n        "        if not text:\\n",\n        "            raise HTTPException(status_code=502, detail=\\"The loaded model returned an empty transcript\\")\\n",\n        "        return {\\n",\n        "            \\"text\\": text,\\n",\n        "            \\"segments\\": result.get(\\"segments\\", []),\\n",\n        "            \\"language\\": result.get(\\"language\\", language),\\n",\n        "            \\"model\\": MODEL_ID,\\n",\n        "            \\"upstream_model\\": UPSTREAM_MODEL,\\n",\n        "            \\"response_format\\": response_format,\\n",\n        "        }\\n",\n        "    finally:\\n",\n        "        if path is not None:\\n",\n        "            path.unlink(missing_ok=True)\\n",\n        "        REQUEST_SLOTS.release()\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import subprocess\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.request\\n",\n        "\\n",\n        "\\n",\n        "def read_health(url: str, timeout: float = 2.0):\\n",\n        "    try:\\n",\n        "        with urllib.request.urlopen(url.rstrip(\\"/\\") + \\"/health\\", timeout=timeout) as response:\\n",\n        "            if response.status != 200:\\n",\n        "                return None\\n",\n        "            payload = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            return payload if isinstance(payload, dict) else None\\n",\n        "    except Exception:\\n",\n        "        return None\\n",\n        "\\n",\n        "\\n",\n        "def is_exact_worker(payload) -> bool:\\n",\n        "    return bool(\\n",\n        "        isinstance(payload, dict)\\n",\n        "        and payload.get(\\"ready\\") is True\\n",\n        "        and str(payload.get(\\"device\\", \\"\\")).strip().lower() == \\"cuda\\"\\n",\n        "        and str(payload.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "        and str(payload.get(\\"worker_revision\\", \\"\\")) == WORKER_REVISION\\n",\n        "    )\\n",\n        "\\n",\n        "\\n",\n        "def wait_for_local_health():\\n",\n        "    deadline = time.time() + 60\\n",\n        "    while time.time() < deadline:\\n",\n        "        health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "        if health is not None:\\n",\n        "            return health\\n",\n        "        time.sleep(0.5)\\n",\n        "    raise RuntimeError(\\"The local STT worker did not become healthy\\")\\n",\n        "\\n",\n        "\\n",\n        "def run_server():\\n",\n        "    import uvicorn\\n",\n        "    uvicorn.run(app, host=\\"127.0.0.1\\", port=8000, log_level=\\"warning\\")\\n",\n        "\\n",\n        "\\n",\n        "# Re-running this cell in the same Colab runtime must not create a second\\n",\n        "# Uvicorn server.  The existing app functions use the refreshed notebook\\n",\n        "# globals, so the same exact model can safely be reused.  A different model\\n",\n        "# must use a fresh Colab runtime to avoid silently serving the wrong worker.\\n",\n        "local_health = read_health(\\"http://127.0.0.1:8000\\")\\n",\n        "if local_health is None:\\n",\n        "    threading.Thread(target=run_server, daemon=True).start()\\n",\n        "    local_health = wait_for_local_health()\\n",\n        "    print(\\"Started local LA Studio STT worker on port 8000\\")\\n",\n        "elif is_exact_worker(local_health):\\n",\n        "    print(\\"Reusing the existing local LA Studio STT worker on port 8000\\")\\n",\n        "else:\\n",\n        "    current_model = str(local_health.get(\\"model\\", \\"unknown\\"))\\n",\n        "    current_revision = str(local_health.get(\\"worker_revision\\", \\"unknown\\"))\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"Port 8000 serves LA Studio model \'{current_model}\' (worker revision \'{current_revision}\'), \\"\\n",\n        "        f\\"not the required \'{MODEL_ID}\' revision \'{WORKER_REVISION}\'. \\"\\n",\n        "        \\"Use Runtime > Disconnect and delete runtime, then Run all for this exact model.\\"\\n",\n        "    )\\n",\n        "\\n",\n        "if not is_exact_worker(local_health):\\n",\n        "    raise RuntimeError(\\"The local worker did not confirm the selected exact CUDA model\\")\\n",\n        "\\n",\n        "\\n",\n        "def valid_cloudflared(path: str) -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [path, \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> str:\\n",\n        "    path = \\"/content/cloudflared\\"\\n",\n        "    if valid_cloudflared(path):\\n",\n        "        return path\\n",\n        "    result = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if result.returncode == 0:\\n",\n        "        subprocess.run([\\"chmod\\", \\"+x\\", path], check=True)\\n",\n        "    if result.returncode != 0 or not valid_cloudflared(path):\\n",\n        "        detail = result.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not obtain a working cloudflared binary: \\" + detail)\\n",\n        "    return path\\n",\n        "\\n",\n        "\\n",\n        "existing_url = os.environ.get(\\"LA_STUDIO_COLAB_STT_URL\\", \\"\\").strip()\\n",\n        "existing_health = read_health(existing_url, timeout=4.0) if existing_url else None\\n",\n        "if is_exact_worker(existing_health):\\n",\n        "    worker_url = existing_url\\n",\n        "    print(\\"Reusing the existing public Cloudflare tunnel\\")\\n",\n        "else:\\n",\n        "    cloudflared_path = ensure_cloudflared()\\n",\n        "    process = subprocess.Popen(\\n",\n        "        [cloudflared_path, \\"tunnel\\", \\"--url\\", \\"http://127.0.0.1:8000\\", \\"--no-autoupdate\\"],\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        text=True,\\n",\n        "        bufsize=1,\\n",\n        "    )\\n",\n        "    lines = queue.Queue()\\n",\n        "\\n",\n        "    def collect_output():\\n",\n        "        assert process.stdout is not None\\n",\n        "        for line in process.stdout:\\n",\n        "            lines.put(line)\\n",\n        "\\n",\n        "    threading.Thread(target=collect_output, daemon=True).start()\\n",\n        "    worker_url = \\"\\"\\n",\n        "    deadline = time.time() + 90\\n",\n        "    while time.time() < deadline and not worker_url:\\n",\n        "        if process.poll() is not None:\\n",\n        "            raise RuntimeError(\\"cloudflared exited before creating a public tunnel\\")\\n",\n        "        try:\\n",\n        "            line = lines.get(timeout=1)\\n",\n        "        except queue.Empty:\\n",\n        "            continue\\n",\n        "        match = re.search(r\\"https://[a-z0-9-]+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "        if match:\\n",\n        "            # Colab\'s self-request to its own public quick tunnel is not a\\n",\n        "            # reliable readiness test. The desktop Check Colab action is the\\n",\n        "            # authoritative public endpoint + token + exact-model verification.\\n",\n        "            worker_url = match.group(0)\\n",\n        "            print(\\"Cloudflare tunnel URL created. Verify it with Check Colab in LA Studio.\\")\\n",\n        "\\n",\n        "    if not worker_url:\\n",\n        "        process.terminate()\\n",\n        "        raise RuntimeError(\\n",\n        "            \\"cloudflared did not publish a trycloudflare URL within 90 seconds. \\"\\n",\n        "            \\"Run the launch cell once more; if it repeats, reset the Colab runtime and check that internet access is available.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_URL\\"] = worker_url\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_TOKEN\\"] = TOKEN\\n",\n        "os.environ[\\"LA_STUDIO_COLAB_STT_MODEL\\"] = MODEL_ID\\n",\n        "\\n",\n        "print(\\"\\\\nLA Studio Colab STT worker is ready\\")\\n",\n        "print(\\"MODEL:\\", MODEL_ID)\\n",\n        "print(\\"URL:\\", worker_url)\\n",\n        "print(\\"TOKEN:\\", TOKEN)\\n",\n        "print(\\"\\\\nPaste the URL and TOKEN into LA Studio. Keep this cell running.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "language": "python",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "stt",\n      "family_id": "whisper.cpp",\n      "upstream_model": "large-v3",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/subtitle_ocr/LA_STUDIO_SUBTITLE_OCR_PP_OCRV5_GPU.ipynb': '{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "# LA Studio Subtitle OCR — PP-OCRv5 Multilingual 3.1\\n",\n    "\\n",\n    "This direct CUDA notebook is independent of API Gateway. It loads the\\n",\n    "PP-OCRv5 multilingual family (PaddleOCR 3.1.1, Apache-2.0) and accepts\\n",\n    "only cropped PNG subtitle frames, never a source video.\\n",\n    "\\n",\n    "1. Choose **Runtime → Change runtime type → GPU**.\\n",\n    "2. Run all cells.\\n",\n    "3. In Subtitle OCR choose **Colab GPU**, open Configure / check Colab,\\n",\n    "   and paste only the temporary URL and token printed below.\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "!nvidia-smi\\n",\n    "# Keep OCR out of Colab\'s mutable global site-packages. A\\n",\n    "# global install can leave Pillow 12\'s ImageText.py beside an\\n",\n    "# older PIL._typing.py, which causes the `_Ink` error reported\\n",\n    "# in this notebook. Do not create a venv: recent Colab Python\\n",\n    "# images can have a broken ensurepip bootstrap.\\n",\n    "import os\\n",\n    "import platform\\n",\n    "import shutil\\n",\n    "import subprocess\\n",\n    "import sys\\n",\n    "from pathlib import Path\\n",\n    "\\n",\n    "BOOTSTRAP_REVISION = \\"subtitle-ocr-bootstrap-2026-08-23.18\\"\\n",\n    "print(\\"LA Studio Subtitle OCR bootstrap:\\", BOOTSTRAP_REVISION)\\n",\n    "print(\\"This revision uses a dedicated package directory; it never creates a venv or calls ensurepip.\\")\\n",\n    "\\n",\n    "OCR_SITE_PACKAGES = Path(\\"/content/la_studio_subtitle_ocr_site\\")\\n",\n    "# Remove only the old app-owned bootstrap directories. This\\n",\n    "# makes Run all safe after a notebook revision that used a\\n",\n    "# broken ensurepip virtual environment, without touching any\\n",\n    "# user files in /content.\\n",\n    "shutil.rmtree(Path(\\"/content/la_studio_subtitle_ocr_venv\\"), ignore_errors=True)\\n",\n    "shutil.rmtree(OCR_SITE_PACKAGES, ignore_errors=True)\\n",\n    "OCR_SITE_PACKAGES.mkdir(parents=True, exist_ok=True)\\n",\n    "BOOTSTRAP_ENV = os.environ.copy()\\n",\n    "BOOTSTRAP_ENV.pop(\\"PYTHONPATH\\", None)\\n",\n    "BOOTSTRAP_ENV[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n    "BOOTSTRAP_ENV[\\"PADDLE_PDX_MODEL_SOURCE\\"] = \\"BOS\\"\\n",\n    "\\n",\n    "if sys.platform != \\"linux\\" or platform.machine().lower() not in {\\"x86_64\\", \\"amd64\\"}:\\n",\n    "    raise RuntimeError(\\n",\n    "        \\"LA Studio Subtitle OCR exact CUDA notebook requires a Linux x86_64 Colab GPU runtime.\\"\\n",\n    "    )\\n",\n    "python_tag = f\\"cp{sys.version_info.major}{sys.version_info.minor}\\"\\n",\n    "# The Paddle index repeatedly times out while resolving a\\n",\n    "# fresh Colab image. Use the exact CUDA 11.8 wheel for the\\n",\n    "# current Python ABI rather than letting pip discover it via\\n",\n    "# an extra index.\\n",\n    "PADDLE_GPU_WHEEL = (\\n",\n    "    \\"https://paddle-whl.bj.bcebos.com/stable/cu118/paddlepaddle-gpu/\\"\\n",\n    "    f\\"paddlepaddle_gpu-3.1.0-{python_tag}-{python_tag}-linux_x86_64.whl\\"\\n",\n    ")\\n",\n    "\\n",\n    "def bootstrap_pip(*arguments):\\n",\n    "    command = [sys.executable, \\"-m\\", \\"pip\\", *arguments]\\n",\n    "    result = subprocess.run(command, env=BOOTSTRAP_ENV,\\n",\n    "                            text=True, stdout=subprocess.PIPE,\\n",\n    "                            stderr=subprocess.STDOUT)\\n",\n    "    output = result.stdout or \\"\\"\\n",\n    "    if output:\\n",\n    "        print(output)\\n",\n    "    if result.returncode:\\n",\n    "        raise RuntimeError(\\n",\n    "            \\"LA Studio Subtitle OCR dependency bootstrap failed with exit code \\"\\n",\n    "            + str(result.returncode) + \\".\\\\nCommand: \\" + \\" \\".join(command)\\n",\n    "            + \\"\\\\n\\\\n---- pip output (last 12,000 characters) ----\\\\n\\"\\n",\n    "            + output[-12000:])\\n",\n    "\\n",\n    "def ocr_pip(*arguments):\\n",\n    "    # --target alone can still treat a globally installed\\n",\n    "    # requirement as satisfied on some Colab images. Force\\n",\n    "    # every package into this directory so the worker cannot\\n",\n    "    # combine a new ImageText.py with an old PIL._typing.py.\\n",\n    "    bootstrap_pip(\\"install\\", \\"--target\\", str(OCR_SITE_PACKAGES),\\n",\n    "                  \\"--ignore-installed\\", \\"--disable-pip-version-check\\",\\n",\n    "                  \\"--retries\\", \\"4\\", \\"--timeout\\", \\"120\\", *arguments)\\n",\n    "\\n",\n    "OCR_PYTHON = sys.executable\\n",\n    "OCR_ENV = os.environ.copy()\\n",\n    "OCR_ENV[\\"PYTHONPATH\\"] = str(OCR_SITE_PACKAGES)\\n",\n    "OCR_ENV[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n    "OCR_ENV[\\"PADDLE_PDX_MODEL_SOURCE\\"] = \\"BOS\\"\\n",\n    "# Install the exact CUDA wheel first. This direct URL avoids\\n",\n    "# the timing-out Paddle index while still letting pip install\\n",\n    "# Paddle\'s own declared runtime dependencies into the same\\n",\n    "# isolated directory. The wheel URL pins the GPU runtime, so\\n",\n    "# the resolver cannot substitute a CPU Paddle build.\\n",\n    "ocr_pip(\\"--no-cache-dir\\", \\"--upgrade\\", \\"--force-reinstall\\",\\n",\n    "        PADDLE_GPU_WHEEL)\\n",\n    "\\n",\n    "# PaddleOCR 3.1.1 advertises broad PaddleX extras\\n",\n    "# (`ie,multimodal,ocr,trans`) that bring unrelated LLM/document\\n",\n    "# dependencies and source-only GPUtil into the resolver.\\n",\n    "# This worker needs only the pinned image OCR group; install\\n",\n    "# PaddleOCR itself without its broad dependency metadata.\\n",\n    "ocr_pip(\\"--no-cache-dir\\", \\"--upgrade\\", \\"--force-reinstall\\",\\n",\n    "        \\"--only-binary=:all:\\",\\n",\n    "        \\"paddlex[ocr]==3.1.0\\", \\"PyYAML==6.0.2\\", \\"typing-extensions==4.15.0\\",\\n",\n    "        # Colab can publish a compatible Pillow patch release\\n",\n    "        # (for example 12.3.0) before this notebook is rerun.\\n",\n    "        # Require the supported major/minor family, not one\\n",\n    "        # fragile patch version; the probe below still checks\\n",\n    "        # the actual API and package isolation.\\n",\n    "        \\"Pillow>=12.0.0,<13.0.0\\", \\"fastapi==0.115.12\\", \\"uvicorn==0.34.3\\",\\n",\n    "        \\"python-multipart==0.0.20\\")\\n",\n    "# PaddleX 3.1 imports langchain.docstore while importing its\\n",\n    "# pipeline registry, even for an image-only OCR worker. The\\n",\n    "# older 0.2 stack is incompatible with Colab Python 3.13\\n",\n    "# because it requires numpy<2. Version 0.3.27 keeps the\\n",\n    "# required legacy docstore module, supports Python 3.13, and\\n",\n    "# does not constrain NumPy below 2.\\n",\n    "ocr_pip(\\"--no-cache-dir\\", \\"--upgrade\\", \\"--force-reinstall\\",\\n",\n    "        \\"--only-binary=:all:\\", \\"langchain==0.3.27\\")\\n",\n    "ocr_pip(\\"--no-cache-dir\\", \\"--upgrade\\", \\"--force-reinstall\\",\\n",\n    "        \\"--only-binary=:all:\\", \\"--no-deps\\", \\"paddleocr==3.1.1\\")\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# Fail in this explicit dependency probe rather than after the\\n",\n    "# service has started. This uses the same interpreter plus\\n",\n    "# package-directory environment that will launch Uvicorn.\\n",\n    "from textwrap import dedent\\n",\n    "\\n",\n    "probe = dedent(r\\"\\"\\"\\n",\n    "from importlib.metadata import version\\n",\n    "import os\\n",\n    "import tempfile\\n",\n    "from pathlib import Path\\n",\n    "import PIL\\n",\n    "from PIL import Image, ImageText\\n",\n    "from PIL._typing import _Ink\\n",\n    "import paddle\\n",\n    "import paddleocr\\n",\n    "import paddlex\\n",\n    "import langchain\\n",\n    "from langchain.docstore.document import Document\\n",\n    "from paddleocr import PaddleOCR\\n",\n    "\\n",\n    "assert version(\\"paddlepaddle-gpu\\") == \\"3.1.0\\", version(\\"paddlepaddle-gpu\\")\\n",\n    "assert version(\\"paddlex\\") == \\"3.1.0\\", version(\\"paddlex\\")\\n",\n    "assert version(\\"paddleocr\\") == \\"3.1.1\\", version(\\"paddleocr\\")\\n",\n    "pillow_version = version(\\"pillow\\")\\n",\n    "pillow_major_minor = tuple(int(part) for part in pillow_version.split(\\".\\")[:2])\\n",\n    "assert (12, 0) <= pillow_major_minor < (13, 0), pillow_version\\n",\n    "dedicated_site = str(Path(os.environ[\\"LA_STUDIO_OCR_SITE\\"]).resolve())\\n",\n    "for package in (PIL, paddle, paddleocr, paddlex, langchain):\\n",\n    "    assert str(Path(package.__file__).resolve()).startswith(dedicated_site), (package.__name__, package.__file__)\\n",\n    "assert Document is not None\\n",\n    "assert paddle.device.is_compiled_with_cuda(), \\"Choose a Colab GPU runtime; CPU fallback is disabled.\\"\\n",\n    "paddle.device.set_device(\\"gpu:0\\")\\n",\n    "probe_path = None\\n",\n    "try:\\n",\n    "    with tempfile.NamedTemporaryFile(prefix=\\"la-studio-subtitle-bootstrap-\\", suffix=\\".png\\", delete=False) as handle:\\n",\n    "        probe_path = Path(handle.name)\\n",\n    "    Image.new(\\"RGB\\", (640, 160), \\"white\\").save(probe_path, format=\\"PNG\\")\\n",\n    "    engine = PaddleOCR(\\n",\n    "        lang=\\"en\\", ocr_version=\\"PP-OCRv5\\", device=\\"gpu:0\\",\\n",\n    "        use_doc_orientation_classify=False,\\n",\n    "        use_doc_unwarping=False,\\n",\n    "        use_textline_orientation=False,\\n",\n    "    )\\n",\n    "    for _ in engine.predict(str(probe_path)):\\n",\n    "        pass\\n",\n    "finally:\\n",\n    "    if probe_path is not None:\\n",\n    "        probe_path.unlink(missing_ok=True)\\n",\n    "print(\\"Verified isolated PP-OCRv5 CUDA inference:\\", paddle.__version__, version(\\"paddlex\\"), version(\\"paddleocr\\"), version(\\"pillow\\"))\\n",\n    "\\"\\"\\"\\n",\n    ")\\n",\n    "OCR_ENV[\\"LA_STUDIO_OCR_SITE\\"] = str(OCR_SITE_PACKAGES)\\n",\n    "probe_result = subprocess.run(\\n",\n    "    [OCR_PYTHON, \\"-c\\", probe], env=OCR_ENV, text=True,\\n",\n    "    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)\\n",\n    "probe_output = probe_result.stdout or \\"\\"\\n",\n    "if probe_output:\\n",\n    "    print(probe_output)\\n",\n    "if probe_result.returncode:\\n",\n    "    raise RuntimeError(\\n",\n    "        \\"LA Studio Subtitle OCR isolated-stack probe failed with exit code \\"\\n",\n    "        + str(probe_result.returncode)\\n",\n    "        + \\". The worker was not launched.\\\\n\\\\n\\"\\n",\n    "        + \\"---- OCR probe output (last 12,000 characters) ----\\\\n\\"\\n",\n    "        + probe_output[-12000:])\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "from pathlib import Path\\n",\n    "\\n",\n    "WORKER = Path(\'/content/la_studio_subtitle_ocr_worker.py\')\\n",\n    "WORKER.write_text(\'import json\\\\nimport os\\\\nimport secrets\\\\nimport tempfile\\\\nimport threading\\\\nfrom pathlib import Path\\\\n\\\\nimport paddle\\\\nfrom fastapi import Depends, FastAPI, File, Form, Header, HTTPException, Request, UploadFile\\\\nfrom fastapi.responses import JSONResponse\\\\nfrom PIL import Image\\\\nfrom paddleocr import PaddleOCR\\\\n\\\\nif not paddle.device.is_compiled_with_cuda():\\\\n    raise RuntimeError(\\"CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.\\")\\\\npaddle.device.set_device(\\"gpu:0\\")\\\\nGPU_NAME = paddle.device.cuda.get_device_name(0)\\\\n\\\\nMODEL_ID = \\"pp-ocrv5-multilingual-3.1\\"\\\\nMODEL_NAME = \\"PP-OCRv5 Multilingual 3.1\\"\\\\nUPSTREAM_MODEL = \\"PaddlePaddle/PaddleOCR PP-OCRv5\\"\\\\nUPSTREAM_VERSION = \\"PaddleOCR 3.1.1\\"\\\\nLICENSE = \\"Apache-2.0\\"\\\\nWORKER_REVISION = \\"subtitle-ocr-2026-08-23.18\\"\\\\nRESPONSE_CONTRACT = \\"subtitle-ocr-crops-v1\\"\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_SUBTITLE_OCR_TOKEN\\"]\\\\nMAX_UPLOAD_BYTES = 16 * 1024 * 1024\\\\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\\\\nENGINE_LOCK = threading.Lock()\\\\nENGINES: dict[str, PaddleOCR] = {}\\\\n\\\\n# The desktop has stored legacy Tesseract codes since Subtitle OCR started.\\\\n# Map them explicitly to the PP-OCRv5 language profiles; unsupported codes are\\\\n# rejected instead of routed to an arbitrary model or a CPU fallback.\\\\nLANGUAGE_PROFILES = {\\\\n    \\"eng\\": \\"en\\", \\"en\\": \\"en\\",\\\\n    \\"vie\\": \\"vi\\", \\"vi\\": \\"vi\\",\\\\n    \\"chi_sim\\": \\"ch\\", \\"chi_tra\\": \\"chinese_cht\\", \\"ch\\": \\"ch\\", \\"zh\\": \\"ch\\",\\\\n    \\"jpn\\": \\"japan\\", \\"ja\\": \\"japan\\",\\\\n    \\"kor\\": \\"korean\\", \\"ko\\": \\"korean\\",\\\\n}\\\\n\\\\n\\\\ndef authorize(authorization: str = Header(default=\\"\\")):\\\\n    if not secrets.compare_digest(authorization, \\"Bearer \\" + TOKEN):\\\\n        raise HTTPException(status_code=401, detail=\\"invalid or missing bearer token\\")\\\\n\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the matching notebook.\\",\\\\n        )\\\\n\\\\n\\\\ndef resolve_profile(language: str) -> str:\\\\n    # A composite Tesseract value (for example eng+chi_sim) is intentionally\\\\n    # rejected. The user must choose one visible OCR language per run, which\\\\n    # makes the actual PP-OCR profile and expected quality unambiguous.\\\\n    normalized = language.strip().lower()\\\\n    if normalized not in LANGUAGE_PROFILES:\\\\n        raise HTTPException(status_code=422, detail=f\\"unsupported Subtitle OCR language: {language}\\")\\\\n    return LANGUAGE_PROFILES[normalized]\\\\n\\\\n\\\\ndef engine_for(profile: str) -> PaddleOCR:\\\\n    with ENGINE_LOCK:\\\\n        engine = ENGINES.get(profile)\\\\n        if engine is None:\\\\n            engine = PaddleOCR(\\\\n                lang=profile,\\\\n                ocr_version=\\"PP-OCRv5\\",\\\\n                device=\\"gpu:0\\",\\\\n                use_doc_orientation_classify=False,\\\\n                use_doc_unwarping=False,\\\\n                use_textline_orientation=False,\\\\n            )\\\\n            ENGINES[profile] = engine\\\\n        return engine\\\\n\\\\n\\\\ndef result_fields(result: object) -> tuple[list[str], list[float]]:\\\\n    # PaddleOCR 3.x result objects expose a JSON-compatible payload. Keep this\\\\n    # adapter tolerant of the documented result wrappers, but never invent text\\\\n    # when the exact model detected none in a sampled subtitle crop.\\\\n    payload = result\\\\n    if hasattr(payload, \\"json\\"):\\\\n        payload = payload.json\\\\n        if callable(payload):\\\\n            payload = payload()\\\\n    if isinstance(payload, str):\\\\n        payload = json.loads(payload)\\\\n    if not isinstance(payload, dict):\\\\n        return [], []\\\\n    data = payload.get(\\"res\\", payload)\\\\n    if not isinstance(data, dict):\\\\n        return [], []\\\\n    texts = data.get(\\"rec_texts\\", data.get(\\"text\\", []))\\\\n    scores = data.get(\\"rec_scores\\", data.get(\\"scores\\", []))\\\\n    if isinstance(texts, str):\\\\n        texts = [texts]\\\\n    if not isinstance(texts, list):\\\\n        texts = []\\\\n    if not isinstance(scores, list):\\\\n        scores = []\\\\n    clean_texts = [str(value).strip() for value in texts if str(value).strip()]\\\\n    clean_scores = [float(value) for value in scores[:len(clean_texts)] if isinstance(value, (int, float))]\\\\n    return clean_texts, clean_scores\\\\n\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Subtitle OCR - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n\\\\n@app.on_event(\\"startup\\")\\\\ndef verify_cuda_inference_before_ready() -> None:\\\\n    \\"\\"\\"Load and execute the exact PP-OCRv5 stack before /health says ready.\\\\n\\\\n    A successful Python import is not enough: a mismatched Paddle/PaddleX\\\\n    installation can still fail only when a CUDA inference pipeline is built.\\\\n    The blank image deliberately has no subtitle text; successful completion\\\\n    proves model construction plus a real GPU inference without inventing OCR\\\\n    output. Other language profiles remain lazy-loaded when requested.\\\\n    \\"\\"\\"\\\\n    probe_path: Path | None = None\\\\n    try:\\\\n        with tempfile.NamedTemporaryFile(prefix=\\"la-studio-subtitle-probe-\\", suffix=\\".png\\", delete=False) as handle:\\\\n            probe_path = Path(handle.name)\\\\n        Image.new(\\"RGB\\", (640, 160), \\"white\\").save(probe_path, format=\\"PNG\\")\\\\n        for _ in engine_for(\\"en\\").predict(str(probe_path)):\\\\n            pass\\\\n        print(\\"PP-OCRv5 CUDA startup inference passed for profile en\\", flush=True)\\\\n    finally:\\\\n        if probe_path is not None:\\\\n            probe_path.unlink(missing_ok=True)\\\\n\\\\n\\\\n@app.exception_handler(Exception)\\\\nasync def unhandled_exception(request: Request, error: Exception):\\\\n    detail = f\\"Subtitle OCR worker internal error: {type(error).__name__}: {str(error)[:300]}\\"\\\\n    print(detail, flush=True)\\\\n    return JSONResponse(status_code=500, content={\\"detail\\": detail})\\\\n\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": GPU_NAME,\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"upstream_version\\": UPSTREAM_VERSION,\\\\n        \\"license\\": LICENSE,\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"response_contract\\": RESPONSE_CONTRACT,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"subtitle-ocr\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"upstream_version\\": UPSTREAM_VERSION,\\\\n                \\"license\\": LICENSE,\\\\n                \\"languages\\": [\\"vi\\", \\"ch\\", \\"japan\\", \\"korean\\", \\"en\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n                \\"response_contract\\": RESPONSE_CONTRACT,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n\\\\n@app.post(\\"/v1/ocr/subtitles\\")\\\\nasync def recognize_subtitle(\\\\n    model: str = Form(...),\\\\n    language: str = Form(...),\\\\n    file: UploadFile = File(...),\\\\n    _: None = Depends(authorize),\\\\n):\\\\n    require_exact_model(model)\\\\n    profile = resolve_profile(language)\\\\n    if file.content_type not in {\\"image/png\\", \\"application/octet-stream\\"}:\\\\n        raise HTTPException(status_code=415, detail=\\"Subtitle OCR accepts only PNG crop frames\\")\\\\n    data = await file.read(MAX_UPLOAD_BYTES + 1)\\\\n    if not data or len(data) > MAX_UPLOAD_BYTES or not data.startswith(b\\"\\\\\\\\x89PNG\\\\\\\\r\\\\\\\\n\\\\\\\\x1a\\\\\\\\n\\"):\\\\n        raise HTTPException(status_code=400, detail=\\"Subtitle OCR crop must be a non-empty PNG no larger than 16 MiB\\")\\\\n    if not INFERENCE_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"worker is busy; retry shortly\\")\\\\n    path = None\\\\n    try:\\\\n        with tempfile.NamedTemporaryFile(prefix=\\"la-studio-subtitle-\\", suffix=\\".png\\", delete=False) as handle:\\\\n            handle.write(data)\\\\n            path = Path(handle.name)\\\\n        texts: list[str] = []\\\\n        scores: list[float] = []\\\\n        for result in engine_for(profile).predict(str(path)):\\\\n            current_texts, current_scores = result_fields(result)\\\\n            texts.extend(current_texts)\\\\n            scores.extend(current_scores)\\\\n        text = \\" \\".join(texts).strip()\\\\n        confidence = sum(scores) / len(scores) if scores else 0.0\\\\n        return {\\"text\\": text, \\"confidence\\": max(0.0, min(1.0, confidence))}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(status_code=503, detail=f\\"PP-OCRv5 inference failed: {type(error).__name__}: {str(error)[:300]}\\") from error\\\\n    finally:\\\\n        if path is not None:\\\\n            path.unlink(missing_ok=True)\\\\n        INFERENCE_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n    "print(\'Worker source:\', WORKER)\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "MODEL_ID = \'pp-ocrv5-multilingual-3.1\'\\n",\n    "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n    "import json\\n",\n    "import os\\n",\n    "import queue\\n",\n    "import re\\n",\n    "import secrets\\n",\n    "import signal\\n",\n    "import socket\\n",\n    "import subprocess\\n",\n    "import sys\\n",\n    "import threading\\n",\n    "import time\\n",\n    "import urllib.error\\n",\n    "import urllib.request\\n",\n    "from pathlib import Path\\n",\n    "\\n",\n    "CAPABILITY_LABEL = \'Subtitle OCR\'\\n",\n    "MODEL_ID = \'pp-ocrv5-multilingual-3.1\'\\n",\n    "PORT = 3955\\n",\n    "TOKEN_ENV = \'LA_STUDIO_COLAB_SUBTITLE_OCR_TOKEN\'\\n",\n    "URL_ENV = \'LA_STUDIO_COLAB_SUBTITLE_OCR_URL\'\\n",\n    "MODEL_ENV = \'LA_STUDIO_COLAB_SUBTITLE_OCR_MODEL\'\\n",\n    "WORKER_LOG = Path(\'/content/la_studio_subtitle_ocr_worker.log\')\\n",\n    "WORKER_MODULE = \'la_studio_subtitle_ocr_worker\'\\n",\n    "WORKER_PYTHON = sys.executable\\n",\n    "WORKER_PYTHON_ISOLATED = False\\n",\n    "WORKER_ENVIRONMENT = {\\"PADDLE_PDX_MODEL_SOURCE\\": \\"BOS\\", \\"PYTHONNOUSERSITE\\": \\"1\\", \\"PYTHONPATH\\": \\"/content/la_studio_subtitle_ocr_site\\"}\\n",\n    "REQUIRES_CUDA = True\\n",\n    "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n    "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n    "TOKEN = secrets.token_urlsafe(32)\\n",\n    "\\n",\n    "\\n",\n    "def port_is_occupied(port: int) -> bool:\\n",\n    "    try:\\n",\n    "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n    "            return True\\n",\n    "    except OSError:\\n",\n    "        return False\\n",\n    "\\n",\n    "\\n",\n    "def process_cmdline(pid: int) -> str:\\n",\n    "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n    "    try:\\n",\n    "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n    "            \\"utf-8\\", errors=\\"replace\\"\\n",\n    "        ).strip()\\n",\n    "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n    "        return \\"\\"\\n",\n    "\\n",\n    "\\n",\n    "def all_processes():\\n",\n    "    for entry in Path(\\"/proc\\").iterdir():\\n",\n    "        if not entry.name.isdigit():\\n",\n    "            continue\\n",\n    "        pid = int(entry.name)\\n",\n    "        command = process_cmdline(pid)\\n",\n    "        if command:\\n",\n    "            yield pid, command\\n",\n    "\\n",\n    "\\n",\n    "def listening_processes(port: int) -> dict[int, str]:\\n",\n    "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n    "    target_port = f\\"{port:04X}\\"\\n",\n    "    socket_inodes = set()\\n",\n    "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n    "        try:\\n",\n    "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n    "        except FileNotFoundError:\\n",\n    "            continue\\n",\n    "        for line in lines:\\n",\n    "            fields = line.split()\\n",\n    "            if len(fields) < 10:\\n",\n    "                continue\\n",\n    "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n    "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n    "                socket_inodes.add(inode)\\n",\n    "    if not socket_inodes:\\n",\n    "        return {}\\n",\n    "\\n",\n    "    listeners = {}\\n",\n    "    for entry in Path(\\"/proc\\").iterdir():\\n",\n    "        if not entry.name.isdigit():\\n",\n    "            continue\\n",\n    "        try:\\n",\n    "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n    "        except (FileNotFoundError, PermissionError):\\n",\n    "            continue\\n",\n    "        for descriptor in descriptors:\\n",\n    "            try:\\n",\n    "                target = os.readlink(descriptor)\\n",\n    "            except (FileNotFoundError, PermissionError, OSError):\\n",\n    "                continue\\n",\n    "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n    "            if match and match.group(1) in socket_inodes:\\n",\n    "                pid = int(entry.name)\\n",\n    "                listeners[pid] = process_cmdline(pid)\\n",\n    "                break\\n",\n    "    return listeners\\n",\n    "\\n",\n    "\\n",\n    "def stop_pid(pid: int) -> None:\\n",\n    "    if pid == os.getpid():\\n",\n    "        return\\n",\n    "    try:\\n",\n    "        os.kill(pid, signal.SIGTERM)\\n",\n    "    except ProcessLookupError:\\n",\n    "        return\\n",\n    "    deadline = time.monotonic() + 10\\n",\n    "    while time.monotonic() < deadline:\\n",\n    "        try:\\n",\n    "            os.kill(pid, 0)\\n",\n    "        except ProcessLookupError:\\n",\n    "            return\\n",\n    "        time.sleep(0.2)\\n",\n    "    try:\\n",\n    "        os.kill(pid, signal.SIGKILL)\\n",\n    "    except ProcessLookupError:\\n",\n    "        pass\\n",\n    "\\n",\n    "\\n",\n    "def reclaim_previous_la_studio_worker() -> None:\\n",\n    "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n    "\\n",\n    "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n    "    created a new token but aborted before it could replace the old worker,\\n",\n    "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n    "    the exact generated module name and never terminate a foreign listener.\\n",\n    "    \\"\\"\\"\\n",\n    "    stopped = []\\n",\n    "    for pid, command in listening_processes(PORT).items():\\n",\n    "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n    "            stop_pid(pid)\\n",\n    "            stopped.append(f\\"worker PID {pid}\\")\\n",\n    "\\n",\n    "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n    "    for pid, command in all_processes():\\n",\n    "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n    "            stop_pid(pid)\\n",\n    "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n    "\\n",\n    "    deadline = time.monotonic() + 12\\n",\n    "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n    "        time.sleep(0.2)\\n",\n    "    if stopped:\\n",\n    "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n    "\\n",\n    "    if port_is_occupied(PORT):\\n",\n    "        listeners = listening_processes(PORT)\\n",\n    "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n    "        raise RuntimeError(\\n",\n    "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n    "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n    "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n    "        )\\n",\n    "\\n",\n    "\\n",\n    "def worker_log_tail() -> str:\\n",\n    "    try:\\n",\n    "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n    "    except FileNotFoundError:\\n",\n    "        return \\"(worker log was not created)\\"\\n",\n    "\\n",\n    "\\n",\n    "def stop_process(process) -> None:\\n",\n    "    if process is None or process.poll() is not None:\\n",\n    "        return\\n",\n    "    process.terminate()\\n",\n    "    try:\\n",\n    "        process.wait(timeout=10)\\n",\n    "    except subprocess.TimeoutExpired:\\n",\n    "        process.kill()\\n",\n    "\\n",\n    "\\n",\n    "reclaim_previous_la_studio_worker()\\n",\n    "\\n",\n    "env = os.environ.copy()\\n",\n    "env[TOKEN_ENV] = TOKEN\\n",\n    "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n    "env.update(WORKER_ENVIRONMENT)\\n",\n    "if WORKER_PYTHON_ISOLATED:\\n",\n    "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n    "    # bleed into a dedicated worker virtual environment.\\n",\n    "    env.pop(\\"PYTHONPATH\\", None)\\n",\n    "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n    "worker = None\\n",\n    "tunnel = None\\n",\n    "\\n",\n    "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n    "    worker = subprocess.Popen(\\n",\n    "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_subtitle_ocr_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n    "        cwd=\\"/content\\",\\n",\n    "        env=env,\\n",\n    "        stdout=worker_output,\\n",\n    "        stderr=subprocess.STDOUT,\\n",\n    "    )\\n",\n    "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n    "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n    "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n    "    last_error = \\"worker has not answered /health yet\\"\\n",\n    "    next_report = time.monotonic()\\n",\n    "    while time.monotonic() < deadline:\\n",\n    "        exit_code = worker.poll()\\n",\n    "        if exit_code is not None:\\n",\n    "            raise RuntimeError(\\n",\n    "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n    "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n    "            )\\n",\n    "        try:\\n",\n    "            request = urllib.request.Request(\\n",\n    "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n    "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n    "            )\\n",\n    "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n    "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n    "            if (response.status == 200\\n",\n    "                    and health.get(\\"ready\\") is True\\n",\n    "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n    "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n    "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n    "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n    "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n    "                break\\n",\n    "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n    "        except urllib.error.HTTPError as error:\\n",\n    "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n    "        except Exception as error:\\n",\n    "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n    "        if time.monotonic() >= next_report:\\n",\n    "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n    "            next_report = time.monotonic() + 30\\n",\n    "        time.sleep(2)\\n",\n    "    else:\\n",\n    "        stop_process(worker)\\n",\n    "        raise RuntimeError(\\n",\n    "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n    "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n    "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n    "        )\\n",\n    "\\n",\n    "\\n",\n    "def cloudflared_ready() -> bool:\\n",\n    "    try:\\n",\n    "        return subprocess.run(\\n",\n    "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n    "            check=False,\\n",\n    "        ).returncode == 0\\n",\n    "    except OSError:\\n",\n    "        return False\\n",\n    "\\n",\n    "\\n",\n    "def ensure_cloudflared() -> None:\\n",\n    "    if cloudflared_ready():\\n",\n    "        return\\n",\n    "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n    "    download = subprocess.run(\\n",\n    "        [\\n",\n    "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n    "            \\"--output\\", package_path,\\n",\n    "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n    "        ],\\n",\n    "        text=True,\\n",\n    "        stdout=subprocess.PIPE,\\n",\n    "        stderr=subprocess.STDOUT,\\n",\n    "        check=False,\\n",\n    "    )\\n",\n    "    if download.returncode != 0:\\n",\n    "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n    "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n    "    install = subprocess.run(\\n",\n    "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n    "        stderr=subprocess.STDOUT, check=False,\\n",\n    "    )\\n",\n    "    if install.returncode != 0 or not cloudflared_ready():\\n",\n    "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n    "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n    "\\n",\n    "\\n",\n    "ensure_cloudflared()\\n",\n    "tunnel = subprocess.Popen(\\n",\n    "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n    "    stdout=subprocess.PIPE,\\n",\n    "    stderr=subprocess.STDOUT,\\n",\n    "    text=True,\\n",\n    "    bufsize=1,\\n",\n    ")\\n",\n    "tunnel_lines = queue.Queue()\\n",\n    "\\n",\n    "\\n",\n    "def collect_tunnel_output() -> None:\\n",\n    "    assert tunnel.stdout is not None\\n",\n    "    for line in tunnel.stdout:\\n",\n    "        tunnel_lines.put(line)\\n",\n    "\\n",\n    "\\n",\n    "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n    "public_url = \\"\\"\\n",\n    "recent_tunnel_lines = []\\n",\n    "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n    "while time.monotonic() < deadline and not public_url:\\n",\n    "    if tunnel.poll() is not None:\\n",\n    "        break\\n",\n    "    try:\\n",\n    "        line = tunnel_lines.get(timeout=1)\\n",\n    "    except queue.Empty:\\n",\n    "        continue\\n",\n    "    recent_tunnel_lines.append(line.rstrip())\\n",\n    "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n    "    print(line, end=\\"\\")\\n",\n    "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n    "    if match:\\n",\n    "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n    "        # bearer-token, capability, and exact-model verification.\\n",\n    "        public_url = match.group(0)\\n",\n    "\\n",\n    "if not public_url:\\n",\n    "    stop_process(tunnel)\\n",\n    "    stop_process(worker)\\n",\n    "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n    "    raise RuntimeError(\\n",\n    "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n    "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n    "    )\\n",\n    "\\n",\n    "os.environ[URL_ENV] = public_url\\n",\n    "os.environ[TOKEN_ENV] = TOKEN\\n",\n    "os.environ[MODEL_ENV] = MODEL_ID\\n",\n    "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n    "print(URL_ENV + \\"=\\" + public_url)\\n",\n    "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n    "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n    "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# ==============================================================================\\n",\n    "# 📥 LƯU FILE TRỰC TIẾP VÀO THƯ MỤC DỰ ÁN TRÊN MÁY TÍNH (FILE SYSTEM ACCESS API)\\n",\n    "# ==============================================================================\\n",\n    "import base64\\n",\n    "import glob\\n",\n    "import json\\n",\n    "import os\\n",\n    "from IPython.display import HTML, display\\n",\n    "\\n",\n    "# Thu thập tất cả các file kết quả vừa tạo\\n",\n    "result_files = {}\\n",\n    "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n    "    for f in glob.glob(pattern):\\n",\n    "        name = os.path.basename(f)\\n",\n    "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n    "            with open(f, \'rb\') as fp:\\n",\n    "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n    "\\n",\n    "if not result_files:\\n",\n    "    print(\\"⚠️ Chưa có file kết quả mới để lưu.\\")\\n",\n    "else:\\n",\n    "    print(f\\"✅ Đã tìm thấy {len(result_files)} file kết quả: {\', \'.join(result_files.keys())}\\")\\n",\n    "    print(\\"👉 Bấm nút bên dưới và chọn thư mục \'LA-Studio/out/colab-live\' để lưu thẳng vào máy:\\")\\n",\n    "    \\n",\n    "    files_json = json.dumps(result_files)\\n",\n    "    html_code = f\\"\\"\\"\\n",\n    "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n    "        📁 Chọn Thư Mục & Lưu File Trực Tiếp Vào Máy\\n",\n    "    </button>\\n",\n    "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n    "    <script>\\n",\n    "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n    "        const log = document.getElementById(\'statusLog\');\\n",\n    "        try {{\\n",\n    "            if (!window.showDirectoryPicker) {{\\n",\n    "                log.innerText = \'Trình duyệt không hỗ trợ File System Access API. Đang dùng tải thông thường...\';\\n",\n    "                return;\\n",\n    "            }}\\n",\n    "            log.innerText = \'Đang mở hộp thoại chọn thư mục...\';\\n",\n    "            const dirHandle = await window.showDirectoryPicker();\\n",\n    "            const files = {files_json};\\n",\n    "            for (const [name, b64] of Object.entries(files)) {{\\n",\n    "                log.innerText = \'Đang ghi file: \' + name + \'...\';\\n",\n    "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n    "                const writable = await fileHandle.createWritable();\\n",\n    "                const byteCharacters = atob(b64);\\n",\n    "                const byteNumbers = new Array(byteCharacters.length);\\n",\n    "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n    "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n    "                }}\\n",\n    "                const byteArray = new Uint8Array(byteNumbers);\\n",\n    "                await writable.write(byteArray);\\n",\n    "                await writable.close();\\n",\n    "            }}\\n",\n    "            log.innerText = \'🎉 Đã lưu thành công toàn bộ file vào thư mục bạn chọn!\';\\n",\n    "        }} catch (err) {{\\n",\n    "            if (err.name !== \'AbortError\') {{\\n",\n    "                log.innerText = \'Lỗi: \' + err.message;\\n",\n    "            }} else {{\\n",\n    "                log.innerText = \'Đã hủy chọn thư mục.\';\\n",\n    "            }}\\n",\n    "        }}\\n",\n    "    }};\\n",\n    "    </script>\\n",\n    "    \\"\\"\\"\\n",\n    "    display(HTML(html_code))\\n"\n   ]\n  }\n ],\n "metadata": {\n  "accelerator": "GPU",\n  "colab": {\n   "gpuType": "T4",\n   "provenance": []\n  },\n  "kernelspec": {\n   "display_name": "Python 3",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python"\n  },\n  "la_studio": {\n   "capability": "subtitle-ocr",\n   "family_id": "pp-ocrv5-multilingual-3.1",\n   "upstream_model": "PaddlePaddle/PaddleOCR PP-OCRv5",\n   "upstream_version": "PaddleOCR 3.1.1",\n   "license": "Apache-2.0",\n   "contract_version": 1,\n   "device": "cuda",\n   "cpu_fallback": false\n  }\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}\n',
    'notebooks/translation/LA_STUDIO_TRANSLATION_HY_MT2_1_8B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio translation \\u2014 Tencent Hy-MT2 1.8B\\n",\n        "\\n",\n        "This notebook loads exactly `hy-mt2-1.8b` (`tencent/Hy-MT2-1.8B`) on CUDA.\\n",\n        "It is independent from API Gateway and rejects every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into the matching LA Studio feature.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"transformers>=5.6.0,<6\\" \\"accelerate>=1.12,<2\\" \\"sentencepiece==0.2.1\\" \\"safetensors>=0.6,<1\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_translation_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport re\\\\nimport secrets\\\\nimport threading\\\\n\\\\nimport torch\\\\nfrom fastapi import Depends, FastAPI, Header, HTTPException\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.\\")\\\\n\\\\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\\\\n\\\\nMODEL_ID = \\"hy-mt2-1.8b\\"\\\\nMODEL_NAME = \\"Tencent Hy-MT2 1.8B\\"\\\\nUPSTREAM_MODEL = \\"tencent/Hy-MT2-1.8B\\"\\\\nUPSTREAM_REVISION = \\"9a341cd1b679d3efd23b46e847b01745a71ed792\\"\\\\nLANGUAGE_NAMES = {\\\\n    \\"zh\\": \\"Chinese\\", \\"en\\": \\"English\\", \\"fr\\": \\"French\\", \\"pt\\": \\"Portuguese\\",\\\\n    \\"es\\": \\"Spanish\\", \\"ja\\": \\"Japanese\\", \\"tr\\": \\"Turkish\\", \\"ru\\": \\"Russian\\",\\\\n    \\"ar\\": \\"Arabic\\", \\"ko\\": \\"Korean\\", \\"th\\": \\"Thai\\", \\"it\\": \\"Italian\\",\\\\n    \\"de\\": \\"German\\", \\"vi\\": \\"Vietnamese\\", \\"ms\\": \\"Malay\\", \\"id\\": \\"Indonesian\\",\\\\n    \\"tl\\": \\"Filipino\\", \\"hi\\": \\"Hindi\\", \\"zh-hant\\": \\"Traditional Chinese\\",\\\\n    \\"pl\\": \\"Polish\\", \\"cs\\": \\"Czech\\", \\"nl\\": \\"Dutch\\", \\"km\\": \\"Khmer\\",\\\\n    \\"my\\": \\"Burmese\\", \\"fa\\": \\"Persian\\", \\"gu\\": \\"Gujarati\\", \\"ur\\": \\"Urdu\\",\\\\n    \\"te\\": \\"Telugu\\", \\"mr\\": \\"Marathi\\", \\"he\\": \\"Hebrew\\", \\"bn\\": \\"Bengali\\",\\\\n    \\"ta\\": \\"Tamil\\", \\"uk\\": \\"Ukrainian\\", \\"bo\\": \\"Tibetan\\", \\"kk\\": \\"Kazakh\\",\\\\n    \\"mn\\": \\"Mongolian\\", \\"ug\\": \\"Uyghur\\", \\"yue\\": \\"Cantonese\\",\\\\n}\\\\nSUPPORTED_LANGUAGES = list(LANGUAGE_NAMES)\\\\n\\\\nTOKENIZER = AutoTokenizer.from_pretrained(\\\\n    UPSTREAM_MODEL, revision=UPSTREAM_REVISION, trust_remote_code=True\\\\n)\\\\nMODEL = AutoModelForCausalLM.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    revision=UPSTREAM_REVISION,\\\\n    dtype=torch.bfloat16,\\\\n    device_map={\\"\\": 0},\\\\n    trust_remote_code=True,\\\\n    low_cpu_mem_usage=True,\\\\n).eval()\\\\n\\\\ndef translate_exact(texts: list[str], source: str, target: str) -> list[str]:\\\\n    source_key = source.lower()\\\\n    target_key = target.lower()\\\\n    if source_key not in LANGUAGE_NAMES or target_key not in LANGUAGE_NAMES:\\\\n        raise HTTPException(status_code=422, detail=f\\"unsupported Hy-MT2 language pair: {source} -> {target}\\")\\\\n    results = []\\\\n    for text in texts:\\\\n        prompt = (\\\\n            f\\"Translate the following text from {LANGUAGE_NAMES[source_key]} into {LANGUAGE_NAMES[target_key]}. \\"\\\\n            \\"Only output the translated result without any additional explanation:\\\\\\\\n\\"\\\\n            f\\"{text}\\"\\\\n        )\\\\n        inputs = TOKENIZER.apply_chat_template(\\\\n            [{\\"role\\": \\"user\\", \\"content\\": prompt}],\\\\n            add_generation_prompt=True,\\\\n            return_tensors=\\"pt\\",\\\\n            return_dict=True,\\\\n        ).to(\\"cuda\\")\\\\n        with torch.inference_mode():\\\\n            output = MODEL.generate(\\\\n                **inputs,\\\\n                max_new_tokens=512,\\\\n                # Translation needs reproducible decoding. Sampling can end at\\\\n                # EOS immediately and was the only adapter that could emit an\\\\n                # empty response nondeterministically for the same segment.\\\\n                do_sample=False,\\\\n            )\\\\n        generated = output[0][inputs[\\"input_ids\\"].shape[-1]:]\\\\n        results.append(TOKENIZER.decode(generated, skip_special_tokens=True).strip())\\\\n    return results\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TRANSLATION_TOKEN\\"]\\\\nWORKER_REVISION = \\"translation-2026-07-30.3\\"\\\\nRESPONSE_CONTRACT = \\"translation-patches-v3\\"\\\\nMAX_TRANSLATION_SEGMENTS = 128\\\\nMAX_TRANSLATION_CHARS = 50000\\\\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\n# A translation result must never make the rest of a dubbing job disappear.\\\\n# The worker retries blank model output once, then preserves the source in a\\\\n# visible needs-review patch. This is deliberately not reported as a completed\\\\n# translation: the editor can show it for review while later segments continue.\\\\nNONLEXICAL_UTTERANCES = {\\\\n    \\"ah\\", \\"aha\\", \\"eh\\", \\"er\\", \\"ha\\", \\"haha\\", \\"heh\\", \\"hmm\\", \\"hm\\", \\"ho\\", \\"oh\\",\\\\n    \\"uh\\", \\"um\\", \\"wow\\", \\"\\u55ef\\", \\"\\u55ef\\u54fc\\", \\"\\u554a\\", \\"\\u554a\\u54c8\\", \\"\\u54ce\\", \\"\\u54ce\\u5440\\", \\"\\u8bf6\\", \\"\\u6b38\\",\\\\n    \\"\\u54c8\\", \\"\\u54c8\\u54c8\\", \\"\\u5475\\", \\"\\u5475\\u5475\\", \\"\\u563f\\", \\"\\u54fc\\", \\"\\u5509\\", \\"\\u5440\\", \\"\\u5443\\", \\"\\u54e6\\", \\"\\u54e6\\u54e6\\",\\\\n    \\"\\u5594\\", \\"\\u5662\\", \\"\\u54c7\\", \\"\\u5514\\",\\\\n}\\\\n\\\\ndef is_nonlexical_utterance(text: str) -> bool:\\\\n    normalized = re.sub(r\\"[^\\\\\\\\w]\\", \\"\\", text, flags=re.UNICODE).casefold()\\\\n    return normalized in NONLEXICAL_UTTERANCES\\\\n\\\\ndef retry_empty_translations(texts: list[str], translated: list[str], source: str, target: str) -> list[str]:\\\\n    if not isinstance(translated, list) or len(translated) != len(texts):\\\\n        return translated\\\\n    empty_indices = [\\\\n        index for index, value in enumerate(translated)\\\\n        if not isinstance(value, str) or not value.strip()\\\\n    ]\\\\n    if not empty_indices:\\\\n        return translated\\\\n    # Retry only the affected source strings with the same selected, pinned\\\\n    # model. If the retry itself fails, the patch builder below keeps the source\\\\n    # and flags it for review instead of terminating unrelated segments.\\\\n    try:\\\\n        retry_values = translate_exact([texts[index] for index in empty_indices], source, target)\\\\n    except Exception:\\\\n        return translated\\\\n    if not isinstance(retry_values, list) or len(retry_values) != len(empty_indices):\\\\n        return translated\\\\n    for index, retry_value in zip(empty_indices, retry_values):\\\\n        if isinstance(retry_value, str) and retry_value.strip():\\\\n            translated[index] = retry_value\\\\n    return translated\\\\n\\\\ndef make_translation_patches(segments: list[\\"TranslationSegment\\"], translated: list[str]) -> list[dict]:\\\\n    if len(translated) != len(segments):\\\\n        raise RuntimeError(\\"model returned a different number of translations\\")\\\\n    patches = []\\\\n    for index, (item, value) in enumerate(zip(segments, translated), start=1):\\\\n        target = value.strip() if isinstance(value, str) else \\"\\"\\\\n        if target:\\\\n            patches.append({\\"id\\": item.id, \\"targetText\\": target, \\"state\\": \\"translated\\"})\\\\n            continue\\\\n        source = item.sourceText.strip()\\\\n        if is_nonlexical_utterance(source):\\\\n            patches.append({\\\\n                \\"id\\": item.id,\\\\n                \\"targetText\\": source,\\\\n                \\"state\\": \\"needs-review\\",\\\\n                \\"translationDiagnostic\\": (\\\\n                    \\"The exact translation model returned no lexical text for this short vocal reaction; \\"\\\\n                    \\"the source was preserved for review.\\"\\\\n                ),\\\\n            })\\\\n            continue\\\\n        patches.append({\\\\n            \\"id\\": item.id,\\\\n            \\"targetText\\": source,\\\\n            \\"state\\": \\"needs-review\\",\\\\n            \\"translationDiagnostic\\": (\\\\n                \\"The exact translation model returned no text after retry; the source was preserved \\"\\\\n                \\"and later segments continued. Review this segment before export.\\"\\\\n            ),\\\\n        })\\\\n    return patches\\\\n\\\\ndef authorize(authorization: str = Header(default=\\"\\")):\\\\n    if not secrets.compare_digest(authorization, \\"Bearer \\" + TOKEN):\\\\n        raise HTTPException(status_code=401, detail=\\"invalid or missing bearer token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\nclass TranslationSegment(BaseModel):\\\\n    id: str = Field(min_length=1, max_length=128)\\\\n    sourceText: str = Field(min_length=1, max_length=5000)\\\\n\\\\nclass TranslationRequest(BaseModel):\\\\n    model: str\\\\n    source_language: str = Field(min_length=2, max_length=12)\\\\n    target_language: str = Field(min_length=2, max_length=12)\\\\n    segments: list[TranslationSegment]\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Translation - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"response_contract\\": RESPONSE_CONTRACT,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"response_contract\\": RESPONSE_CONTRACT,\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"translation\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n                \\"response_contract\\": RESPONSE_CONTRACT,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/translations\\")\\\\ndef translate(request: TranslationRequest, _: None = Depends(authorize)):\\\\n    require_exact_model(request.model)\\\\n    if not request.segments:\\\\n        raise HTTPException(status_code=400, detail=\\"segments must not be empty\\")\\\\n    texts = [item.sourceText.strip() for item in request.segments]\\\\n    if any(not text for text in texts):\\\\n        raise HTTPException(status_code=400, detail=\\"each segment needs sourceText\\")\\\\n    if len(texts) > MAX_TRANSLATION_SEGMENTS or sum(map(len, texts)) > MAX_TRANSLATION_CHARS:\\\\n        raise HTTPException(status_code=413, detail=\\"translation request is too large\\")\\\\n    if not INFERENCE_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"worker is busy; retry shortly\\")\\\\n    try:\\\\n        translated = translate_exact(\\\\n            texts,\\\\n            request.source_language.strip(),\\\\n            request.target_language.strip(),\\\\n        )\\\\n        translated = retry_empty_translations(\\\\n            texts,\\\\n            translated,\\\\n            request.source_language.strip(),\\\\n            request.target_language.strip(),\\\\n        )\\\\n        return {\\"patches\\": make_translation_patches(request.segments, translated)}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} translation failed: {type(error).__name__}: {str(error)[:300]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        INFERENCE_SLOTS.release()\' + \'\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'hy-mt2-1.8b\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'translation\'\\n",\n        "MODEL_ID = \'hy-mt2-1.8b\'\\n",\n        "PORT = 3943\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TRANSLATION_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TRANSLATION_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TRANSLATION_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_translation_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_translation_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_translation_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "translation",\n      "family_id": "hy-mt2-1.8b",\n      "upstream_model": "tencent/Hy-MT2-1.8B",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/translation/LA_STUDIO_TRANSLATION_M2M100_418M_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio translation \\u2014 M2M-100 418M\\n",\n        "\\n",\n        "This notebook loads exactly `m2m100-418m` (`facebook/m2m100_418M`) on CUDA.\\n",\n        "It is independent from API Gateway and rejects every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into the matching LA Studio feature.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"transformers==4.57.3\\" \\"accelerate==1.12.0\\" \\"sentencepiece==0.2.1\\" \\"safetensors==0.6.2\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_translation_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport re\\\\nimport secrets\\\\nimport threading\\\\n\\\\nimport torch\\\\nfrom fastapi import Depends, FastAPI, Header, HTTPException\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.\\")\\\\n\\\\nfrom transformers import M2M100ForConditionalGeneration, M2M100Tokenizer\\\\n\\\\nMODEL_ID = \\"m2m100-418m\\"\\\\nMODEL_NAME = \\"M2M-100 418M\\"\\\\nUPSTREAM_MODEL = \\"facebook/m2m100_418M\\"\\\\nUPSTREAM_REVISION = \\"55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636\\"\\\\nSUPPORTED_LANGUAGES = \\"101 M2M100 language codes\\"\\\\n\\\\nTOKENIZER = M2M100Tokenizer.from_pretrained(UPSTREAM_MODEL, revision=UPSTREAM_REVISION)\\\\nMODEL = M2M100ForConditionalGeneration.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    revision=UPSTREAM_REVISION,\\\\n    torch_dtype=torch.float16,\\\\n    low_cpu_mem_usage=True,\\\\n).to(\\"cuda\\").eval()\\\\n\\\\ndef translate_exact(texts: list[str], source: str, target: str) -> list[str]:\\\\n    source = source.lower()\\\\n    target = target.lower()\\\\n    try:\\\\n        TOKENIZER.src_lang = source\\\\n        target_id = TOKENIZER.get_lang_id(target)\\\\n    except KeyError as error:\\\\n        raise HTTPException(status_code=422, detail=f\\"unsupported M2M100 language pair: {source} -> {target}\\") from error\\\\n    def generate(batch: list[str], **generation_options: object) -> list[str]:\\\\n        inputs = TOKENIZER(batch, return_tensors=\\"pt\\", padding=True, truncation=True, max_length=512).to(\\"cuda\\")\\\\n        with torch.inference_mode():\\\\n            output = MODEL.generate(\\\\n                **inputs,\\\\n                forced_bos_token_id=target_id,\\\\n                max_new_tokens=512,\\\\n                **generation_options,\\\\n            )\\\\n        return TOKENIZER.batch_decode(output, skip_special_tokens=True)\\\\n\\\\n    translated = generate(texts)\\\\n    # Greedy decoding can terminate at EOS immediately for a noisy but valid\\\\n    # ASR segment. Retry only those blanks with the same pinned M2M checkpoint\\\\n    # and a deterministic beam search; this is not a source-text fallback.\\\\n    for index, value in enumerate(translated):\\\\n        if isinstance(value, str) and value.strip():\\\\n            continue\\\\n        retry_text = \\" \\".join(texts[index].split())\\\\n        retry = generate(\\\\n            [retry_text],\\\\n            num_beams=4,\\\\n            min_new_tokens=1,\\\\n            early_stopping=True,\\\\n            repetition_penalty=1.05,\\\\n        )\\\\n        translated[index] = retry[0] if retry else \\"\\"\\\\n    return translated\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TRANSLATION_TOKEN\\"]\\\\nWORKER_REVISION = \\"translation-2026-07-30.3\\"\\\\nRESPONSE_CONTRACT = \\"translation-patches-v3\\"\\\\nMAX_TRANSLATION_SEGMENTS = 128\\\\nMAX_TRANSLATION_CHARS = 50000\\\\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\n# A translation result must never make the rest of a dubbing job disappear.\\\\n# The worker retries blank model output once, then preserves the source in a\\\\n# visible needs-review patch. This is deliberately not reported as a completed\\\\n# translation: the editor can show it for review while later segments continue.\\\\nNONLEXICAL_UTTERANCES = {\\\\n    \\"ah\\", \\"aha\\", \\"eh\\", \\"er\\", \\"ha\\", \\"haha\\", \\"heh\\", \\"hmm\\", \\"hm\\", \\"ho\\", \\"oh\\",\\\\n    \\"uh\\", \\"um\\", \\"wow\\", \\"\\u55ef\\", \\"\\u55ef\\u54fc\\", \\"\\u554a\\", \\"\\u554a\\u54c8\\", \\"\\u54ce\\", \\"\\u54ce\\u5440\\", \\"\\u8bf6\\", \\"\\u6b38\\",\\\\n    \\"\\u54c8\\", \\"\\u54c8\\u54c8\\", \\"\\u5475\\", \\"\\u5475\\u5475\\", \\"\\u563f\\", \\"\\u54fc\\", \\"\\u5509\\", \\"\\u5440\\", \\"\\u5443\\", \\"\\u54e6\\", \\"\\u54e6\\u54e6\\",\\\\n    \\"\\u5594\\", \\"\\u5662\\", \\"\\u54c7\\", \\"\\u5514\\",\\\\n}\\\\n\\\\ndef is_nonlexical_utterance(text: str) -> bool:\\\\n    normalized = re.sub(r\\"[^\\\\\\\\w]\\", \\"\\", text, flags=re.UNICODE).casefold()\\\\n    return normalized in NONLEXICAL_UTTERANCES\\\\n\\\\ndef retry_empty_translations(texts: list[str], translated: list[str], source: str, target: str) -> list[str]:\\\\n    if not isinstance(translated, list) or len(translated) != len(texts):\\\\n        return translated\\\\n    empty_indices = [\\\\n        index for index, value in enumerate(translated)\\\\n        if not isinstance(value, str) or not value.strip()\\\\n    ]\\\\n    if not empty_indices:\\\\n        return translated\\\\n    # Retry only the affected source strings with the same selected, pinned\\\\n    # model. If the retry itself fails, the patch builder below keeps the source\\\\n    # and flags it for review instead of terminating unrelated segments.\\\\n    try:\\\\n        retry_values = translate_exact([texts[index] for index in empty_indices], source, target)\\\\n    except Exception:\\\\n        return translated\\\\n    if not isinstance(retry_values, list) or len(retry_values) != len(empty_indices):\\\\n        return translated\\\\n    for index, retry_value in zip(empty_indices, retry_values):\\\\n        if isinstance(retry_value, str) and retry_value.strip():\\\\n            translated[index] = retry_value\\\\n    return translated\\\\n\\\\ndef make_translation_patches(segments: list[\\"TranslationSegment\\"], translated: list[str]) -> list[dict]:\\\\n    if len(translated) != len(segments):\\\\n        raise RuntimeError(\\"model returned a different number of translations\\")\\\\n    patches = []\\\\n    for index, (item, value) in enumerate(zip(segments, translated), start=1):\\\\n        target = value.strip() if isinstance(value, str) else \\"\\"\\\\n        if target:\\\\n            patches.append({\\"id\\": item.id, \\"targetText\\": target, \\"state\\": \\"translated\\"})\\\\n            continue\\\\n        source = item.sourceText.strip()\\\\n        if is_nonlexical_utterance(source):\\\\n            patches.append({\\\\n                \\"id\\": item.id,\\\\n                \\"targetText\\": source,\\\\n                \\"state\\": \\"needs-review\\",\\\\n                \\"translationDiagnostic\\": (\\\\n                    \\"The exact translation model returned no lexical text for this short vocal reaction; \\"\\\\n                    \\"the source was preserved for review.\\"\\\\n                ),\\\\n            })\\\\n            continue\\\\n        patches.append({\\\\n            \\"id\\": item.id,\\\\n            \\"targetText\\": source,\\\\n            \\"state\\": \\"needs-review\\",\\\\n            \\"translationDiagnostic\\": (\\\\n                \\"The exact translation model returned no text after retry; the source was preserved \\"\\\\n                \\"and later segments continued. Review this segment before export.\\"\\\\n            ),\\\\n        })\\\\n    return patches\\\\n\\\\ndef authorize(authorization: str = Header(default=\\"\\")):\\\\n    if not secrets.compare_digest(authorization, \\"Bearer \\" + TOKEN):\\\\n        raise HTTPException(status_code=401, detail=\\"invalid or missing bearer token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\nclass TranslationSegment(BaseModel):\\\\n    id: str = Field(min_length=1, max_length=128)\\\\n    sourceText: str = Field(min_length=1, max_length=5000)\\\\n\\\\nclass TranslationRequest(BaseModel):\\\\n    model: str\\\\n    source_language: str = Field(min_length=2, max_length=12)\\\\n    target_language: str = Field(min_length=2, max_length=12)\\\\n    segments: list[TranslationSegment]\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Translation - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"response_contract\\": RESPONSE_CONTRACT,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"response_contract\\": RESPONSE_CONTRACT,\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"translation\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n                \\"response_contract\\": RESPONSE_CONTRACT,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/translations\\")\\\\ndef translate(request: TranslationRequest, _: None = Depends(authorize)):\\\\n    require_exact_model(request.model)\\\\n    if not request.segments:\\\\n        raise HTTPException(status_code=400, detail=\\"segments must not be empty\\")\\\\n    texts = [item.sourceText.strip() for item in request.segments]\\\\n    if any(not text for text in texts):\\\\n        raise HTTPException(status_code=400, detail=\\"each segment needs sourceText\\")\\\\n    if len(texts) > MAX_TRANSLATION_SEGMENTS or sum(map(len, texts)) > MAX_TRANSLATION_CHARS:\\\\n        raise HTTPException(status_code=413, detail=\\"translation request is too large\\")\\\\n    if not INFERENCE_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"worker is busy; retry shortly\\")\\\\n    try:\\\\n        translated = translate_exact(\\\\n            texts,\\\\n            request.source_language.strip(),\\\\n            request.target_language.strip(),\\\\n        )\\\\n        translated = retry_empty_translations(\\\\n            texts,\\\\n            translated,\\\\n            request.source_language.strip(),\\\\n            request.target_language.strip(),\\\\n        )\\\\n        return {\\"patches\\": make_translation_patches(request.segments, translated)}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} translation failed: {type(error).__name__}: {str(error)[:300]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        INFERENCE_SLOTS.release()\' + \'\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'m2m100-418m\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'translation\'\\n",\n        "MODEL_ID = \'m2m100-418m\'\\n",\n        "PORT = 3943\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TRANSLATION_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TRANSLATION_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TRANSLATION_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_translation_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_translation_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_translation_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "translation",\n      "family_id": "m2m100-418m",\n      "upstream_model": "facebook/m2m100_418M",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/translation/LA_STUDIO_TRANSLATION_MADLAD400_3B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio translation \\u2014 MADLAD-400 3B MT\\n",\n        "\\n",\n        "This notebook loads exactly `madlad400-3b-mt` (`google/madlad400-3b-mt`) on CUDA.\\n",\n        "It is independent from API Gateway and rejects every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into the matching LA Studio feature.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"transformers==4.57.3\\" \\"accelerate==1.12.0\\" \\"sentencepiece==0.2.1\\" \\"safetensors==0.6.2\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_translation_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport re\\\\nimport secrets\\\\nimport threading\\\\n\\\\nimport torch\\\\nfrom fastapi import Depends, FastAPI, Header, HTTPException\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.\\")\\\\n\\\\nfrom transformers import AutoModelForSeq2SeqLM, AutoTokenizer\\\\n\\\\nMODEL_ID = \\"madlad400-3b-mt\\"\\\\nMODEL_NAME = \\"MADLAD-400 3B MT\\"\\\\nUPSTREAM_MODEL = \\"google/madlad400-3b-mt\\"\\\\nUPSTREAM_REVISION = \\"fa184c675da0b5c9e1c8694fccd4e12e2d422094\\"\\\\nSUPPORTED_LANGUAGES = \\"419 MADLAD language codes\\"\\\\n\\\\nTOKENIZER = AutoTokenizer.from_pretrained(UPSTREAM_MODEL, revision=UPSTREAM_REVISION)\\\\nMODEL = AutoModelForSeq2SeqLM.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    revision=UPSTREAM_REVISION,\\\\n    torch_dtype=torch.float16,\\\\n    low_cpu_mem_usage=True,\\\\n).to(\\"cuda\\").eval()\\\\n\\\\ndef translate_exact(texts: list[str], source: str, target: str) -> list[str]:\\\\n    del source\\\\n    target = target.lower()\\\\n    tag = f\\"<2{target}>\\"\\\\n    if TOKENIZER.convert_tokens_to_ids(tag) == TOKENIZER.unk_token_id:\\\\n        raise HTTPException(status_code=422, detail=f\\"unsupported MADLAD target language: {target}\\")\\\\n    prompts = [f\\"{tag} {text}\\" for text in texts]\\\\n    inputs = TOKENIZER(prompts, return_tensors=\\"pt\\", padding=True, truncation=True, max_length=512).to(\\"cuda\\")\\\\n    with torch.inference_mode():\\\\n        output = MODEL.generate(**inputs, max_new_tokens=512)\\\\n    return TOKENIZER.batch_decode(output, skip_special_tokens=True)\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TRANSLATION_TOKEN\\"]\\\\nWORKER_REVISION = \\"translation-2026-07-30.3\\"\\\\nRESPONSE_CONTRACT = \\"translation-patches-v3\\"\\\\nMAX_TRANSLATION_SEGMENTS = 128\\\\nMAX_TRANSLATION_CHARS = 50000\\\\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\n# A translation result must never make the rest of a dubbing job disappear.\\\\n# The worker retries blank model output once, then preserves the source in a\\\\n# visible needs-review patch. This is deliberately not reported as a completed\\\\n# translation: the editor can show it for review while later segments continue.\\\\nNONLEXICAL_UTTERANCES = {\\\\n    \\"ah\\", \\"aha\\", \\"eh\\", \\"er\\", \\"ha\\", \\"haha\\", \\"heh\\", \\"hmm\\", \\"hm\\", \\"ho\\", \\"oh\\",\\\\n    \\"uh\\", \\"um\\", \\"wow\\", \\"\\u55ef\\", \\"\\u55ef\\u54fc\\", \\"\\u554a\\", \\"\\u554a\\u54c8\\", \\"\\u54ce\\", \\"\\u54ce\\u5440\\", \\"\\u8bf6\\", \\"\\u6b38\\",\\\\n    \\"\\u54c8\\", \\"\\u54c8\\u54c8\\", \\"\\u5475\\", \\"\\u5475\\u5475\\", \\"\\u563f\\", \\"\\u54fc\\", \\"\\u5509\\", \\"\\u5440\\", \\"\\u5443\\", \\"\\u54e6\\", \\"\\u54e6\\u54e6\\",\\\\n    \\"\\u5594\\", \\"\\u5662\\", \\"\\u54c7\\", \\"\\u5514\\",\\\\n}\\\\n\\\\ndef is_nonlexical_utterance(text: str) -> bool:\\\\n    normalized = re.sub(r\\"[^\\\\\\\\w]\\", \\"\\", text, flags=re.UNICODE).casefold()\\\\n    return normalized in NONLEXICAL_UTTERANCES\\\\n\\\\ndef retry_empty_translations(texts: list[str], translated: list[str], source: str, target: str) -> list[str]:\\\\n    if not isinstance(translated, list) or len(translated) != len(texts):\\\\n        return translated\\\\n    empty_indices = [\\\\n        index for index, value in enumerate(translated)\\\\n        if not isinstance(value, str) or not value.strip()\\\\n    ]\\\\n    if not empty_indices:\\\\n        return translated\\\\n    # Retry only the affected source strings with the same selected, pinned\\\\n    # model. If the retry itself fails, the patch builder below keeps the source\\\\n    # and flags it for review instead of terminating unrelated segments.\\\\n    try:\\\\n        retry_values = translate_exact([texts[index] for index in empty_indices], source, target)\\\\n    except Exception:\\\\n        return translated\\\\n    if not isinstance(retry_values, list) or len(retry_values) != len(empty_indices):\\\\n        return translated\\\\n    for index, retry_value in zip(empty_indices, retry_values):\\\\n        if isinstance(retry_value, str) and retry_value.strip():\\\\n            translated[index] = retry_value\\\\n    return translated\\\\n\\\\ndef make_translation_patches(segments: list[\\"TranslationSegment\\"], translated: list[str]) -> list[dict]:\\\\n    if len(translated) != len(segments):\\\\n        raise RuntimeError(\\"model returned a different number of translations\\")\\\\n    patches = []\\\\n    for index, (item, value) in enumerate(zip(segments, translated), start=1):\\\\n        target = value.strip() if isinstance(value, str) else \\"\\"\\\\n        if target:\\\\n            patches.append({\\"id\\": item.id, \\"targetText\\": target, \\"state\\": \\"translated\\"})\\\\n            continue\\\\n        source = item.sourceText.strip()\\\\n        if is_nonlexical_utterance(source):\\\\n            patches.append({\\\\n                \\"id\\": item.id,\\\\n                \\"targetText\\": source,\\\\n                \\"state\\": \\"needs-review\\",\\\\n                \\"translationDiagnostic\\": (\\\\n                    \\"The exact translation model returned no lexical text for this short vocal reaction; \\"\\\\n                    \\"the source was preserved for review.\\"\\\\n                ),\\\\n            })\\\\n            continue\\\\n        patches.append({\\\\n            \\"id\\": item.id,\\\\n            \\"targetText\\": source,\\\\n            \\"state\\": \\"needs-review\\",\\\\n            \\"translationDiagnostic\\": (\\\\n                \\"The exact translation model returned no text after retry; the source was preserved \\"\\\\n                \\"and later segments continued. Review this segment before export.\\"\\\\n            ),\\\\n        })\\\\n    return patches\\\\n\\\\ndef authorize(authorization: str = Header(default=\\"\\")):\\\\n    if not secrets.compare_digest(authorization, \\"Bearer \\" + TOKEN):\\\\n        raise HTTPException(status_code=401, detail=\\"invalid or missing bearer token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\nclass TranslationSegment(BaseModel):\\\\n    id: str = Field(min_length=1, max_length=128)\\\\n    sourceText: str = Field(min_length=1, max_length=5000)\\\\n\\\\nclass TranslationRequest(BaseModel):\\\\n    model: str\\\\n    source_language: str = Field(min_length=2, max_length=12)\\\\n    target_language: str = Field(min_length=2, max_length=12)\\\\n    segments: list[TranslationSegment]\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Translation - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"response_contract\\": RESPONSE_CONTRACT,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(_: None = Depends(authorize)):\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"response_contract\\": RESPONSE_CONTRACT,\\\\n        \\"worker_revision\\": WORKER_REVISION,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"translation\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n                \\"response_contract\\": RESPONSE_CONTRACT,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/translations\\")\\\\ndef translate(request: TranslationRequest, _: None = Depends(authorize)):\\\\n    require_exact_model(request.model)\\\\n    if not request.segments:\\\\n        raise HTTPException(status_code=400, detail=\\"segments must not be empty\\")\\\\n    texts = [item.sourceText.strip() for item in request.segments]\\\\n    if any(not text for text in texts):\\\\n        raise HTTPException(status_code=400, detail=\\"each segment needs sourceText\\")\\\\n    if len(texts) > MAX_TRANSLATION_SEGMENTS or sum(map(len, texts)) > MAX_TRANSLATION_CHARS:\\\\n        raise HTTPException(status_code=413, detail=\\"translation request is too large\\")\\\\n    if not INFERENCE_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"worker is busy; retry shortly\\")\\\\n    try:\\\\n        translated = translate_exact(\\\\n            texts,\\\\n            request.source_language.strip(),\\\\n            request.target_language.strip(),\\\\n        )\\\\n        translated = retry_empty_translations(\\\\n            texts,\\\\n            translated,\\\\n            request.source_language.strip(),\\\\n            request.target_language.strip(),\\\\n        )\\\\n        return {\\"patches\\": make_translation_patches(request.segments, translated)}\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} translation failed: {type(error).__name__}: {str(error)[:300]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        INFERENCE_SLOTS.release()\' + \'\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'madlad400-3b-mt\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'translation\'\\n",\n        "MODEL_ID = \'madlad400-3b-mt\'\\n",\n        "PORT = 3943\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TRANSLATION_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TRANSLATION_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TRANSLATION_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_translation_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_translation_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_translation_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "translation",\n      "family_id": "madlad400-3b-mt",\n      "upstream_model": "google/madlad400-3b-mt",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/tts/LA_STUDIO_TTS_KOKORO_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 Kokoro 82M\\n",\n        "\\n",\n        "This notebook loads exactly `kokoro` (`hexgrad/Kokoro-82M`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "!apt-get -qq update && apt-get -qq install -y espeak-ng\\n",\n        "%pip install -q \\"git+https://github.com/hexgrad/kokoro.git@dfb907a02bba\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nfrom functools import lru_cache\\\\nfrom kokoro import KPipeline\\\\n\\\\nMODEL_ID = \\"kokoro\\"\\\\nMODEL_NAME = \\"Kokoro 82M\\"\\\\nUPSTREAM_MODEL = \\"hexgrad/Kokoro-82M\\"\\\\nLANGUAGE_CODES = {\\"en\\": \\"a\\", \\"en-us\\": \\"a\\", \\"en-gb\\": \\"b\\", \\"ja\\": \\"j\\", \\"zh\\": \\"z\\", \\"es\\": \\"e\\", \\"fr\\": \\"f\\", \\"hi\\": \\"h\\", \\"it\\": \\"i\\", \\"pt-br\\": \\"p\\"}\\\\nSUPPORTED_LANGUAGES = sorted(LANGUAGE_CODES)\\\\nSUPPORTED_VOICES = [\\"af_heart\\", \\"af_bella\\", \\"af_nicole\\", \\"am_adam\\", \\"am_michael\\", \\"bf_emma\\", \\"bm_george\\"]\\\\n\\\\n@lru_cache(maxsize=10)\\\\ndef pipeline_for(code: str):\\\\n    pipeline = KPipeline(lang_code=code)\\\\n    moved = False\\\\n    for attribute in (\\"model\\", \\"kokoro_model\\"):\\\\n        candidate = getattr(pipeline, attribute, None)\\\\n        if hasattr(candidate, \\"to\\"):\\\\n            candidate.to(\\"cuda\\")\\\\n            moved = True\\\\n    if not moved:\\\\n        raise RuntimeError(\\"Kokoro did not expose a CUDA-movable model\\")\\\\n    return pipeline\\\\n\\\\npipeline_for(\\"a\\")\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    language = request.language.strip().lower() or \\"en\\"\\\\n    code = LANGUAGE_CODES.get(language)\\\\n    if not code:\\\\n        raise HTTPException(status_code=422, detail=\\"unsupported Kokoro language\\")\\\\n    voice = request.voice.strip().lower() or \\"af_heart\\"\\\\n    if voice not in SUPPORTED_VOICES:\\\\n        raise HTTPException(status_code=422, detail=\\"unsupported Kokoro voice\\")\\\\n    chunks = [\\\\n        np.asarray(audio, dtype=np.float32).reshape(-1)\\\\n        for _, _, audio in pipeline_for(code)(request.input, voice=voice, speed=request.speed)\\\\n    ]\\\\n    chunks = [chunk for chunk in chunks if chunk.size]\\\\n    if not chunks:\\\\n        raise RuntimeError(\\"Kokoro returned no chunks\\")\\\\n    return np.concatenate(chunks), 24000\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'kokoro\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "kokoro",\n      "upstream_model": "hexgrad/Kokoro-82M",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/tts/LA_STUDIO_TTS_KOKORO_VIETNAMESE_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 Kokoro Vietnamese\\n",\n        "\\n",\n        "This notebook loads exactly `kokoro-vietnamese` (`contextboxai/Kokoro-Vietnamese`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"git+https://github.com/iamdinhthuan/Kokoro-Vietnamese.git@a249afe5555a\\" \\"onnxruntime-gpu==1.22.0\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nfrom functools import lru_cache\\\\nfrom kokoro_vietnamese.core import list_voices\\\\nfrom kokoro_vietnamese.onnx_cli import KokoroVietnameseONNX\\\\n\\\\nMODEL_ID = \\"kokoro-vietnamese\\"\\\\nMODEL_NAME = \\"Kokoro Vietnamese\\"\\\\nUPSTREAM_MODEL = \\"contextboxai/Kokoro-Vietnamese\\"\\\\nSUPPORTED_LANGUAGES = [\\"vi\\"]\\\\nSUPPORTED_VOICES = list(list_voices()) or [\\"diem_trinh\\"]\\\\n\\\\n@lru_cache(maxsize=24)\\\\ndef model_for(voice: str):\\\\n    runtime = KokoroVietnameseONNX(voice=voice, device=\\"cuda\\")\\\\n    if \\"CUDAExecutionProvider\\" not in runtime.session.get_providers():\\\\n        raise RuntimeError(\\"Kokoro Vietnamese ONNX did not activate CUDAExecutionProvider\\")\\\\n    return runtime\\\\n\\\\nmodel_for(SUPPORTED_VOICES[0])\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    if request.language.strip().lower() not in (\\"\\", \\"auto\\", \\"vi\\", \\"vi-vn\\"):\\\\n        raise HTTPException(status_code=422, detail=\\"Kokoro Vietnamese supports Vietnamese only\\")\\\\n    voice = request.voice.strip() or SUPPORTED_VOICES[0]\\\\n    if voice not in SUPPORTED_VOICES:\\\\n        raise HTTPException(status_code=422, detail=\\"unsupported Kokoro Vietnamese voice\\")\\\\n    audio, _phonemes = model_for(voice).synthesize(request.input, speed=request.speed)\\\\n    return audio, 24000\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'kokoro-vietnamese\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "kokoro-vietnamese",\n      "upstream_model": "contextboxai/Kokoro-Vietnamese",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/tts/LA_STUDIO_TTS_QWEN3_CUSTOMVOICE_1_7B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 Qwen3-TTS CustomVoice 1.7B\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3-tts-1.7b-customvoice` (`Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"qwen-tts==0.1.1\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nfrom qwen_tts import Qwen3TTSModel\\\\n\\\\nMODEL_ID = \\"qwen3-tts-1.7b-customvoice\\"\\\\nMODEL_NAME = \\"Qwen3-TTS CustomVoice 1.7B\\"\\\\nUPSTREAM_MODEL = \\"Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\\"\\\\nSUPPORTED_LANGUAGES = [\\"Auto\\", \\"Chinese\\", \\"English\\", \\"Japanese\\", \\"Korean\\", \\"German\\", \\"French\\", \\"Russian\\", \\"Portuguese\\", \\"Spanish\\", \\"Italian\\"]\\\\nSUPPORTED_VOICES = [\\"Aiden\\", \\"Dylan\\", \\"Eric\\", \\"Ono_Anna\\", \\"Ryan\\", \\"Serena\\", \\"Sohee\\", \\"Uncle_Fu\\", \\"Vivian\\"]\\\\nMODEL = Qwen3TTSModel.from_pretrained(UPSTREAM_MODEL, device_map=\\"cuda:0\\", dtype=torch.float16, attn_implementation=\\"sdpa\\")\\\\n\\\\ndef qwen_language(value: str):\\\\n    normalized = value.strip().lower()\\\\n    mapping = {\\"auto\\": \\"Auto\\", \\"zh\\": \\"Chinese\\", \\"en\\": \\"English\\", \\"ja\\": \\"Japanese\\", \\"ko\\": \\"Korean\\", \\"de\\": \\"German\\", \\"fr\\": \\"French\\", \\"ru\\": \\"Russian\\", \\"pt\\": \\"Portuguese\\", \\"es\\": \\"Spanish\\", \\"it\\": \\"Italian\\"}\\\\n    return mapping.get(normalized, value.strip().title() or \\"Auto\\")\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    speaker = request.voice.strip() or \\"Aiden\\"\\\\n    if speaker.lower() not in {voice.lower() for voice in SUPPORTED_VOICES}:\\\\n        raise HTTPException(status_code=422, detail=\\"unsupported Qwen3 CustomVoice speaker\\")\\\\n    canonical = next(voice for voice in SUPPORTED_VOICES if voice.lower() == speaker.lower())\\\\n    instruct = str(request.settings.get(\\"instruct\\", \\"\\")).strip()\\\\n    wavs, sample_rate = MODEL.generate_custom_voice(\\\\n        text=request.input,\\\\n        language=qwen_language(request.language),\\\\n        speaker=canonical,\\\\n        instruct=instruct,\\\\n    )\\\\n    return wavs[0], sample_rate\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'qwen3-tts-1.7b-customvoice\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "qwen3-tts-1.7b-customvoice",\n      "upstream_model": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/tts/LA_STUDIO_TTS_VIBEVOICE_0_5B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 VibeVoice Realtime 0.5B\\n",\n        "\\n",\n        "This notebook loads exactly `vibevoice` (`microsoft/VibeVoice-Realtime-0.5B`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "!git clone --quiet https://github.com/microsoft/VibeVoice.git /content/VibeVoice\\n",\n        "!git -C /content/VibeVoice checkout --quiet 94da20d98b2f\\n",\n        "%pip install -q -e \\"/content/VibeVoice[streamingtts]\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n",\n        "!bash /content/VibeVoice/demo/download_experimental_voices.sh\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nimport copy\\\\nimport glob\\\\nfrom pathlib import Path\\\\nfrom transformers.cache_utils import DynamicCache\\\\nfrom transformers.modeling_outputs import BaseModelOutputWithPast\\\\nfrom vibevoice.modular.modeling_vibevoice_streaming_inference import VibeVoiceStreamingForConditionalGenerationInference\\\\nfrom vibevoice.processor.vibevoice_streaming_processor import VibeVoiceStreamingProcessor\\\\n\\\\nMODEL_ID = \\"vibevoice\\"\\\\nMODEL_NAME = \\"VibeVoice Realtime 0.5B\\"\\\\nUPSTREAM_MODEL = \\"microsoft/VibeVoice-Realtime-0.5B\\"\\\\n# The pinned VibeVoice Realtime 0.5B release is English-only. Advertising\\\\n# unsupported languages here would let the desktop UI select a route the\\\\n# upstream model explicitly describes as unpredictable.\\\\nSUPPORTED_LANGUAGES = [\\"en\\"]\\\\nVOICE_DIR = Path(\\"/content/VibeVoice/demo/voices/streaming_model\\")\\\\nVOICE_FILES = {Path(path).stem.lower(): path for path in glob.glob(str(VOICE_DIR / \\"**\\" / \\"*.pt\\"), recursive=True)}\\\\nif not VOICE_FILES:\\\\n    raise RuntimeError(\\"VibeVoice voice presets were not downloaded\\")\\\\nSUPPORTED_VOICES = sorted(VOICE_FILES)\\\\nPROCESSOR = VibeVoiceStreamingProcessor.from_pretrained(UPSTREAM_MODEL)\\\\nMODEL = VibeVoiceStreamingForConditionalGenerationInference.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    torch_dtype=torch.bfloat16,\\\\n    device_map=\\"cuda\\",\\\\n    attn_implementation=\\"sdpa\\",\\\\n)\\\\nMODEL.eval()\\\\nMODEL.set_ddpm_inference_steps(num_steps=5)\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    voice = request.voice.strip().lower() or SUPPORTED_VOICES[0]\\\\n    if voice not in VOICE_FILES:\\\\n        raise HTTPException(status_code=422, detail=\\"unsupported VibeVoice preset\\")\\\\n    with torch.serialization.safe_globals([BaseModelOutputWithPast, DynamicCache]):\\\\n        cached = torch.load(VOICE_FILES[voice], map_location=\\"cuda\\", weights_only=True)\\\\n    inputs = PROCESSOR.process_input_with_cached_prompt(\\\\n        text=request.input.replace(\\"\\u2019\\", \\"\\\\\'\\").replace(\\"\\u201c\\", \\\\\'\\"\\\\\').replace(\\"\\u201d\\", \\\\\'\\"\\\\\'),\\\\n        cached_prompt=cached,\\\\n        padding=True,\\\\n        return_tensors=\\"pt\\",\\\\n        return_attention_mask=True,\\\\n    )\\\\n    for key, value in inputs.items():\\\\n        if torch.is_tensor(value):\\\\n            inputs[key] = value.to(\\"cuda\\")\\\\n    outputs = MODEL.generate(\\\\n        **inputs,\\\\n        max_new_tokens=None,\\\\n        cfg_scale=float(request.settings.get(\\"cfg_scale\\", 1.5)),\\\\n        tokenizer=PROCESSOR.tokenizer,\\\\n        generation_config={\\"do_sample\\": False},\\\\n        verbose=False,\\\\n        all_prefilled_outputs=copy.deepcopy(cached),\\\\n    )\\\\n    if not outputs.speech_outputs or outputs.speech_outputs[0] is None:\\\\n        raise RuntimeError(\\"VibeVoice returned no speech output\\")\\\\n    return outputs.speech_outputs[0].detach().float().cpu().numpy(), 24000\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'vibevoice\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "vibevoice",\n      "upstream_model": "microsoft/VibeVoice-Realtime-0.5B",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/tts/LA_STUDIO_TTS_VIENEU_V2_TURBO_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 VieNeu-TTS v2 Turbo\\n",\n        "\\n",\n        "This notebook loads exactly `vieneu-tts-v2-turbo` (`pnnbao-ump/VieNeu-TTS-v2-Turbo`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"torch==2.8.0\\" \\"torchaudio==2.8.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q --upgrade --force-reinstall --no-deps \\"torchvision==0.23.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q \\"transformers==4.57.6\\" \\"git+https://github.com/pnnbao97/VieNeu-TTS.git@f56ce97ffb37\\" \\"onnxruntime-gpu==1.22.0\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n",\n        "\\n",\n        "\\n",\n        "# Colab can retain an older torchvision after torch is upgraded. Transformers\\n",\n        "# then masks the binary mismatch as a missing PreTrainedModel/Qwen3 class.\\n",\n        "import importlib.metadata as package_metadata\\n",\n        "import traceback\\n",\n        "\\n",\n        "import torch\\n",\n        "import torchvision\\n",\n        "\\n",\n        "print(\\"PyTorch stack:\\", torch.__version__, torchvision.__version__)\\n",\n        "assert torch.cuda.is_available(), \\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU.\\"\\n",\n        "assert package_metadata.version(\\"torchvision\\").split(\\"+\\")[0] == \\"0.23.0\\", \\"VieNeu requires torchvision 0.23.0 with torch 2.8.0.\\"\\n",\n        "try:\\n",\n        "    from transformers import PreTrainedModel\\n",\n        "    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM\\n",\n        "except Exception as error:\\n",\n        "    traceback.print_exc()\\n",\n        "    raise RuntimeError(\\n",\n        "        \\"The Colab PyTorch/Transformers stack is not importable for VieNeu. \\"\\n",\n        "        \\"Restart the runtime, rerun this install cell, then run all cells again.\\"\\n",\n        "    ) from error\\n",\n        "print(\\"Transformers imports verified for VieNeu:\\", PreTrainedModel.__name__, Qwen3ForCausalLM.__name__)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nfrom vieneu import Vieneu\\\\n\\\\nMODEL_ID = \\"vieneu-tts-v2-turbo\\"\\\\nMODEL_NAME = \\"VieNeu-TTS v2 Turbo\\"\\\\nUPSTREAM_MODEL = \\"pnnbao-ump/VieNeu-TTS-v2-Turbo\\"\\\\nSUPPORTED_LANGUAGES = [\\"vi\\", \\"en\\"]\\\\nMODEL = Vieneu(mode=\\"turbo_gpu\\", device=\\"cuda\\", backend=\\"standard\\", backbone_repo=UPSTREAM_MODEL)\\\\nif getattr(MODEL, \\"device\\", \\"\\") != \\"cuda\\":\\\\n    raise RuntimeError(\\"VieNeu v2 Turbo did not load on CUDA\\")\\\\nVOICE_ROWS = MODEL.list_preset_voices()\\\\nSUPPORTED_VOICES = [str(row[1]) for row in VOICE_ROWS] if VOICE_ROWS else [\\"auto\\"]\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    voice = request.voice.strip()\\\\n    kwargs = {\\"text\\": request.input}\\\\n    if voice and voice.lower() != \\"auto\\":\\\\n        kwargs[\\"voice\\"] = voice\\\\n    audio = MODEL.infer(**kwargs)\\\\n    return audio, 24000\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'vieneu-tts-v2-turbo\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "vieneu-tts-v2-turbo",\n      "upstream_model": "pnnbao-ump/VieNeu-TTS-v2-Turbo",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/tts/LA_STUDIO_TTS_VIENEU_V3_TURBO_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 VieNeu-TTS v3 Turbo\\n",\n        "\\n",\n        "This notebook loads exactly `vieneu-tts-v3-turbo` (`pnnbao-ump/VieNeu-TTS-v3-Turbo`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"torch==2.8.0\\" \\"torchaudio==2.8.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q --upgrade --force-reinstall --no-deps \\"torchvision==0.23.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q \\"transformers==4.57.6\\" \\"git+https://github.com/pnnbao97/VieNeu-TTS.git@f56ce97ffb37\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n",\n        "\\n",\n        "\\n",\n        "# Colab can retain an older torchvision after torch is upgraded. Transformers\\n",\n        "# then masks the binary mismatch as a missing PreTrainedModel/Qwen3 class.\\n",\n        "import importlib.metadata as package_metadata\\n",\n        "import traceback\\n",\n        "\\n",\n        "import torch\\n",\n        "import torchvision\\n",\n        "\\n",\n        "print(\\"PyTorch stack:\\", torch.__version__, torchvision.__version__)\\n",\n        "assert torch.cuda.is_available(), \\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU.\\"\\n",\n        "assert package_metadata.version(\\"torchvision\\").split(\\"+\\")[0] == \\"0.23.0\\", \\"VieNeu requires torchvision 0.23.0 with torch 2.8.0.\\"\\n",\n        "try:\\n",\n        "    from transformers import PreTrainedModel\\n",\n        "    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM\\n",\n        "except Exception as error:\\n",\n        "    traceback.print_exc()\\n",\n        "    raise RuntimeError(\\n",\n        "        \\"The Colab PyTorch/Transformers stack is not importable for VieNeu. \\"\\n",\n        "        \\"Restart the runtime, rerun this install cell, then run all cells again.\\"\\n",\n        "    ) from error\\n",\n        "print(\\"Transformers imports verified for VieNeu:\\", PreTrainedModel.__name__, Qwen3ForCausalLM.__name__)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nfrom vieneu import Vieneu\\\\n\\\\nMODEL_ID = \\"vieneu-tts-v3-turbo\\"\\\\nMODEL_NAME = \\"VieNeu-TTS v3 Turbo\\"\\\\nUPSTREAM_MODEL = \\"pnnbao-ump/VieNeu-TTS-v3-Turbo\\"\\\\nSUPPORTED_LANGUAGES = [\\"vi\\", \\"en\\"]\\\\nMODEL = Vieneu(mode=\\"v3turbo\\", device=\\"cuda\\", backend=\\"pytorch\\", backbone_repo=UPSTREAM_MODEL)\\\\nif getattr(MODEL, \\"backend\\", \\"\\") != \\"pytorch\\":\\\\n    raise RuntimeError(\\"VieNeu v3 Turbo did not activate the PyTorch CUDA backend\\")\\\\nVOICE_ROWS = MODEL.list_preset_voices()\\\\nSUPPORTED_VOICES = [str(row[1]) for row in VOICE_ROWS] if VOICE_ROWS else [\\"auto\\"]\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    voice = request.voice.strip()\\\\n    kwargs = {\\"text\\": request.input}\\\\n    if voice and voice.lower() != \\"auto\\":\\\\n        kwargs[\\"voice\\"] = voice\\\\n    style = str(request.settings.get(\\"style\\", \\"\\")).strip()\\\\n    if style:\\\\n        kwargs[\\"style\\"] = style\\\\n    audio = MODEL.infer(**kwargs)\\\\n    return audio, 48000\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'vieneu-tts-v3-turbo\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "vieneu-tts-v3-turbo",\n      "upstream_model": "pnnbao-ump/VieNeu-TTS-v3-Turbo",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/tts/LA_STUDIO_TTS_VOXCPM2_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 VoxCPM2\\n",\n        "\\n",\n        "This notebook loads exactly `voxcpm2` (`openbmb/VoxCPM2`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "!git clone --quiet https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM\\n",\n        "!git -C /content/VoxCPM checkout --quiet 616d3d3e630a\\n",\n        "%pip install -q -e /content/VoxCPM \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nfrom voxcpm import VoxCPM\\\\n\\\\nMODEL_ID = \\"voxcpm2\\"\\\\nMODEL_NAME = \\"VoxCPM2\\"\\\\nUPSTREAM_MODEL = \\"openbmb/VoxCPM2\\"\\\\nSUPPORTED_LANGUAGES = [\\"auto\\", \\"vi\\", \\"en\\", \\"zh\\", \\"ja\\", \\"ko\\", \\"fr\\", \\"de\\", \\"es\\", \\"it\\", \\"pt\\", \\"th\\"]\\\\nSUPPORTED_VOICES = [\\"auto\\"]\\\\nMODEL = VoxCPM.from_pretrained(UPSTREAM_MODEL, load_denoiser=False, optimize=True, device=\\"cuda\\")\\\\nif \\"cuda\\" not in str(MODEL.model.device).lower():\\\\n    raise RuntimeError(\\"VoxCPM2 did not load on CUDA\\")\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    audio = MODEL.generate(\\\\n        text=request.input,\\\\n        cfg_value=float(request.settings.get(\\"cfg_value\\", 2.0)),\\\\n        inference_timesteps=int(request.settings.get(\\"inference_timesteps\\", 10)),\\\\n        normalize=bool(request.settings.get(\\"normalize\\", True)),\\\\n        seed=request.settings.get(\\"seed\\"),\\\\n    )\\\\n    return audio, 48000\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'voxcpm2\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "voxcpm2",\n      "upstream_model": "openbmb/VoxCPM2",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_cloning/LA_STUDIO_TTS_OMNIVOICE_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio TTS \\u2014 OmniVoice\\n",\n        "\\n",\n        "This notebook loads exactly `omnivoice` (`k2-fsa/OmniVoice`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s TTS panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"git+https://github.com/k2-fsa/OmniVoice.git@468e927ba371\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_tts_worker.py\')\\n",\n        "WORKER.write_text(\'import io\\\\nimport os\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_TTS_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass SpeechRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice: str = Field(default=\\"auto\\", max_length=160)\\\\n    language: str = Field(default=\\"auto\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\\\\n    response_format: str = \\"wav\\"\\\\n    settings: dict = Field(default_factory=dict)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef wav_response(samples, sample_rate: int):\\\\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0:\\\\n        raise RuntimeError(\\"the selected model returned no audio\\")\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    if not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned non-finite audio\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\nfrom omnivoice import OmniVoice\\\\n\\\\nMODEL_ID = \\"omnivoice\\"\\\\nMODEL_NAME = \\"OmniVoice\\"\\\\nUPSTREAM_MODEL = \\"k2-fsa/OmniVoice\\"\\\\nSUPPORTED_LANGUAGES = [\\"auto\\", \\"vi\\", \\"en\\", \\"zh\\", \\"ja\\", \\"ko\\", \\"fr\\", \\"es\\", \\"de\\"]\\\\nSUPPORTED_VOICES = [\\"auto\\"]\\\\nMODEL = OmniVoice.from_pretrained(UPSTREAM_MODEL, device_map=\\"cuda:0\\", dtype=torch.float16)\\\\nif not next(MODEL.parameters()).is_cuda:\\\\n    raise RuntimeError(\\"OmniVoice did not load on CUDA\\")\\\\n\\\\ndef synthesize_exact_model(request: SpeechRequest):\\\\n    kwargs = {\\"text\\": request.input, \\"speed\\": request.speed}\\\\n    language = request.language.strip().lower()\\\\n    if language and language != \\"auto\\":\\\\n        kwargs[\\"language_id\\"] = language\\\\n    audio = MODEL.generate(**kwargs)\\\\n    return audio[0] if isinstance(audio, (list, tuple)) else audio, 24000\\\\n\\\\napp = FastAPI(title=f\\"LA Studio TTS \\u2014 {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"tts\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"languages\\": SUPPORTED_LANGUAGES,\\\\n                \\"voices\\": SUPPORTED_VOICES,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/speech\\")\\\\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if request.response_format.strip().lower() != \\"wav\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns WAV audio only\\")\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab TTS worker is busy; retry shortly\\")\\\\n    try:\\\\n        samples, sample_rate = synthesize_exact_model(request)\\\\n        return wav_response(samples, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'TTS\'\\n",\n        "MODEL_ID = \'omnivoice\'\\n",\n        "PORT = 3921\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_TTS_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_TTS_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_TTS_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_tts_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_tts_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_tts_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "tts",\n      "family_id": "omnivoice",\n      "upstream_model": "k2-fsa/OmniVoice",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_cloning/LA_STUDIO_VOICE_CLONE_OMNIVOICE_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Cloning - OmniVoice\\n",\n        "\\n",\n        "This notebook loads exactly `omnivoice` (`k2-fsa/OmniVoice`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Cloning panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"git+https://github.com/k2-fsa/OmniVoice.git@468e927ba371\\" \\"soundfile==0.13.1\\" \\"python-multipart==0.0.20\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_clone_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom omnivoice import OmniVoice\\\\n\\\\nMODEL_ID = \\"omnivoice\\"\\\\nMODEL_NAME = \\"OmniVoice\\"\\\\nUPSTREAM_MODEL = \\"k2-fsa/OmniVoice\\"\\\\nMODEL = OmniVoice.from_pretrained(UPSTREAM_MODEL, device_map=\\"cuda:0\\", dtype=torch.float16)\\\\n\\\\ndef prepare_exact_profile(profile):\\\\n    # OmniVoice auto-transcribes the reference with Whisper when ref_text is\\\\n    # omitted. Pass the keyword only when the user supplied an exact transcript.\\\\n    kwargs = {\\"ref_audio\\": profile[\\"ref_audio\\"]}\\\\n    if profile[\\"ref_text\\"]:\\\\n        kwargs[\\"ref_text\\"] = profile[\\"ref_text\\"]\\\\n    return MODEL.create_voice_clone_prompt(**kwargs)\\\\n\\\\ndef clone_with_exact_model(profile, request):\\\\n    audio = MODEL.generate(\\\\n        text=request.text,\\\\n        voice_clone_prompt=profile[\\"state\\"],\\\\n        speed=request.speed,\\\\n        num_step=request.num_step,\\\\n    )\\\\n    return audio, 24000\\\\n\\\\nimport io\\\\nimport os\\\\nimport shutil\\\\nimport tempfile\\\\nimport threading\\\\nimport uuid\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\\"]\\\\nDATA_DIR = Path(\\"/content/la-studio-voice-clone-data\\") / MODEL_ID\\\\nDATA_DIR.mkdir(parents=True, exist_ok=True)\\\\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nMODEL_LOCK = threading.Lock()\\\\nSTATE_LOCK = threading.Lock()\\\\nPROFILES = {}\\\\nJOBS = {}\\\\n\\\\nclass GenerationRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    profile_id: str = Field(min_length=1, max_length=160)\\\\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    language: str = Field(default=\\"vi\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\\\\n    num_step: int = Field(default=32, ge=1, le=64)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(model: str) -> None:\\\\n    if model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef write_wav(path: Path, value, sample_rate: int) -> None:\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise RuntimeError(\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    sf.write(path, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n\\\\ndef public_job(job_id: str):\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        return {key: value for key, value in job.items() if key not in {\\"audio_path\\", \\"cancelled\\"}}\\\\n\\\\ndef fail_job(job_id: str, error: Exception) -> None:\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if job:\\\\n            job.update({\\\\n                \\"status\\": \\"failed\\",\\\\n                \\"stage\\": \\"failed\\",\\\\n                \\"error\\": {\\"message\\": f\\"{type(error).__name__}: {str(error)[:300]}\\"},\\\\n            })\\\\n\\\\ndef build_profile(job_id: str, profile_id: str) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES[profile_id]\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"prepare_profile\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            state = prepare_exact_profile(profile)\\\\n        with STATE_LOCK:\\\\n            profile[\\"state\\"] = state\\\\n            JOBS[job_id].update({\\\\n                \\"status\\": \\"succeeded\\",\\\\n                \\"stage\\": \\"complete\\",\\\\n                \\"percent\\": 100,\\\\n                \\"result\\": {\\"id\\": profile_id, \\"model\\": MODEL_ID},\\\\n            })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES.get(request.profile_id)\\\\n            if not profile:\\\\n                raise RuntimeError(\\"voice profile no longer exists\\")\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"generate\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            audio, sample_rate = clone_with_exact_model(profile, request)\\\\n        output_path = DATA_DIR / f\\"{job_id}.wav\\"\\\\n        write_wav(output_path, audio, sample_rate)\\\\n        with STATE_LOCK:\\\\n            if JOBS[job_id].get(\\"cancelled\\"):\\\\n                JOBS[job_id].update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n                output_path.unlink(missing_ok=True)\\\\n            else:\\\\n                JOBS[job_id].update({\\\\n                    \\"status\\": \\"succeeded\\",\\\\n                    \\"stage\\": \\"complete\\",\\\\n                    \\"percent\\": 100,\\\\n                    \\"audio_path\\": str(output_path),\\\\n                    \\"result\\": {\\"model\\": MODEL_ID, \\"sample_rate\\": int(sample_rate)},\\\\n                })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Cloning - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-cloning\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"reference_formats\\": [\\"wav\\", \\"mp3\\", \\"flac\\"],\\\\n                \\"reference_duration_seconds\\": {\\"min\\": 3, \\"max\\": 30},\\\\n                \\"requires_consent\\": True,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v2/jobs/profile\\", status_code=202)\\\\nasync def create_profile(\\\\n    model: str = Form(...),\\\\n    name: str = Form(...),\\\\n    consent_confirmed: bool = Form(...),\\\\n    ref_text: str = Form(default=\\"\\"),\\\\n    language: str = Form(default=\\"vi\\"),\\\\n    separate_music: bool = Form(default=False),\\\\n    ref_audio: UploadFile = File(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if not consent_confirmed:\\\\n        raise HTTPException(status_code=403, detail=\\"explicit voice-cloning consent is required\\")\\\\n    if not name.strip():\\\\n        raise HTTPException(status_code=422, detail=\\"profile name is required\\")\\\\n    suffix = Path(ref_audio.filename or \\"\\").suffix.lower()\\\\n    if suffix not in {\\".wav\\", \\".mp3\\", \\".flac\\"}:\\\\n        raise HTTPException(status_code=415, detail=\\"reference audio must be WAV, MP3, or FLAC\\")\\\\n    profile_id = uuid.uuid4().hex\\\\n    reference_path = DATA_DIR / f\\"{profile_id}{suffix}\\"\\\\n    size = 0\\\\n    with reference_path.open(\\"wb\\") as output:\\\\n        while chunk := await ref_audio.read(1024 * 1024):\\\\n            size += len(chunk)\\\\n            if size > MAX_REFERENCE_BYTES:\\\\n                reference_path.unlink(missing_ok=True)\\\\n                raise HTTPException(status_code=413, detail=\\"reference audio exceeds 256 MB\\")\\\\n            output.write(chunk)\\\\n    try:\\\\n        info = sf.info(reference_path)\\\\n        duration = float(info.frames) / float(info.samplerate)\\\\n    except Exception as error:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=f\\"reference audio cannot be decoded: {error}\\") from error\\\\n    if duration < 3.0 or duration > 30.0:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=\\"reference audio must be between 3 and 30 seconds\\")\\\\n    job_id = uuid.uuid4().hex\\\\n    profile = {\\\\n        \\"id\\": profile_id,\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"name\\": name.strip(),\\\\n        \\"ref_audio\\": str(reference_path),\\\\n        \\"ref_text\\": ref_text.strip(),\\\\n        \\"language\\": language.strip() or \\"vi\\",\\\\n        \\"separate_music\\": bool(separate_music),\\\\n    }\\\\n    with STATE_LOCK:\\\\n        PROFILES[profile_id] = profile\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.post(\\"/v2/jobs/generation\\", status_code=202)\\\\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    require_exact_model(request.model)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.get(request.profile_id)\\\\n        if not profile:\\\\n            raise HTTPException(status_code=404, detail=\\"voice profile not found\\")\\\\n        if profile[\\"model\\"] != MODEL_ID:\\\\n            raise HTTPException(status_code=409, detail=\\"voice profile belongs to a different model worker\\")\\\\n        job_id = uuid.uuid4().hex\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}/audio\\")\\\\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        path = Path(job.get(\\"audio_path\\", \\"\\")) if job else None\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n    if job.get(\\"status\\") != \\"succeeded\\" or not path or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"voice job audio is not ready\\")\\\\n    return Response(path.read_bytes(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}\\")\\\\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return public_job(job_id)\\\\n\\\\n@app.delete(\\"/v2/jobs/{job_id}\\")\\\\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        job[\\"cancelled\\"] = True\\\\n        if job[\\"status\\"] == \\"queued\\":\\\\n            job.update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n    return {\\"cancelled\\": True}\\\\n\\\\n@app.delete(\\"/v1/profiles/{profile_id}\\")\\\\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.pop(profile_id, None)\\\\n    if profile:\\\\n        Path(profile[\\"ref_audio\\"]).unlink(missing_ok=True)\\\\n    return {\\"deleted\\": bool(profile)}\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Cloning\'\\n",\n        "MODEL_ID = \'omnivoice\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_clone_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_clone_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_clone_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-cloning",\n      "family_id": "omnivoice",\n      "upstream_model": "k2-fsa/OmniVoice",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_cloning/LA_STUDIO_VOICE_CLONE_QWEN3_BASE_0_6B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Cloning - Qwen3-TTS Base 0.6B\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3-tts-0.6b-base` (`Qwen/Qwen3-TTS-12Hz-0.6B-Base`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Cloning panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"qwen-tts==0.1.1\\" \\"soundfile==0.13.1\\" \\"python-multipart==0.0.20\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_clone_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom qwen_tts import Qwen3TTSModel\\\\n\\\\nMODEL_ID = \\"qwen3-tts-0.6b-base\\"\\\\nMODEL_NAME = \\"Qwen3-TTS Base 0.6B\\"\\\\nUPSTREAM_MODEL = \\"Qwen/Qwen3-TTS-12Hz-0.6B-Base\\"\\\\nMODEL = Qwen3TTSModel.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    device_map=\\"cuda:0\\",\\\\n    dtype=torch.bfloat16,\\\\n    attn_implementation=\\"sdpa\\",\\\\n)\\\\n\\\\ndef qwen_language(value: str):\\\\n    mapping = {\\"auto\\": \\"Auto\\", \\"zh\\": \\"Chinese\\", \\"en\\": \\"English\\", \\"ja\\": \\"Japanese\\", \\"ko\\": \\"Korean\\", \\"de\\": \\"German\\", \\"fr\\": \\"French\\", \\"ru\\": \\"Russian\\", \\"pt\\": \\"Portuguese\\", \\"es\\": \\"Spanish\\", \\"it\\": \\"Italian\\", \\"vi\\": \\"Auto\\"}\\\\n    return mapping.get(value.strip().lower(), value.strip().title() or \\"Auto\\")\\\\n\\\\ndef prepare_exact_profile(profile):\\\\n    if not profile[\\"ref_text\\"]:\\\\n        # Qwen supports speaker-only cloning. It avoids making the transcript\\\\n        # a form requirement, with a clear quality trade-off for this mode.\\\\n        return MODEL.create_voice_clone_prompt(\\\\n            ref_audio=profile[\\"ref_audio\\"],\\\\n            x_vector_only_mode=True,\\\\n        )\\\\n    return MODEL.create_voice_clone_prompt(\\\\n        ref_audio=profile[\\"ref_audio\\"],\\\\n        ref_text=profile[\\"ref_text\\"],\\\\n        x_vector_only_mode=False,\\\\n    )\\\\n\\\\ndef clone_with_exact_model(profile, request):\\\\n    wavs, sample_rate = MODEL.generate_voice_clone(\\\\n        text=request.text,\\\\n        language=qwen_language(request.language),\\\\n        voice_clone_prompt=profile[\\"state\\"],\\\\n    )\\\\n    return wavs[0], sample_rate\\\\n\\\\nimport io\\\\nimport os\\\\nimport shutil\\\\nimport tempfile\\\\nimport threading\\\\nimport uuid\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\\"]\\\\nDATA_DIR = Path(\\"/content/la-studio-voice-clone-data\\") / MODEL_ID\\\\nDATA_DIR.mkdir(parents=True, exist_ok=True)\\\\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nMODEL_LOCK = threading.Lock()\\\\nSTATE_LOCK = threading.Lock()\\\\nPROFILES = {}\\\\nJOBS = {}\\\\n\\\\nclass GenerationRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    profile_id: str = Field(min_length=1, max_length=160)\\\\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    language: str = Field(default=\\"vi\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\\\\n    num_step: int = Field(default=32, ge=1, le=64)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(model: str) -> None:\\\\n    if model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef write_wav(path: Path, value, sample_rate: int) -> None:\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise RuntimeError(\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    sf.write(path, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n\\\\ndef public_job(job_id: str):\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        return {key: value for key, value in job.items() if key not in {\\"audio_path\\", \\"cancelled\\"}}\\\\n\\\\ndef fail_job(job_id: str, error: Exception) -> None:\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if job:\\\\n            job.update({\\\\n                \\"status\\": \\"failed\\",\\\\n                \\"stage\\": \\"failed\\",\\\\n                \\"error\\": {\\"message\\": f\\"{type(error).__name__}: {str(error)[:300]}\\"},\\\\n            })\\\\n\\\\ndef build_profile(job_id: str, profile_id: str) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES[profile_id]\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"prepare_profile\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            state = prepare_exact_profile(profile)\\\\n        with STATE_LOCK:\\\\n            profile[\\"state\\"] = state\\\\n            JOBS[job_id].update({\\\\n                \\"status\\": \\"succeeded\\",\\\\n                \\"stage\\": \\"complete\\",\\\\n                \\"percent\\": 100,\\\\n                \\"result\\": {\\"id\\": profile_id, \\"model\\": MODEL_ID},\\\\n            })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES.get(request.profile_id)\\\\n            if not profile:\\\\n                raise RuntimeError(\\"voice profile no longer exists\\")\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"generate\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            audio, sample_rate = clone_with_exact_model(profile, request)\\\\n        output_path = DATA_DIR / f\\"{job_id}.wav\\"\\\\n        write_wav(output_path, audio, sample_rate)\\\\n        with STATE_LOCK:\\\\n            if JOBS[job_id].get(\\"cancelled\\"):\\\\n                JOBS[job_id].update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n                output_path.unlink(missing_ok=True)\\\\n            else:\\\\n                JOBS[job_id].update({\\\\n                    \\"status\\": \\"succeeded\\",\\\\n                    \\"stage\\": \\"complete\\",\\\\n                    \\"percent\\": 100,\\\\n                    \\"audio_path\\": str(output_path),\\\\n                    \\"result\\": {\\"model\\": MODEL_ID, \\"sample_rate\\": int(sample_rate)},\\\\n                })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Cloning - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-cloning\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"reference_formats\\": [\\"wav\\", \\"mp3\\", \\"flac\\"],\\\\n                \\"reference_duration_seconds\\": {\\"min\\": 3, \\"max\\": 30},\\\\n                \\"requires_consent\\": True,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v2/jobs/profile\\", status_code=202)\\\\nasync def create_profile(\\\\n    model: str = Form(...),\\\\n    name: str = Form(...),\\\\n    consent_confirmed: bool = Form(...),\\\\n    ref_text: str = Form(default=\\"\\"),\\\\n    language: str = Form(default=\\"vi\\"),\\\\n    separate_music: bool = Form(default=False),\\\\n    ref_audio: UploadFile = File(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if not consent_confirmed:\\\\n        raise HTTPException(status_code=403, detail=\\"explicit voice-cloning consent is required\\")\\\\n    if not name.strip():\\\\n        raise HTTPException(status_code=422, detail=\\"profile name is required\\")\\\\n    suffix = Path(ref_audio.filename or \\"\\").suffix.lower()\\\\n    if suffix not in {\\".wav\\", \\".mp3\\", \\".flac\\"}:\\\\n        raise HTTPException(status_code=415, detail=\\"reference audio must be WAV, MP3, or FLAC\\")\\\\n    profile_id = uuid.uuid4().hex\\\\n    reference_path = DATA_DIR / f\\"{profile_id}{suffix}\\"\\\\n    size = 0\\\\n    with reference_path.open(\\"wb\\") as output:\\\\n        while chunk := await ref_audio.read(1024 * 1024):\\\\n            size += len(chunk)\\\\n            if size > MAX_REFERENCE_BYTES:\\\\n                reference_path.unlink(missing_ok=True)\\\\n                raise HTTPException(status_code=413, detail=\\"reference audio exceeds 256 MB\\")\\\\n            output.write(chunk)\\\\n    try:\\\\n        info = sf.info(reference_path)\\\\n        duration = float(info.frames) / float(info.samplerate)\\\\n    except Exception as error:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=f\\"reference audio cannot be decoded: {error}\\") from error\\\\n    if duration < 3.0 or duration > 30.0:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=\\"reference audio must be between 3 and 30 seconds\\")\\\\n    job_id = uuid.uuid4().hex\\\\n    profile = {\\\\n        \\"id\\": profile_id,\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"name\\": name.strip(),\\\\n        \\"ref_audio\\": str(reference_path),\\\\n        \\"ref_text\\": ref_text.strip(),\\\\n        \\"language\\": language.strip() or \\"vi\\",\\\\n        \\"separate_music\\": bool(separate_music),\\\\n    }\\\\n    with STATE_LOCK:\\\\n        PROFILES[profile_id] = profile\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.post(\\"/v2/jobs/generation\\", status_code=202)\\\\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    require_exact_model(request.model)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.get(request.profile_id)\\\\n        if not profile:\\\\n            raise HTTPException(status_code=404, detail=\\"voice profile not found\\")\\\\n        if profile[\\"model\\"] != MODEL_ID:\\\\n            raise HTTPException(status_code=409, detail=\\"voice profile belongs to a different model worker\\")\\\\n        job_id = uuid.uuid4().hex\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}/audio\\")\\\\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        path = Path(job.get(\\"audio_path\\", \\"\\")) if job else None\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n    if job.get(\\"status\\") != \\"succeeded\\" or not path or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"voice job audio is not ready\\")\\\\n    return Response(path.read_bytes(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}\\")\\\\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return public_job(job_id)\\\\n\\\\n@app.delete(\\"/v2/jobs/{job_id}\\")\\\\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        job[\\"cancelled\\"] = True\\\\n        if job[\\"status\\"] == \\"queued\\":\\\\n            job.update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n    return {\\"cancelled\\": True}\\\\n\\\\n@app.delete(\\"/v1/profiles/{profile_id}\\")\\\\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.pop(profile_id, None)\\\\n    if profile:\\\\n        Path(profile[\\"ref_audio\\"]).unlink(missing_ok=True)\\\\n    return {\\"deleted\\": bool(profile)}\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Cloning\'\\n",\n        "MODEL_ID = \'qwen3-tts-0.6b-base\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_clone_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_clone_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_clone_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-cloning",\n      "family_id": "qwen3-tts-0.6b-base",\n      "upstream_model": "Qwen/Qwen3-TTS-12Hz-0.6B-Base",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_cloning/LA_STUDIO_VOICE_CLONE_QWEN3_BASE_1_7B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Cloning - Qwen3-TTS Base 1.7B\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3-tts-1.7b-base` (`Qwen/Qwen3-TTS-12Hz-1.7B-Base`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Cloning panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"qwen-tts==0.1.1\\" \\"soundfile==0.13.1\\" \\"python-multipart==0.0.20\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_clone_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom qwen_tts import Qwen3TTSModel\\\\n\\\\nMODEL_ID = \\"qwen3-tts-1.7b-base\\"\\\\nMODEL_NAME = \\"Qwen3-TTS Base 1.7B\\"\\\\nUPSTREAM_MODEL = \\"Qwen/Qwen3-TTS-12Hz-1.7B-Base\\"\\\\nMODEL = Qwen3TTSModel.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    device_map=\\"cuda:0\\",\\\\n    dtype=torch.bfloat16,\\\\n    attn_implementation=\\"sdpa\\",\\\\n)\\\\n\\\\ndef qwen_language(value: str):\\\\n    mapping = {\\"auto\\": \\"Auto\\", \\"zh\\": \\"Chinese\\", \\"en\\": \\"English\\", \\"ja\\": \\"Japanese\\", \\"ko\\": \\"Korean\\", \\"de\\": \\"German\\", \\"fr\\": \\"French\\", \\"ru\\": \\"Russian\\", \\"pt\\": \\"Portuguese\\", \\"es\\": \\"Spanish\\", \\"it\\": \\"Italian\\", \\"vi\\": \\"Auto\\"}\\\\n    return mapping.get(value.strip().lower(), value.strip().title() or \\"Auto\\")\\\\n\\\\ndef prepare_exact_profile(profile):\\\\n    if not profile[\\"ref_text\\"]:\\\\n        # Qwen supports speaker-only cloning. It avoids making the transcript\\\\n        # a form requirement, with a clear quality trade-off for this mode.\\\\n        return MODEL.create_voice_clone_prompt(\\\\n            ref_audio=profile[\\"ref_audio\\"],\\\\n            x_vector_only_mode=True,\\\\n        )\\\\n    return MODEL.create_voice_clone_prompt(\\\\n        ref_audio=profile[\\"ref_audio\\"],\\\\n        ref_text=profile[\\"ref_text\\"],\\\\n        x_vector_only_mode=False,\\\\n    )\\\\n\\\\ndef clone_with_exact_model(profile, request):\\\\n    wavs, sample_rate = MODEL.generate_voice_clone(\\\\n        text=request.text,\\\\n        language=qwen_language(request.language),\\\\n        voice_clone_prompt=profile[\\"state\\"],\\\\n    )\\\\n    return wavs[0], sample_rate\\\\n\\\\nimport io\\\\nimport os\\\\nimport shutil\\\\nimport tempfile\\\\nimport threading\\\\nimport uuid\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\\"]\\\\nDATA_DIR = Path(\\"/content/la-studio-voice-clone-data\\") / MODEL_ID\\\\nDATA_DIR.mkdir(parents=True, exist_ok=True)\\\\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nMODEL_LOCK = threading.Lock()\\\\nSTATE_LOCK = threading.Lock()\\\\nPROFILES = {}\\\\nJOBS = {}\\\\n\\\\nclass GenerationRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    profile_id: str = Field(min_length=1, max_length=160)\\\\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    language: str = Field(default=\\"vi\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\\\\n    num_step: int = Field(default=32, ge=1, le=64)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(model: str) -> None:\\\\n    if model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef write_wav(path: Path, value, sample_rate: int) -> None:\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise RuntimeError(\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    sf.write(path, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n\\\\ndef public_job(job_id: str):\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        return {key: value for key, value in job.items() if key not in {\\"audio_path\\", \\"cancelled\\"}}\\\\n\\\\ndef fail_job(job_id: str, error: Exception) -> None:\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if job:\\\\n            job.update({\\\\n                \\"status\\": \\"failed\\",\\\\n                \\"stage\\": \\"failed\\",\\\\n                \\"error\\": {\\"message\\": f\\"{type(error).__name__}: {str(error)[:300]}\\"},\\\\n            })\\\\n\\\\ndef build_profile(job_id: str, profile_id: str) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES[profile_id]\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"prepare_profile\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            state = prepare_exact_profile(profile)\\\\n        with STATE_LOCK:\\\\n            profile[\\"state\\"] = state\\\\n            JOBS[job_id].update({\\\\n                \\"status\\": \\"succeeded\\",\\\\n                \\"stage\\": \\"complete\\",\\\\n                \\"percent\\": 100,\\\\n                \\"result\\": {\\"id\\": profile_id, \\"model\\": MODEL_ID},\\\\n            })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES.get(request.profile_id)\\\\n            if not profile:\\\\n                raise RuntimeError(\\"voice profile no longer exists\\")\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"generate\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            audio, sample_rate = clone_with_exact_model(profile, request)\\\\n        output_path = DATA_DIR / f\\"{job_id}.wav\\"\\\\n        write_wav(output_path, audio, sample_rate)\\\\n        with STATE_LOCK:\\\\n            if JOBS[job_id].get(\\"cancelled\\"):\\\\n                JOBS[job_id].update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n                output_path.unlink(missing_ok=True)\\\\n            else:\\\\n                JOBS[job_id].update({\\\\n                    \\"status\\": \\"succeeded\\",\\\\n                    \\"stage\\": \\"complete\\",\\\\n                    \\"percent\\": 100,\\\\n                    \\"audio_path\\": str(output_path),\\\\n                    \\"result\\": {\\"model\\": MODEL_ID, \\"sample_rate\\": int(sample_rate)},\\\\n                })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Cloning - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-cloning\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"reference_formats\\": [\\"wav\\", \\"mp3\\", \\"flac\\"],\\\\n                \\"reference_duration_seconds\\": {\\"min\\": 3, \\"max\\": 30},\\\\n                \\"requires_consent\\": True,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v2/jobs/profile\\", status_code=202)\\\\nasync def create_profile(\\\\n    model: str = Form(...),\\\\n    name: str = Form(...),\\\\n    consent_confirmed: bool = Form(...),\\\\n    ref_text: str = Form(default=\\"\\"),\\\\n    language: str = Form(default=\\"vi\\"),\\\\n    separate_music: bool = Form(default=False),\\\\n    ref_audio: UploadFile = File(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if not consent_confirmed:\\\\n        raise HTTPException(status_code=403, detail=\\"explicit voice-cloning consent is required\\")\\\\n    if not name.strip():\\\\n        raise HTTPException(status_code=422, detail=\\"profile name is required\\")\\\\n    suffix = Path(ref_audio.filename or \\"\\").suffix.lower()\\\\n    if suffix not in {\\".wav\\", \\".mp3\\", \\".flac\\"}:\\\\n        raise HTTPException(status_code=415, detail=\\"reference audio must be WAV, MP3, or FLAC\\")\\\\n    profile_id = uuid.uuid4().hex\\\\n    reference_path = DATA_DIR / f\\"{profile_id}{suffix}\\"\\\\n    size = 0\\\\n    with reference_path.open(\\"wb\\") as output:\\\\n        while chunk := await ref_audio.read(1024 * 1024):\\\\n            size += len(chunk)\\\\n            if size > MAX_REFERENCE_BYTES:\\\\n                reference_path.unlink(missing_ok=True)\\\\n                raise HTTPException(status_code=413, detail=\\"reference audio exceeds 256 MB\\")\\\\n            output.write(chunk)\\\\n    try:\\\\n        info = sf.info(reference_path)\\\\n        duration = float(info.frames) / float(info.samplerate)\\\\n    except Exception as error:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=f\\"reference audio cannot be decoded: {error}\\") from error\\\\n    if duration < 3.0 or duration > 30.0:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=\\"reference audio must be between 3 and 30 seconds\\")\\\\n    job_id = uuid.uuid4().hex\\\\n    profile = {\\\\n        \\"id\\": profile_id,\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"name\\": name.strip(),\\\\n        \\"ref_audio\\": str(reference_path),\\\\n        \\"ref_text\\": ref_text.strip(),\\\\n        \\"language\\": language.strip() or \\"vi\\",\\\\n        \\"separate_music\\": bool(separate_music),\\\\n    }\\\\n    with STATE_LOCK:\\\\n        PROFILES[profile_id] = profile\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.post(\\"/v2/jobs/generation\\", status_code=202)\\\\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    require_exact_model(request.model)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.get(request.profile_id)\\\\n        if not profile:\\\\n            raise HTTPException(status_code=404, detail=\\"voice profile not found\\")\\\\n        if profile[\\"model\\"] != MODEL_ID:\\\\n            raise HTTPException(status_code=409, detail=\\"voice profile belongs to a different model worker\\")\\\\n        job_id = uuid.uuid4().hex\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}/audio\\")\\\\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        path = Path(job.get(\\"audio_path\\", \\"\\")) if job else None\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n    if job.get(\\"status\\") != \\"succeeded\\" or not path or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"voice job audio is not ready\\")\\\\n    return Response(path.read_bytes(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}\\")\\\\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return public_job(job_id)\\\\n\\\\n@app.delete(\\"/v2/jobs/{job_id}\\")\\\\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        job[\\"cancelled\\"] = True\\\\n        if job[\\"status\\"] == \\"queued\\":\\\\n            job.update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n    return {\\"cancelled\\": True}\\\\n\\\\n@app.delete(\\"/v1/profiles/{profile_id}\\")\\\\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.pop(profile_id, None)\\\\n    if profile:\\\\n        Path(profile[\\"ref_audio\\"]).unlink(missing_ok=True)\\\\n    return {\\"deleted\\": bool(profile)}\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Cloning\'\\n",\n        "MODEL_ID = \'qwen3-tts-1.7b-base\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_clone_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_clone_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_clone_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-cloning",\n      "family_id": "qwen3-tts-1.7b-base",\n      "upstream_model": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_cloning/LA_STUDIO_VOICE_CLONE_VIENEU_V2_TURBO_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Cloning - VieNeu-TTS v2 Turbo\\n",\n        "\\n",\n        "This notebook loads exactly `vieneu-tts-v2-turbo` (`pnnbao-ump/VieNeu-TTS-v2-Turbo`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Cloning panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"torch==2.8.0\\" \\"torchaudio==2.8.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q --upgrade --force-reinstall --no-deps \\"torchvision==0.23.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q \\"transformers==4.57.6\\" \\"git+https://github.com/pnnbao97/VieNeu-TTS.git@f56ce97ffb37\\" \\"soundfile==0.13.1\\" \\"python-multipart==0.0.20\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n",\n        "\\n",\n        "# Colab can retain an older torchvision after torch is upgraded. Transformers\\n",\n        "# then masks the binary mismatch as a missing PreTrainedModel/Qwen3 class.\\n",\n        "import importlib.metadata as package_metadata\\n",\n        "import traceback\\n",\n        "\\n",\n        "import torch\\n",\n        "import torchvision\\n",\n        "\\n",\n        "print(\\"PyTorch stack:\\", torch.__version__, torchvision.__version__)\\n",\n        "assert torch.cuda.is_available(), \\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU.\\"\\n",\n        "assert package_metadata.version(\\"torchvision\\").split(\\"+\\")[0] == \\"0.23.0\\", \\"VieNeu requires torchvision 0.23.0 with torch 2.8.0.\\"\\n",\n        "try:\\n",\n        "    from transformers import PreTrainedModel\\n",\n        "    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM\\n",\n        "except Exception as error:\\n",\n        "    traceback.print_exc()\\n",\n        "    raise RuntimeError(\\n",\n        "        \\"The Colab PyTorch/Transformers stack is not importable for VieNeu. \\"\\n",\n        "        \\"Restart the runtime, rerun this install cell, then run all cells again.\\"\\n",\n        "    ) from error\\n",\n        "print(\\"Transformers imports verified for VieNeu:\\", PreTrainedModel.__name__, Qwen3ForCausalLM.__name__)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_clone_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom vieneu import Vieneu\\\\n\\\\nMODEL_ID = \\"vieneu-tts-v2-turbo\\"\\\\nMODEL_NAME = \\"VieNeu-TTS v2 Turbo\\"\\\\nUPSTREAM_MODEL = \\"pnnbao-ump/VieNeu-TTS-v2-Turbo\\"\\\\nMODEL = Vieneu(mode=\\"turbo_gpu\\", device=\\"cuda\\", backend=\\"standard\\", backbone_repo=UPSTREAM_MODEL)\\\\n\\\\ndef prepare_exact_profile(profile):\\\\n    return MODEL.encode_reference(profile[\\"ref_audio\\"])\\\\n\\\\ndef clone_with_exact_model(profile, request):\\\\n    audio = MODEL.infer(text=request.text, voice=profile[\\"state\\"])\\\\n    return audio, 24000\\\\n\\\\nimport io\\\\nimport os\\\\nimport shutil\\\\nimport tempfile\\\\nimport threading\\\\nimport uuid\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\\"]\\\\nDATA_DIR = Path(\\"/content/la-studio-voice-clone-data\\") / MODEL_ID\\\\nDATA_DIR.mkdir(parents=True, exist_ok=True)\\\\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nMODEL_LOCK = threading.Lock()\\\\nSTATE_LOCK = threading.Lock()\\\\nPROFILES = {}\\\\nJOBS = {}\\\\n\\\\nclass GenerationRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    profile_id: str = Field(min_length=1, max_length=160)\\\\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    language: str = Field(default=\\"vi\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\\\\n    num_step: int = Field(default=32, ge=1, le=64)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(model: str) -> None:\\\\n    if model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef write_wav(path: Path, value, sample_rate: int) -> None:\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise RuntimeError(\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    sf.write(path, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n\\\\ndef public_job(job_id: str):\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        return {key: value for key, value in job.items() if key not in {\\"audio_path\\", \\"cancelled\\"}}\\\\n\\\\ndef fail_job(job_id: str, error: Exception) -> None:\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if job:\\\\n            job.update({\\\\n                \\"status\\": \\"failed\\",\\\\n                \\"stage\\": \\"failed\\",\\\\n                \\"error\\": {\\"message\\": f\\"{type(error).__name__}: {str(error)[:300]}\\"},\\\\n            })\\\\n\\\\ndef build_profile(job_id: str, profile_id: str) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES[profile_id]\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"prepare_profile\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            state = prepare_exact_profile(profile)\\\\n        with STATE_LOCK:\\\\n            profile[\\"state\\"] = state\\\\n            JOBS[job_id].update({\\\\n                \\"status\\": \\"succeeded\\",\\\\n                \\"stage\\": \\"complete\\",\\\\n                \\"percent\\": 100,\\\\n                \\"result\\": {\\"id\\": profile_id, \\"model\\": MODEL_ID},\\\\n            })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES.get(request.profile_id)\\\\n            if not profile:\\\\n                raise RuntimeError(\\"voice profile no longer exists\\")\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"generate\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            audio, sample_rate = clone_with_exact_model(profile, request)\\\\n        output_path = DATA_DIR / f\\"{job_id}.wav\\"\\\\n        write_wav(output_path, audio, sample_rate)\\\\n        with STATE_LOCK:\\\\n            if JOBS[job_id].get(\\"cancelled\\"):\\\\n                JOBS[job_id].update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n                output_path.unlink(missing_ok=True)\\\\n            else:\\\\n                JOBS[job_id].update({\\\\n                    \\"status\\": \\"succeeded\\",\\\\n                    \\"stage\\": \\"complete\\",\\\\n                    \\"percent\\": 100,\\\\n                    \\"audio_path\\": str(output_path),\\\\n                    \\"result\\": {\\"model\\": MODEL_ID, \\"sample_rate\\": int(sample_rate)},\\\\n                })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Cloning - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-cloning\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"reference_formats\\": [\\"wav\\", \\"mp3\\", \\"flac\\"],\\\\n                \\"reference_duration_seconds\\": {\\"min\\": 3, \\"max\\": 30},\\\\n                \\"requires_consent\\": True,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v2/jobs/profile\\", status_code=202)\\\\nasync def create_profile(\\\\n    model: str = Form(...),\\\\n    name: str = Form(...),\\\\n    consent_confirmed: bool = Form(...),\\\\n    ref_text: str = Form(default=\\"\\"),\\\\n    language: str = Form(default=\\"vi\\"),\\\\n    separate_music: bool = Form(default=False),\\\\n    ref_audio: UploadFile = File(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if not consent_confirmed:\\\\n        raise HTTPException(status_code=403, detail=\\"explicit voice-cloning consent is required\\")\\\\n    if not name.strip():\\\\n        raise HTTPException(status_code=422, detail=\\"profile name is required\\")\\\\n    suffix = Path(ref_audio.filename or \\"\\").suffix.lower()\\\\n    if suffix not in {\\".wav\\", \\".mp3\\", \\".flac\\"}:\\\\n        raise HTTPException(status_code=415, detail=\\"reference audio must be WAV, MP3, or FLAC\\")\\\\n    profile_id = uuid.uuid4().hex\\\\n    reference_path = DATA_DIR / f\\"{profile_id}{suffix}\\"\\\\n    size = 0\\\\n    with reference_path.open(\\"wb\\") as output:\\\\n        while chunk := await ref_audio.read(1024 * 1024):\\\\n            size += len(chunk)\\\\n            if size > MAX_REFERENCE_BYTES:\\\\n                reference_path.unlink(missing_ok=True)\\\\n                raise HTTPException(status_code=413, detail=\\"reference audio exceeds 256 MB\\")\\\\n            output.write(chunk)\\\\n    try:\\\\n        info = sf.info(reference_path)\\\\n        duration = float(info.frames) / float(info.samplerate)\\\\n    except Exception as error:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=f\\"reference audio cannot be decoded: {error}\\") from error\\\\n    if duration < 3.0 or duration > 30.0:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=\\"reference audio must be between 3 and 30 seconds\\")\\\\n    job_id = uuid.uuid4().hex\\\\n    profile = {\\\\n        \\"id\\": profile_id,\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"name\\": name.strip(),\\\\n        \\"ref_audio\\": str(reference_path),\\\\n        \\"ref_text\\": ref_text.strip(),\\\\n        \\"language\\": language.strip() or \\"vi\\",\\\\n        \\"separate_music\\": bool(separate_music),\\\\n    }\\\\n    with STATE_LOCK:\\\\n        PROFILES[profile_id] = profile\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.post(\\"/v2/jobs/generation\\", status_code=202)\\\\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    require_exact_model(request.model)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.get(request.profile_id)\\\\n        if not profile:\\\\n            raise HTTPException(status_code=404, detail=\\"voice profile not found\\")\\\\n        if profile[\\"model\\"] != MODEL_ID:\\\\n            raise HTTPException(status_code=409, detail=\\"voice profile belongs to a different model worker\\")\\\\n        job_id = uuid.uuid4().hex\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}/audio\\")\\\\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        path = Path(job.get(\\"audio_path\\", \\"\\")) if job else None\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n    if job.get(\\"status\\") != \\"succeeded\\" or not path or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"voice job audio is not ready\\")\\\\n    return Response(path.read_bytes(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}\\")\\\\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return public_job(job_id)\\\\n\\\\n@app.delete(\\"/v2/jobs/{job_id}\\")\\\\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        job[\\"cancelled\\"] = True\\\\n        if job[\\"status\\"] == \\"queued\\":\\\\n            job.update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n    return {\\"cancelled\\": True}\\\\n\\\\n@app.delete(\\"/v1/profiles/{profile_id}\\")\\\\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.pop(profile_id, None)\\\\n    if profile:\\\\n        Path(profile[\\"ref_audio\\"]).unlink(missing_ok=True)\\\\n    return {\\"deleted\\": bool(profile)}\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Cloning\'\\n",\n        "MODEL_ID = \'vieneu-tts-v2-turbo\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_clone_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_clone_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_clone_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-cloning",\n      "family_id": "vieneu-tts-v2-turbo",\n      "upstream_model": "pnnbao-ump/VieNeu-TTS-v2-Turbo",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_cloning/LA_STUDIO_VOICE_CLONE_VIENEU_V3_TURBO_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Cloning - VieNeu-TTS v3 Turbo\\n",\n        "\\n",\n        "This notebook loads exactly `vieneu-tts-v3-turbo` (`pnnbao-ump/VieNeu-TTS-v3-Turbo`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Cloning panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"torch==2.8.0\\" \\"torchaudio==2.8.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q --upgrade --force-reinstall --no-deps \\"torchvision==0.23.0\\" --index-url https://download.pytorch.org/whl/cu128\\n",\n        "%pip install -q \\"transformers==4.57.6\\" \\"git+https://github.com/pnnbao97/VieNeu-TTS.git@f56ce97ffb37\\" \\"soundfile==0.13.1\\" \\"python-multipart==0.0.20\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n",\n        "\\n",\n        "# Colab can retain an older torchvision after torch is upgraded. Transformers\\n",\n        "# then masks the binary mismatch as a missing PreTrainedModel/Qwen3 class.\\n",\n        "import importlib.metadata as package_metadata\\n",\n        "import traceback\\n",\n        "\\n",\n        "import torch\\n",\n        "import torchvision\\n",\n        "\\n",\n        "print(\\"PyTorch stack:\\", torch.__version__, torchvision.__version__)\\n",\n        "assert torch.cuda.is_available(), \\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU.\\"\\n",\n        "assert package_metadata.version(\\"torchvision\\").split(\\"+\\")[0] == \\"0.23.0\\", \\"VieNeu requires torchvision 0.23.0 with torch 2.8.0.\\"\\n",\n        "try:\\n",\n        "    from transformers import PreTrainedModel\\n",\n        "    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM\\n",\n        "except Exception as error:\\n",\n        "    traceback.print_exc()\\n",\n        "    raise RuntimeError(\\n",\n        "        \\"The Colab PyTorch/Transformers stack is not importable for VieNeu. \\"\\n",\n        "        \\"Restart the runtime, rerun this install cell, then run all cells again.\\"\\n",\n        "    ) from error\\n",\n        "print(\\"Transformers imports verified for VieNeu:\\", PreTrainedModel.__name__, Qwen3ForCausalLM.__name__)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_clone_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom vieneu import Vieneu\\\\n\\\\nMODEL_ID = \\"vieneu-tts-v3-turbo\\"\\\\nMODEL_NAME = \\"VieNeu-TTS v3 Turbo\\"\\\\nUPSTREAM_MODEL = \\"pnnbao-ump/VieNeu-TTS-v3-Turbo\\"\\\\nMODEL = Vieneu(mode=\\"v3turbo\\", device=\\"cuda\\", backend=\\"pytorch\\", backbone_repo=UPSTREAM_MODEL)\\\\n\\\\ndef prepare_exact_profile(profile):\\\\n    return MODEL.encode_reference(profile[\\"ref_audio\\"], denoise=True)\\\\n\\\\ndef clone_with_exact_model(profile, request):\\\\n    speaker_emb, ref_codes = profile[\\"state\\"]\\\\n    voice = {\\"speaker_emb\\": speaker_emb, \\"codes\\": ref_codes}\\\\n    audio = MODEL.infer(text=request.text, voice=voice, denoise=False)\\\\n    return audio, 48000\\\\n\\\\nimport io\\\\nimport os\\\\nimport shutil\\\\nimport tempfile\\\\nimport threading\\\\nimport uuid\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\\"]\\\\nDATA_DIR = Path(\\"/content/la-studio-voice-clone-data\\") / MODEL_ID\\\\nDATA_DIR.mkdir(parents=True, exist_ok=True)\\\\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nMODEL_LOCK = threading.Lock()\\\\nSTATE_LOCK = threading.Lock()\\\\nPROFILES = {}\\\\nJOBS = {}\\\\n\\\\nclass GenerationRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    profile_id: str = Field(min_length=1, max_length=160)\\\\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    language: str = Field(default=\\"vi\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\\\\n    num_step: int = Field(default=32, ge=1, le=64)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(model: str) -> None:\\\\n    if model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef write_wav(path: Path, value, sample_rate: int) -> None:\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise RuntimeError(\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    sf.write(path, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n\\\\ndef public_job(job_id: str):\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        return {key: value for key, value in job.items() if key not in {\\"audio_path\\", \\"cancelled\\"}}\\\\n\\\\ndef fail_job(job_id: str, error: Exception) -> None:\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if job:\\\\n            job.update({\\\\n                \\"status\\": \\"failed\\",\\\\n                \\"stage\\": \\"failed\\",\\\\n                \\"error\\": {\\"message\\": f\\"{type(error).__name__}: {str(error)[:300]}\\"},\\\\n            })\\\\n\\\\ndef build_profile(job_id: str, profile_id: str) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES[profile_id]\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"prepare_profile\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            state = prepare_exact_profile(profile)\\\\n        with STATE_LOCK:\\\\n            profile[\\"state\\"] = state\\\\n            JOBS[job_id].update({\\\\n                \\"status\\": \\"succeeded\\",\\\\n                \\"stage\\": \\"complete\\",\\\\n                \\"percent\\": 100,\\\\n                \\"result\\": {\\"id\\": profile_id, \\"model\\": MODEL_ID},\\\\n            })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES.get(request.profile_id)\\\\n            if not profile:\\\\n                raise RuntimeError(\\"voice profile no longer exists\\")\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"generate\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            audio, sample_rate = clone_with_exact_model(profile, request)\\\\n        output_path = DATA_DIR / f\\"{job_id}.wav\\"\\\\n        write_wav(output_path, audio, sample_rate)\\\\n        with STATE_LOCK:\\\\n            if JOBS[job_id].get(\\"cancelled\\"):\\\\n                JOBS[job_id].update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n                output_path.unlink(missing_ok=True)\\\\n            else:\\\\n                JOBS[job_id].update({\\\\n                    \\"status\\": \\"succeeded\\",\\\\n                    \\"stage\\": \\"complete\\",\\\\n                    \\"percent\\": 100,\\\\n                    \\"audio_path\\": str(output_path),\\\\n                    \\"result\\": {\\"model\\": MODEL_ID, \\"sample_rate\\": int(sample_rate)},\\\\n                })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Cloning - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-cloning\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"reference_formats\\": [\\"wav\\", \\"mp3\\", \\"flac\\"],\\\\n                \\"reference_duration_seconds\\": {\\"min\\": 3, \\"max\\": 30},\\\\n                \\"requires_consent\\": True,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v2/jobs/profile\\", status_code=202)\\\\nasync def create_profile(\\\\n    model: str = Form(...),\\\\n    name: str = Form(...),\\\\n    consent_confirmed: bool = Form(...),\\\\n    ref_text: str = Form(default=\\"\\"),\\\\n    language: str = Form(default=\\"vi\\"),\\\\n    separate_music: bool = Form(default=False),\\\\n    ref_audio: UploadFile = File(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if not consent_confirmed:\\\\n        raise HTTPException(status_code=403, detail=\\"explicit voice-cloning consent is required\\")\\\\n    if not name.strip():\\\\n        raise HTTPException(status_code=422, detail=\\"profile name is required\\")\\\\n    suffix = Path(ref_audio.filename or \\"\\").suffix.lower()\\\\n    if suffix not in {\\".wav\\", \\".mp3\\", \\".flac\\"}:\\\\n        raise HTTPException(status_code=415, detail=\\"reference audio must be WAV, MP3, or FLAC\\")\\\\n    profile_id = uuid.uuid4().hex\\\\n    reference_path = DATA_DIR / f\\"{profile_id}{suffix}\\"\\\\n    size = 0\\\\n    with reference_path.open(\\"wb\\") as output:\\\\n        while chunk := await ref_audio.read(1024 * 1024):\\\\n            size += len(chunk)\\\\n            if size > MAX_REFERENCE_BYTES:\\\\n                reference_path.unlink(missing_ok=True)\\\\n                raise HTTPException(status_code=413, detail=\\"reference audio exceeds 256 MB\\")\\\\n            output.write(chunk)\\\\n    try:\\\\n        info = sf.info(reference_path)\\\\n        duration = float(info.frames) / float(info.samplerate)\\\\n    except Exception as error:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=f\\"reference audio cannot be decoded: {error}\\") from error\\\\n    if duration < 3.0 or duration > 30.0:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=\\"reference audio must be between 3 and 30 seconds\\")\\\\n    job_id = uuid.uuid4().hex\\\\n    profile = {\\\\n        \\"id\\": profile_id,\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"name\\": name.strip(),\\\\n        \\"ref_audio\\": str(reference_path),\\\\n        \\"ref_text\\": ref_text.strip(),\\\\n        \\"language\\": language.strip() or \\"vi\\",\\\\n        \\"separate_music\\": bool(separate_music),\\\\n    }\\\\n    with STATE_LOCK:\\\\n        PROFILES[profile_id] = profile\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.post(\\"/v2/jobs/generation\\", status_code=202)\\\\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    require_exact_model(request.model)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.get(request.profile_id)\\\\n        if not profile:\\\\n            raise HTTPException(status_code=404, detail=\\"voice profile not found\\")\\\\n        if profile[\\"model\\"] != MODEL_ID:\\\\n            raise HTTPException(status_code=409, detail=\\"voice profile belongs to a different model worker\\")\\\\n        job_id = uuid.uuid4().hex\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}/audio\\")\\\\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        path = Path(job.get(\\"audio_path\\", \\"\\")) if job else None\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n    if job.get(\\"status\\") != \\"succeeded\\" or not path or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"voice job audio is not ready\\")\\\\n    return Response(path.read_bytes(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}\\")\\\\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return public_job(job_id)\\\\n\\\\n@app.delete(\\"/v2/jobs/{job_id}\\")\\\\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        job[\\"cancelled\\"] = True\\\\n        if job[\\"status\\"] == \\"queued\\":\\\\n            job.update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n    return {\\"cancelled\\": True}\\\\n\\\\n@app.delete(\\"/v1/profiles/{profile_id}\\")\\\\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.pop(profile_id, None)\\\\n    if profile:\\\\n        Path(profile[\\"ref_audio\\"]).unlink(missing_ok=True)\\\\n    return {\\"deleted\\": bool(profile)}\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Cloning\'\\n",\n        "MODEL_ID = \'vieneu-tts-v3-turbo\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_clone_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_clone_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_clone_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-cloning",\n      "family_id": "vieneu-tts-v3-turbo",\n      "upstream_model": "pnnbao-ump/VieNeu-TTS-v3-Turbo",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_cloning/LA_STUDIO_VOICE_CLONE_VOXCPM2_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Cloning - VoxCPM2\\n",\n        "\\n",\n        "This notebook loads exactly `voxcpm2` (`openbmb/VoxCPM2`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Cloning panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "!git clone --quiet https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM\\n",\n        "!git -C /content/VoxCPM checkout --quiet 616d3d3e630a\\n",\n        "%pip install -q -e /content/VoxCPM \\"soundfile==0.13.1\\" \\"python-multipart==0.0.20\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_clone_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom voxcpm import VoxCPM\\\\n\\\\nMODEL_ID = \\"voxcpm2\\"\\\\nMODEL_NAME = \\"VoxCPM2\\"\\\\nUPSTREAM_MODEL = \\"openbmb/VoxCPM2\\"\\\\nMODEL = VoxCPM.from_pretrained(UPSTREAM_MODEL, load_denoiser=False, optimize=True, device=\\"cuda\\")\\\\n\\\\ndef prepare_exact_profile(profile):\\\\n    return {\\"ref_audio\\": profile[\\"ref_audio\\"], \\"ref_text\\": profile[\\"ref_text\\"]}\\\\n\\\\ndef clone_with_exact_model(profile, request):\\\\n    kwargs = {\\\\n        \\"text\\": request.text,\\\\n        \\"reference_wav_path\\": profile[\\"state\\"][\\"ref_audio\\"],\\\\n        \\"cfg_value\\": 2.0,\\\\n        \\"inference_timesteps\\": max(1, min(request.num_step, 50)),\\\\n    }\\\\n    # VoxCPM can clone from reference audio alone. An optional transcript adds\\\\n    # the stronger prompt-guided mode when the user provides one.\\\\n    if profile[\\"state\\"][\\"ref_text\\"]:\\\\n        kwargs.update({\\\\n            \\"prompt_wav_path\\": profile[\\"state\\"][\\"ref_audio\\"],\\\\n            \\"prompt_text\\": profile[\\"state\\"][\\"ref_text\\"],\\\\n        })\\\\n    audio = MODEL.generate(**kwargs)\\\\n    return audio, int(MODEL.tts_model.sample_rate)\\\\n\\\\nimport io\\\\nimport os\\\\nimport shutil\\\\nimport tempfile\\\\nimport threading\\\\nimport uuid\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\\"]\\\\nDATA_DIR = Path(\\"/content/la-studio-voice-clone-data\\") / MODEL_ID\\\\nDATA_DIR.mkdir(parents=True, exist_ok=True)\\\\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nMODEL_LOCK = threading.Lock()\\\\nSTATE_LOCK = threading.Lock()\\\\nPROFILES = {}\\\\nJOBS = {}\\\\n\\\\nclass GenerationRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    profile_id: str = Field(min_length=1, max_length=160)\\\\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    language: str = Field(default=\\"vi\\", max_length=40)\\\\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\\\\n    num_step: int = Field(default=32, ge=1, le=64)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(model: str) -> None:\\\\n    if model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef write_wav(path: Path, value, sample_rate: int) -> None:\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise RuntimeError(\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    sf.write(path, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n\\\\ndef public_job(job_id: str):\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        return {key: value for key, value in job.items() if key not in {\\"audio_path\\", \\"cancelled\\"}}\\\\n\\\\ndef fail_job(job_id: str, error: Exception) -> None:\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if job:\\\\n            job.update({\\\\n                \\"status\\": \\"failed\\",\\\\n                \\"stage\\": \\"failed\\",\\\\n                \\"error\\": {\\"message\\": f\\"{type(error).__name__}: {str(error)[:300]}\\"},\\\\n            })\\\\n\\\\ndef build_profile(job_id: str, profile_id: str) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES[profile_id]\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"prepare_profile\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            state = prepare_exact_profile(profile)\\\\n        with STATE_LOCK:\\\\n            profile[\\"state\\"] = state\\\\n            JOBS[job_id].update({\\\\n                \\"status\\": \\"succeeded\\",\\\\n                \\"stage\\": \\"complete\\",\\\\n                \\"percent\\": 100,\\\\n                \\"result\\": {\\"id\\": profile_id, \\"model\\": MODEL_ID},\\\\n            })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\\\\n    try:\\\\n        with STATE_LOCK:\\\\n            profile = PROFILES.get(request.profile_id)\\\\n            if not profile:\\\\n                raise RuntimeError(\\"voice profile no longer exists\\")\\\\n            JOBS[job_id].update({\\"status\\": \\"running\\", \\"stage\\": \\"generate\\", \\"percent\\": 10})\\\\n        with MODEL_LOCK:\\\\n            audio, sample_rate = clone_with_exact_model(profile, request)\\\\n        output_path = DATA_DIR / f\\"{job_id}.wav\\"\\\\n        write_wav(output_path, audio, sample_rate)\\\\n        with STATE_LOCK:\\\\n            if JOBS[job_id].get(\\"cancelled\\"):\\\\n                JOBS[job_id].update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n                output_path.unlink(missing_ok=True)\\\\n            else:\\\\n                JOBS[job_id].update({\\\\n                    \\"status\\": \\"succeeded\\",\\\\n                    \\"stage\\": \\"complete\\",\\\\n                    \\"percent\\": 100,\\\\n                    \\"audio_path\\": str(output_path),\\\\n                    \\"result\\": {\\"model\\": MODEL_ID, \\"sample_rate\\": int(sample_rate)},\\\\n                })\\\\n    except Exception as error:\\\\n        fail_job(job_id, error)\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Cloning - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-cloning\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"reference_formats\\": [\\"wav\\", \\"mp3\\", \\"flac\\"],\\\\n                \\"reference_duration_seconds\\": {\\"min\\": 3, \\"max\\": 30},\\\\n                \\"requires_consent\\": True,\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v2/jobs/profile\\", status_code=202)\\\\nasync def create_profile(\\\\n    model: str = Form(...),\\\\n    name: str = Form(...),\\\\n    consent_confirmed: bool = Form(...),\\\\n    ref_text: str = Form(default=\\"\\"),\\\\n    language: str = Form(default=\\"vi\\"),\\\\n    separate_music: bool = Form(default=False),\\\\n    ref_audio: UploadFile = File(...),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if not consent_confirmed:\\\\n        raise HTTPException(status_code=403, detail=\\"explicit voice-cloning consent is required\\")\\\\n    if not name.strip():\\\\n        raise HTTPException(status_code=422, detail=\\"profile name is required\\")\\\\n    suffix = Path(ref_audio.filename or \\"\\").suffix.lower()\\\\n    if suffix not in {\\".wav\\", \\".mp3\\", \\".flac\\"}:\\\\n        raise HTTPException(status_code=415, detail=\\"reference audio must be WAV, MP3, or FLAC\\")\\\\n    profile_id = uuid.uuid4().hex\\\\n    reference_path = DATA_DIR / f\\"{profile_id}{suffix}\\"\\\\n    size = 0\\\\n    with reference_path.open(\\"wb\\") as output:\\\\n        while chunk := await ref_audio.read(1024 * 1024):\\\\n            size += len(chunk)\\\\n            if size > MAX_REFERENCE_BYTES:\\\\n                reference_path.unlink(missing_ok=True)\\\\n                raise HTTPException(status_code=413, detail=\\"reference audio exceeds 256 MB\\")\\\\n            output.write(chunk)\\\\n    try:\\\\n        info = sf.info(reference_path)\\\\n        duration = float(info.frames) / float(info.samplerate)\\\\n    except Exception as error:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=f\\"reference audio cannot be decoded: {error}\\") from error\\\\n    if duration < 3.0 or duration > 30.0:\\\\n        reference_path.unlink(missing_ok=True)\\\\n        raise HTTPException(status_code=422, detail=\\"reference audio must be between 3 and 30 seconds\\")\\\\n    job_id = uuid.uuid4().hex\\\\n    profile = {\\\\n        \\"id\\": profile_id,\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"name\\": name.strip(),\\\\n        \\"ref_audio\\": str(reference_path),\\\\n        \\"ref_text\\": ref_text.strip(),\\\\n        \\"language\\": language.strip() or \\"vi\\",\\\\n        \\"separate_music\\": bool(separate_music),\\\\n    }\\\\n    with STATE_LOCK:\\\\n        PROFILES[profile_id] = profile\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.post(\\"/v2/jobs/generation\\", status_code=202)\\\\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    require_exact_model(request.model)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.get(request.profile_id)\\\\n        if not profile:\\\\n            raise HTTPException(status_code=404, detail=\\"voice profile not found\\")\\\\n        if profile[\\"model\\"] != MODEL_ID:\\\\n            raise HTTPException(status_code=409, detail=\\"voice profile belongs to a different model worker\\")\\\\n        job_id = uuid.uuid4().hex\\\\n        JOBS[job_id] = {\\"id\\": job_id, \\"status\\": \\"queued\\", \\"stage\\": \\"queued\\", \\"percent\\": 0}\\\\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\\\\n    return public_job(job_id)\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}/audio\\")\\\\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        path = Path(job.get(\\"audio_path\\", \\"\\")) if job else None\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n    if job.get(\\"status\\") != \\"succeeded\\" or not path or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"voice job audio is not ready\\")\\\\n    return Response(path.read_bytes(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\n@app.get(\\"/v2/jobs/{job_id}\\")\\\\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return public_job(job_id)\\\\n\\\\n@app.delete(\\"/v2/jobs/{job_id}\\")\\\\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        job = JOBS.get(job_id)\\\\n        if not job:\\\\n            raise HTTPException(status_code=404, detail=\\"voice job not found\\")\\\\n        job[\\"cancelled\\"] = True\\\\n        if job[\\"status\\"] == \\"queued\\":\\\\n            job.update({\\"status\\": \\"cancelled\\", \\"stage\\": \\"cancelled\\", \\"percent\\": 0})\\\\n    return {\\"cancelled\\": True}\\\\n\\\\n@app.delete(\\"/v1/profiles/{profile_id}\\")\\\\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with STATE_LOCK:\\\\n        profile = PROFILES.pop(profile_id, None)\\\\n    if profile:\\\\n        Path(profile[\\"ref_audio\\"]).unlink(missing_ok=True)\\\\n    return {\\"deleted\\": bool(profile)}\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Cloning\'\\n",\n        "MODEL_ID = \'voxcpm2\'\\n",\n        "PORT = 3923\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_CLONE_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_clone_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_clone_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_clone_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-cloning",\n      "family_id": "voxcpm2",\n      "upstream_model": "openbmb/VoxCPM2",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_design/LA_STUDIO_VOICE_DESIGN_OMNIVOICE_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Design - OmniVoice\\n",\n        "\\n",\n        "This notebook loads exactly `omnivoice` (`k2-fsa/OmniVoice`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Design panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"git+https://github.com/k2-fsa/OmniVoice.git@468e927ba371\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_design_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom omnivoice import OmniVoice\\\\n\\\\nMODEL_ID = \\"omnivoice\\"\\\\nMODEL_NAME = \\"OmniVoice\\"\\\\nUPSTREAM_MODEL = \\"k2-fsa/OmniVoice\\"\\\\nMODEL = OmniVoice.from_pretrained(UPSTREAM_MODEL, device_map=\\"cuda:0\\", dtype=torch.float16)\\\\n\\\\ndef design_with_exact_model(request):\\\\n    instruction = \\", \\".join(part for part in (request.voice_description.strip(), request.style.strip()) if part)\\\\n    kwargs = {\\"text\\": request.input, \\"instruct\\": instruction}\\\\n    if request.language.strip().lower() not in (\\"\\", \\"auto\\"):\\\\n        kwargs[\\"language_id\\"] = request.language.strip().lower()\\\\n    audio = MODEL.generate(**kwargs)\\\\n    return audio, 24000\\\\n\\\\nimport io\\\\nimport os\\\\nimport re\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass VoiceDesignRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice_description: str = Field(min_length=1, max_length=2000)\\\\n    style: str = Field(default=\\"\\", max_length=1000)\\\\n    language: str = Field(default=\\"en\\", max_length=40)\\\\n    temperature: float = Field(default=0.9, ge=0.1, le=2.0)\\\\n    seed: int = Field(default=-1, ge=-1)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef wav_response(value, sample_rate: int):\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Design - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-design\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/voice_designs\\")\\\\ndef voice_design(request: VoiceDesignRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab voice-design worker is busy; retry shortly\\")\\\\n    try:\\\\n        audio, sample_rate = design_with_exact_model(request)\\\\n        return wav_response(audio, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} voice design failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Design\'\\n",\n        "MODEL_ID = \'omnivoice\'\\n",\n        "PORT = 3924\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_design_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_design_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_design_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-design",\n      "family_id": "omnivoice",\n      "upstream_model": "k2-fsa/OmniVoice",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_design/LA_STUDIO_VOICE_DESIGN_QWEN3_1_7B_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Design - Qwen3-TTS VoiceDesign 1.7B\\n",\n        "\\n",\n        "This notebook loads exactly `qwen3-tts-1.7b-voicedesign` (`Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Design panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"qwen-tts==0.1.1\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_design_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom qwen_tts import Qwen3TTSModel\\\\n\\\\nMODEL_ID = \\"qwen3-tts-1.7b-voicedesign\\"\\\\nMODEL_NAME = \\"Qwen3-TTS VoiceDesign 1.7B\\"\\\\nUPSTREAM_MODEL = \\"Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign\\"\\\\nMODEL = Qwen3TTSModel.from_pretrained(\\\\n    UPSTREAM_MODEL,\\\\n    device_map=\\"cuda:0\\",\\\\n    dtype=torch.bfloat16,\\\\n    attn_implementation=\\"sdpa\\",\\\\n)\\\\n\\\\ndef qwen_language(value: str):\\\\n    mapping = {\\"auto\\": \\"Auto\\", \\"zh\\": \\"Chinese\\", \\"en\\": \\"English\\", \\"ja\\": \\"Japanese\\", \\"ko\\": \\"Korean\\", \\"de\\": \\"German\\", \\"fr\\": \\"French\\", \\"ru\\": \\"Russian\\", \\"pt\\": \\"Portuguese\\", \\"es\\": \\"Spanish\\", \\"it\\": \\"Italian\\"}\\\\n    return mapping.get(value.strip().lower(), value.strip().title() or \\"Auto\\")\\\\n\\\\ndef design_with_exact_model(request):\\\\n    instruction = \\", \\".join(part for part in (request.voice_description.strip(), request.style.strip()) if part)\\\\n    if request.seed >= 0:\\\\n        torch.manual_seed(request.seed)\\\\n        torch.cuda.manual_seed_all(request.seed)\\\\n    wavs, sample_rate = MODEL.generate_voice_design(\\\\n        text=request.input,\\\\n        language=qwen_language(request.language),\\\\n        instruct=instruction,\\\\n        temperature=request.temperature,\\\\n    )\\\\n    return wavs[0], sample_rate\\\\n\\\\nimport io\\\\nimport os\\\\nimport re\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass VoiceDesignRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice_description: str = Field(min_length=1, max_length=2000)\\\\n    style: str = Field(default=\\"\\", max_length=1000)\\\\n    language: str = Field(default=\\"en\\", max_length=40)\\\\n    temperature: float = Field(default=0.9, ge=0.1, le=2.0)\\\\n    seed: int = Field(default=-1, ge=-1)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef wav_response(value, sample_rate: int):\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Design - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-design\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/voice_designs\\")\\\\ndef voice_design(request: VoiceDesignRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab voice-design worker is busy; retry shortly\\")\\\\n    try:\\\\n        audio, sample_rate = design_with_exact_model(request)\\\\n        return wav_response(audio, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} voice design failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Design\'\\n",\n        "MODEL_ID = \'qwen3-tts-1.7b-voicedesign\'\\n",\n        "PORT = 3924\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_design_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_design_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_design_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-design",\n      "family_id": "qwen3-tts-1.7b-voicedesign",\n      "upstream_model": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_design/LA_STUDIO_VOICE_DESIGN_VOXCPM2_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio Voice Design - VoxCPM2\\n",\n        "\\n",\n        "This notebook loads exactly `voxcpm2` (`openbmb/VoxCPM2`) on CUDA.\\n",\n        "It is independent from API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio\'s Voice Design panel.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "!git clone --quiet https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM\\n",\n        "!git -C /content/VoxCPM checkout --quiet 616d3d3e630a\\n",\n        "%pip install -q -e /content/VoxCPM \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\"\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_voice_design_worker.py\')\\n",\n        "WORKER.write_text(\'import torch\\\\n\\\\nfrom voxcpm import VoxCPM\\\\n\\\\nMODEL_ID = \\"voxcpm2\\"\\\\nMODEL_NAME = \\"VoxCPM2\\"\\\\nUPSTREAM_MODEL = \\"openbmb/VoxCPM2\\"\\\\nMODEL = VoxCPM.from_pretrained(UPSTREAM_MODEL, load_denoiser=False, optimize=True, device=\\"cuda\\")\\\\n\\\\ndef design_with_exact_model(request):\\\\n    instruction = \\", \\".join(part for part in (request.voice_description.strip(), request.style.strip()) if part)\\\\n    instruction = re.sub(r\\"[()\\uff08\\uff09]\\", \\"\\", instruction).strip()\\\\n    text = f\\"({instruction}){request.input}\\"\\\\n    audio = MODEL.generate(\\\\n        text=text,\\\\n        cfg_value=2.0,\\\\n        inference_timesteps=10,\\\\n        seed=None if request.seed < 0 else request.seed,\\\\n    )\\\\n    return audio, int(MODEL.tts_model.sample_rate)\\\\n\\\\nimport io\\\\nimport os\\\\nimport re\\\\nimport threading\\\\n\\\\nimport numpy as np\\\\nimport soundfile as sf\\\\nimport torch\\\\nfrom fastapi import FastAPI, Header, HTTPException\\\\nfrom fastapi.responses import Response\\\\nfrom pydantic import BaseModel, Field\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.\\")\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN\\"]\\\\nMAX_INPUT_CHARS = 4000\\\\nMAX_OUTPUT_SECONDS = 300\\\\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\\\\n\\\\nclass VoiceDesignRequest(BaseModel):\\\\n    model: str = Field(min_length=1, max_length=120)\\\\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\\\\n    voice_description: str = Field(min_length=1, max_length=2000)\\\\n    style: str = Field(default=\\"\\", max_length=1000)\\\\n    language: str = Field(default=\\"en\\", max_length=40)\\\\n    temperature: float = Field(default=0.9, ge=0.1, le=2.0)\\\\n    seed: int = Field(default=-1, ge=-1)\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef audio_array(value):\\\\n    if isinstance(value, (list, tuple)):\\\\n        if not value:\\\\n            raise RuntimeError(\\"the selected model returned no audio\\")\\\\n        value = value[0]\\\\n    if torch.is_tensor(value):\\\\n        value = value.detach().float().cpu().numpy()\\\\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\\\\n    if audio.size == 0 or not np.isfinite(audio).all():\\\\n        raise RuntimeError(\\"the selected model returned invalid audio\\")\\\\n    return audio\\\\n\\\\ndef wav_response(value, sample_rate: int):\\\\n    audio = audio_array(value)\\\\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\\\\n        raise HTTPException(status_code=413, detail=\\"generated audio exceeds the five minute output limit\\")\\\\n    peak = float(np.max(np.abs(audio)))\\\\n    if peak > 1.2:\\\\n        audio = audio / peak\\\\n    output = io.BytesIO()\\\\n    sf.write(output, audio, int(sample_rate), format=\\"WAV\\", subtype=\\"PCM_16\\")\\\\n    return Response(output.getvalue(), media_type=\\"audio/wav\\", headers={\\"Cache-Control\\": \\"no-store\\"})\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Design - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-design\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"formats\\": [\\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/voice_designs\\")\\\\ndef voice_design(request: VoiceDesignRequest, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if request.model.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{request.model}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n    if not REQUEST_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab voice-design worker is busy; retry shortly\\")\\\\n    try:\\\\n        audio, sample_rate = design_with_exact_model(request)\\\\n        return wav_response(audio, sample_rate)\\\\n    except HTTPException:\\\\n        raise\\\\n    except Exception as error:\\\\n        raise HTTPException(\\\\n            status_code=503,\\\\n            detail=f\\"{MODEL_NAME} voice design failed: {type(error).__name__}: {str(error)[:240]}\\",\\\\n        ) from error\\\\n    finally:\\\\n        REQUEST_SLOTS.release()\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Design\'\\n",\n        "MODEL_ID = \'voxcpm2\'\\n",\n        "PORT = 3924\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_VOICE_DESIGN_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_voice_design_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_voice_design_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_voice_design_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-design",\n      "family_id": "voxcpm2",\n      "upstream_model": "openbmb/VoxCPM2",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/voice_separation/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_GPU.ipynb': '{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "# LA Studio voice-isolation - Spleeter 2-stem FP16\\n",\n    "\\n",\n    "This notebook runs exactly `sherpa-onnx-spleeter-2stems-fp16` from the declared k2-fsa artifact on the temporary **Colab GPU worker**. The LA Studio worker and launcher are embedded in this notebook; no LA Studio GitHub repository or repository token is required at runtime.\\n",\n    "\\n",\n    "The worker performs a CUDA startup probe before it prints a URL. It also sends long audio as bounded, overlapping segments, so the Spleeter FP16 CUDA convolution plan remains within the verified shape.\\n",\n    "\\n",\n    "1. Choose **Runtime -> Change runtime type -> GPU**.\\n",\n    "2. Run all cells. The final cell must print `startup probe: passed`.\\n",\n    "3. Copy the printed URL and token to Dubbing -> Colab setup, then press **Check Colab**.\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "!nvidia-smi\\n",\n    "%pip install -q --upgrade --no-cache-dir \\"onnxruntime-gpu==1.21.0\\" \\"kaldi-native-fbank\\" \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"python-multipart==0.0.20\\"\\n",\n    "\\n",\n    "import torch\\n",\n    "if not torch.cuda.is_available():\\n",\n    "    raise RuntimeError(\'No Colab CUDA GPU is available. Select Runtime > Change runtime type > GPU, then restart and Run all.\')\\n",\n    "print(\'Colab CUDA:\', torch.cuda.get_device_name(0))\\n",\n    "\\n",\n    "!wget -q --show-progress -O /content/spleeter.tar.bz2 https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2\\n",\n    "!tar -xjf /content/spleeter.tar.bz2 -C /content\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "from hashlib import sha256\\n",\n    "from pathlib import Path\\n",\n    "\\n",\n    "EMBEDDED_WORKERS = {\\n",\n    "    \'la_studio_separation_worker.py\': (\'307861926e13ff9849b04594074b573b1da063b1791c56dc2f502ab64991c5af\', \'\\"\\"\\"Temporary Direct Colab worker for the exact Spleeter 2-stem FP16 artifact.\\\\n\\\\nThe worker deliberately uses ONNX Runtime\\\\\'s CUDA provider directly.  The\\\\nsherpa-onnx source-separation wrapper fixes its CUDA convolution search to\\\\nHEURISTIC, which fails on some current Colab cuDNN 9 images.  This worker uses\\\\nthe same upstream FP16 ONNX files, but chooses ORT\\\\\'s documented DEFAULT\\\\nconvolution algorithm instead and bounds every inference input.\\\\n\\"\\"\\"\\\\n\\\\nimport math\\\\nimport os\\\\nimport secrets\\\\nimport shutil\\\\nimport subprocess\\\\nimport threading\\\\nimport traceback\\\\nfrom pathlib import Path\\\\n\\\\nimport kaldi_native_fbank as knf\\\\nimport numpy as np\\\\nimport torch\\\\nimport onnxruntime as ort\\\\nimport soundfile as sf\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import FileResponse\\\\n\\\\n\\\\nWORKER_CONTRACT = \\"spleeter-cuda-safe-20260816.1\\"\\\\nMODEL_ID = \\"sherpa-onnx-spleeter-2stems-fp16\\"\\\\nMODEL_NAME = \\"Spleeter 2-stem FP16\\"\\\\nUPSTREAM_MODEL = \\"k2-fsa/sherpa-onnx-spleeter-2stems-fp16\\"\\\\nARTIFACT_URL = (\\\\n    \\"https://github.com/k2-fsa/sherpa-onnx/releases/download/\\"\\\\n    \\"source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2\\"\\\\n)\\\\nMODEL_ROOT = Path(\\"/content/sherpa-onnx-spleeter-2stems-fp16\\")\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_SEPARATION_TOKEN\\"]\\\\nROOT = Path(\\"/content/la-studio-separation-jobs\\") / MODEL_ID\\\\nROOT.mkdir(parents=True, exist_ok=True)\\\\n\\\\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\\\\nMAX_AUDIO_SECONDS = 30 * 60\\\\nARTIFACT_TTL_SECONDS = 1800\\\\n# A 20-second core has at most two 512-frame Spleeter splits.  The previous\\\\n# worker sent a complete long video through one CUDA call, producing a large\\\\n# dynamic Conv shape that caused CUDNN_FE_HEURISTIC_QUERY_FAILED on Colab.\\\\nCORE_SECONDS = 20.0\\\\nCONTEXT_SECONDS = 1.5\\\\nPROBE_SECONDS = CORE_SECONDS\\\\n\\\\nALLOWED_CONTENT_TYPES = {\\\\n    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp4\\", \\"audio/webm\\",\\\\n    \\"audio/ogg\\", \\"audio/flac\\", \\"video/mp4\\", \\"video/webm\\", \\"video/quicktime\\",\\\\n    \\"video/x-matroska\\", \\"application/octet-stream\\",\\\\n}\\\\nALLOWED_EXTENSIONS = {\\\\n    \\".wav\\", \\".mp3\\", \\".m4a\\", \\".mp4\\", \\".webm\\", \\".ogg\\", \\".flac\\", \\".mkv\\", \\".mov\\", \\".avi\\",\\\\n}\\\\nCUDA_OPTIONS = {\\\\n    # DEFAULT avoids the cuDNN heuristic-plan query that failed in the old\\\\n    # sherpa-onnx wrapper.  The exact model remains FP16 and executes on CUDA.\\\\n    \\"cudnn_conv_algo_search\\": \\"DEFAULT\\",\\\\n    \\"cudnn_conv_use_max_workspace\\": \\"1\\",\\\\n    \\"do_copy_in_default_stream\\": \\"1\\",\\\\n    \\"arena_extend_strategy\\": \\"kSameAsRequested\\",\\\\n}\\\\n\\\\nif not torch.cuda.is_available():\\\\n    raise RuntimeError(\\"Colab GPU is not available; select a GPU runtime before starting this worker\\")\\\\n\\\\n\\\\ndef _cuda_session(path: Path) -> ort.InferenceSession:\\\\n    if \\"CUDAExecutionProvider\\" not in ort.get_available_providers():\\\\n        raise RuntimeError(\\"ONNX Runtime CUDAExecutionProvider is unavailable in this Colab runtime\\")\\\\n    options = ort.SessionOptions()\\\\n    options.intra_op_num_threads = 1\\\\n    options.inter_op_num_threads = 1\\\\n    session = ort.InferenceSession(\\\\n        str(path),\\\\n        sess_options=options,\\\\n        providers=[(\\"CUDAExecutionProvider\\", CUDA_OPTIONS), \\"CPUExecutionProvider\\"],\\\\n    )\\\\n    if not session.get_providers() or session.get_providers()[0] != \\"CUDAExecutionProvider\\":\\\\n        raise RuntimeError(\\"The exact Spleeter ONNX session did not bind CUDAExecutionProvider\\")\\\\n    return session\\\\n\\\\n\\\\nclass ExactSpleeterCuda:\\\\n    def __init__(self) -> None:\\\\n        vocals = MODEL_ROOT / \\"vocals.fp16.onnx\\"\\\\n        accompaniment = MODEL_ROOT / \\"accompaniment.fp16.onnx\\"\\\\n        if not vocals.is_file() or not accompaniment.is_file():\\\\n            raise RuntimeError(\\"The exact Spleeter FP16 ONNX artifacts are missing\\")\\\\n        self.vocals = _cuda_session(vocals)\\\\n        self.accompaniment = _cuda_session(accompaniment)\\\\n        self.stft_config = knf.StftConfig(\\\\n            n_fft=4096,\\\\n            hop_length=1024,\\\\n            win_length=4096,\\\\n            center=False,\\\\n            window_type=\\"hann\\",\\\\n        )\\\\n\\\\n    @staticmethod\\\\n    def _stft(samples: np.ndarray, channel: int) -> tuple[np.ndarray, np.ndarray]:\\\\n        result = knf.Stft(knf.StftConfig(\\\\n            n_fft=4096, hop_length=1024, win_length=4096,\\\\n            center=False, window_type=\\"hann\\",\\\\n        ))(samples[:, channel].tolist())\\\\n        real = np.asarray(result.real, dtype=np.float32).reshape(result.num_frames, -1)\\\\n        imag = np.asarray(result.imag, dtype=np.float32).reshape(result.num_frames, -1)\\\\n        return real, imag\\\\n\\\\n    def process(self, sample_rate: int, samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\\\\n        if sample_rate != 44100:\\\\n            raise RuntimeError(f\\"expected 44100 Hz worker input, received {sample_rate}\\")\\\\n        if samples.ndim != 2 or samples.shape[1] != 2 or samples.shape[0] == 0:\\\\n            raise RuntimeError(\\"expected non-empty stereo audio\\")\\\\n        real0, imag0 = self._stft(samples, 0)\\\\n        real1, imag1 = self._stft(samples, 1)\\\\n        if real0.shape[0] == 0 or real1.shape[0] == 0:\\\\n            raise RuntimeError(\\"audio is too short for Spleeter analysis\\")\\\\n        frame_count = real0.shape[0]\\\\n        if real1.shape[0] != frame_count:\\\\n            raise RuntimeError(\\"stereo channel frame counts differ\\")\\\\n\\\\n        magnitude0 = np.sqrt(real0[:, :1024] ** 2 + imag0[:, :1024] ** 2).astype(np.float32)\\\\n        magnitude1 = np.sqrt(real1[:, :1024] ** 2 + imag1[:, :1024] ** 2).astype(np.float32)\\\\n        padded_frames = int(math.ceil(frame_count / 512.0) * 512)\\\\n        if padded_frames != frame_count:\\\\n            padding = ((0, padded_frames - frame_count), (0, 0))\\\\n            magnitude0 = np.pad(magnitude0, padding)\\\\n            magnitude1 = np.pad(magnitude1, padding)\\\\n        model_input = np.ascontiguousarray(\\\\n            np.stack((magnitude0, magnitude1), axis=0).reshape(2, -1, 512, 1024),\\\\n            dtype=np.float32,\\\\n        )\\\\n        vocals_spec = self.vocals.run(None, {self.vocals.get_inputs()[0].name: model_input})[0]\\\\n        accompaniment_spec = self.accompaniment.run(\\\\n            None, {self.accompaniment.get_inputs()[0].name: model_input}\\\\n        )[0]\\\\n        denominator = vocals_spec ** 2 + accompaniment_spec ** 2 + 1e-10\\\\n        masks = (\\\\n            (vocals_spec ** 2 + 5e-11) / denominator,\\\\n            (accompaniment_spec ** 2 + 5e-11) / denominator,\\\\n        )\\\\n\\\\n        stems: list[np.ndarray] = []\\\\n        for mask in masks:\\\\n            channels: list[np.ndarray] = []\\\\n            for channel, (real, imag) in enumerate(((real0, imag0), (real1, imag1))):\\\\n                channel_mask = mask[channel].reshape(-1, 1024)[:frame_count]\\\\n                channel_mask = np.pad(channel_mask, ((0, 0), (0, real.shape[1] - 1024)))\\\\n                masked = knf.StftResult(\\\\n                    real=(channel_mask * real).reshape(-1).tolist(),\\\\n                    imag=(channel_mask * imag).reshape(-1).tolist(),\\\\n                    num_frames=frame_count,\\\\n                )\\\\n                waveform = knf.IStft(self.stft_config)(masked)\\\\n                channels.append(np.asarray(waveform, dtype=np.float32))\\\\n            stem = np.column_stack(channels)\\\\n            stems.append(stem)\\\\n        return stems[0], stems[1]\\\\n\\\\n\\\\n# Constructing and running the same bounded shape before /health is exposed\\\\n# proves CUDA works for this exact model.  An unsupported Colab image fails in\\\\n# the notebook cell, rather than accepting a URL and later failing at a random\\\\n# workflow step.\\\\nSEPARATOR = ExactSpleeterCuda()\\\\n_probe = np.zeros((int(44100 * PROBE_SECONDS), 2), dtype=np.float32)\\\\n_probe_vocals, _probe_background = SEPARATOR.process(44100, _probe)\\\\nif _probe_vocals.shape[0] == 0 or _probe_background.shape[0] == 0:\\\\n    raise RuntimeError(\\"exact Spleeter CUDA startup probe produced empty audio\\")\\\\ndel _probe, _probe_vocals, _probe_background\\\\n\\\\nJOB_SLOTS = threading.BoundedSemaphore(1)\\\\nJOB_LOCK = threading.Lock()\\\\nJOBS: dict[str, dict] = {}\\\\n\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=(f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. \\"\\\\n                    \\"Open the notebook for the selected model.\\"),\\\\n        )\\\\n\\\\n\\\\ndef media_duration_seconds(path: Path) -> float:\\\\n    probe = subprocess.run(\\\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-show_entries\\", \\"format=duration\\",\\\\n         \\"-of\\", \\"default=nokey=1:noprint_wrappers=1\\", str(path)],\\\\n        text=True, capture_output=True,\\\\n    )\\\\n    try:\\\\n        duration = float(probe.stdout.strip())\\\\n    except ValueError:\\\\n        duration = 0.0\\\\n    if probe.returncode != 0 or duration <= 0.0:\\\\n        raise HTTPException(status_code=415, detail=\\"media is unsupported or could not be decoded\\")\\\\n    return duration\\\\n\\\\n\\\\ndef update(job_id: str, **values) -> dict:\\\\n    with JOB_LOCK:\\\\n        job = dict(JOBS.get(job_id, {}))\\\\n        job.update(values)\\\\n        JOBS[job_id] = job\\\\n        return job\\\\n\\\\n\\\\ndef is_cancelled(job_id: str) -> bool:\\\\n    with JOB_LOCK:\\\\n        return bool(JOBS.get(job_id, {}).get(\\"cancel_requested\\", False))\\\\n\\\\n\\\\ndef cleanup(job_id: str) -> None:\\\\n    with JOB_LOCK:\\\\n        job = JOBS.pop(job_id, None)\\\\n    if job:\\\\n        shutil.rmtree(job.get(\\"directory\\", \\"\\"), ignore_errors=True)\\\\n\\\\n\\\\ndef _fit_piece(stem: np.ndarray, start: int, length: int) -> np.ndarray:\\\\n    piece = stem[start:start + length]\\\\n    if piece.shape[0] >= length:\\\\n        return piece[:length]\\\\n    return np.pad(piece, ((0, length - piece.shape[0]), (0, 0)))\\\\n\\\\n\\\\ndef separate_bounded(job_id: str, samples: np.ndarray, sample_rate: int) -> tuple[np.ndarray, np.ndarray]:\\\\n    core = int(CORE_SECONDS * sample_rate)\\\\n    context = int(CONTEXT_SECONDS * sample_rate)\\\\n    total = samples.shape[0]\\\\n    pieces = max(1, math.ceil(total / core))\\\\n    vocals_parts: list[np.ndarray] = []\\\\n    background_parts: list[np.ndarray] = []\\\\n    for index, core_start in enumerate(range(0, total, core), start=1):\\\\n        if is_cancelled(job_id):\\\\n            raise RuntimeError(\\"Separation cancelled\\")\\\\n        core_end = min(total, core_start + core)\\\\n        window_start = max(0, core_start - context)\\\\n        window_end = min(total, core_end + context)\\\\n        update(\\\\n            job_id,\\\\n            status=\\"running\\",\\\\n            progress=20 + int(65 * (index - 1) / pieces),\\\\n            detail=f\\"{MODEL_NAME} CUDA segment {index}/{pieces}\\",\\\\n        )\\\\n        vocals, background = SEPARATOR.process(sample_rate, samples[window_start:window_end])\\\\n        trim = core_start - window_start\\\\n        core_length = core_end - core_start\\\\n        vocals_parts.append(_fit_piece(vocals, trim, core_length))\\\\n        background_parts.append(_fit_piece(background, trim, core_length))\\\\n        update(\\\\n            job_id,\\\\n            status=\\"running\\",\\\\n            progress=20 + int(65 * index / pieces),\\\\n            detail=f\\"{MODEL_NAME} CUDA segment {index}/{pieces} complete\\",\\\\n        )\\\\n    return np.concatenate(vocals_parts, axis=0), np.concatenate(background_parts, axis=0)\\\\n\\\\n\\\\ndef concise_failure(error: Exception) -> str:\\\\n    text = str(error).replace(\\"\\\\\\\\n\\", \\" \\").strip()\\\\n    if \\"CUDNN\\" in text.upper() or \\"CUDA\\" in text.upper():\\\\n        return (\\"The verified Colab CUDA worker failed during Spleeter inference. \\"\\\\n                \\"No local model was started. Stop this job, reopen the current Spleeter notebook, \\"\\\\n                \\"and use its startup probe before reconnecting. Full worker detail is in the Colab output.\\")\\\\n    return f\\"{type(error).__name__}: {text[:600]}\\"\\\\n\\\\n\\\\ndef run_job(job_id: str, directory: Path, source: Path, output_format: str) -> None:\\\\n    try:\\\\n        update(job_id, status=\\"running\\", progress=12, detail=\\"Decoding media for bounded CUDA separation\\")\\\\n        wav_path = directory / \\"source-44100-stereo.wav\\"\\\\n        subprocess.run(\\\\n            [\\"ffmpeg\\", \\"-y\\", \\"-v\\", \\"error\\", \\"-i\\", str(source), \\"-vn\\", \\"-acodec\\", \\"pcm_s16le\\",\\\\n             \\"-ar\\", \\"44100\\", \\"-ac\\", \\"2\\", str(wav_path)],\\\\n            check=True,\\\\n        )\\\\n        samples, sample_rate = sf.read(wav_path, dtype=\\"float32\\", always_2d=True)\\\\n        samples = np.ascontiguousarray(samples, dtype=np.float32)\\\\n        vocals_data, background_data = separate_bounded(job_id, samples, sample_rate)\\\\n        if is_cancelled(job_id):\\\\n            update(job_id, status=\\"cancelled\\", progress=0, detail=\\"Separation cancelled\\")\\\\n            return\\\\n        update(job_id, status=\\"running\\", progress=90, detail=\\"Writing separated CUDA stems\\")\\\\n        suffix = \\".wav\\" if output_format == \\"wav\\" else \\".flac\\"\\\\n        vocals = directory / (\\"vocals\\" + suffix)\\\\n        background = directory / (\\"background\\" + suffix)\\\\n        # FLAC is lossless and typically reduces the 44.1 kHz stereo transfer\\\\n        # by far more than 50%; PCM WAV remains the explicit compatibility\\\\n        # choice for an operator who needs it.\\\\n        sf.write(vocals, vocals_data, sample_rate,\\\\n                 format=\\"WAV\\" if output_format == \\"wav\\" else \\"FLAC\\",\\\\n                 subtype=\\"PCM_16\\")\\\\n        sf.write(background, background_data, sample_rate,\\\\n                 format=\\"WAV\\" if output_format == \\"wav\\" else \\"FLAC\\",\\\\n                 subtype=\\"PCM_16\\")\\\\n        update(\\\\n            job_id, status=\\"ready\\", progress=100, detail=\\"Separated CUDA stems are ready\\",\\\\n            vocals=str(vocals), background=str(background),\\\\n            artifact_format=output_format, artifacts_ready=True,\\\\n        )\\\\n    except Exception as error:\\\\n        traceback.print_exc()\\\\n        update(job_id, status=\\"failed\\", progress=0, detail=concise_failure(error))\\\\n    finally:\\\\n        threading.Timer(ARTIFACT_TTL_SECONDS, cleanup, args=[job_id]).start()\\\\n        JOB_SLOTS.release()\\\\n\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Isolation - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"onnxruntime\\": ort.__version__,\\\\n        \\"cuda_provider_options\\": CUDA_OPTIONS,\\\\n        \\"bounded_core_seconds\\": CORE_SECONDS,\\\\n        \\"startup_probe\\": \\"passed\\",\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-isolation\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"artifact_url\\": ARTIFACT_URL,\\\\n                \\"stems\\": [\\"vocals\\", \\"background\\"],\\\\n                \\"formats\\": [\\"flac\\", \\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n\\\\n@app.post(\\"/v1/audio/separations\\")\\\\nasync def create_separation(\\\\n    file: UploadFile = File(...),\\\\n    stems: str = Form(\\"vocals,background\\"),\\\\n    model: str = Form(...),\\\\n    output_format: str = Form(\\"flac\\"),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if stems != \\"vocals,background\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns vocals and background stems\\")\\\\n    output_format = output_format.strip().lower()\\\\n    if output_format not in {\\"flac\\", \\"wav\\"}:\\\\n        raise HTTPException(status_code=422, detail=\\"output_format must be flac or wav\\")\\\\n    suffix = Path(file.filename or \\"source.wav\\").suffix.lower() or \\".wav\\"\\\\n    if suffix not in ALLOWED_EXTENSIONS:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported media filename extension\\")\\\\n    if file.content_type and file.content_type not in ALLOWED_CONTENT_TYPES:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported media MIME type\\")\\\\n    if not JOB_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab separation worker is busy; retry shortly\\")\\\\n    job_id = secrets.token_urlsafe(18)\\\\n    directory = ROOT / job_id\\\\n    directory.mkdir(parents=True, exist_ok=True)\\\\n    source = directory / (\\"source\\" + suffix)\\\\n    try:\\\\n        with source.open(\\"wb\\") as output:\\\\n            while chunk := await file.read(1024 * 1024):\\\\n                output.write(chunk)\\\\n                if output.tell() > MAX_UPLOAD_BYTES:\\\\n                    raise HTTPException(status_code=413, detail=\\"media exceeds 512 MB upload limit\\")\\\\n        if source.stat().st_size <= 0:\\\\n            raise HTTPException(status_code=413, detail=\\"media must not be empty\\")\\\\n        if media_duration_seconds(source) > MAX_AUDIO_SECONDS:\\\\n            raise HTTPException(status_code=413, detail=\\"media exceeds the 30 minute duration limit\\")\\\\n    except Exception:\\\\n        shutil.rmtree(directory, ignore_errors=True)\\\\n        JOB_SLOTS.release()\\\\n        raise\\\\n    finally:\\\\n        await file.close()\\\\n    update(\\\\n        job_id, status=\\"queued\\", progress=10,\\\\n        detail=f\\"Media uploaded; {MODEL_NAME} CUDA job is queued\\",\\\\n        directory=str(directory), cancel_requested=False,\\\\n    )\\\\n    threading.Thread(target=run_job, args=(job_id, directory, source, output_format), daemon=True).start()\\\\n    return {\\"job_id\\": job_id, \\"status\\": \\"queued\\", \\"progress\\": 10,\\\\n            \\"artifact_format\\": output_format}\\\\n\\\\n\\\\n@app.get(\\"/v1/audio/separations/{job_id}\\")\\\\ndef separation_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with JOB_LOCK:\\\\n        job = dict(JOBS.get(job_id, {}))\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"separation job not found\\")\\\\n    return {key: job.get(key) for key in (\\"status\\", \\"progress\\", \\"detail\\", \\"artifact_format\\", \\"artifacts_ready\\") if key in job} | {\\"job_id\\": job_id}\\\\n\\\\n\\\\n@app.get(\\"/v1/audio/separations/{job_id}/artifacts/{stem}\\")\\\\ndef artifact(job_id: str, stem: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if stem not in {\\"vocals\\", \\"background\\"}:\\\\n        raise HTTPException(status_code=404, detail=\\"unknown stem\\")\\\\n    with JOB_LOCK:\\\\n        job = dict(JOBS.get(job_id, {}))\\\\n    path = Path(job.get(stem, \\"\\"))\\\\n    if job.get(\\"status\\") != \\"ready\\" or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"stem is not ready\\")\\\\n    output_format = job.get(\\"artifact_format\\", \\"wav\\")\\\\n    media_type = \\"audio/flac\\" if output_format == \\"flac\\" else \\"audio/wav\\"\\\\n    return FileResponse(path, media_type=media_type, filename=stem + \\".\\" + output_format)\\\\n\\\\n\\\\n@app.delete(\\"/v1/audio/separations/{job_id}\\")\\\\ndef cancel_separation(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with JOB_LOCK:\\\\n        if job_id not in JOBS:\\\\n            raise HTTPException(status_code=404, detail=\\"separation job not found\\")\\\\n        JOBS[job_id][\\"cancel_requested\\"] = True\\\\n        JOBS[job_id][\\"status\\"] = \\"cancelling\\"\\\\n    return {\\"job_id\\": job_id, \\"status\\": \\"cancelling\\"}\\\\n\'),\\n",\n    "    \'la_studio_separation_launcher.py\': (\'9ca893f8e06826bb875e68a7e364cdb43be515882b8438f01eb71465874375d1\', \'\\"\\"\\"Launch the exact Spleeter Direct Colab worker and a temporary tunnel.\\"\\"\\"\\\\n\\\\nimport json\\\\nimport os\\\\nimport re\\\\nimport secrets\\\\nimport socket\\\\nimport subprocess\\\\nimport sys\\\\nimport time\\\\nimport urllib.error\\\\nimport urllib.request\\\\nfrom pathlib import Path\\\\n\\\\n\\\\nWORKER_CONTRACT = \\"spleeter-cuda-safe-20260816.1\\"\\\\nMODEL_ID = \\"sherpa-onnx-spleeter-2stems-fp16\\"\\\\nCAPABILITY_LABEL = \\"Voice Isolation\\"\\\\nPORT = 3924\\\\nTOKEN_ENV = \\"LA_STUDIO_COLAB_SEPARATION_TOKEN\\"\\\\nURL_ENV = \\"LA_STUDIO_COLAB_SEPARATION_URL\\"\\\\nMODEL_ENV = \\"LA_STUDIO_COLAB_SEPARATION_MODEL\\"\\\\nWORKER_LOG = Path(\\"/content/la_studio_separation_worker.log\\")\\\\nTUNNEL_LOG = Path(\\"/content/la_studio_separation_tunnel.log\\")\\\\nSTARTUP_TIMEOUT_SECONDS = 20 * 60\\\\nTUNNEL_TIMEOUT_SECONDS = 90\\\\nPUBLIC_TUNNEL_VERIFY_TIMEOUT_SECONDS = 120\\\\n\\\\n\\\\ndef port_is_occupied(port: int) -> bool:\\\\n    try:\\\\n        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\\\n            return True\\\\n    except OSError:\\\\n        return False\\\\n\\\\n\\\\ndef tail(path: Path, limit: int = 12000) -> str:\\\\n    try:\\\\n        return path.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-limit:]\\\\n    except FileNotFoundError:\\\\n        return \\"(log was not created)\\"\\\\n\\\\n\\\\ndef stop(process: subprocess.Popen | None) -> None:\\\\n    if process is None or process.poll() is not None:\\\\n        return\\\\n    process.terminate()\\\\n    try:\\\\n        process.wait(timeout=10)\\\\n    except subprocess.TimeoutExpired:\\\\n        process.kill()\\\\n\\\\n\\\\ndef cloudflared_ready() -> bool:\\\\n    try:\\\\n        return subprocess.run(\\\\n            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL,\\\\n            stderr=subprocess.DEVNULL, check=False,\\\\n        ).returncode == 0\\\\n    except OSError:\\\\n        # A fresh Colab runtime normally has no cloudflared executable yet.\\\\n        # subprocess.run raises FileNotFoundError in that case rather than\\\\n        # returning a non-zero status.\\\\n        return False\\\\n\\\\n\\\\ndef ensure_cloudflared() -> None:\\\\n    if cloudflared_ready():\\\\n        return\\\\n    package_path = \\"/content/la-studio-cloudflared.deb\\"\\\\n    download = subprocess.run(\\\\n        [\\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\\\n         \\"--output\\", package_path,\\\\n         \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\"],\\\\n        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,\\\\n    )\\\\n    if download.returncode != 0:\\\\n        raise RuntimeError(\\"Could not download cloudflared: \\" + (download.stdout[-1200:] or \\"no output\\"))\\\\n    install = subprocess.run([\\"dpkg\\", \\"-i\\", package_path], text=True,\\\\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)\\\\n    if install.returncode != 0 or not cloudflared_ready():\\\\n        raise RuntimeError(\\"Could not install cloudflared: \\" + (install.stdout[-1200:] or \\"no output\\"))\\\\n\\\\n\\\\ndef verify_public_tunnel(public_url: str, token: str) -> tuple[bool, str]:\\\\n    \\"\\"\\"Attempt a Colab-side check of the public hostname.\\\\n\\\\n    This is useful diagnostic evidence, but it is not authoritative: a\\\\n    Colab runtime can fail to resolve a newly-created trycloudflare hostname\\\\n    even while the desktop can reach it.  The desktop\\\\\'s Check Colab action is\\\\n    the authoritative authenticated capability/model verification.\\\\n    \\"\\"\\"\\\\n    try:\\\\n        request = urllib.request.Request(\\\\n            public_url.rstrip(\\"/\\") + \\"/health\\",\\\\n            headers={\\"Authorization\\": \\"Bearer \\" + token},\\\\n        )\\\\n        with urllib.request.urlopen(request, timeout=12) as response:\\\\n            health = json.loads(response.read().decode(\\"utf-8\\"))\\\\n        if (response.status == 200 and health.get(\\"ready\\") is True\\\\n                and str(health.get(\\"device\\", \\"\\")).lower() == \\"cuda\\"\\\\n                and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\\\n                and health.get(\\"cpu_fallback\\") is False\\\\n                and health.get(\\"startup_probe\\") == \\"passed\\"):\\\\n            return True, \\"verified\\"\\\\n        return False, \\"unexpected public /health response: \\" + json.dumps(health, ensure_ascii=False)\\\\n    except urllib.error.HTTPError as error:\\\\n        return False, f\\"public /health returned HTTP {error.code}: \\" + error.read().decode(\\\\n            \\"utf-8\\", errors=\\"replace\\"\\\\n        )[:1000]\\\\n    except Exception as error:\\\\n        return False, f\\"public /health is not reachable: {type(error).__name__}: {error}\\"\\\\n\\\\n\\\\nif port_is_occupied(PORT):\\\\n    raise RuntimeError(\\\\n        f\\"Port {PORT} is occupied by an earlier Colab worker. Use Runtime > Disconnect and delete runtime, \\"\\\\n        \\"then Run all once for this exact model.\\"\\\\n    )\\\\n\\\\ntoken = secrets.token_urlsafe(32)\\\\nenvironment = os.environ.copy()\\\\nenvironment[TOKEN_ENV] = token\\\\nenvironment[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\\\nwith WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\\\n    worker = subprocess.Popen(\\\\n        [sys.executable, \\"-m\\", \\"uvicorn\\", \\"la_studio_separation_worker:app\\",\\\\n         \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\\\n        cwd=\\"/content\\", env=environment, stdout=worker_output, stderr=subprocess.STDOUT,\\\\n        start_new_session=True,\\\\n    )\\\\n\\\\nprint(f\\"Starting exact CUDA {CAPABILITY_LABEL} worker; it must pass its bounded Spleeter startup probe.\\")\\\\ndeadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\\\nlast_error = \\"worker has not answered /health yet\\"\\\\nwhile time.monotonic() < deadline:\\\\n    if worker.poll() is not None:\\\\n        raise RuntimeError(\\\\n            f\\"The exact-model worker exited before becoming CUDA-ready (exit code {worker.returncode}).\\\\\\\\n\\\\\\\\n\\"\\\\n            \\"---- worker log ----\\\\\\\\n\\" + tail(WORKER_LOG)\\\\n        )\\\\n    try:\\\\n        request = urllib.request.Request(f\\"http://127.0.0.1:{PORT}/health\\",\\\\n                                         headers={\\"Authorization\\": \\"Bearer \\" + token})\\\\n        with urllib.request.urlopen(request, timeout=10) as response:\\\\n            health = json.loads(response.read().decode(\\"utf-8\\"))\\\\n        if (response.status == 200 and health.get(\\"ready\\") is True\\\\n                and str(health.get(\\"device\\", \\"\\")).lower() == \\"cuda\\"\\\\n                and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\\\n                and health.get(\\"cpu_fallback\\") is False\\\\n                and health.get(\\"startup_probe\\") == \\"passed\\"):\\\\n            print(\\"Exact CUDA worker passed startup probe:\\", health)\\\\n            break\\\\n        last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\\\n    except urllib.error.HTTPError as error:\\\\n        last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\\\n    except Exception as error:\\\\n        last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\\\n    time.sleep(2)\\\\nelse:\\\\n    stop(worker)\\\\n    raise RuntimeError(\\\\n        f\\"The exact-model worker did not become CUDA-ready within {STARTUP_TIMEOUT_SECONDS // 60} minutes. \\"\\\\n        f\\"Last check: {last_error}\\\\\\\\n\\\\\\\\n---- worker log ----\\\\\\\\n\\" + tail(WORKER_LOG)\\\\n    )\\\\n\\\\nensure_cloudflared()\\\\nwith TUNNEL_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as tunnel_output:\\\\n    tunnel = subprocess.Popen(\\\\n        [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\\\n        stdout=tunnel_output, stderr=subprocess.STDOUT, start_new_session=True,\\\\n    )\\\\n\\\\npublic_url = \\"\\"\\\\npublic_tunnel_verified = False\\\\ncandidate_url = \\"\\"\\\\nlast_tunnel_error = \\"cloudflared has not published a public URL yet\\"\\\\ndeadline = time.monotonic() + max(TUNNEL_TIMEOUT_SECONDS, PUBLIC_TUNNEL_VERIFY_TIMEOUT_SECONDS)\\\\nwhile time.monotonic() < deadline and not public_url:\\\\n    if tunnel.poll() is not None:\\\\n        last_tunnel_error = f\\"cloudflared exited with code {tunnel.returncode}\\"\\\\n        break\\\\n    match = re.search(r\\"https://[^\\\\\\\\s\\\\\\\\\\"\\\\\']+\\\\\\\\.trycloudflare\\\\\\\\.com\\", tail(TUNNEL_LOG, 4000))\\\\n    if match:\\\\n        candidate_url = match.group(0)\\\\n        verified, last_tunnel_error = verify_public_tunnel(candidate_url, token)\\\\n        if verified:\\\\n            public_url = candidate_url\\\\n            public_tunnel_verified = True\\\\n            break\\\\n    time.sleep(2)\\\\n\\\\nif not public_url:\\\\n    # A Quick Tunnel may be healthy from the desktop even when this Colab\\\\n    # runtime cannot resolve its just-created DNS name.  The exact CUDA worker\\\\n    # was already verified locally above.  Preserve a candidate only for this\\\\n    # narrow DNS/connectivity failure, and let the desktop prove the public\\\\n    # endpoint before it can run any job.  Never publish a candidate after an\\\\n    # HTTP/auth/model-contract mismatch, or after either local process exited.\\\\n    if (candidate_url\\\\n            and last_tunnel_error.startswith(\\"public /health is not reachable:\\")\\\\n            and worker.poll() is None\\\\n            and tunnel.poll() is None):\\\\n        public_url = candidate_url\\\\n    else:\\\\n        stop(tunnel)\\\\n        stop(worker)\\\\n        raise RuntimeError(\\\\n            \\"cloudflared did not create a verified public trycloudflare endpoint within \\"\\\\n            f\\"{max(TUNNEL_TIMEOUT_SECONDS, PUBLIC_TUNNEL_VERIFY_TIMEOUT_SECONDS)} seconds. \\"\\\\n            f\\"Last check: {last_tunnel_error}\\\\\\\\n\\"\\\\n            \\"---- cloudflared log ----\\\\\\\\n\\" + tail(TUNNEL_LOG, 4000)\\\\n        )\\\\n\\\\nprint(\\"\\\\\\\\nLA Studio exact-model Colab worker is ready\\")\\\\nif public_tunnel_verified:\\\\n    print(\\"Verified the public Cloudflare tunnel against this exact CUDA worker.\\")\\\\nelse:\\\\n    print(\\\\n        \\"Cloudflare emitted a public endpoint, but this Colab runtime could not \\"\\\\n        \\"complete its own DNS/public-health check. The desktop Check Colab action \\"\\\\n        \\"must now verify the public endpoint, bearer token, capability, and exact model.\\"\\\\n    )\\\\nprint(URL_ENV + \\"=\\" + public_url)\\\\nprint(TOKEN_ENV + \\"=\\" + token)\\\\nprint(MODEL_ENV + \\"=\\" + MODEL_ID)\\\\nprint(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\\\n\')\\n",\n    "}\\n",\n    "for destination, (expected_sha256, source) in EMBEDDED_WORKERS.items():\\n",\n    "    actual_sha256 = sha256(source.encode(\'utf-8\')).hexdigest()\\n",\n    "    if actual_sha256 != expected_sha256:\\n",\n    "        raise RuntimeError(f\'Embedded worker integrity check failed for {destination}: {actual_sha256}\')\\n",\n    "    Path(\'/content\', destination).write_text(source, encoding=\'utf-8\')\\n",\n    "print(\'Embedded verified exact-model CUDA worker and launcher templates.\')\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "!python /content/la_studio_separation_launcher.py\\n"\n   ]\n  }\n ],\n "metadata": {\n  "accelerator": "GPU",\n  "colab": {\n   "gpuType": "T4",\n   "provenance": []\n  },\n  "kernelspec": {\n   "display_name": "Python 3",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python"\n  },\n  "la_studio": {\n   "capability": "voice-isolation",\n   "family_id": "sherpa-onnx-spleeter-2stems-fp16",\n   "upstream_model": "k2-fsa/sherpa-onnx-spleeter-2stems-fp16",\n   "contract_version": 1,\n   "device": "cuda",\n   "cpu_fallback": false,\n   "worker_contract": "spleeter-cuda-safe-20260816.1",\n   "worker_source": "embedded-local",\n   "worker_templates": [\n    "notebooks/workers/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_WORKER.py",\n    "notebooks/workers/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_LAUNCHER.py"\n   ],\n   "embedded_worker_sha256": {\n    "la_studio_separation_worker.py": "307861926e13ff9849b04594074b573b1da063b1791c56dc2f502ab64991c5af",\n    "la_studio_separation_launcher.py": "9ca893f8e06826bb875e68a7e364cdb43be515882b8438f01eb71465874375d1"\n   },\n   "artifact_url": "https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2"\n  }\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}\n',
    'notebooks/voice_separation/LA_STUDIO_SEPARATION_UVR_VOCALS_GPU.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# LA Studio voice-isolation \\u2014 UVR MDX-Net Vocals FT\\n",\n        "\\n",\n        "This notebook loads exactly `sherpa-onnx-uvr-vocals-ft` (`k2-fsa/sherpa-onnx-uvr-vocals-ft`) on CUDA.\\n",\n        "It does not use API Gateway and refuses every other model ID.\\n",\n        "\\n",\n        "1. Choose **Runtime \\u2192 Change runtime type \\u2192 GPU**.\\n",\n        "2. Run all cells.\\n",\n        "3. Copy the printed URL and token into LA Studio.\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "!nvidia-smi\\n",\n        "%pip install -q \\"sherpa-onnx==1.13.4+cuda12.cudnn9\\" --find-links https://k2-fsa.github.io/sherpa/onnx/cuda.html\\n",\n        "%pip install -q \\"soundfile==0.13.1\\" \\"fastapi==0.115.12\\" \\"uvicorn==0.34.3\\" \\"python-multipart==0.0.20\\"\\n",\n        "\\n",\n        "!wget -q --show-progress -O /content/UVR-MDX-NET-Voc_FT.onnx https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/UVR-MDX-NET-Voc_FT.onnx\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "from pathlib import Path\\n",\n        "\\n",\n        "WORKER = Path(\'/content/la_studio_separation_worker.py\')\\n",\n        "WORKER.write_text(\'import os\\\\nimport secrets\\\\nimport shutil\\\\nimport subprocess\\\\nimport threading\\\\nimport time\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport sherpa_onnx\\\\nimport soundfile as sf\\\\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\\\\nfrom fastapi.responses import FileResponse\\\\n\\\\nif \\"+cuda\\" not in sherpa_onnx.__version__:\\\\n    raise RuntimeError(\\"The installed sherpa-onnx wheel is not CUDA-enabled\\")\\\\n\\\\nMODEL_ID = \\"sherpa-onnx-uvr-vocals-ft\\"\\\\nMODEL_NAME = \\"UVR MDX-Net Vocals FT\\"\\\\nUPSTREAM_MODEL = \\"k2-fsa/sherpa-onnx-uvr-vocals-ft\\"\\\\nARTIFACT_URL = \\"https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/UVR-MDX-NET-Voc_FT.onnx\\"\\\\nCONFIG = sherpa_onnx.OfflineSourceSeparationConfig(\\\\n    model=sherpa_onnx.OfflineSourceSeparationModelConfig(\\\\n        uvr=sherpa_onnx.OfflineSourceSeparationUvrModelConfig(\\\\n            model=\\"/content/UVR-MDX-NET-Voc_FT.onnx\\",\\\\n        ),\\\\n        num_threads=1,\\\\n        debug=False,\\\\n        provider=\\"cuda\\",\\\\n    )\\\\n)\\\\nif not CONFIG.validate():\\\\n    raise RuntimeError(\\"The exact sherpa-onnx CUDA separation configuration is invalid\\")\\\\nSEPARATOR = sherpa_onnx.OfflineSourceSeparation(CONFIG)\\\\n\\\\nTOKEN = os.environ[\\"LA_STUDIO_COLAB_SEPARATION_TOKEN\\"]\\\\nROOT = Path(\\"/content/la-studio-separation-jobs\\") / MODEL_ID\\\\nROOT.mkdir(parents=True, exist_ok=True)\\\\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\\\\nMAX_AUDIO_SECONDS = 30 * 60\\\\nARTIFACT_TTL_SECONDS = 1800\\\\nALLOWED_CONTENT_TYPES = {\\\\n    \\"audio/wav\\", \\"audio/x-wav\\", \\"audio/mpeg\\", \\"audio/mp4\\", \\"audio/webm\\",\\\\n    \\"audio/ogg\\", \\"audio/flac\\", \\"video/mp4\\", \\"video/webm\\", \\"video/quicktime\\",\\\\n    \\"video/x-matroska\\", \\"application/octet-stream\\",\\\\n}\\\\nALLOWED_EXTENSIONS = {\\".wav\\", \\".mp3\\", \\".m4a\\", \\".mp4\\", \\".webm\\", \\".ogg\\", \\".flac\\", \\".mkv\\", \\".mov\\", \\".avi\\"}\\\\nJOB_SLOTS = threading.BoundedSemaphore(1)\\\\nJOB_LOCK = threading.Lock()\\\\nJOBS = {}\\\\n\\\\ndef authorize(authorization: str | None) -> None:\\\\n    if authorization != \\"Bearer \\" + TOKEN:\\\\n        raise HTTPException(status_code=401, detail=\\"invalid worker token\\")\\\\n\\\\ndef require_exact_model(requested: str) -> None:\\\\n    if requested.strip().lower() != MODEL_ID:\\\\n        raise HTTPException(\\\\n            status_code=409,\\\\n            detail=f\\"This worker loaded \\\\\'{MODEL_ID}\\\\\', but LA Studio requested \\\\\'{requested}\\\\\'. Open the notebook for the selected model.\\",\\\\n        )\\\\n\\\\ndef media_duration_seconds(path: Path) -> float:\\\\n    probe = subprocess.run(\\\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-show_entries\\", \\"format=duration\\",\\\\n         \\"-of\\", \\"default=nokey=1:noprint_wrappers=1\\", str(path)],\\\\n        text=True, capture_output=True,\\\\n    )\\\\n    try:\\\\n        duration = float(probe.stdout.strip())\\\\n    except ValueError:\\\\n        duration = 0.0\\\\n    if probe.returncode != 0 or duration <= 0.0:\\\\n        raise HTTPException(status_code=415, detail=\\"media is unsupported or could not be decoded\\")\\\\n    return duration\\\\n\\\\ndef update(job_id: str, **values) -> dict:\\\\n    with JOB_LOCK:\\\\n        job = dict(JOBS.get(job_id, {}))\\\\n        job.update(values)\\\\n        JOBS[job_id] = job\\\\n        return job\\\\n\\\\ndef cleanup(job_id: str) -> None:\\\\n    with JOB_LOCK:\\\\n        job = JOBS.pop(job_id, None)\\\\n    if job:\\\\n        shutil.rmtree(job.get(\\"directory\\", \\"\\"), ignore_errors=True)\\\\n\\\\ndef run_job(job_id: str, directory: Path, source: Path, output_format: str) -> None:\\\\n    try:\\\\n        update(job_id, status=\\"running\\", progress=20, detail=f\\"{MODEL_NAME} is separating vocals on CUDA\\")\\\\n        wav_path = directory / \\"source-44100-stereo.wav\\"\\\\n        subprocess.run(\\\\n            [\\"ffmpeg\\", \\"-y\\", \\"-v\\", \\"error\\", \\"-i\\", str(source), \\"-vn\\", \\"-acodec\\", \\"pcm_s16le\\",\\\\n             \\"-ar\\", \\"44100\\", \\"-ac\\", \\"2\\", str(wav_path)],\\\\n            check=True,\\\\n        )\\\\n        samples, sample_rate = sf.read(wav_path, dtype=\\"float32\\", always_2d=True)\\\\n        samples = np.ascontiguousarray(samples.T)\\\\n        output = SEPARATOR.process(sample_rate=sample_rate, samples=samples)\\\\n        if len(output.stems) != 2:\\\\n            raise RuntimeError(f\\"expected two stems, received {len(output.stems)}\\")\\\\n        suffix = \\".wav\\" if output_format == \\"wav\\" else \\".flac\\"\\\\n        file_format = \\"WAV\\" if output_format == \\"wav\\" else \\"FLAC\\"\\\\n        vocals = directory / (\\"vocals\\" + suffix)\\\\n        background = directory / (\\"background\\" + suffix)\\\\n        sf.write(vocals, np.asarray(output.stems[0].data).T, output.sample_rate,\\\\n                 format=file_format, subtype=\\"PCM_16\\")\\\\n        sf.write(background, np.asarray(output.stems[1].data).T, output.sample_rate,\\\\n                 format=file_format, subtype=\\"PCM_16\\")\\\\n        with JOB_LOCK:\\\\n            cancelled = JOBS.get(job_id, {}).get(\\"cancel_requested\\", False)\\\\n        if cancelled:\\\\n            vocals.unlink(missing_ok=True)\\\\n            background.unlink(missing_ok=True)\\\\n            update(job_id, status=\\"cancelled\\", progress=0, detail=\\"Separation cancelled\\")\\\\n        else:\\\\n            update(\\\\n                job_id, status=\\"ready\\", progress=100, detail=\\"Separated stems are ready\\",\\\\n                vocals=str(vocals), background=str(background),\\\\n                artifact_format=output_format, artifacts_ready=True,\\\\n            )\\\\n    except Exception as error:\\\\n        update(job_id, status=\\"failed\\", progress=0, detail=f\\"{type(error).__name__}: {str(error)[:1800]}\\")\\\\n    finally:\\\\n        threading.Timer(ARTIFACT_TTL_SECONDS, cleanup, args=[job_id]).start()\\\\n        JOB_SLOTS.release()\\\\n\\\\napp = FastAPI(title=f\\"LA Studio Voice Isolation - {MODEL_NAME}\\", docs_url=None, redoc_url=None, openapi_url=None)\\\\n\\\\n@app.get(\\"/health\\")\\\\n@app.get(\\"/v1/health\\")\\\\ndef health(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"status\\": \\"ready\\",\\\\n        \\"ready\\": True,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"model\\": MODEL_ID,\\\\n        \\"variant\\": \\"fixed\\",\\\\n        \\"upstream_model\\": UPSTREAM_MODEL,\\\\n        \\"sherpa_onnx\\": sherpa_onnx.__version__,\\\\n        \\"cpu_fallback\\": False,\\\\n    }\\\\n\\\\n@app.get(\\"/v1/capabilities\\")\\\\ndef capabilities(authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    return {\\\\n        \\"contract_version\\": 1,\\\\n        \\"device\\": \\"cuda\\",\\\\n        \\"capabilities\\": [{\\\\n            \\"id\\": \\"voice-isolation\\",\\\\n            \\"models\\": [{\\\\n                \\"id\\": MODEL_ID,\\\\n                \\"name\\": MODEL_NAME,\\\\n                \\"variant\\": \\"fixed\\",\\\\n                \\"upstream_model\\": UPSTREAM_MODEL,\\\\n                \\"artifact_url\\": ARTIFACT_URL,\\\\n                \\"stems\\": [\\"vocals\\", \\"background\\"],\\\\n                \\"formats\\": [\\"flac\\", \\"wav\\"],\\\\n                \\"device\\": \\"cuda\\",\\\\n                \\"loaded\\": True,\\\\n            }],\\\\n        }],\\\\n    }\\\\n\\\\n@app.post(\\"/v1/audio/separations\\")\\\\nasync def create_separation(\\\\n    file: UploadFile = File(...),\\\\n    stems: str = Form(\\"vocals,background\\"),\\\\n    model: str = Form(...),\\\\n    output_format: str = Form(\\"flac\\"),\\\\n    authorization: str | None = Header(default=None),\\\\n):\\\\n    authorize(authorization)\\\\n    require_exact_model(model)\\\\n    if stems != \\"vocals,background\\":\\\\n        raise HTTPException(status_code=422, detail=\\"this worker returns vocals and background stems\\")\\\\n    output_format = output_format.strip().lower()\\\\n    if output_format not in {\\"flac\\", \\"wav\\"}:\\\\n        raise HTTPException(status_code=422, detail=\\"output_format must be flac or wav\\")\\\\n    suffix = Path(file.filename or \\"source.wav\\").suffix.lower() or \\".wav\\"\\\\n    if suffix not in ALLOWED_EXTENSIONS:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported media filename extension\\")\\\\n    if file.content_type and file.content_type not in ALLOWED_CONTENT_TYPES:\\\\n        raise HTTPException(status_code=415, detail=\\"unsupported media MIME type\\")\\\\n    if not JOB_SLOTS.acquire(blocking=False):\\\\n        raise HTTPException(status_code=429, detail=\\"the Colab separation worker is busy; retry shortly\\")\\\\n    job_id = secrets.token_urlsafe(18)\\\\n    directory = ROOT / job_id\\\\n    directory.mkdir(parents=True, exist_ok=True)\\\\n    source = directory / (\\"source\\" + suffix)\\\\n    try:\\\\n        with source.open(\\"wb\\") as output:\\\\n            while chunk := await file.read(1024 * 1024):\\\\n                output.write(chunk)\\\\n                if output.tell() > MAX_UPLOAD_BYTES:\\\\n                    raise HTTPException(status_code=413, detail=\\"media exceeds 512 MB upload limit\\")\\\\n        if source.stat().st_size <= 0:\\\\n            raise HTTPException(status_code=413, detail=\\"media must not be empty\\")\\\\n        if media_duration_seconds(source) > MAX_AUDIO_SECONDS:\\\\n            raise HTTPException(status_code=413, detail=\\"media exceeds the 30 minute duration limit\\")\\\\n    except Exception:\\\\n        shutil.rmtree(directory, ignore_errors=True)\\\\n        JOB_SLOTS.release()\\\\n        raise\\\\n    finally:\\\\n        await file.close()\\\\n    update(\\\\n        job_id, status=\\"queued\\", progress=10, detail=f\\"Media uploaded; {MODEL_NAME} CUDA job is queued\\",\\\\n        directory=str(directory), cancel_requested=False,\\\\n    )\\\\n    threading.Thread(target=run_job, args=(job_id, directory, source, output_format), daemon=True).start()\\\\n    return {\\"job_id\\": job_id, \\"status\\": \\"queued\\", \\"progress\\": 10,\\\\n            \\"artifact_format\\": output_format}\\\\n\\\\n@app.get(\\"/v1/audio/separations/{job_id}\\")\\\\ndef separation_status(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with JOB_LOCK:\\\\n        job = dict(JOBS.get(job_id, {}))\\\\n    if not job:\\\\n        raise HTTPException(status_code=404, detail=\\"separation job not found\\")\\\\n    return {key: job.get(key) for key in (\\"status\\", \\"progress\\", \\"detail\\", \\"artifact_format\\", \\"artifacts_ready\\") if key in job} | {\\"job_id\\": job_id}\\\\n\\\\n@app.get(\\"/v1/audio/separations/{job_id}/artifacts/{stem}\\")\\\\ndef artifact(job_id: str, stem: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    if stem not in {\\"vocals\\", \\"background\\"}:\\\\n        raise HTTPException(status_code=404, detail=\\"unknown stem\\")\\\\n    with JOB_LOCK:\\\\n        job = dict(JOBS.get(job_id, {}))\\\\n    path = Path(job.get(stem, \\"\\"))\\\\n    if job.get(\\"status\\") != \\"ready\\" or not path.is_file():\\\\n        raise HTTPException(status_code=409, detail=\\"stem is not ready\\")\\\\n    output_format = job.get(\\"artifact_format\\", \\"wav\\")\\\\n    media_type = \\"audio/wav\\" if output_format == \\"wav\\" else \\"audio/flac\\"\\\\n    return FileResponse(path, media_type=media_type, filename=stem + \\".\\" + output_format)\\\\n\\\\n@app.delete(\\"/v1/audio/separations/{job_id}\\")\\\\ndef cancel_separation(job_id: str, authorization: str | None = Header(default=None)):\\\\n    authorize(authorization)\\\\n    with JOB_LOCK:\\\\n        if job_id not in JOBS:\\\\n            raise HTTPException(status_code=404, detail=\\"separation job not found\\")\\\\n        JOBS[job_id][\\"cancel_requested\\"] = True\\\\n        JOBS[job_id][\\"status\\"] = \\"cancelling\\"\\\\n    return {\\"job_id\\": job_id, \\"status\\": \\"cancelling\\"}\\\\n\', encoding=\'utf-8\')\\n",\n        "print(\'Worker source:\', WORKER)\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "MODEL_ID = \'sherpa-onnx-uvr-vocals-ft\'\\n",\n        "# LA Studio worker launch contract: launch-2026-08-06.1\\n",\n        "import json\\n",\n        "import os\\n",\n        "import queue\\n",\n        "import re\\n",\n        "import secrets\\n",\n        "import signal\\n",\n        "import socket\\n",\n        "import subprocess\\n",\n        "import sys\\n",\n        "import threading\\n",\n        "import time\\n",\n        "import urllib.error\\n",\n        "import urllib.request\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "CAPABILITY_LABEL = \'Voice Isolation\'\\n",\n        "MODEL_ID = \'sherpa-onnx-uvr-vocals-ft\'\\n",\n        "PORT = 3924\\n",\n        "TOKEN_ENV = \'LA_STUDIO_COLAB_SEPARATION_TOKEN\'\\n",\n        "URL_ENV = \'LA_STUDIO_COLAB_SEPARATION_URL\'\\n",\n        "MODEL_ENV = \'LA_STUDIO_COLAB_SEPARATION_MODEL\'\\n",\n        "WORKER_LOG = Path(\'/content/la_studio_separation_worker.log\')\\n",\n        "WORKER_MODULE = \'la_studio_separation_worker\'\\n",\n        "WORKER_PYTHON = sys.executable\\n",\n        "WORKER_PYTHON_ISOLATED = False\\n",\n        "WORKER_ENVIRONMENT = {}\\n",\n        "REQUIRES_CUDA = True\\n",\n        "STARTUP_TIMEOUT_SECONDS = 20 * 60\\n",\n        "TUNNEL_TIMEOUT_SECONDS = 90\\n",\n        "TOKEN = secrets.token_urlsafe(32)\\n",\n        "\\n",\n        "\\n",\n        "def port_is_occupied(port: int) -> bool:\\n",\n        "    try:\\n",\n        "        with socket.create_connection((\\"127.0.0.1\\", port), timeout=0.5):\\n",\n        "            return True\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def process_cmdline(pid: int) -> str:\\n",\n        "    \\"\\"\\"Read a Linux process command line without depending on psutil.\\"\\"\\"\\n",\n        "    try:\\n",\n        "        return Path(f\\"/proc/{pid}/cmdline\\").read_bytes().replace(b\\"\\\\0\\", b\\" \\").decode(\\n",\n        "            \\"utf-8\\", errors=\\"replace\\"\\n",\n        "        ).strip()\\n",\n        "    except (FileNotFoundError, PermissionError, ProcessLookupError):\\n",\n        "        return \\"\\"\\n",\n        "\\n",\n        "\\n",\n        "def all_processes():\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        pid = int(entry.name)\\n",\n        "        command = process_cmdline(pid)\\n",\n        "        if command:\\n",\n        "            yield pid, command\\n",\n        "\\n",\n        "\\n",\n        "def listening_processes(port: int) -> dict[int, str]:\\n",\n        "    \\"\\"\\"Return PIDs listening on a local TCP port via /proc socket ownership.\\"\\"\\"\\n",\n        "    target_port = f\\"{port:04X}\\"\\n",\n        "    socket_inodes = set()\\n",\n        "    for table_name in (\\"/proc/net/tcp\\", \\"/proc/net/tcp6\\"):\\n",\n        "        try:\\n",\n        "            lines = Path(table_name).read_text(encoding=\\"utf-8\\").splitlines()[1:]\\n",\n        "        except FileNotFoundError:\\n",\n        "            continue\\n",\n        "        for line in lines:\\n",\n        "            fields = line.split()\\n",\n        "            if len(fields) < 10:\\n",\n        "                continue\\n",\n        "            local_address, state, inode = fields[1], fields[3], fields[9]\\n",\n        "            if state == \\"0A\\" and local_address.rsplit(\\":\\", 1)[-1].upper() == target_port:\\n",\n        "                socket_inodes.add(inode)\\n",\n        "    if not socket_inodes:\\n",\n        "        return {}\\n",\n        "\\n",\n        "    listeners = {}\\n",\n        "    for entry in Path(\\"/proc\\").iterdir():\\n",\n        "        if not entry.name.isdigit():\\n",\n        "            continue\\n",\n        "        try:\\n",\n        "            descriptors = (entry / \\"fd\\").iterdir()\\n",\n        "        except (FileNotFoundError, PermissionError):\\n",\n        "            continue\\n",\n        "        for descriptor in descriptors:\\n",\n        "            try:\\n",\n        "                target = os.readlink(descriptor)\\n",\n        "            except (FileNotFoundError, PermissionError, OSError):\\n",\n        "                continue\\n",\n        "            match = re.fullmatch(r\\"socket:\\\\\\\\[(\\\\\\\\d+)\\\\\\\\]\\", target)\\n",\n        "            if match and match.group(1) in socket_inodes:\\n",\n        "                pid = int(entry.name)\\n",\n        "                listeners[pid] = process_cmdline(pid)\\n",\n        "                break\\n",\n        "    return listeners\\n",\n        "\\n",\n        "\\n",\n        "def stop_pid(pid: int) -> None:\\n",\n        "    if pid == os.getpid():\\n",\n        "        return\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGTERM)\\n",\n        "    except ProcessLookupError:\\n",\n        "        return\\n",\n        "    deadline = time.monotonic() + 10\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        try:\\n",\n        "            os.kill(pid, 0)\\n",\n        "        except ProcessLookupError:\\n",\n        "            return\\n",\n        "        time.sleep(0.2)\\n",\n        "    try:\\n",\n        "        os.kill(pid, signal.SIGKILL)\\n",\n        "    except ProcessLookupError:\\n",\n        "        pass\\n",\n        "\\n",\n        "\\n",\n        "def reclaim_previous_la_studio_worker() -> None:\\n",\n        "    \\"\\"\\"Stop only an older LA Studio worker/tunnel for this exact local port.\\n",\n        "\\n",\n        "    Re-running a Colab cell keeps child processes alive.  The previous launch\\n",\n        "    created a new token but aborted before it could replace the old worker,\\n",\n        "    forcing users to destroy the whole GPU runtime.  We identify ownership by\\n",\n        "    the exact generated module name and never terminate a foreign listener.\\n",\n        "    \\"\\"\\"\\n",\n        "    stopped = []\\n",\n        "    for pid, command in listening_processes(PORT).items():\\n",\n        "        if WORKER_MODULE in command and \\"uvicorn\\" in command:\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"worker PID {pid}\\")\\n",\n        "\\n",\n        "    endpoint = f\\"http://127.0.0.1:{PORT}\\"\\n",\n        "    for pid, command in all_processes():\\n",\n        "        if (\\"cloudflared\\" in command and \\"tunnel\\" in command and endpoint in command):\\n",\n        "            stop_pid(pid)\\n",\n        "            stopped.append(f\\"tunnel PID {pid}\\")\\n",\n        "\\n",\n        "    deadline = time.monotonic() + 12\\n",\n        "    while port_is_occupied(PORT) and time.monotonic() < deadline:\\n",\n        "        time.sleep(0.2)\\n",\n        "    if stopped:\\n",\n        "        print(\\"Stopped previous LA Studio \\" + \\", \\".join(stopped) + \\".\\")\\n",\n        "\\n",\n        "    if port_is_occupied(PORT):\\n",\n        "        listeners = listening_processes(PORT)\\n",\n        "        foreign_pids = sorted(listeners) or [\\"unknown\\"]\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"Port {PORT} is occupied by a process that is not the previous LA Studio \\"\\n",\n        "            f\\"{CAPABILITY_LABEL} worker (PID(s): {\', \'.join(map(str, foreign_pids))}). \\"\\n",\n        "            \\"Choose a fresh Colab runtime rather than terminating an unrelated process.\\"\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def worker_log_tail() -> str:\\n",\n        "    try:\\n",\n        "        return WORKER_LOG.read_text(encoding=\\"utf-8\\", errors=\\"replace\\")[-12000:]\\n",\n        "    except FileNotFoundError:\\n",\n        "        return \\"(worker log was not created)\\"\\n",\n        "\\n",\n        "\\n",\n        "def stop_process(process) -> None:\\n",\n        "    if process is None or process.poll() is not None:\\n",\n        "        return\\n",\n        "    process.terminate()\\n",\n        "    try:\\n",\n        "        process.wait(timeout=10)\\n",\n        "    except subprocess.TimeoutExpired:\\n",\n        "        process.kill()\\n",\n        "\\n",\n        "\\n",\n        "reclaim_previous_la_studio_worker()\\n",\n        "\\n",\n        "env = os.environ.copy()\\n",\n        "env[TOKEN_ENV] = TOKEN\\n",\n        "env[\\"PYTHONUNBUFFERED\\"] = \\"1\\"\\n",\n        "env.update(WORKER_ENVIRONMENT)\\n",\n        "if WORKER_PYTHON_ISOLATED:\\n",\n        "    # Do not let Colab\'s global site-packages or a notebook-level PYTHONPATH\\n",\n        "    # bleed into a dedicated worker virtual environment.\\n",\n        "    env.pop(\\"PYTHONPATH\\", None)\\n",\n        "    env[\\"PYTHONNOUSERSITE\\"] = \\"1\\"\\n",\n        "worker = None\\n",\n        "tunnel = None\\n",\n        "\\n",\n        "with WORKER_LOG.open(\\"w\\", encoding=\\"utf-8\\", buffering=1) as worker_output:\\n",\n        "    worker = subprocess.Popen(\\n",\n        "        [WORKER_PYTHON, \\"-m\\", \\"uvicorn\\", \'la_studio_separation_worker:app\', \\"--host\\", \\"127.0.0.1\\", \\"--port\\", str(PORT)],\\n",\n        "        cwd=\\"/content\\",\\n",\n        "        env=env,\\n",\n        "        stdout=worker_output,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "    )\\n",\n        "    worker_kind = \\"exact CUDA\\" if REQUIRES_CUDA else \\"dedicated Colab CPU\\"\\n",\n        "    print(f\\"Starting {worker_kind} {CAPABILITY_LABEL} worker.\\")\\n",\n        "    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\\n",\n        "    last_error = \\"worker has not answered /health yet\\"\\n",\n        "    next_report = time.monotonic()\\n",\n        "    while time.monotonic() < deadline:\\n",\n        "        exit_code = worker.poll()\\n",\n        "        if exit_code is not None:\\n",\n        "            raise RuntimeError(\\n",\n        "                f\\"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\\\\n\\\\n\\"\\n",\n        "                \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "            )\\n",\n        "        try:\\n",\n        "            request = urllib.request.Request(\\n",\n        "                f\\"http://127.0.0.1:{PORT}/health\\",\\n",\n        "                headers={\\"Authorization\\": \\"Bearer \\" + TOKEN},\\n",\n        "            )\\n",\n        "            with urllib.request.urlopen(request, timeout=10) as response:\\n",\n        "                health = json.loads(response.read().decode(\\"utf-8\\"))\\n",\n        "            if (response.status == 200\\n",\n        "                    and health.get(\\"ready\\") is True\\n",\n        "                    and str(health.get(\\"device\\", \\"\\")).lower()\\n",\n        "                        == (\\"cuda\\" if REQUIRES_CUDA else \\"colab-cpu\\")\\n",\n        "                    and str(health.get(\\"model\\", \\"\\")).strip().lower() == MODEL_ID\\n",\n        "                    and health.get(\\"cpu_fallback\\") is False):\\n",\n        "                print(worker_kind.title() + \\" worker is ready:\\", health)\\n",\n        "                break\\n",\n        "            last_error = \\"unexpected /health response: \\" + json.dumps(health, ensure_ascii=False)\\n",\n        "        except urllib.error.HTTPError as error:\\n",\n        "            last_error = f\\"/health returned HTTP {error.code}: \\" + error.read().decode(\\"utf-8\\", errors=\\"replace\\")[:1000]\\n",\n        "        except Exception as error:\\n",\n        "            last_error = f\\"/health is not ready: {type(error).__name__}: {error}\\"\\n",\n        "        if time.monotonic() >= next_report:\\n",\n        "            print(f\\"Waiting for the {worker_kind} worker...\\", last_error)\\n",\n        "            next_report = time.monotonic() + 30\\n",\n        "        time.sleep(2)\\n",\n        "    else:\\n",\n        "        stop_process(worker)\\n",\n        "        raise RuntimeError(\\n",\n        "            f\\"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within \\"\\n",\n        "            f\\"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\\\\n\\\\n\\"\\n",\n        "            \\"---- LA Studio worker log (last 12,000 characters) ----\\\\n\\" + worker_log_tail()\\n",\n        "        )\\n",\n        "\\n",\n        "\\n",\n        "def cloudflared_ready() -> bool:\\n",\n        "    try:\\n",\n        "        return subprocess.run(\\n",\n        "            [\\"cloudflared\\", \\"--version\\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n",\n        "            check=False,\\n",\n        "        ).returncode == 0\\n",\n        "    except OSError:\\n",\n        "        return False\\n",\n        "\\n",\n        "\\n",\n        "def ensure_cloudflared() -> None:\\n",\n        "    if cloudflared_ready():\\n",\n        "        return\\n",\n        "    package_path = \\"/content/la-studio-cloudflared.deb\\"\\n",\n        "    download = subprocess.run(\\n",\n        "        [\\n",\n        "            \\"curl\\", \\"--fail\\", \\"--location\\", \\"--retry\\", \\"4\\", \\"--retry-all-errors\\",\\n",\n        "            \\"--output\\", package_path,\\n",\n        "            \\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\\",\\n",\n        "        ],\\n",\n        "        text=True,\\n",\n        "        stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT,\\n",\n        "        check=False,\\n",\n        "    )\\n",\n        "    if download.returncode != 0:\\n",\n        "        detail = download.stdout[-1200:].strip() or \\"no download output\\"\\n",\n        "        raise RuntimeError(\\"Could not download cloudflared: \\" + detail)\\n",\n        "    install = subprocess.run(\\n",\n        "        [\\"dpkg\\", \\"-i\\", package_path], text=True, stdout=subprocess.PIPE,\\n",\n        "        stderr=subprocess.STDOUT, check=False,\\n",\n        "    )\\n",\n        "    if install.returncode != 0 or not cloudflared_ready():\\n",\n        "        detail = install.stdout[-1200:].strip() or \\"no installation output\\"\\n",\n        "        raise RuntimeError(\\"Could not install cloudflared: \\" + detail)\\n",\n        "\\n",\n        "\\n",\n        "ensure_cloudflared()\\n",\n        "tunnel = subprocess.Popen(\\n",\n        "    [\\"cloudflared\\", \\"tunnel\\", \\"--url\\", f\\"http://127.0.0.1:{PORT}\\", \\"--no-autoupdate\\"],\\n",\n        "    stdout=subprocess.PIPE,\\n",\n        "    stderr=subprocess.STDOUT,\\n",\n        "    text=True,\\n",\n        "    bufsize=1,\\n",\n        ")\\n",\n        "tunnel_lines = queue.Queue()\\n",\n        "\\n",\n        "\\n",\n        "def collect_tunnel_output() -> None:\\n",\n        "    assert tunnel.stdout is not None\\n",\n        "    for line in tunnel.stdout:\\n",\n        "        tunnel_lines.put(line)\\n",\n        "\\n",\n        "\\n",\n        "threading.Thread(target=collect_tunnel_output, daemon=True).start()\\n",\n        "public_url = \\"\\"\\n",\n        "recent_tunnel_lines = []\\n",\n        "deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS\\n",\n        "while time.monotonic() < deadline and not public_url:\\n",\n        "    if tunnel.poll() is not None:\\n",\n        "        break\\n",\n        "    try:\\n",\n        "        line = tunnel_lines.get(timeout=1)\\n",\n        "    except queue.Empty:\\n",\n        "        continue\\n",\n        "    recent_tunnel_lines.append(line.rstrip())\\n",\n        "    recent_tunnel_lines = recent_tunnel_lines[-10:]\\n",\n        "    print(line, end=\\"\\")\\n",\n        "    match = re.search(r\\"https://[^\\\\s\\\\\\"\']+\\\\.trycloudflare\\\\.com\\", line)\\n",\n        "    if match:\\n",\n        "        # The desktop Check Colab action is the authoritative public endpoint,\\n",\n        "        # bearer-token, capability, and exact-model verification.\\n",\n        "        public_url = match.group(0)\\n",\n        "\\n",\n        "if not public_url:\\n",\n        "    stop_process(tunnel)\\n",\n        "    stop_process(worker)\\n",\n        "    tail = \\"\\\\n\\".join(recent_tunnel_lines) or \\"(no cloudflared output)\\"\\n",\n        "    raise RuntimeError(\\n",\n        "        f\\"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\\\\n\\"\\n",\n        "        \\"---- cloudflared output ----\\\\n\\" + tail\\n",\n        "    )\\n",\n        "\\n",\n        "os.environ[URL_ENV] = public_url\\n",\n        "os.environ[TOKEN_ENV] = TOKEN\\n",\n        "os.environ[MODEL_ENV] = MODEL_ID\\n",\n        "print(\\"\\\\nLA Studio exact-model Colab worker is ready\\")\\n",\n        "print(URL_ENV + \\"=\\" + public_url)\\n",\n        "print(TOKEN_ENV + \\"=\\" + TOKEN)\\n",\n        "print(MODEL_ENV + \\"=\\" + MODEL_ID)\\n",\n        "print(\\"Click Check Colab in the matching LA Studio feature before running it.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "# ==============================================================================\\n",\n        "# \\ud83d\\udce5 L\\u01afU FILE TR\\u1ef0C TI\\u1ebeP V\\u00c0O TH\\u01af M\\u1ee4C D\\u1ef0 \\u00c1N TR\\u00caN M\\u00c1Y T\\u00cdNH (FILE SYSTEM ACCESS API)\\n",\n        "# ==============================================================================\\n",\n        "import base64\\n",\n        "import glob\\n",\n        "import json\\n",\n        "import os\\n",\n        "from IPython.display import HTML, display\\n",\n        "\\n",\n        "# Thu th\\u1eadp t\\u1ea5t c\\u1ea3 c\\u00e1c file k\\u1ebft qu\\u1ea3 v\\u1eeba t\\u1ea1o\\n",\n        "result_files = {}\\n",\n        "for pattern in [\'/content/*.wav\', \'/content/*.srt\', \'/content/*.json\', \'/content/*/*/*.wav\', \'/content/*/*/*.srt\']:\\n",\n        "    for f in glob.glob(pattern):\\n",\n        "        name = os.path.basename(f)\\n",\n        "        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:\\n",\n        "            with open(f, \'rb\') as fp:\\n",\n        "                result_files[name] = base64.b64encode(fp.read()).decode(\'utf-8\')\\n",\n        "\\n",\n        "if not result_files:\\n",\n        "    print(\\"\\u26a0\\ufe0f Ch\\u01b0a c\\u00f3 file k\\u1ebft qu\\u1ea3 m\\u1edbi \\u0111\\u1ec3 l\\u01b0u.\\")\\n",\n        "else:\\n",\n        "    print(f\\"\\u2705 \\u0110\\u00e3 t\\u00ecm th\\u1ea5y {len(result_files)} file k\\u1ebft qu\\u1ea3: {\', \'.join(result_files.keys())}\\")\\n",\n        "    print(\\"\\ud83d\\udc49 B\\u1ea5m n\\u00fat b\\u00ean d\\u01b0\\u1edbi v\\u00e0 ch\\u1ecdn th\\u01b0 m\\u1ee5c \'LA-Studio/out/colab-live\' \\u0111\\u1ec3 l\\u01b0u th\\u1eb3ng v\\u00e0o m\\u00e1y:\\")\\n",\n        "    \\n",\n        "    files_json = json.dumps(result_files)\\n",\n        "    html_code = f\\"\\"\\"\\n",\n        "    <button id=\\"saveBtn\\" style=\\"background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;\\">\\n",\n        "        \\ud83d\\udcc1 Ch\\u1ecdn Th\\u01b0 M\\u1ee5c & L\\u01b0u File Tr\\u1ef1c Ti\\u1ebfp V\\u00e0o M\\u00e1y\\n",\n        "    </button>\\n",\n        "    <div id=\\"statusLog\\" style=\\"margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;\\"></div>\\n",\n        "    <script>\\n",\n        "    document.getElementById(\'saveBtn\').onclick = async () => {{\\n",\n        "        const log = document.getElementById(\'statusLog\');\\n",\n        "        try {{\\n",\n        "            if (!window.showDirectoryPicker) {{\\n",\n        "                log.innerText = \'Tr\\u00ecnh duy\\u1ec7t kh\\u00f4ng h\\u1ed7 tr\\u1ee3 File System Access API. \\u0110ang d\\u00f9ng t\\u1ea3i th\\u00f4ng th\\u01b0\\u1eddng...\';\\n",\n        "                return;\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\u0110ang m\\u1edf h\\u1ed9p tho\\u1ea1i ch\\u1ecdn th\\u01b0 m\\u1ee5c...\';\\n",\n        "            const dirHandle = await window.showDirectoryPicker();\\n",\n        "            const files = {files_json};\\n",\n        "            for (const [name, b64] of Object.entries(files)) {{\\n",\n        "                log.innerText = \'\\u0110ang ghi file: \' + name + \'...\';\\n",\n        "                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});\\n",\n        "                const writable = await fileHandle.createWritable();\\n",\n        "                const byteCharacters = atob(b64);\\n",\n        "                const byteNumbers = new Array(byteCharacters.length);\\n",\n        "                for (let i = 0; i < byteCharacters.length; i++) {{\\n",\n        "                    byteNumbers[i] = byteCharacters.charCodeAt(i);\\n",\n        "                }}\\n",\n        "                const byteArray = new Uint8Array(byteNumbers);\\n",\n        "                await writable.write(byteArray);\\n",\n        "                await writable.close();\\n",\n        "            }}\\n",\n        "            log.innerText = \'\\ud83c\\udf89 \\u0110\\u00e3 l\\u01b0u th\\u00e0nh c\\u00f4ng to\\u00e0n b\\u1ed9 file v\\u00e0o th\\u01b0 m\\u1ee5c b\\u1ea1n ch\\u1ecdn!\';\\n",\n        "        }} catch (err) {{\\n",\n        "            if (err.name !== \'AbortError\') {{\\n",\n        "                log.innerText = \'L\\u1ed7i: \' + err.message;\\n",\n        "            }} else {{\\n",\n        "                log.innerText = \'\\u0110\\u00e3 h\\u1ee7y ch\\u1ecdn th\\u01b0 m\\u1ee5c.\';\\n",\n        "            }}\\n",\n        "        }}\\n",\n        "    }};\\n",\n        "    </script>\\n",\n        "    \\"\\"\\"\\n",\n        "    display(HTML(html_code))\\n"\n      ]\n    }\n  ],\n  "metadata": {\n    "accelerator": "GPU",\n    "colab": {\n      "gpuType": "T4",\n      "provenance": []\n    },\n    "kernelspec": {\n      "display_name": "Python 3",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "la_studio": {\n      "capability": "voice-isolation",\n      "family_id": "sherpa-onnx-uvr-vocals-ft",\n      "upstream_model": "k2-fsa/sherpa-onnx-uvr-vocals-ft",\n      "contract_version": 1,\n      "device": "cuda",\n      "cpu_fallback": false,\n      "artifact_url": "https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/UVR-MDX-NET-Voc_FT.onnx"\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}',
    'notebooks/workers/LA_STUDIO_UNIFIED_DUBBING_COORDINATOR.py': '#!/usr/bin/env python3\n"""One-tunnel coordinator for real LA Studio Dubbing Colab workers.\n\nThe coordinator deliberately does not implement inference itself.  It starts\nthe selected *exact* notebook workers on private loopback ports, waits for each\nworker\'s real CUDA /health response, and exposes them through one authenticated\nCloudflare URL:\n\n    /v1/unified/<capability>/<model>/<the normal worker route>\n\nThis keeps the direct per-model notebooks valid while allowing the optional\nUnified Dubbing setup in the desktop app to use one URL and token.  A worker\nthat fails to install, load CUDA, or pass its normal health check prevents the\ncoordinator from becoming ready; it is never reported as a successful fake\nroute.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport asyncio\nimport json\nimport os\nimport re\nimport secrets\nimport shutil\nimport socket\nimport subprocess\nimport sys\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport httpx\nimport uvicorn\nfrom fastapi import FastAPI, HTTPException, Request\nfrom fastapi.responses import JSONResponse, StreamingResponse\n\n\nCOORDINATOR_REVISION = "unified-dubbing-coordinator-2026-08-22.1"\nLISTEN_HOST = "127.0.0.1"\nLISTEN_PORT = 3960\nTOKEN_ENVIRONMENTS = {\n    "stt": "LA_STUDIO_COLAB_STT_TOKEN",\n    "subtitle-ocr": "LA_STUDIO_COLAB_SUBTITLE_OCR_TOKEN",\n    "translation": "LA_STUDIO_COLAB_TRANSLATION_TOKEN",\n    "tts": "LA_STUDIO_COLAB_TTS_TOKEN",\n    "voice-isolation": "LA_STUDIO_COLAB_SEPARATION_TOKEN",\n    "forced-alignment": "LA_STUDIO_COLAB_ALIGNMENT_TOKEN",\n    "llm": "LA_STUDIO_COLAB_LLM_TOKEN",\n}\nHOP_BY_HOP_HEADERS = {\n    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",\n    "te", "trailers", "transfer-encoding", "upgrade", "host",\n}\nSAFE_SLUG = re.compile(r"^[a-z0-9][a-z0-9._-]{0,127}$")\n\n\n@dataclass(frozen=True)\nclass WorkerSpec:\n    capability: str\n    model: str\n    notebook: Path\n\n\n@dataclass\nclass RunningWorker:\n    spec: WorkerSpec\n    port: int\n    process: subprocess.Popen[str]\n    log_path: Path\n\n    @property\n    def base_url(self) -> str:\n        return f"http://{LISTEN_HOST}:{self.port}"\n\n\ndef require_slug(value: str, label: str) -> str:\n    if not isinstance(value, str) or not SAFE_SLUG.fullmatch(value):\n        raise ValueError(f"{label} must be a lowercase model/capability slug")\n    return value\n\n\ndef find_free_port() -> int:\n    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as candidate:\n        candidate.bind((LISTEN_HOST, 0))\n        return int(candidate.getsockname()[1])\n\n\ndef read_notebook(path: Path) -> dict[str, Any]:\n    with path.open("r", encoding="utf-8") as stream:\n        return json.load(stream)\n\n\ndef notebook_source(cell: dict[str, Any]) -> str:\n    source = cell.get("source", "")\n    return "".join(source) if isinstance(source, list) else str(source)\n\n\ndef metadata_for_notebook(path: Path) -> tuple[str, str] | None:\n    metadata = read_notebook(path).get("metadata", {}).get("la_studio", {})\n    capability = metadata.get("capability")\n    model = metadata.get("family_id") or metadata.get("model_id")\n    if not isinstance(capability, str) or not isinstance(model, str):\n        return None\n    return capability, model\n\n\ndef discover_exact_notebook(source_root: Path, capability: str, model: str) -> Path:\n    # The embedded source tree stores exact workers under capability\n    # directories. A flat glob silently makes every prewarmed model look\n    # missing when the coordinator is given its self-contained bundle.\n    for notebook in sorted((source_root / "notebooks").rglob("*.ipynb")):\n        metadata = metadata_for_notebook(notebook)\n        if metadata == (capability, model):\n            return notebook\n    raise RuntimeError(\n        f"No exact generated notebook exists for {capability}/{model}. "\n        "Choose an exact model listed by the Dubbing app, then regenerate this notebook."\n    )\n\n\ndef literal_worker_source(document: dict[str, Any]) -> str | None:\n    """Return a static Path(...).write_text(<worker>) payload from a notebook."""\n    def static_string(node: ast.AST) -> str | None:\n        if isinstance(node, ast.Constant) and isinstance(node.value, str):\n            return node.value\n        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Add):\n            left = static_string(node.left)\n            right = static_string(node.right)\n            return left + right if left is not None and right is not None else None\n        return None\n\n    for cell in document.get("cells", []):\n        if cell.get("cell_type") != "code":\n            continue\n        try:\n            tree = ast.parse(notebook_source(cell))\n        except SyntaxError:\n            continue\n        for statement in ast.walk(tree):\n            if not isinstance(statement, ast.Call) or not isinstance(statement.func, ast.Attribute):\n                continue\n            if statement.func.attr != "write_text" or not statement.args:\n                continue\n            value = static_string(statement.args[0])\n            if value is not None and "FastAPI" in value:\n                return value\n    return None\n\n\ndef stt_worker_source(document: dict[str, Any]) -> str | None:\n    """The STT generator declares its app directly instead of write_text()."""\n    for cell in document.get("cells", []):\n        source = notebook_source(cell)\n        if "app = FastAPI" in source and "@app.get(\\"/health\\")" in source:\n            return source\n    return None\n\n\ndef worker_source_for(spec: WorkerSpec, source_root: Path) -> str:\n    if spec.capability == "voice-isolation" and spec.model == "sherpa-onnx-spleeter-2stems-fp16":\n        return (source_root / "notebooks" / "workers" /\n                "LA_STUDIO_SEPARATION_SPLEETER_2STEMS_WORKER.py").read_text(encoding="utf-8")\n    document = read_notebook(spec.notebook)\n    source = literal_worker_source(document)\n    if source is None and spec.capability == "stt":\n        source = stt_worker_source(document)\n        if source:\n            source = re.sub(\n                r"TOKEN\\s*=\\s*secrets\\.token_urlsafe\\(32\\)",\n                "TOKEN = os.environ[\'LA_STUDIO_COLAB_STT_TOKEN\']",\n                source,\n            )\n    if source is None:\n        raise RuntimeError(\n            f"Could not extract the exact worker source for {spec.capability}/{spec.model}. "\n            "The notebook must keep a static FastAPI worker source."\n        )\n    return source\n\n\ndef install_shell_lines(document: dict[str, Any], runtime: Path) -> None:\n    """Run only explicit package/artifact setup commands, never notebook launch cells."""\n    for cell in document.get("cells", []):\n        if cell.get("cell_type") != "code":\n            continue\n        source = notebook_source(cell)\n        if "uvicorn" in source and ("cloudflared" in source or "tunnel" in source):\n            continue\n        for raw_line in source.splitlines():\n            line = raw_line.strip()\n            if line.startswith("%pip "):\n                arguments = [sys.executable, "-m", "pip", *line[5:].strip().split()]\n            elif line.startswith("!"):\n                command = line[1:].strip()\n                if command.startswith(("python ", "python3 ")) or "cloudflared" in command:\n                    continue\n                arguments = ["bash", "-lc", command]\n            else:\n                continue\n            subprocess.run(arguments, cwd=runtime, check=True)\n\n\ndef run_ocr_bootstrap(document: dict[str, Any], runtime: Path) -> None:\n    """Use the OCR notebook\'s isolated bootstrap rather than mixing Paddle globally."""\n    for cell in document.get("cells", []):\n        source = notebook_source(cell)\n        if "OCR_SITE_PACKAGES" not in source or "BOOTSTRAP_REVISION" not in source:\n            continue\n        source = source.replace("!nvidia-smi", "subprocess.run([\'nvidia-smi\'], check=True)")\n        bootstrap = runtime / "la_studio_ocr_bootstrap.py"\n        bootstrap.write_text(source, encoding="utf-8")\n        subprocess.run([sys.executable, str(bootstrap)], cwd=runtime, check=True)\n        return\n    raise RuntimeError("The selected Subtitle OCR notebook has no recognized isolated bootstrap cell")\n\n\ndef prepare_worker(spec: WorkerSpec, source_root: Path, runtime: Path) -> tuple[Path, dict[str, str]]:\n    document = read_notebook(spec.notebook)\n    if spec.capability == "subtitle-ocr":\n        run_ocr_bootstrap(document, runtime)\n    else:\n        install_shell_lines(document, runtime)\n    # Uvicorn imports a Python module, so exact model IDs such as\n    # ``whisper.cpp`` and ``m2m100-418m`` cannot be used verbatim as names.\n    module_stem = re.sub(r"[^A-Za-z0-9_]", "_", f"worker_{spec.capability}_{spec.model}")\n    worker_path = runtime / f"{module_stem}.py"\n    worker_path.write_text(worker_source_for(spec, source_root), encoding="utf-8")\n    environment = os.environ.copy()\n    environment["PYTHONUNBUFFERED"] = "1"\n    environment["LA_STUDIO_UNIFIED_DUBBING_TOKEN"] = environment["LA_STUDIO_UNIFIED_DUBBING_TOKEN"]\n    for token_environment in TOKEN_ENVIRONMENTS.values():\n        environment[token_environment] = environment["LA_STUDIO_UNIFIED_DUBBING_TOKEN"]\n    if spec.capability == "subtitle-ocr":\n        isolated_site = runtime / "la_studio_subtitle_ocr_site"\n        environment["PYTHONPATH"] = str(isolated_site)\n        environment["PYTHONNOUSERSITE"] = "1"\n    return worker_path, environment\n\n\ndef health_payload(base_url: str, token: str) -> dict[str, Any] | None:\n    try:\n        response = httpx.get(\n            f"{base_url}/health", headers={"Authorization": f"Bearer {token}"}, timeout=10.0\n        )\n        response.raise_for_status()\n        payload = response.json()\n        return payload if isinstance(payload, dict) else None\n    except (httpx.HTTPError, ValueError):\n        return None\n\n\ndef wait_for_exact_health(worker: RunningWorker, token: str, timeout_seconds: float = 420.0) -> None:\n    deadline = time.monotonic() + timeout_seconds\n    while time.monotonic() < deadline:\n        if worker.process.poll() is not None:\n            tail = worker.log_path.read_text(encoding="utf-8", errors="replace")[-12000:]\n            raise RuntimeError(\n                f"{worker.spec.capability}/{worker.spec.model} exited before readiness "\n                f"(exit {worker.process.returncode}).\\n{tail}"\n            )\n        payload = health_payload(worker.base_url, token)\n        if payload and payload.get("ready") is True:\n            returned_model = str(payload.get("model") or payload.get("family_id") or "")\n            if returned_model and returned_model != worker.spec.model:\n                raise RuntimeError(\n                    f"Worker identity mismatch: expected {worker.spec.model}, got {returned_model}"\n                )\n            if str(payload.get("device", "cuda")).lower() != "cuda":\n                raise RuntimeError(\n                    f"{worker.spec.capability}/{worker.spec.model} is not CUDA-ready: {payload}"\n                )\n            return\n        time.sleep(1.0)\n    raise RuntimeError(\n        f"Timed out waiting for actual CUDA health from {worker.spec.capability}/{worker.spec.model}. "\n        f"See {worker.log_path}."\n    )\n\n\nclass UnifiedCoordinator:\n    def __init__(self, source_root: Path, runtime: Path, token: str):\n        self.source_root = source_root\n        self.runtime = runtime\n        self.token = token\n        self.workers: dict[tuple[str, str], RunningWorker] = {}\n\n    def start(self, selections: list[dict[str, Any]]) -> None:\n        if not selections:\n            raise RuntimeError("UNIFIED_WORKERS is empty; configure at least one exact Dubbing model")\n        self.runtime.mkdir(parents=True, exist_ok=True)\n        for selection in selections:\n            capability = require_slug(selection.get("capability"), "capability")\n            model = require_slug(selection.get("model"), "model")\n            key = (capability, model)\n            if key in self.workers:\n                continue\n            notebook = discover_exact_notebook(self.source_root, capability, model)\n            spec = WorkerSpec(capability, model, notebook)\n            worker_path, environment = prepare_worker(spec, self.source_root, self.runtime)\n            port = find_free_port()\n            log_path = self.runtime / f"{capability}-{model}.log"\n            with log_path.open("w", encoding="utf-8") as log:\n                process = subprocess.Popen(\n                    [sys.executable, "-m", "uvicorn", f"{worker_path.stem}:app",\n                     "--host", LISTEN_HOST, "--port", str(port)],\n                    cwd=self.runtime, env=environment, stdout=log, stderr=subprocess.STDOUT, text=True,\n                )\n            worker = RunningWorker(spec, port, process, log_path)\n            wait_for_exact_health(worker, self.token)\n            self.workers[key] = worker\n\n    def worker_for(self, capability: str, model: str) -> RunningWorker:\n        worker = self.workers.get((capability, model))\n        if worker is None:\n            raise HTTPException(\n                status_code=404,\n                detail=(f"{capability}/{model} was not prewarmed by this unified notebook. "\n                        "Add that exact model to UNIFIED_WORKERS and run the notebook again."),\n            )\n        return worker\n\n    def health(self) -> dict[str, Any]:\n        result: list[dict[str, Any]] = []\n        for worker in self.workers.values():\n            payload = health_payload(worker.base_url, self.token)\n            if not payload or payload.get("ready") is not True:\n                raise HTTPException(status_code=503, detail=f"Worker lost readiness: {worker.spec}")\n            result.append({"capability": worker.spec.capability, "model": worker.spec.model, "health": payload})\n        return {"ready": True, "coordinator": COORDINATOR_REVISION, "workers": result}\n\n    def stop(self) -> None:\n        for worker in self.workers.values():\n            if worker.process.poll() is None:\n                worker.process.terminate()\n\n\nCOORDINATOR: UnifiedCoordinator | None = None\nAPP = FastAPI(title="LA Studio unified Dubbing coordinator")\n\n\ndef require_authorization(request: Request) -> None:\n    expected = f"Bearer {os.environ[\'LA_STUDIO_UNIFIED_DUBBING_TOKEN\']}"\n    if not secrets.compare_digest(request.headers.get("authorization", ""), expected):\n        raise HTTPException(status_code=401, detail="Invalid LA Studio unified session token")\n\n\n@APP.get("/health")\nasync def coordinator_health(request: Request) -> JSONResponse:\n    require_authorization(request)\n    if COORDINATOR is None:\n        raise HTTPException(status_code=503, detail="Coordinator has not finished prewarming exact workers")\n    return JSONResponse(COORDINATOR.health())\n\n\n@APP.get("/v1/capabilities")\nasync def capabilities(request: Request) -> JSONResponse:\n    require_authorization(request)\n    if COORDINATOR is None:\n        raise HTTPException(status_code=503, detail="Coordinator has not finished prewarming exact workers")\n    rows = [{"capability": worker.spec.capability, "model": worker.spec.model}\n            for worker in COORDINATOR.workers.values()]\n    return JSONResponse({"ready": True, "routes": rows})\n\n\n@APP.api_route("/v1/unified/{capability}/{model}/{route:path}", methods=["GET", "POST", "PUT", "PATCH", "DELETE", "HEAD", "OPTIONS"])\nasync def proxy(capability: str, model: str, route: str, request: Request) -> StreamingResponse:\n    require_authorization(request)\n    if COORDINATOR is None:\n        raise HTTPException(status_code=503, detail="Coordinator has not finished prewarming exact workers")\n    worker = COORDINATOR.worker_for(require_slug(capability, "capability"), require_slug(model, "model"))\n    target = f"{worker.base_url}/{route.lstrip(\'/\')}"\n    if request.url.query:\n        target += f"?{request.url.query}"\n    headers = {key: value for key, value in request.headers.items() if key.lower() not in HOP_BY_HOP_HEADERS}\n    client = httpx.AsyncClient(timeout=httpx.Timeout(connect=30.0, read=None, write=None, pool=30.0))\n    try:\n        upstream_request = client.build_request(request.method, target, headers=headers, content=request.stream())\n        upstream = await client.send(upstream_request, stream=True)\n    except httpx.HTTPError as error:\n        await client.aclose()\n        raise HTTPException(status_code=502, detail=f"Configured unified worker request failed: {error}") from error\n\n    async def response_body():\n        try:\n            async for chunk in upstream.aiter_raw():\n                yield chunk\n        finally:\n            await upstream.aclose()\n            await client.aclose()\n\n    response_headers = {key: value for key, value in upstream.headers.items() if key.lower() not in HOP_BY_HOP_HEADERS}\n    return StreamingResponse(response_body(), status_code=upstream.status_code, headers=response_headers)\n\n\ndef ensure_cloudflared(runtime: Path) -> str:\n    found = shutil.which("cloudflared")\n    if found:\n        return found\n    destination = runtime / "cloudflared"\n    subprocess.run([\n        "curl", "--fail", "--location", "--retry", "3", "--output", str(destination),\n        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",\n    ], check=True)\n    destination.chmod(0o755)\n    return str(destination)\n\n\ndef start_tunnel(runtime: Path) -> tuple[subprocess.Popen[str], str]:\n    cloudflared = ensure_cloudflared(runtime)\n    log_path = runtime / "unified-tunnel.log"\n    log = log_path.open("w", encoding="utf-8")\n    tunnel = subprocess.Popen(\n        [cloudflared, "tunnel", "--url", f"http://{LISTEN_HOST}:{LISTEN_PORT}", "--no-autoupdate"],\n        stdout=log, stderr=subprocess.STDOUT, text=True,\n    )\n    pattern = re.compile(r"https://[-a-z0-9]+\\.trycloudflare\\.com", re.IGNORECASE)\n    deadline = time.monotonic() + 90.0\n    while time.monotonic() < deadline:\n        if tunnel.poll() is not None:\n            raise RuntimeError(f"Cloudflare tunnel exited early:\\n{log_path.read_text(encoding=\'utf-8\', errors=\'replace\')[-8000:]}")\n        text = log_path.read_text(encoding="utf-8", errors="replace")\n        match = pattern.search(text)\n        if match:\n            return tunnel, match.group(0)\n        time.sleep(0.5)\n    tunnel.terminate()\n    raise RuntimeError("Timed out waiting for the verified public Cloudflare tunnel URL")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--source-root", required=True, type=Path)\n    parser.add_argument("--config", required=True, type=Path)\n    parser.add_argument("--runtime", default=Path("/content/la_studio_unified_dubbing"), type=Path)\n    arguments = parser.parse_args()\n    if not arguments.source_root.is_dir():\n        raise RuntimeError(f"Source checkout does not exist: {arguments.source_root}")\n    selections = json.loads(arguments.config.read_text(encoding="utf-8"))\n    if not isinstance(selections, list):\n        raise RuntimeError("Unified workers config must be a JSON list")\n    os.environ.setdefault("LA_STUDIO_UNIFIED_DUBBING_TOKEN", secrets.token_urlsafe(32))\n    global COORDINATOR\n    COORDINATOR = UnifiedCoordinator(arguments.source_root, arguments.runtime, os.environ["LA_STUDIO_UNIFIED_DUBBING_TOKEN"])\n    try:\n        COORDINATOR.start(selections)\n        tunnel, public_url = start_tunnel(arguments.runtime)\n        print("\\nLA Studio Unified Dubbing coordinator is ready.")\n        print(f"LA_STUDIO_UNIFIED_DUBBING_URL={public_url}")\n        print(f"LA_STUDIO_UNIFIED_DUBBING_TOKEN={os.environ[\'LA_STUDIO_UNIFIED_DUBBING_TOKEN\']}")\n        print("Paste these once in Dubbing > Project setup > Unified Colab (optional).")\n        uvicorn.run(APP, host=LISTEN_HOST, port=LISTEN_PORT, log_level="info")\n        tunnel.terminate()\n    finally:\n        if COORDINATOR is not None:\n            COORDINATOR.stop()\n\n\nif __name__ == "__main__":\n    main()\n',
    'notebooks/workers/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_WORKER.py': '"""Temporary Direct Colab worker for the exact Spleeter 2-stem FP16 artifact.\n\nThe worker deliberately uses ONNX Runtime\'s CUDA provider directly.  The\nsherpa-onnx source-separation wrapper fixes its CUDA convolution search to\nHEURISTIC, which fails on some current Colab cuDNN 9 images.  This worker uses\nthe same upstream FP16 ONNX files, but chooses ORT\'s documented DEFAULT\nconvolution algorithm instead and bounds every inference input.\n"""\n\nimport math\nimport os\nimport secrets\nimport shutil\nimport subprocess\nimport threading\nimport traceback\nfrom pathlib import Path\n\nimport kaldi_native_fbank as knf\nimport numpy as np\nimport torch\nimport onnxruntime as ort\nimport soundfile as sf\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\nfrom fastapi.responses import FileResponse\n\n\nWORKER_CONTRACT = "spleeter-cuda-safe-20260816.1"\nMODEL_ID = "sherpa-onnx-spleeter-2stems-fp16"\nMODEL_NAME = "Spleeter 2-stem FP16"\nUPSTREAM_MODEL = "k2-fsa/sherpa-onnx-spleeter-2stems-fp16"\nARTIFACT_URL = (\n    "https://github.com/k2-fsa/sherpa-onnx/releases/download/"\n    "source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2"\n)\nMODEL_ROOT = Path("/content/sherpa-onnx-spleeter-2stems-fp16")\nTOKEN = os.environ["LA_STUDIO_COLAB_SEPARATION_TOKEN"]\nROOT = Path("/content/la-studio-separation-jobs") / MODEL_ID\nROOT.mkdir(parents=True, exist_ok=True)\n\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\nMAX_AUDIO_SECONDS = 30 * 60\nARTIFACT_TTL_SECONDS = 1800\n# A 20-second core has at most two 512-frame Spleeter splits.  The previous\n# worker sent a complete long video through one CUDA call, producing a large\n# dynamic Conv shape that caused CUDNN_FE_HEURISTIC_QUERY_FAILED on Colab.\nCORE_SECONDS = 20.0\nCONTEXT_SECONDS = 1.5\nPROBE_SECONDS = CORE_SECONDS\n\nALLOWED_CONTENT_TYPES = {\n    "audio/wav", "audio/x-wav", "audio/mpeg", "audio/mp4", "audio/webm",\n    "audio/ogg", "audio/flac", "video/mp4", "video/webm", "video/quicktime",\n    "video/x-matroska", "application/octet-stream",\n}\nALLOWED_EXTENSIONS = {\n    ".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac", ".mkv", ".mov", ".avi",\n}\nCUDA_OPTIONS = {\n    # DEFAULT avoids the cuDNN heuristic-plan query that failed in the old\n    # sherpa-onnx wrapper.  The exact model remains FP16 and executes on CUDA.\n    "cudnn_conv_algo_search": "DEFAULT",\n    "cudnn_conv_use_max_workspace": "1",\n    "do_copy_in_default_stream": "1",\n    "arena_extend_strategy": "kSameAsRequested",\n}\n\nif not torch.cuda.is_available():\n    raise RuntimeError("Colab GPU is not available; select a GPU runtime before starting this worker")\n\n\ndef _cuda_session(path: Path) -> ort.InferenceSession:\n    if "CUDAExecutionProvider" not in ort.get_available_providers():\n        raise RuntimeError("ONNX Runtime CUDAExecutionProvider is unavailable in this Colab runtime")\n    options = ort.SessionOptions()\n    options.intra_op_num_threads = 1\n    options.inter_op_num_threads = 1\n    session = ort.InferenceSession(\n        str(path),\n        sess_options=options,\n        providers=[("CUDAExecutionProvider", CUDA_OPTIONS), "CPUExecutionProvider"],\n    )\n    if not session.get_providers() or session.get_providers()[0] != "CUDAExecutionProvider":\n        raise RuntimeError("The exact Spleeter ONNX session did not bind CUDAExecutionProvider")\n    return session\n\n\nclass ExactSpleeterCuda:\n    def __init__(self) -> None:\n        vocals = MODEL_ROOT / "vocals.fp16.onnx"\n        accompaniment = MODEL_ROOT / "accompaniment.fp16.onnx"\n        if not vocals.is_file() or not accompaniment.is_file():\n            raise RuntimeError("The exact Spleeter FP16 ONNX artifacts are missing")\n        self.vocals = _cuda_session(vocals)\n        self.accompaniment = _cuda_session(accompaniment)\n        self.stft_config = knf.StftConfig(\n            n_fft=4096,\n            hop_length=1024,\n            win_length=4096,\n            center=False,\n            window_type="hann",\n        )\n\n    @staticmethod\n    def _stft(samples: np.ndarray, channel: int) -> tuple[np.ndarray, np.ndarray]:\n        result = knf.Stft(knf.StftConfig(\n            n_fft=4096, hop_length=1024, win_length=4096,\n            center=False, window_type="hann",\n        ))(samples[:, channel].tolist())\n        real = np.asarray(result.real, dtype=np.float32).reshape(result.num_frames, -1)\n        imag = np.asarray(result.imag, dtype=np.float32).reshape(result.num_frames, -1)\n        return real, imag\n\n    def process(self, sample_rate: int, samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n        if sample_rate != 44100:\n            raise RuntimeError(f"expected 44100 Hz worker input, received {sample_rate}")\n        if samples.ndim != 2 or samples.shape[1] != 2 or samples.shape[0] == 0:\n            raise RuntimeError("expected non-empty stereo audio")\n        real0, imag0 = self._stft(samples, 0)\n        real1, imag1 = self._stft(samples, 1)\n        if real0.shape[0] == 0 or real1.shape[0] == 0:\n            raise RuntimeError("audio is too short for Spleeter analysis")\n        frame_count = real0.shape[0]\n        if real1.shape[0] != frame_count:\n            raise RuntimeError("stereo channel frame counts differ")\n\n        magnitude0 = np.sqrt(real0[:, :1024] ** 2 + imag0[:, :1024] ** 2).astype(np.float32)\n        magnitude1 = np.sqrt(real1[:, :1024] ** 2 + imag1[:, :1024] ** 2).astype(np.float32)\n        padded_frames = int(math.ceil(frame_count / 512.0) * 512)\n        if padded_frames != frame_count:\n            padding = ((0, padded_frames - frame_count), (0, 0))\n            magnitude0 = np.pad(magnitude0, padding)\n            magnitude1 = np.pad(magnitude1, padding)\n        model_input = np.ascontiguousarray(\n            np.stack((magnitude0, magnitude1), axis=0).reshape(2, -1, 512, 1024),\n            dtype=np.float32,\n        )\n        vocals_spec = self.vocals.run(None, {self.vocals.get_inputs()[0].name: model_input})[0]\n        accompaniment_spec = self.accompaniment.run(\n            None, {self.accompaniment.get_inputs()[0].name: model_input}\n        )[0]\n        denominator = vocals_spec ** 2 + accompaniment_spec ** 2 + 1e-10\n        masks = (\n            (vocals_spec ** 2 + 5e-11) / denominator,\n            (accompaniment_spec ** 2 + 5e-11) / denominator,\n        )\n\n        stems: list[np.ndarray] = []\n        for mask in masks:\n            channels: list[np.ndarray] = []\n            for channel, (real, imag) in enumerate(((real0, imag0), (real1, imag1))):\n                channel_mask = mask[channel].reshape(-1, 1024)[:frame_count]\n                channel_mask = np.pad(channel_mask, ((0, 0), (0, real.shape[1] - 1024)))\n                masked = knf.StftResult(\n                    real=(channel_mask * real).reshape(-1).tolist(),\n                    imag=(channel_mask * imag).reshape(-1).tolist(),\n                    num_frames=frame_count,\n                )\n                waveform = knf.IStft(self.stft_config)(masked)\n                channels.append(np.asarray(waveform, dtype=np.float32))\n            stem = np.column_stack(channels)\n            stems.append(stem)\n        return stems[0], stems[1]\n\n\n# Constructing and running the same bounded shape before /health is exposed\n# proves CUDA works for this exact model.  An unsupported Colab image fails in\n# the notebook cell, rather than accepting a URL and later failing at a random\n# workflow step.\nSEPARATOR = ExactSpleeterCuda()\n_probe = np.zeros((int(44100 * PROBE_SECONDS), 2), dtype=np.float32)\n_probe_vocals, _probe_background = SEPARATOR.process(44100, _probe)\nif _probe_vocals.shape[0] == 0 or _probe_background.shape[0] == 0:\n    raise RuntimeError("exact Spleeter CUDA startup probe produced empty audio")\ndel _probe, _probe_vocals, _probe_background\n\nJOB_SLOTS = threading.BoundedSemaphore(1)\nJOB_LOCK = threading.Lock()\nJOBS: dict[str, dict] = {}\n\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=(f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. "\n                    "Open the notebook for the selected model."),\n        )\n\n\ndef media_duration_seconds(path: Path) -> float:\n    probe = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "default=nokey=1:noprint_wrappers=1", str(path)],\n        text=True, capture_output=True,\n    )\n    try:\n        duration = float(probe.stdout.strip())\n    except ValueError:\n        duration = 0.0\n    if probe.returncode != 0 or duration <= 0.0:\n        raise HTTPException(status_code=415, detail="media is unsupported or could not be decoded")\n    return duration\n\n\ndef update(job_id: str, **values) -> dict:\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n        job.update(values)\n        JOBS[job_id] = job\n        return job\n\n\ndef is_cancelled(job_id: str) -> bool:\n    with JOB_LOCK:\n        return bool(JOBS.get(job_id, {}).get("cancel_requested", False))\n\n\ndef cleanup(job_id: str) -> None:\n    with JOB_LOCK:\n        job = JOBS.pop(job_id, None)\n    if job:\n        shutil.rmtree(job.get("directory", ""), ignore_errors=True)\n\n\ndef _fit_piece(stem: np.ndarray, start: int, length: int) -> np.ndarray:\n    piece = stem[start:start + length]\n    if piece.shape[0] >= length:\n        return piece[:length]\n    return np.pad(piece, ((0, length - piece.shape[0]), (0, 0)))\n\n\ndef separate_bounded(job_id: str, samples: np.ndarray, sample_rate: int) -> tuple[np.ndarray, np.ndarray]:\n    core = int(CORE_SECONDS * sample_rate)\n    context = int(CONTEXT_SECONDS * sample_rate)\n    total = samples.shape[0]\n    pieces = max(1, math.ceil(total / core))\n    vocals_parts: list[np.ndarray] = []\n    background_parts: list[np.ndarray] = []\n    for index, core_start in enumerate(range(0, total, core), start=1):\n        if is_cancelled(job_id):\n            raise RuntimeError("Separation cancelled")\n        core_end = min(total, core_start + core)\n        window_start = max(0, core_start - context)\n        window_end = min(total, core_end + context)\n        update(\n            job_id,\n            status="running",\n            progress=20 + int(65 * (index - 1) / pieces),\n            detail=f"{MODEL_NAME} CUDA segment {index}/{pieces}",\n        )\n        vocals, background = SEPARATOR.process(sample_rate, samples[window_start:window_end])\n        trim = core_start - window_start\n        core_length = core_end - core_start\n        vocals_parts.append(_fit_piece(vocals, trim, core_length))\n        background_parts.append(_fit_piece(background, trim, core_length))\n        update(\n            job_id,\n            status="running",\n            progress=20 + int(65 * index / pieces),\n            detail=f"{MODEL_NAME} CUDA segment {index}/{pieces} complete",\n        )\n    return np.concatenate(vocals_parts, axis=0), np.concatenate(background_parts, axis=0)\n\n\ndef concise_failure(error: Exception) -> str:\n    text = str(error).replace("\\n", " ").strip()\n    if "CUDNN" in text.upper() or "CUDA" in text.upper():\n        return ("The verified Colab CUDA worker failed during Spleeter inference. "\n                "No local model was started. Stop this job, reopen the current Spleeter notebook, "\n                "and use its startup probe before reconnecting. Full worker detail is in the Colab output.")\n    return f"{type(error).__name__}: {text[:600]}"\n\n\ndef run_job(job_id: str, directory: Path, source: Path, output_format: str) -> None:\n    try:\n        update(job_id, status="running", progress=12, detail="Decoding media for bounded CUDA separation")\n        wav_path = directory / "source-44100-stereo.wav"\n        subprocess.run(\n            ["ffmpeg", "-y", "-v", "error", "-i", str(source), "-vn", "-acodec", "pcm_s16le",\n             "-ar", "44100", "-ac", "2", str(wav_path)],\n            check=True,\n        )\n        samples, sample_rate = sf.read(wav_path, dtype="float32", always_2d=True)\n        samples = np.ascontiguousarray(samples, dtype=np.float32)\n        vocals_data, background_data = separate_bounded(job_id, samples, sample_rate)\n        if is_cancelled(job_id):\n            update(job_id, status="cancelled", progress=0, detail="Separation cancelled")\n            return\n        update(job_id, status="running", progress=90, detail="Writing separated CUDA stems")\n        suffix = ".wav" if output_format == "wav" else ".flac"\n        vocals = directory / ("vocals" + suffix)\n        background = directory / ("background" + suffix)\n        # FLAC is lossless and typically reduces the 44.1 kHz stereo transfer\n        # by far more than 50%; PCM WAV remains the explicit compatibility\n        # choice for an operator who needs it.\n        sf.write(vocals, vocals_data, sample_rate,\n                 format="WAV" if output_format == "wav" else "FLAC",\n                 subtype="PCM_16")\n        sf.write(background, background_data, sample_rate,\n                 format="WAV" if output_format == "wav" else "FLAC",\n                 subtype="PCM_16")\n        update(\n            job_id, status="ready", progress=100, detail="Separated CUDA stems are ready",\n            vocals=str(vocals), background=str(background),\n            artifact_format=output_format, artifacts_ready=True,\n        )\n    except Exception as error:\n        traceback.print_exc()\n        update(job_id, status="failed", progress=0, detail=concise_failure(error))\n    finally:\n        threading.Timer(ARTIFACT_TTL_SECONDS, cleanup, args=[job_id]).start()\n        JOB_SLOTS.release()\n\n\napp = FastAPI(title=f"LA Studio Voice Isolation - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "onnxruntime": ort.__version__,\n        "cuda_provider_options": CUDA_OPTIONS,\n        "bounded_core_seconds": CORE_SECONDS,\n        "startup_probe": "passed",\n        "cpu_fallback": False,\n    }\n\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "voice-isolation",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "artifact_url": ARTIFACT_URL,\n                "stems": ["vocals", "background"],\n                "formats": ["flac", "wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n\n@app.post("/v1/audio/separations")\nasync def create_separation(\n    file: UploadFile = File(...),\n    stems: str = Form("vocals,background"),\n    model: str = Form(...),\n    output_format: str = Form("flac"),\n    authorization: str | None = Header(default=None),\n):\n    authorize(authorization)\n    require_exact_model(model)\n    if stems != "vocals,background":\n        raise HTTPException(status_code=422, detail="this worker returns vocals and background stems")\n    output_format = output_format.strip().lower()\n    if output_format not in {"flac", "wav"}:\n        raise HTTPException(status_code=422, detail="output_format must be flac or wav")\n    suffix = Path(file.filename or "source.wav").suffix.lower() or ".wav"\n    if suffix not in ALLOWED_EXTENSIONS:\n        raise HTTPException(status_code=415, detail="unsupported media filename extension")\n    if file.content_type and file.content_type not in ALLOWED_CONTENT_TYPES:\n        raise HTTPException(status_code=415, detail="unsupported media MIME type")\n    if not JOB_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab separation worker is busy; retry shortly")\n    job_id = secrets.token_urlsafe(18)\n    directory = ROOT / job_id\n    directory.mkdir(parents=True, exist_ok=True)\n    source = directory / ("source" + suffix)\n    try:\n        with source.open("wb") as output:\n            while chunk := await file.read(1024 * 1024):\n                output.write(chunk)\n                if output.tell() > MAX_UPLOAD_BYTES:\n                    raise HTTPException(status_code=413, detail="media exceeds 512 MB upload limit")\n        if source.stat().st_size <= 0:\n            raise HTTPException(status_code=413, detail="media must not be empty")\n        if media_duration_seconds(source) > MAX_AUDIO_SECONDS:\n            raise HTTPException(status_code=413, detail="media exceeds the 30 minute duration limit")\n    except Exception:\n        shutil.rmtree(directory, ignore_errors=True)\n        JOB_SLOTS.release()\n        raise\n    finally:\n        await file.close()\n    update(\n        job_id, status="queued", progress=10,\n        detail=f"Media uploaded; {MODEL_NAME} CUDA job is queued",\n        directory=str(directory), cancel_requested=False,\n    )\n    threading.Thread(target=run_job, args=(job_id, directory, source, output_format), daemon=True).start()\n    return {"job_id": job_id, "status": "queued", "progress": 10,\n            "artifact_format": output_format}\n\n\n@app.get("/v1/audio/separations/{job_id}")\ndef separation_status(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n    if not job:\n        raise HTTPException(status_code=404, detail="separation job not found")\n    return {key: job.get(key) for key in ("status", "progress", "detail", "artifact_format", "artifacts_ready") if key in job} | {"job_id": job_id}\n\n\n@app.get("/v1/audio/separations/{job_id}/artifacts/{stem}")\ndef artifact(job_id: str, stem: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if stem not in {"vocals", "background"}:\n        raise HTTPException(status_code=404, detail="unknown stem")\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n    path = Path(job.get(stem, ""))\n    if job.get("status") != "ready" or not path.is_file():\n        raise HTTPException(status_code=409, detail="stem is not ready")\n    output_format = job.get("artifact_format", "wav")\n    media_type = "audio/flac" if output_format == "flac" else "audio/wav"\n    return FileResponse(path, media_type=media_type, filename=stem + "." + output_format)\n\n\n@app.delete("/v1/audio/separations/{job_id}")\ndef cancel_separation(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail="separation job not found")\n        JOBS[job_id]["cancel_requested"] = True\n        JOBS[job_id]["status"] = "cancelling"\n    return {"job_id": job_id, "status": "cancelling"}\n',
}
SOURCE_ROOT = Path('/content/la-studio-unified-source')
shutil.rmtree(SOURCE_ROOT, ignore_errors=True)
for relative_path, payload in EMBEDDED_UNIFIED_FILES.items():
    target = SOURCE_ROOT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(payload, encoding='utf-8')
COORDINATOR_PATH = Path('/content/la_studio_unified_dubbing_coordinator.py')
COORDINATOR_PATH.write_text(
    EMBEDDED_UNIFIED_FILES['notebooks/workers/LA_STUDIO_UNIFIED_DUBBING_COORDINATOR.py'],
    encoding='utf-8',
)
print(f'Wrote {len(EMBEDDED_UNIFIED_FILES)} embedded coordinator inputs.')


In [ ]:
# This cell remains running while LA Studio uses the unified worker.
# It prints one URL and one token only after every selected exact CUDA worker is healthy.
!python3 /content/la_studio_unified_dubbing_coordinator.py \
    --source-root /content/la-studio-unified-source \
    --config /content/la_studio_unified_workers.json
